In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import re

# =========================
# 入出力
# =========================
FINS_DIR = Path(r"C:\Users\yongr\Project\jquants_fins_summary_10y_parquet\daily_parquet")
PRICE_MONTH_END_PATH = Path(r"C:\Users\yongr\Project\merged_data_all_stocks\factors\price_month_end.parquet")
OUT_SNAPSHOT_PATH = Path(r"C:\Users\yongr\Project\merged_data_all_stocks\factors\month_end_snapshot_REBUILT.parquet")

DIAG_FINS_COLS_CSV = Path(r"C:\Users\yongr\Project\merged_data_all_stocks\factors\diag_fins_columns_summary.csv")
DIAG_COVERAGE_CSV  = Path(r"C:\Users\yongr\Project\merged_data_all_stocks\factors\diag_fin_coverage_by_month.csv")

START = "2016-01-01"
END   = "2026-01-31"

date_pat = re.compile(r"date=(\d{4}-\d{2}-\d{2})\.parquet$")

def list_files_in_range(folder: Path, start: str, end: str):
    start = pd.Timestamp(start)
    end = pd.Timestamp(end)
    files = []
    for p in folder.glob("date=*.parquet"):
        m = date_pat.search(str(p))
        if not m:
            continue
        d = pd.Timestamp(m.group(1))
        if start <= d <= end:
            files.append((d, p))
    files.sort(key=lambda x: x[0])
    return files

def month_end(ts: pd.Series) -> pd.Series:
    return ts.dt.to_period("M").dt.to_timestamp("M")

# =========================
# Step2-0: price_month_end 読み込み
# =========================
print("="*120)
print("[Step2-0] load price_month_end")
print("="*120)

price = pd.read_parquet(PRICE_MONTH_END_PATH)
price["Date"] = pd.to_datetime(price["Date"], errors="coerce")
price["MonthEnd"] = pd.to_datetime(price["MonthEnd"], errors="coerce")
price["Code"] = price["Code"].astype(str)
price["AdjustedClose"] = pd.to_numeric(price["AdjustedClose"], errors="coerce")

assert price.duplicated(["Code","MonthEnd"]).sum() == 0
print(f"price rows={len(price):,} codes={price['Code'].nunique():,} months={price['MonthEnd'].nunique():,} Date.max={price['Date'].max().date()}")

# =========================
# Step2-1: fins 列名の実態を把握（最初に全体の列バリエーション診断）
# =========================
print("\n" + "="*120)
print("[Step2-1] scan fins columns (diagnostics)")
print("="*120)

fin_files = list_files_in_range(FINS_DIR, START, END)
print(f"fins files: {len(fin_files)}  ({fin_files[0][0].date()} .. {fin_files[-1][0].date()})")

col_counter = {}
sample_cols = None

# まずは軽く列名だけ収集（全ファイル読みでもOKだが重いなら間引き可）
for i, (d, fp) in enumerate(fin_files, 1):
    fin = pd.read_parquet(fp)
    cols = tuple(fin.columns.tolist())
    col_counter[cols] = col_counter.get(cols, 0) + 1
    if sample_cols is None:
        sample_cols = fin.columns.tolist()
    if i % 500 == 0 or i == len(fin_files):
        print(f"  scanned {i}/{len(fin_files)}")

# 上位の列構成を保存
rows = []
for cols, cnt in sorted(col_counter.items(), key=lambda x: x[1], reverse=True)[:20]:
    rows.append({"count": cnt, "n_cols": len(cols), "cols": "|".join(cols)})
pd.DataFrame(rows).to_csv(DIAG_FINS_COLS_CSV, index=False, encoding="utf-8-sig")
print(f"✅ saved: {DIAG_FINS_COLS_CSV}")

# =========================
# Step2-2: fins 読み込み（必要列だけ抽出）→ asof 用に snapshot_date を付与
# =========================
print("\n" + "="*120)
print("[Step2-2] load fins (light) + attach snapshot_date")
print("="*120)

# ここで「候補列」を広めに定義しておく（実際に存在するものだけ残す）
CAND = {
    "Code": ["Code"],
    "DiscDate": ["DiscDate"],
    "ROE": ["ROE"],
    "INV_Growth": ["INV_Growth"],
    "Eq_yen": ["Eq_yen", "EqJPY", "Equity_yen"],
    "TA_yen": ["TA_yen", "Assets_yen"],
    "NP_yen": ["NP_yen", "NetProfit_yen"],
    "Eq": ["Eq", "Equity"],
    "TA": ["TA", "TotalAssets"],
    "NP": ["NP", "NetProfit"],
    # 株数候補（あれば MarketCap 作成に使える）
    "SharesOut": ["SharesOut", "ShOutFY", "TrShFY", "ShareOutstanding", "ShOut"]
}

def pick_col(fin_cols, candidates):
    for c in candidates:
        if c in fin_cols:
            return c
    return None

fin_parts = []
for i, (d, fp) in enumerate(fin_files, 1):
    fin = pd.read_parquet(fp)
    cols = fin.columns.tolist()

    code_col = pick_col(cols, CAND["Code"])
    if code_col is None:
        raise KeyError(f"fins missing Code: {fp}")

    # このファイルで使える列だけ集める
    picked = {}
    for k, cands in CAND.items():
        cc = pick_col(cols, cands)
        if cc is not None:
            picked[k] = cc

    use_cols = list(set(picked.values()))
    fin = fin[use_cols].copy()
    fin = fin.rename(columns={v: k for k, v in picked.items()})

    fin["Code"] = fin["Code"].astype(str)
    fin = fin[fin["Code"].notna() & ~fin["Code"].isin(["None","nan",""])].copy()
    fin["snapshot_date"] = pd.Timestamp(d)

    if "DiscDate" in fin.columns:
        fin["DiscDate"] = pd.to_datetime(fin["DiscDate"], errors="coerce")

    fin_parts.append(fin)

    if i % 300 == 0 or i == len(fin_files):
        print(f"  loaded {i}/{len(fin_files)}: {d.date()}")

fin_all = pd.concat(fin_parts, ignore_index=True)
print(f"fin_all rows={len(fin_all):,} codes={fin_all['Code'].nunique():,} snapshot_date range={fin_all['snapshot_date'].min().date()}..{fin_all['snapshot_date'].max().date()}")

# =========================
# Step2-3: asof結合（Codeごと）
# =========================
print("\n" + "="*120)
print("[Step2-3] merge_asof by Code (price.Date <= fin.snapshot_date)")
print("="*120)

left = price[["Code","MonthEnd","Date","AdjustedClose"]].copy()
left = left.sort_values(["Code","Date"])

right = fin_all.sort_values(["Code","snapshot_date"])

out_parts = []
codes = left["Code"].unique()
print(f"asof codes: {len(codes):,}")

for j, cd in enumerate(codes, 1):
    l = left[left["Code"] == cd].sort_values("Date")
    r = right[right["Code"] == cd].sort_values("snapshot_date")
    if len(r) == 0:
        out_parts.append(l)
        continue

    l2 = pd.merge_asof(
        l, r,
        left_on="Date", right_on="snapshot_date",
        direction="backward",
        allow_exact_matches=True
    )
    out_parts.append(l2)

    if j % 400 == 0 or j == len(codes):
        print(f"  asof {j}/{len(codes)}")

asof_df = pd.concat(out_parts, ignore_index=True)

# =========================
# Step2-4: MarketCap/BM_Ratio生成（可能な範囲で）
# =========================
print("\n" + "="*120)
print("[Step2-4] derive MarketCap and BM_Ratio if possible")
print("="*120)

# 数値化
for c in ["ROE","INV_Growth","Eq_yen","Eq","SharesOut"]:
    if c in asof_df.columns:
        asof_df[c] = pd.to_numeric(asof_df[c], errors="coerce")

# MarketCap：SharesOutが取れた場合のみ作る（取れないなら別途対応）
if "SharesOut" in asof_df.columns:
    asof_df["MarketCap"] = asof_df["AdjustedClose"] * asof_df["SharesOut"]
else:
    asof_df["MarketCap"] = np.nan

# BM_Ratio：Eq_yenがあればそれを使う。なければ Eq を使う（単位は後で補正）
if "Eq_yen" in asof_df.columns:
    book = asof_df["Eq_yen"]
elif "Eq" in asof_df.columns:
    book = asof_df["Eq"]
else:
    book = np.nan

asof_df["BM_Ratio"] = np.where(
    asof_df["MarketCap"].notna() & (asof_df["MarketCap"] > 0),
    book / asof_df["MarketCap"],
    np.nan
)

# month_end_snapshot として整形
snap = asof_df[[
    "Code","MonthEnd","Date","AdjustedClose",
    "MarketCap","BM_Ratio",
    "ROE","INV_Growth"
]].copy()

# 一意性チェック
dup = snap.duplicated(["Code","MonthEnd"]).sum()
print(f"snap rows={len(snap):,} dup(Code,MonthEnd)={dup:,}")
if dup != 0:
    raise RuntimeError("snapが (Code,MonthEnd) で一意ではありません（asof側が壊れてます）")

# カバレッジ診断（月ごとにMarketCap/BM/ROE/INVがどれだけ埋まっているか）
cov = (snap.groupby("MonthEnd")
       .agg(n=("Code","size"),
            mcap_notna=("MarketCap", lambda s: float(s.notna().mean())),
            bm_notna=("BM_Ratio", lambda s: float(s.notna().mean())),
            roe_notna=("ROE", lambda s: float(s.notna().mean())),
            inv_notna=("INV_Growth", lambda s: float(s.notna().mean())))
       .reset_index())
for c in ["mcap_notna","bm_notna","roe_notna","inv_notna"]:
    cov[c] = (cov[c]*100).round(2)
cov.to_csv(DIAG_COVERAGE_CSV, index=False, encoding="utf-8-sig")
print(f"✅ saved: {DIAG_COVERAGE_CSV}")

# 保存
OUT_SNAPSHOT_PATH.parent.mkdir(parents=True, exist_ok=True)
snap.to_parquet(OUT_SNAPSHOT_PATH, engine="pyarrow", index=False)
print(f"✅ saved: {OUT_SNAPSHOT_PATH}")
print("columns:", list(snap.columns))
print(f"Date.max={snap['Date'].max().date()} MonthEnd.max={snap['MonthEnd'].max().date()}")
print("coverage(%):",
      f"MarketCap={snap['MarketCap'].notna().mean()*100:.2f}",
      f"BM={snap['BM_Ratio'].notna().mean()*100:.2f}",
      f"ROE={snap['ROE'].notna().mean()*100:.2f}",
      f"INV={snap['INV_Growth'].notna().mean()*100:.2f}",
)


[Step2-0] load price_month_end
price rows=474,003 codes=5,225 months=117 Date.max=2025-09-30

[Step2-1] scan fins columns (diagnostics)
fins files: 2437  (2016-01-15 .. 2026-01-09)
  scanned 500/2437
  scanned 1000/2437
  scanned 1500/2437
  scanned 2000/2437
  scanned 2437/2437
✅ saved: C:\Users\yongr\Project\merged_data_all_stocks\factors\diag_fins_columns_summary.csv

[Step2-2] load fins (light) + attach snapshot_date
  loaded 300/2437: 2017-04-04
  loaded 600/2437: 2018-06-25
  loaded 900/2437: 2019-09-18
  loaded 1200/2437: 2020-12-15
  loaded 1500/2437: 2022-03-09
  loaded 1800/2437: 2023-06-01
  loaded 2100/2437: 2024-08-20
  loaded 2400/2437: 2025-11-12
  loaded 2437/2437: 2026-01-09
fin_all rows=190,872 codes=4,663 snapshot_date range=2016-01-15..2026-01-09

[Step2-3] merge_asof by Code (price.Date <= fin.snapshot_date)
asof codes: 5,225


MergeError: incompatible merge keys [0] dtype('<M8[ns]') and dtype('<M8[s]'), must be the same type

In [ ]:
# -*- coding: utf-8 -*-
"""
年次（10/1）割安高質バックテスト（100株単位・税引後）に
FF5+MOMのリスク制御（C：レジーム減速＋β制約、riskoff=0.5）を統合。

【最小改修で高速化】
- prices_df を Code ごとの辞書 prices_by_code に前処理
- get_near_price() は巨大DFを毎回フィルタせず、prices_by_code[code]のみを探索

税金:
- 簡易：年次（10/1→翌10/1）の最終損益にのみ課税

出力（統合前後＋診断）:
- annual_returns_raw.csv / annual_returns_riskcontrol.csv
- cumulative_curve_raw.csv / cumulative_curve_riskcontrol.csv
- performance_summary_raw.csv / performance_summary_riskcontrol.csv
- regression_ff5mom_raw.csv / regression_ff5mom_riskcontrol.csv
- diag_invest_ratio_monthly.csv

【追加出力（既存は変更しない）】
- daily_equity_curve_raw.csv / daily_equity_curve_riskcontrol.csv
- daily_mdd_summary.csv
- daily_drawdown_events_raw.csv / daily_drawdown_events_riskcontrol.csv
- daily_drawdown_events_max_summary.csv

依存:
  pip install pandas numpy pyarrow tqdm
"""

import warnings
warnings.filterwarnings("ignore")

import os
import logging
from pathlib import Path
from typing import Dict, Tuple, List, Optional

import numpy as np
import pandas as pd
from tqdm import tqdm


# ===================================
# ロギング設定
# ===================================
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler("backtest_october_unit_with_ff5mom_riskcontrol.log", encoding="utf-8"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)


# ===================================
# パス（あなたの環境）
# ===================================
FACTORS_DIR = Path(r"C:\Users\yongr\Project\merged_data_all_stocks\factors")
FF5MOM_FACTOR_PATH = FACTORS_DIR / "ff5_mom_factors_monthly.parquet"

CACHE_FILE = "topix_quarterly_statements.csv"
OHLCV_DIR = "./OHLCV_Adjusted"

OUT_DIR = FACTORS_DIR / "bt_october_unit_with_ff5mom_riskcontrol"
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_ANNUAL_RAW = OUT_DIR / "annual_returns_raw.csv"
OUT_ANNUAL_RC  = OUT_DIR / "annual_returns_riskcontrol.csv"

OUT_CURVE_RAW  = OUT_DIR / "cumulative_curve_raw.csv"
OUT_CURVE_RC   = OUT_DIR / "cumulative_curve_riskcontrol.csv"

OUT_PERF_RAW   = OUT_DIR / "performance_summary_raw.csv"
OUT_PERF_RC    = OUT_DIR / "performance_summary_riskcontrol.csv"

OUT_REG_RAW    = OUT_DIR / "regression_ff5mom_raw.csv"
OUT_REG_RC     = OUT_DIR / "regression_ff5mom_riskcontrol.csv"

OUT_IR_DIAG    = OUT_DIR / "diag_invest_ratio_monthly.csv"

# --- 追加出力（日次）
OUT_DAILY_CURVE_RAW = OUT_DIR / "daily_equity_curve_raw.csv"
OUT_DAILY_CURVE_RC  = OUT_DIR / "daily_equity_curve_riskcontrol.csv"
OUT_DAILY_MDD_SUMMARY = OUT_DIR / "daily_mdd_summary.csv"

OUT_DD_EVENTS_RAW = OUT_DIR / "daily_drawdown_events_raw.csv"
OUT_DD_EVENTS_RC  = OUT_DIR / "daily_drawdown_events_riskcontrol.csv"
OUT_DD_EVENTS_MAX = OUT_DIR / "daily_drawdown_events_max_summary.csv"


# ===================================
# 税・単位株
# ===================================
TAX_RATE = 0.20315
UNIT_SHARES = 100
INITIAL_CAPITAL = 10_000_000


# ===================================
# リスク制御パラメータ（指定）
# ===================================
RISKOFF_RATIO = 0.5

REGIME_LOOKBACK_M = 3
REGIME_OFF_IF_SUM_MKT_LT = 0.0
REGIME_OFF_IF_SUM_WML_LT = 0.0

BETA_CAP_CMA_LT = -0.8
BETA_CAP_ABS_MKT_GT = 0.9

BETA_EST_WINDOW_M = 12
BETA_EST_MIN_OBS = 10

FACTOR_COLS = ["MKT", "SMB", "HML", "RMW", "CMA", "WML"]

BAD_CODE_STRINGS = {"None", "nan", "", "NaN", "NULL", "null"}


# ===================================
# ユーティリティ
# ===================================
def normalize_code(code) -> str:
    if code is None:
        return ""
    s = str(code).strip()
    if s in BAD_CODE_STRINGS:
        return ""
    return s


def ols_alpha_beta(y: np.ndarray, X: np.ndarray) -> Tuple[float, np.ndarray, float]:
    n = len(y)
    if n < 3:
        return np.nan, np.full(X.shape[1], np.nan), np.nan

    X1 = np.column_stack([np.ones(n), X])
    XtX = X1.T @ X1
    try:
        inv = np.linalg.inv(XtX)
    except np.linalg.LinAlgError:
        inv = np.linalg.pinv(XtX)
    b = inv @ (X1.T @ y)

    yhat = X1 @ b
    resid = y - yhat
    sse = float(np.sum(resid**2))
    sst = float(np.sum((y - y.mean())**2))
    r2 = np.nan if sst <= 0 else (1.0 - sse / sst)

    alpha = float(b[0])
    betas = b[1:].astype(float)
    return alpha, betas, float(r2)


def compute_drawdown(cum: pd.Series) -> pd.Series:
    peak = cum.cummax()
    return cum / peak - 1.0


def perf_stats(annual_ret: pd.Series) -> Dict:
    r = annual_ret.dropna().astype(float)
    if r.empty:
        return {"n_years": 0, "CAGR": np.nan, "ann_mean": np.nan, "ann_vol": np.nan, "sharpe0": np.nan, "maxDD": np.nan, "cum_end": np.nan}

    n = len(r)
    cum = (1.0 + r).cumprod()
    years = n
    cagr = float(cum.iloc[-1] ** (1/years) - 1.0) if years > 0 else np.nan
    ann_mean = float(r.mean())
    ann_vol = float(r.std(ddof=1)) if n >= 2 else np.nan
    sharpe0 = float(ann_mean / ann_vol) if ann_vol and ann_vol > 0 else np.nan
    dd = compute_drawdown(cum)
    maxdd = float(dd.min())

    return {
        "n_years": int(n),
        "CAGR": cagr,
        "ann_mean": ann_mean,
        "ann_vol": ann_vol,
        "sharpe0": sharpe0,
        "maxDD": maxdd,
        "cum_end": float(cum.iloc[-1]),
    }


def extract_max_drawdown_event_from_daily_curve(df_daily: pd.DataFrame, label: str = "") -> pd.DataFrame:
    if df_daily is None or df_daily.empty:
        return pd.DataFrame([{
            "label": label,
            "peak_date": pd.NaT,
            "trough_date": pd.NaT,
            "recovery_date": pd.NaT,
            "dd_min": np.nan,
            "peak_equity": np.nan,
            "trough_equity": np.nan,
            "recovery_equity": np.nan,
            "days_to_trough": np.nan,
            "days_to_recovery": np.nan,
            "dd_duration_days": np.nan,
        }])

    df = df_daily.copy()
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df = df.dropna(subset=["Date"]).sort_values("Date").reset_index(drop=True)

    for col in ["equity_total", "dd_total"]:
        if col not in df.columns:
            raise KeyError(f"daily curve missing required column: {col}")

    df["equity_total"] = pd.to_numeric(df["equity_total"], errors="coerce")
    df["dd_total"] = pd.to_numeric(df["dd_total"], errors="coerce")
    df = df.dropna(subset=["equity_total", "dd_total"]).copy()
    if df.empty:
        return pd.DataFrame([{
            "label": label,
            "peak_date": pd.NaT,
            "trough_date": pd.NaT,
            "recovery_date": pd.NaT,
            "dd_min": np.nan,
            "peak_equity": np.nan,
            "trough_equity": np.nan,
            "recovery_equity": np.nan,
            "days_to_trough": np.nan,
            "days_to_recovery": np.nan,
            "dd_duration_days": np.nan,
        }])

    trough_idx = int(df["dd_total"].idxmin())
    dd_min = float(df.loc[trough_idx, "dd_total"])
    trough_date = pd.Timestamp(df.loc[trough_idx, "Date"])
    trough_equity = float(df.loc[trough_idx, "equity_total"])

    df_pre = df.loc[:trough_idx].copy()
    peak_equity = float(df_pre["equity_total"].max())
    peak_idx = int(df_pre["equity_total"].idxmax())
    peak_date = pd.Timestamp(df.loc[peak_idx, "Date"])

    df_post = df.loc[trough_idx:].copy()
    rec = df_post[df_post["equity_total"] >= peak_equity]
    if len(rec) == 0:
        recovery_date = pd.NaT
        recovery_equity = np.nan
        days_to_recovery = np.nan
        dd_duration_days = np.nan
    else:
        rec_idx = int(rec.index[0])
        recovery_date = pd.Timestamp(df.loc[rec_idx, "Date"])
        recovery_equity = float(df.loc[rec_idx, "equity_total"])
        days_to_recovery = int((recovery_date - peak_date).days)
        dd_duration_days = int((recovery_date - peak_date).days)

    days_to_trough = int((trough_date - peak_date).days)

    return pd.DataFrame([{
        "label": label,
        "peak_date": peak_date,
        "trough_date": trough_date,
        "recovery_date": recovery_date,
        "dd_min": dd_min,
        "peak_equity": peak_equity,
        "trough_equity": trough_equity,
        "recovery_equity": recovery_equity,
        "days_to_trough": days_to_trough,
        "days_to_recovery": days_to_recovery,
        "dd_duration_days": dd_duration_days,
    }])


def daily_mdd(df_daily: pd.DataFrame) -> float:
    if df_daily is None or df_daily.empty or "dd_total" not in df_daily.columns:
        return np.nan
    s = pd.to_numeric(df_daily["dd_total"], errors="coerce").dropna()
    return float(s.min()) if len(s) else np.nan


# ===================================
# A. 財務データ読み込み（列名自動判定）
# ===================================
def load_financial_data(cache_filename: str = CACHE_FILE) -> pd.DataFrame:
    if not os.path.exists(cache_filename):
        logger.error(f"財務データファイル '{cache_filename}' が見つかりません")
        return pd.DataFrame()

    try:
        df = pd.read_csv(cache_filename, encoding="utf-8-sig", parse_dates=["DisclosedDate"], low_memory=False)
        logger.info(f"財務データ読み込み成功: {len(df):,}件")

        column_mapping = {
            "IssuedShareTotal": ["IssuedShareTotal", "NumberOfIssuedAndOutstandingSharesAtTheEndOfFiscalYearIncludingTreasuryStock"],
            "Equity": ["Equity", "NetAssets", "TotalEquity"],
            "Profit": ["Profit", "NetIncome", "ProfitAttributableToOwnersOfParent"],
        }

        available_cols = df.columns.tolist()
        required_columns = {}
        for target_col, possible_names in column_mapping.items():
            found = False
            for possible_name in possible_names:
                if possible_name in available_cols:
                    required_columns[target_col] = possible_name
                    found = True
                    break
            if not found:
                required_columns[target_col] = None

        rename_dict = {v: k for k, v in required_columns.items() if v is not None}
        df = df.rename(columns=rename_dict)

        for col in ["IssuedShareTotal", "Equity", "Profit"]:
            if col not in df.columns:
                df[col] = 0

        base_cols = ["Code", "DisclosedDate"]
        if "CompanyName" in df.columns:
            base_cols.append("CompanyName")
        final_cols = base_cols + ["Profit", "Equity", "IssuedShareTotal"]
        df = df[final_cols].copy()

        logger.info(f"使用列: {df.columns.tolist()}")
        return df

    except Exception as e:
        logger.error(f"財務データ読み込みエラー: {e}", exc_info=True)
        return pd.DataFrame()


# ===================================
# B. 株価データ読み込み
# ===================================
def load_existing_price_data(ohlcv_dir: str = OHLCV_DIR) -> pd.DataFrame:
    if not os.path.exists(ohlcv_dir):
        logger.error(f"株価データディレクトリ '{ohlcv_dir}' が見つかりません。")
        return pd.DataFrame()

    csv_files = sorted([f for f in os.listdir(ohlcv_dir)
                        if f.startswith("OHLCV_Adjusted_") and f.endswith(".csv") and f != "OHLCV_Adjusted_TOPIX.csv"])

    if not csv_files:
        logger.error(f"ディレクトリ '{ohlcv_dir}' 内にCSVファイルが見つかりません。")
        return pd.DataFrame()

    logger.info(f"株価ファイル数: {len(csv_files)}個")

    all_dataframes = []
    usecols = ["Date", "Ticker", "AdjustmentClose"]

    for csv_file in tqdm(csv_files, desc="株価ファイル読み込み中"):
        file_path = os.path.join(ohlcv_dir, csv_file)
        try:
            df = pd.read_csv(
                file_path,
                usecols=usecols,
                parse_dates=["Date"],
                dtype={"Ticker": "Int64", "AdjustmentClose": "float32"},
            )
            df = df.drop_duplicates(subset=["Ticker", "Date"], keep="first")
            all_dataframes.append(df)
        except Exception as e:
            logger.warning(f"ファイル読み込みエラー ({csv_file}): {e}")
            continue

    if not all_dataframes:
        return pd.DataFrame()

    logger.info("全ファイルを結合中...")
    df_all = pd.concat(all_dataframes, ignore_index=True)
    logger.info(f"結合完了: {len(df_all):,}件")

    df_all = df_all.rename(columns={"Ticker": "Code", "AdjustmentClose": "Close"})
    df_all["Code"] = df_all["Code"].astype("str").str.replace("<NA>", "0").str.zfill(4)
    df_all = df_all.dropna(subset=["Close"])

    logger.info("重複除去 & ソート中...")
    df_all = df_all.sort_values(["Code", "Date"])
    df_all = df_all.drop_duplicates(subset=["Code", "Date"], keep="first")
    logger.info(f"重複除去後: {len(df_all):,}件")

    return df_all


def build_prices_by_code(prices_df: pd.DataFrame) -> Dict[str, pd.DataFrame]:
    d = {}
    tmp = prices_df[["Code", "Date", "Close"]].copy()
    tmp["Code"] = tmp["Code"].astype(str).map(normalize_code)
    tmp = tmp[tmp["Code"] != ""].copy()
    tmp = tmp.sort_values(["Code", "Date"])

    for code, g in tmp.groupby("Code", sort=False):
        gg = g[["Date", "Close"]].drop_duplicates(subset=["Date"], keep="last").sort_values("Date").copy()
        gg = gg.set_index("Date", drop=False)
        d[code] = gg

    logger.info(f"prices_by_code built: {len(d):,} codes")
    return d


def get_near_price_fast(prices_by_code: Dict[str, pd.DataFrame],
                        code: str,
                        ref_date: pd.Timestamp,
                        kind: str = "last") -> Optional[float]:
    code = normalize_code(code)
    if not code or code not in prices_by_code:
        return None
    g = prices_by_code[code]
    lo = ref_date - pd.Timedelta(days=5)
    hi = ref_date + pd.Timedelta(days=5)
    w = g.loc[(g["Date"] >= lo) & (g["Date"] <= hi)]
    if w.empty:
        return None
    return float(w.iloc[0]["Close"]) if kind == "first" else float(w.iloc[-1]["Close"])


def safe_code_to_int(code_series: pd.Series) -> pd.Series:
    cleaned = code_series.astype(str).str.replace(r"\D", "", regex=True)
    cleaned = cleaned.replace("", "0")
    return pd.to_numeric(cleaned, errors="coerce").fillna(0).astype("int64")


def calculate_market_metrics_fast_chunked(statements_df: pd.DataFrame,
                                          prices_df: pd.DataFrame,
                                          chunk_size: int = 200) -> pd.DataFrame:
    logger.info("時価総額・PBR・ROE計算中（チャンク処理版）...")

    if prices_df.empty:
        logger.error("株価データが空です。")
        return pd.DataFrame()

    req_cols = {"Code", "Date", "Close"}
    if not req_cols.issubset(set(prices_df.columns)):
        logger.error(f"株価データに必要な列が存在しません。存在する列: {prices_df.columns.tolist()}")
        return pd.DataFrame()

    statements_df = statements_df.copy()
    statements_df["Profit"] = pd.to_numeric(statements_df["Profit"], errors="coerce").fillna(0)
    statements_df["Equity"] = pd.to_numeric(statements_df["Equity"], errors="coerce").fillna(0)
    statements_df["IssuedShareTotal"] = pd.to_numeric(statements_df["IssuedShareTotal"], errors="coerce").fillna(1)

    statements_df = statements_df[(statements_df["Equity"] > 0) & (statements_df["IssuedShareTotal"] > 0)]
    logger.info(f"有効な財務データ: {len(statements_df):,}件")

    statements_df["Code_int"] = safe_code_to_int(statements_df["Code"])
    prices_df = prices_df.copy()
    prices_df["Code_int"] = safe_code_to_int(prices_df["Code"])

    statements_df = statements_df[statements_df["Code_int"] > 0]
    prices_df = prices_df[prices_df["Code_int"] > 0]

    statements_df = statements_df.sort_values(["Code_int", "DisclosedDate"]).reset_index(drop=True)
    prices_df = prices_df.sort_values(["Code_int", "Date"]).reset_index(drop=True)

    statements_df = statements_df.drop_duplicates(subset=["Code_int", "DisclosedDate"], keep="first")
    prices_df = prices_df.drop_duplicates(subset=["Code_int", "Date"], keep="first")

    logger.info(f"ソート・重複除去後: 財務 {len(statements_df):,}件, 株価 {len(prices_df):,}件")

    statements_groups = list(statements_df.groupby("Code_int"))
    prices_dict = {code: group for code, group in prices_df.groupby("Code_int")}

    merged_list = []
    num_chunks = (len(statements_groups) + chunk_size - 1) // chunk_size

    for chunk_idx in tqdm(range(num_chunks), desc="マージ処理"):
        start_idx = chunk_idx * chunk_size
        end_idx = min((chunk_idx + 1) * chunk_size, len(statements_groups))
        chunk_groups = statements_groups[start_idx:end_idx]

        for code, stmt_code in chunk_groups:
            if code not in prices_dict:
                continue
            price_code = prices_dict[code]
            if len(price_code) == 0:
                continue

            stmt_code = stmt_code.sort_values("DisclosedDate").reset_index(drop=True)
            price_code = price_code.sort_values("Date").reset_index(drop=True)

            try:
                merged = pd.merge_asof(
                    stmt_code,
                    price_code[["Date", "Close"]],
                    left_on="DisclosedDate",
                    right_on="Date",
                    direction="backward",
                    tolerance=pd.Timedelta(days=10),
                )
                if not merged.empty:
                    merged_list.append(merged)
            except Exception:
                continue

    if not merged_list:
        logger.error("マージ結果が空です")
        return pd.DataFrame()

    df_merged = pd.concat(merged_list, ignore_index=True)
    df_merged = df_merged.dropna(subset=["Close"])
    logger.info(f"マージ完了: {len(df_merged):,}件")

    df_merged["MarketCap"] = df_merged["Close"] * df_merged["IssuedShareTotal"]
    df_merged["PBR"] = df_merged["MarketCap"] / df_merged["Equity"]
    df_merged["ROE"] = (df_merged["Profit"] / df_merged["Equity"]) * 100

    result_cols = ["Code", "DisclosedDate", "Close", "MarketCap", "PBR", "ROE", "Date"]
    if "CompanyName" in df_merged.columns:
        result_cols.insert(1, "CompanyName")

    result_df = df_merged[result_cols].copy()
    result_df = result_df.rename(columns={"Close": "StockPrice", "Date": "PriceDate"})

    mask = (
        (result_df["PBR"] > 0) &
        (result_df["PBR"] < 50) &
        (result_df["ROE"] > -100) &
        (result_df["ROE"] < 100) &
        (result_df["MarketCap"] > 1_000_000_000)
    )
    result_df = result_df[mask].copy()

    logger.info(f"計算完了: {len(result_df):,}件")
    return result_df


def build_unit_share_portfolio(stock_candidates: pd.DataFrame,
                               target_positions: int = 20,
                               initial_capital: float = 10_000_000) -> dict:
    if len(stock_candidates) == 0:
        return {"stocks": [], "shares": [], "prices": [], "amounts": []}

    selected = stock_candidates.head(target_positions).copy()
    capital_per_stock = initial_capital / len(selected)

    stocks, shares_list, prices_list, amounts_list = [], [], [], []

    for _, row in selected.iterrows():
        code = row["Code"]
        price = row["StockPrice"]
        required_amount = price * UNIT_SHARES

        if required_amount <= capital_per_stock:
            shares = int(capital_per_stock // required_amount) * UNIT_SHARES
            if shares > 0:
                stocks.append(code)
                shares_list.append(shares)
                prices_list.append(price)
                amounts_list.append(shares * price)

    return {"stocks": stocks, "shares": shares_list, "prices": prices_list, "amounts": amounts_list}


def make_month_ends(start: pd.Timestamp, end: pd.Timestamp) -> List[pd.Timestamp]:
    m0 = pd.Timestamp(start.year, start.month, 1) + pd.offsets.MonthEnd(0)
    m1 = pd.Timestamp(end.year, end.month, 1) + pd.offsets.MonthEnd(0)
    months = pd.date_range(m0, m1, freq="M")
    return [pd.Timestamp(x).normalize() for x in months]


def build_daily_equity_curve_for_period(
    portfolio: dict,
    start_date: pd.Timestamp,
    end_date: pd.Timestamp,
    prices_by_code: Dict[str, pd.DataFrame],
    invest_ratio_by_monthend: Optional[Dict[pd.Timestamp, float]] = None,
) -> pd.DataFrame:
    stocks = portfolio.get("stocks", [])
    shares = portfolio.get("shares", [])
    if not stocks:
        return pd.DataFrame()

    start_date = pd.to_datetime(start_date).normalize()
    end_date = pd.to_datetime(end_date).normalize()

    date_sets = []
    for code in stocks:
        code = normalize_code(code)
        if code in prices_by_code:
            g = prices_by_code[code]
            d = g[(g["Date"] >= start_date - pd.Timedelta(days=10)) & (g["Date"] <= end_date + pd.Timedelta(days=10))]["Date"]
            if len(d):
                date_sets.append(d)

    if not date_sets:
        return pd.DataFrame()

    dates = pd.Index(sorted(pd.unique(pd.concat(date_sets)))).astype("datetime64[ns]")
    dates = dates[(dates >= start_date) & (dates <= end_date)]
    if len(dates) == 0:
        return pd.DataFrame()

    values = []
    for dt in dates:
        v = 0.0
        ok = False
        for i, code in enumerate(stocks):
            code = normalize_code(code)
            if code not in prices_by_code:
                continue
            p = get_near_price_fast(prices_by_code, code, pd.Timestamp(dt), kind="last")
            if p is None:
                continue
            v += float(shares[i]) * float(p)
            ok = True
        values.append(v if ok else np.nan)

    df = pd.DataFrame({"Date": pd.to_datetime(dates), "equity_stock": values})
    df = df.dropna(subset=["equity_stock"]).copy()
    if df.empty:
        return df

    df = df.sort_values("Date").reset_index(drop=True)
    df["ret_stock"] = df["equity_stock"].pct_change().fillna(0.0)

    df["MonthEnd"] = (df["Date"] + pd.offsets.MonthEnd(0)).dt.normalize()

    if invest_ratio_by_monthend is None:
        df["invest_ratio"] = 1.0
        df["ret_total"] = df["ret_stock"]
    else:
        df["invest_ratio"] = df["MonthEnd"].map(invest_ratio_by_monthend).fillna(1.0).astype(float)
        df["ret_total"] = df["invest_ratio"] * df["ret_stock"]

    df["equity_total"] = (1.0 + df["ret_total"]).cumprod()
    peak = df["equity_total"].cummax()
    df["dd_total"] = df["equity_total"] / peak - 1.0

    return df[["Date", "MonthEnd", "equity_stock", "ret_stock", "invest_ratio", "ret_total", "equity_total", "dd_total"]]


def load_ff5mom_factors_monthly() -> pd.DataFrame:
    fac = pd.read_parquet(FF5MOM_FACTOR_PATH).copy()
    fac["MonthEnd"] = pd.to_datetime(fac["MonthEnd"], errors="coerce").dt.normalize()
    need = ["MonthEnd"] + FACTOR_COLS
    missing = [c for c in need if c not in fac.columns]
    if missing:
        raise KeyError(f"FF5MOM factors missing columns: {missing} in {FF5MOM_FACTOR_PATH}")
    fac = fac[need].sort_values("MonthEnd").reset_index(drop=True)
    return fac


def compute_regime_off(fac: pd.DataFrame, month_end: pd.Timestamp) -> Tuple[bool, Dict]:
    fac2 = fac.set_index("MonthEnd")
    idx = fac2.index[fac2.index < month_end]
    if len(idx) < REGIME_LOOKBACK_M:
        return False, {"regime_ready": False}

    win = idx[-REGIME_LOOKBACK_M:]
    s_mkt = float(pd.to_numeric(fac2.loc[win, "MKT"], errors="coerce").sum())
    s_wml = float(pd.to_numeric(fac2.loc[win, "WML"], errors="coerce").sum())

    # ===== PATCH START (Option 1): OR -> AND =====
    off = (s_mkt < REGIME_OFF_IF_SUM_MKT_LT) and (s_wml < REGIME_OFF_IF_SUM_WML_LT)
    # ===== PATCH END =====

    return bool(off), {"regime_ready": True, "sum_MKT_3m": s_mkt, "sum_WML_3m": s_wml}


def estimate_port_beta(monthly_port_rets: pd.DataFrame, fac: pd.DataFrame, month_end: pd.Timestamp) -> Tuple[bool, Dict]:
    fac2 = fac.set_index("MonthEnd")
    idx = fac2.index[fac2.index < month_end]
    if len(idx) < BETA_EST_WINDOW_M:
        return False, {"beta_ready": False}

    win = idx[-BETA_EST_WINDOW_M:]

    m = monthly_port_rets.copy()
    m["MonthEnd"] = pd.to_datetime(m["MonthEnd"], errors="coerce").dt.normalize()
    m = m.dropna(subset=["MonthEnd", "port_ret"]).copy()

    # MonthEnd重複を1行化（merge validate="one_to_one"維持）
    m = (
        m.groupby("MonthEnd", as_index=False)["port_ret"]
         .apply(lambda s: (1.0 + s.astype(float)).prod() - 1.0)
    )

    df = m[m["MonthEnd"].isin(win)].merge(
        fac, on="MonthEnd", how="left", validate="one_to_one"
    ).dropna(subset=["port_ret"] + FACTOR_COLS)

    if len(df) < BETA_EST_MIN_OBS:
        return False, {"beta_ready": False, "n_obs": int(len(df))}

    y = df["port_ret"].to_numpy(dtype=float)
    X = df[FACTOR_COLS].to_numpy(dtype=float)
    alpha, betas, r2 = ols_alpha_beta(y, X)

    out = {"beta_ready": True, "n_obs": int(len(df)), "r2": float(r2), "alpha": float(alpha)}
    for c, b in zip(FACTOR_COLS, betas):
        out[f"beta_{c}"] = float(b)
    return True, out


def decide_invest_ratio(month_end: pd.Timestamp,
                        fac: pd.DataFrame,
                        port_hist: pd.DataFrame) -> Tuple[float, Dict]:
    invest_ratio = 1.0
    diag = {
        "MonthEnd": month_end,
        "invest_ratio": 1.0,
        "regime_off": False,
        "beta_off": False,
        "regime_ready": False,
        "beta_ready": False,
        "sum_MKT_3m": np.nan,
        "sum_WML_3m": np.nan,
        "beta_MKT": np.nan,
        "beta_CMA": np.nan,
        "beta_n_obs": np.nan,
        "beta_r2": np.nan,
    }

    off_reg, info = compute_regime_off(fac, month_end)
    diag["regime_off"] = bool(off_reg)
    diag["regime_ready"] = bool(info.get("regime_ready", False))
    diag["sum_MKT_3m"] = info.get("sum_MKT_3m", np.nan)
    diag["sum_WML_3m"] = info.get("sum_WML_3m", np.nan)
    if off_reg:
        invest_ratio = min(invest_ratio, RISKOFF_RATIO)

    ok_beta, binfo = estimate_port_beta(port_hist, fac, month_end)
    diag["beta_ready"] = bool(binfo.get("beta_ready", False))
    if binfo.get("beta_ready", False):
        b_mkt = float(binfo.get("beta_MKT", np.nan))
        b_cma = float(binfo.get("beta_CMA", np.nan))
        diag["beta_MKT"] = b_mkt
        diag["beta_CMA"] = b_cma
        diag["beta_n_obs"] = float(binfo.get("n_obs", np.nan))
        diag["beta_r2"] = float(binfo.get("r2", np.nan))

        off_beta = (b_cma < BETA_CAP_CMA_LT) or (abs(b_mkt) > BETA_CAP_ABS_MKT_GT)
        diag["beta_off"] = bool(off_beta)
        if off_beta:
            invest_ratio = min(invest_ratio, RISKOFF_RATIO)

    diag["invest_ratio"] = float(invest_ratio)
    return float(invest_ratio), diag


def compute_monthly_portfolio_returns_with_riskcontrol_fast(
    portfolio: dict,
    start_date: pd.Timestamp,
    end_date: pd.Timestamp,
    prices_by_code: Dict[str, pd.DataFrame],
    fac: pd.DataFrame
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    stocks = portfolio["stocks"]
    shares = portfolio["shares"]
    if not stocks:
        return pd.DataFrame(), pd.DataFrame()

    month_ends = make_month_ends(start_date, end_date)
    boundaries = [start_date] + [me for me in month_ends if (me > start_date) and (me < end_date)] + [end_date]

    rows = []
    ir_rows = []

    port_hist = pd.DataFrame(columns=["MonthEnd", "port_ret"])

    for j in range(len(boundaries) - 1):
        d0 = boundaries[j]
        d1 = boundaries[j + 1]

        me = pd.Timestamp(d0.year, d0.month, 1) + pd.offsets.MonthEnd(0)
        me = pd.Timestamp(me).normalize()

        start_vals = []
        end_vals = []
        for i, code in enumerate(stocks):
            p0 = get_near_price_fast(prices_by_code, code, d0, kind="first")
            p1 = get_near_price_fast(prices_by_code, code, d1, kind="last")
            if p0 is None or p1 is None:
                continue
            sh = shares[i]
            start_vals.append(sh * p0)
            end_vals.append(sh * p1)

        if len(start_vals) == 0:
            continue

        start_v = float(np.sum(start_vals))
        end_v = float(np.sum(end_vals))
        stock_ret = (end_v / start_v) - 1.0

        port_hist = pd.concat([port_hist, pd.DataFrame([{"MonthEnd": me, "port_ret": stock_ret}])], ignore_index=True)

        invest_ratio, diag = decide_invest_ratio(me, fac, port_hist)
        ir_rows.append(diag)

        total_ret = invest_ratio * stock_ret

        rows.append({
            "MonthEnd": me,
            "period_start": d0,
            "period_end": d1,
            "port_ret_stock": stock_ret,
            "invest_ratio": invest_ratio,
            "port_ret_total": total_ret,
        })

    monthly_df = pd.DataFrame(rows)
    ir_diag_df = pd.DataFrame(ir_rows)
    return monthly_df, ir_diag_df


def build_long_candidates(enhanced_financial_data: pd.DataFrame, rebalance_date: pd.Timestamp) -> pd.DataFrame:
    current_data = enhanced_financial_data[enhanced_financial_data["DisclosedDate"] <= rebalance_date].copy()
    current_data = current_data.sort_values("DisclosedDate").groupby("Code").tail(1)

    if len(current_data) < 100:
        return pd.DataFrame()

    current_data["PBR_Rank"] = current_data["PBR"].rank(method="first", ascending=True)
    current_data["ROE_Rank"] = current_data["ROE"].rank(method="first", ascending=False)

    current_data["PBR_Quartile"] = pd.qcut(current_data["PBR_Rank"], q=4, labels=[1, 2, 3, 4])
    current_data["ROE_Quartile"] = pd.qcut(current_data["ROE_Rank"], q=4, labels=[1, 2, 3, 4])

    long_candidates = current_data[
        (current_data["PBR_Quartile"] == 1) &
        (current_data["ROE_Quartile"] == 4)
    ].nsmallest(50, "PBR")

    return long_candidates


def annual_return_from_monthly(monthly_rets: pd.Series) -> float:
    if monthly_rets.empty:
        return 0.0
    return float((1.0 + monthly_rets).prod() - 1.0)


def apply_tax_annual(gross_return: float, initial_capital: float) -> Tuple[float, float]:
    profit = gross_return * initial_capital
    taxable = max(profit, 0.0)
    tax = taxable * TAX_RATE
    net_profit = profit - tax
    net_return = net_profit / initial_capital
    tax_rate_total = tax / initial_capital
    return float(net_return), float(tax_rate_total)


def run_annual_backtest_with_and_without_riskcontrol_fast(
    enhanced_financial_data: pd.DataFrame,
    prices_df: pd.DataFrame,
    prices_by_code: Dict[str, pd.DataFrame],
    fac: pd.DataFrame,
    initial_capital: float = INITIAL_CAPITAL
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    rebalance_dates = [pd.Timestamp(f"{year}-10-01") for year in range(2016, 2026)]

    results_raw = []
    results_rc = []
    diag_ir_all = []

    daily_curves_raw = []
    daily_curves_rc  = []

    for i, rebalance_date in enumerate(tqdm(rebalance_dates[:-1], desc="10月1日リバランス（統合・高速）")):
        next_rebalance = rebalance_dates[i + 1]

        long_candidates = build_long_candidates(enhanced_financial_data, rebalance_date)
        if long_candidates.empty:
            logger.warning(f"{rebalance_date}: データ不足でスキップ")
            continue

        long_portfolio = build_unit_share_portfolio(
            long_candidates,
            target_positions=20,
            initial_capital=initial_capital
        )

        n_long = len(long_portfolio["stocks"])
        inv_amount = float(np.sum(long_portfolio["amounts"])) if n_long > 0 else 0.0
        inv_ratio_raw = inv_amount / initial_capital if initial_capital > 0 else 0.0

        logger.info(f"{rebalance_date.strftime('%Y-%m')}: long={n_long} invest={inv_amount:,.0f} ratio={inv_ratio_raw:.3f}")

        # --- RAW（年次 start->end の価格で計算）
        total_profit = 0.0
        total_start_value = 0.0

        for j, code in enumerate(long_portfolio["stocks"]):
            sh = long_portfolio["shares"][j]
            p0 = long_portfolio["prices"][j]
            p1 = get_near_price_fast(prices_by_code, code, next_rebalance, kind="last")
            if p1 is None:
                continue
            start_value = sh * p0
            end_value = sh * p1
            total_start_value += start_value
            total_profit += (end_value - start_value)

        gross_return_raw = (total_profit / initial_capital) if initial_capital > 0 else 0.0
        net_return_raw, tax_rate_raw = apply_tax_annual(gross_return_raw, initial_capital)

        results_raw.append({
            "date": next_rebalance,
            "strategy_return_gross": gross_return_raw,
            "strategy_return_net": net_return_raw,
            "tax": tax_rate_raw,
            "long_count": n_long,
            "investment_ratio": inv_ratio_raw,
        })

        # --- RiskControl（月次で縮尺）
        monthly_df, ir_df = compute_monthly_portfolio_returns_with_riskcontrol_fast(
            long_portfolio, rebalance_date, next_rebalance, prices_by_code, fac
        )

        if monthly_df.empty:
            gross_return_rc = 0.0
            inv_ratio_rc_avg = 0.0
        else:
            gross_return_rc = annual_return_from_monthly(monthly_df["port_ret_total"])
            inv_ratio_rc_avg = float(monthly_df["invest_ratio"].mean())

        net_return_rc, tax_rate_rc = apply_tax_annual(gross_return_rc, initial_capital)

        results_rc.append({
            "date": next_rebalance,
            "strategy_return_gross": gross_return_rc,
            "strategy_return_net": net_return_rc,
            "tax": tax_rate_rc,
            "long_count": n_long,
            "investment_ratio": inv_ratio_rc_avg,
        })

        if not ir_df.empty:
            ir_df = ir_df.copy()
            ir_df["rebalance_start"] = rebalance_date
            ir_df["rebalance_end"] = next_rebalance
            diag_ir_all.append(ir_df)

        # --- 日次曲線（RAW / RC）を追加生成
        invest_ratio_map = None
        if not ir_df.empty:
            tmp_ir = ir_df.copy()
            tmp_ir["MonthEnd"] = pd.to_datetime(tmp_ir["MonthEnd"], errors="coerce").dt.normalize()
            tmp_ir = tmp_ir.dropna(subset=["MonthEnd"])
            tmp_ir = tmp_ir.sort_values("MonthEnd").drop_duplicates(subset=["MonthEnd"], keep="last")
            invest_ratio_map = dict(zip(tmp_ir["MonthEnd"], tmp_ir["invest_ratio"].astype(float)))

        daily_raw = build_daily_equity_curve_for_period(
            long_portfolio, rebalance_date, next_rebalance, prices_by_code, invest_ratio_by_monthend=None
        )
        if not daily_raw.empty:
            daily_raw["rebalance_start"] = rebalance_date
            daily_raw["rebalance_end"] = next_rebalance
            daily_curves_raw.append(daily_raw)

        daily_rc = build_daily_equity_curve_for_period(
            long_portfolio, rebalance_date, next_rebalance, prices_by_code, invest_ratio_by_monthend=invest_ratio_map
        )
        if not daily_rc.empty:
            daily_rc["rebalance_start"] = rebalance_date
            daily_rc["rebalance_end"] = next_rebalance
            daily_curves_rc.append(daily_rc)

    df_raw = pd.DataFrame(results_raw)
    df_rc = pd.DataFrame(results_rc)
    diag_ir = pd.concat(diag_ir_all, ignore_index=True) if len(diag_ir_all) else pd.DataFrame()

    daily_raw_all = pd.concat(daily_curves_raw, ignore_index=True) if len(daily_curves_raw) else pd.DataFrame()
    daily_rc_all  = pd.concat(daily_curves_rc,  ignore_index=True) if len(daily_curves_rc)  else pd.DataFrame()

    run_annual_backtest_with_and_without_riskcontrol_fast._daily_raw_all = daily_raw_all
    run_annual_backtest_with_and_without_riskcontrol_fast._daily_rc_all = daily_rc_all

    return df_raw, df_rc, diag_ir


def build_annual_factor_from_monthly(fac: pd.DataFrame, start_date: pd.Timestamp, end_date: pd.Timestamp) -> Dict:
    fac2 = fac.copy()
    fac2 = fac2[(fac2["MonthEnd"] >= (start_date + pd.offsets.MonthEnd(0))) &
                (fac2["MonthEnd"] <= (end_date + pd.offsets.MonthEnd(-1)))].copy()
    out = {}
    for c in FACTOR_COLS:
        s = pd.to_numeric(fac2[c], errors="coerce").dropna()
        out[c] = float((1.0 + s).prod() - 1.0) if len(s) else np.nan
    return out


def run_factor_regression(results_df: pd.DataFrame, fac: pd.DataFrame) -> pd.DataFrame:
    if results_df.empty:
        return pd.DataFrame()

    rows = []
    for _, row in results_df.iterrows():
        end_date = pd.Timestamp(row["date"])
        start_date = end_date - pd.DateOffset(years=1)
        ann_fac = build_annual_factor_from_monthly(fac, start_date, end_date)
        rec = {"date": end_date}
        rec.update(ann_fac)
        rec["y"] = float(row["strategy_return_net"])
        rows.append(rec)

    df = pd.DataFrame(rows).dropna(subset=["y"] + FACTOR_COLS).copy()
    if len(df) < 3:
        return pd.DataFrame([{
            "n_years": int(len(df)),
            "R2": np.nan,
            "alpha": np.nan,
            **{f"beta_{c}": np.nan for c in FACTOR_COLS}
        }])

    y = df["y"].to_numpy(dtype=float)
    X = df[FACTOR_COLS].to_numpy(dtype=float)
    alpha, betas, r2 = ols_alpha_beta(y, X)

    out = {"n_years": int(len(df)), "R2": float(r2), "alpha": float(alpha)}
    for c, b in zip(FACTOR_COLS, betas):
        out[f"beta_{c}"] = float(b)
    return pd.DataFrame([out])


def save_annual_bundle(df: pd.DataFrame, out_annual_csv: Path, out_curve_csv: Path, out_perf_csv: Path):
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date").reset_index(drop=True)

    df.to_csv(out_annual_csv, index=False, encoding="utf-8-sig")

    r = df["strategy_return_net"].astype(float)
    cum = (1.0 + r).cumprod()
    dd = compute_drawdown(cum)
    curve = pd.DataFrame({"date": df["date"], "ret": r, "cum": cum, "dd": dd})
    curve.to_csv(out_curve_csv, index=False, encoding="utf-8-sig")

    stats = perf_stats(r)
    perf = pd.DataFrame([stats])
    perf.to_csv(out_perf_csv, index=False, encoding="utf-8-sig")


def main():
    logger.info("=" * 110)
    logger.info("10/1 年次「割安高質」バックテスト + FF5+MOM RiskControl(C) [FAST get_near_price]")
    logger.info("=" * 110)
    logger.info(f"FF5MOM_FACTOR_PATH: {FF5MOM_FACTOR_PATH}")
    logger.info(f"OUT_DIR: {OUT_DIR}")
    logger.info(f"Tax mode: simple annual tax only (TAX_RATE={TAX_RATE})")
    logger.info(f"RiskControl: regime(3m MKT<0 AND WML<0) + beta(CMA<-0.8 or |MKT|>0.9) => riskoff={RISKOFF_RATIO}  [Option1]")
    logger.info(f"Beta estimation window: {BETA_EST_WINDOW_M} months (min_obs={BETA_EST_MIN_OBS})")

    statements_df = load_financial_data()
    if statements_df.empty:
        logger.error("財務データ読み込み失敗")
        return

    prices_df = load_existing_price_data()
    if prices_df.empty:
        logger.error("株価データ読み込み失敗")
        return

    fac = load_ff5mom_factors_monthly()
    prices_by_code = build_prices_by_code(prices_df)

    enhanced_financial_data = calculate_market_metrics_fast_chunked(statements_df, prices_df, chunk_size=200)
    if enhanced_financial_data.empty:
        logger.error("財務指標計算失敗")
        return

    df_raw, df_rc, diag_ir = run_annual_backtest_with_and_without_riskcontrol_fast(
        enhanced_financial_data, prices_df, prices_by_code, fac, initial_capital=INITIAL_CAPITAL
    )

    if df_raw.empty or df_rc.empty:
        logger.error("年次バックテスト結果が空です")
        return

    save_annual_bundle(df_raw, OUT_ANNUAL_RAW, OUT_CURVE_RAW, OUT_PERF_RAW)
    save_annual_bundle(df_rc,  OUT_ANNUAL_RC,  OUT_CURVE_RC,  OUT_PERF_RC)

    reg_raw = run_factor_regression(df_raw, fac)
    reg_rc = run_factor_regression(df_rc, fac)
    reg_raw.to_csv(OUT_REG_RAW, index=False, encoding="utf-8-sig")
    reg_rc.to_csv(OUT_REG_RC, index=False, encoding="utf-8-sig")

    if not diag_ir.empty:
        diag_ir["MonthEnd"] = pd.to_datetime(diag_ir["MonthEnd"])
        diag_ir.sort_values(["rebalance_start", "MonthEnd"], inplace=True)
        diag_ir.to_csv(OUT_IR_DIAG, index=False, encoding="utf-8-sig")

    # ==============================
    # 追加：日次曲線保存 + 日次MDD + 最大DDイベント抽出（ログ出力まで）
    # ==============================
    daily_raw_all = getattr(run_annual_backtest_with_and_without_riskcontrol_fast, "_daily_raw_all", pd.DataFrame())
    daily_rc_all  = getattr(run_annual_backtest_with_and_without_riskcontrol_fast, "_daily_rc_all", pd.DataFrame())

    if not daily_raw_all.empty:
        daily_raw_all.to_csv(OUT_DAILY_CURVE_RAW, index=False, encoding="utf-8-sig")
    if not daily_rc_all.empty:
        daily_rc_all.to_csv(OUT_DAILY_CURVE_RC, index=False, encoding="utf-8-sig")

    mdd_raw_d = daily_mdd(daily_raw_all)
    mdd_rc_d  = daily_mdd(daily_rc_all)

    pd.DataFrame([{
        "daily_maxDD_raw": mdd_raw_d,
        "daily_maxDD_riskcontrol": mdd_rc_d,
        "n_days_raw": int(len(daily_raw_all)) if not daily_raw_all.empty else 0,
        "n_days_riskcontrol": int(len(daily_rc_all)) if not daily_rc_all.empty else 0,
    }]).to_csv(OUT_DAILY_MDD_SUMMARY, index=False, encoding="utf-8-sig")

    ev_raw = extract_max_drawdown_event_from_daily_curve(daily_raw_all, label="RAW")
    ev_rc  = extract_max_drawdown_event_from_daily_curve(daily_rc_all,  label="RISKCONTROL")

    ev_raw.to_csv(OUT_DD_EVENTS_RAW, index=False, encoding="utf-8-sig")
    ev_rc.to_csv(OUT_DD_EVENTS_RC, index=False, encoding="utf-8-sig")
    pd.concat([ev_raw, ev_rc], ignore_index=True).to_csv(OUT_DD_EVENTS_MAX, index=False, encoding="utf-8-sig")

    logger.info("-" * 110)
    logger.info("✅ SAVED (RAW)")
    logger.info(f"  - {OUT_ANNUAL_RAW}")
    logger.info(f"  - {OUT_CURVE_RAW}")
    logger.info(f"  - {OUT_PERF_RAW}")
    logger.info(f"  - {OUT_REG_RAW}")
    logger.info("✅ SAVED (RISKCONTROL)")
    logger.info(f"  - {OUT_ANNUAL_RC}")
    logger.info(f"  - {OUT_CURVE_RC}")
    logger.info(f"  - {OUT_PERF_RC}")
    logger.info(f"  - {OUT_REG_RC}")
    logger.info("✅ DIAGNOSTIC")
    logger.info(f"  - {OUT_IR_DIAG}")
    logger.info("✅ DAILY (ADDED)")
    logger.info(f"  - {OUT_DAILY_CURVE_RAW}")
    logger.info(f"  - {OUT_DAILY_CURVE_RC}")
    logger.info(f"  - {OUT_DAILY_MDD_SUMMARY}")
    logger.info("✅ DAILY DD EVENTS (ADDED)")
    logger.info(f"  - {OUT_DD_EVENTS_RAW}")
    logger.info(f"  - {OUT_DD_EVENTS_RC}")
    logger.info(f"  - {OUT_DD_EVENTS_MAX}")
    logger.info("-" * 110)

    raw_perf = pd.read_csv(OUT_PERF_RAW).iloc[0].to_dict()
    rc_perf  = pd.read_csv(OUT_PERF_RC).iloc[0].to_dict()
    logger.info("[PERF RAW] " + " | ".join([f"{k}={raw_perf.get(k)}" for k in ["n_years","CAGR","ann_mean","ann_vol","sharpe0","maxDD","cum_end"]]))
    logger.info("[PERF RC ] " + " | ".join([f"{k}={rc_perf.get(k)}"  for k in ["n_years","CAGR","ann_mean","ann_vol","sharpe0","maxDD","cum_end"]]))

    logger.info(f"[DAILY MDD] RAW={mdd_raw_d:.6f} | RC={mdd_rc_d:.6f}")
    logger.info("[DAILY MAX DD EVENT RAW] " + " | ".join([f"{k}={ev_raw.iloc[0][k]}" for k in ["peak_date","trough_date","recovery_date","dd_min"]]))
    logger.info("[DAILY MAX DD EVENT  RC] " + " | ".join([f"{k}={ev_rc.iloc[0][k]}"  for k in ["peak_date","trough_date","recovery_date","dd_min"]]))

if __name__ == "__main__":
    main()


2026-01-25 05:33:56,986 - INFO - ==============================================================================================================
2026-01-25 05:33:56,987 - INFO - 10/1 年次「割安高質」バックテスト + FF5+MOM RiskControl(C) [FAST get_near_price]
2026-01-25 05:33:56,987 - INFO - ==============================================================================================================
2026-01-25 05:33:56,988 - INFO - FF5MOM_FACTOR_PATH: C:\Users\yongr\Project\merged_data_all_stocks\factors\ff5_mom_factors_monthly.parquet
2026-01-25 05:33:56,989 - INFO - OUT_DIR: C:\Users\yongr\Project\merged_data_all_stocks\factors\bt_october_unit_with_ff5mom_riskcontrol
2026-01-25 05:33:56,991 - INFO - Tax mode: simple annual tax only (TAX_RATE=0.20315)
2026-01-25 05:33:56,991 - INFO - RiskControl: regime(3m MKT<0 AND WML<0) + beta(CMA<-0.8 or |MKT|>0.9) => riskoff=0.5  [Option1]
2026-01-25 05:33:56,992 - INFO - Beta estimation window: 12 months (min_obs=10)
2026-01-25 05:33:57,697 - INFO - 財務データ読み込み成

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import re

# ============================================================
# 設定
# ============================================================
BARS_DIR = Path(r"C:\Users\yongr\Project\jquants_daily_bars_10y_parquet\daily_parquet")
OUT_PATH = Path(r"C:\Users\yongr\Project\merged_data_all_stocks\factors\price_month_end.parquet")

START = "2016-01-01"
END   = "2026-01-31"

# AdjustedClose の列名揺れ対策（必要なら追加）
ADJ_CLOSE_CANDIDATES = [
    "AdjustedClose",      # 期待
    "AdjustmentClose",    # 揺れ
    "AdjC",               # 短縮名
    "AdjClose",
    "AdjCl",
]

# SharesOut（barsには無いことも多い）
SHARES_CANDIDATES = ["SharesOut", "ShareOutstanding", "ShOut", "Shares"]

# Code汚染判定に使う禁止値（文字列化後）
BAD_CODE_STRINGS = {"None", "nan", "", "NaN", "NULL", "null"}

date_pat = re.compile(r"date=(\d{4}-\d{2}-\d{2})\.parquet$")


# ============================================================
# util
# ============================================================
def list_files_in_range(folder: Path, start: str, end: str):
    start = pd.Timestamp(start)
    end = pd.Timestamp(end)
    files = []
    for p in folder.glob("date=*.parquet"):
        m = date_pat.search(str(p))
        if not m:
            continue
        d = pd.Timestamp(m.group(1))
        if start <= d <= end:
            files.append((d, p))
    files.sort(key=lambda x: x[0])
    if not files:
        raise FileNotFoundError(f"No parquet files found in range: {folder} [{start}..{end}]")
    return files

def month_end(ts: pd.Series) -> pd.Series:
    return ts.dt.to_period("M").dt.to_timestamp("M")

def pick_first_existing(df: pd.DataFrame, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def fail(msg: str):
    raise RuntimeError(msg)

# ============================================================
# main
# ============================================================
def main():
    print("=" * 120)
    print("[Step1 完全版] bars -> price_month_end.parquet（保存＋検査付き）")
    print("=" * 120)
    print(f"[PATH] BARS_DIR: {BARS_DIR}")
    print(f"[PATH] OUT_PATH: {OUT_PATH}")
    print(f"[RANGE] {START} .. {END}")

    bar_files = list_files_in_range(BARS_DIR, START, END)
    print(f"bars files: {len(bar_files)}  ({bar_files[0][0].date()} .. {bar_files[-1][0].date()})")

    chunks = []
    for i, (d, fp) in enumerate(bar_files, 1):
        df = pd.read_parquet(fp)

        # 必須列
        if "Date" not in df.columns or "Code" not in df.columns:
            fail(f"bars missing Date/Code: {fp}")

        # AdjustedClose相当の列を決定
        adj_col = pick_first_existing(df, ADJ_CLOSE_CANDIDATES)
        if adj_col is None:
            print("\n❌ AdjustedClose相当の列が見つかりません")
            print(f"   file: {fp}")
            print(f"   columns: {list(df.columns)}")
            fail(f"AdjustedClose-like column not found: {fp}")

        # SharesOut相当（任意）
        sh_col = pick_first_existing(df, SHARES_CANDIDATES)

        use_cols = ["Date", "Code", adj_col] + ([sh_col] if sh_col else [])
        df = df[use_cols].copy()

        # 列名統一
        df = df.rename(columns={adj_col: "AdjustedClose"})
        if sh_col:
            df = df.rename(columns={sh_col: "SharesOut"})

        # 型整形
        df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
        df["Code"] = df["Code"].astype(str)

        # Code汚染除外（この段階で落とす）
        df = df[df["Code"].notna() & ~df["Code"].isin(BAD_CODE_STRINGS)].copy()
        df = df[df["Date"].notna()].copy()

        # MonthEnd付与
        df["MonthEnd"] = month_end(df["Date"])

        chunks.append(df)

        if i % 200 == 0 or i == len(bar_files):
            print(f"  loaded {i}/{len(bar_files)}: {d.date()}")

    # 結合
    bars_all = pd.concat(chunks, ignore_index=True)

    # 数値化
    bars_all["AdjustedClose"] = pd.to_numeric(bars_all["AdjustedClose"], errors="coerce")
    bars_all = bars_all[bars_all["AdjustedClose"].notna()].copy()

    # 月末（その月の最終営業日の行）を採用
    bars_all = bars_all.sort_values(["Code", "MonthEnd", "Date"], kind="mergesort")
    price_month_end = bars_all.groupby(["Code", "MonthEnd"], as_index=False).tail(1)

    # MarketCap（SharesOutがあれば）
    if "SharesOut" in price_month_end.columns:
        price_month_end["SharesOut"] = pd.to_numeric(price_month_end["SharesOut"], errors="coerce")
        price_month_end["MarketCap"] = price_month_end["AdjustedClose"] * price_month_end["SharesOut"]
    else:
        price_month_end["MarketCap"] = np.nan

    # ============================================================
    # 保存前検査（ここで落とす）
    # ============================================================
    print("\n" + "=" * 120)
    print("[検査] 保存前チェック")
    print("=" * 120)

    # (A) Code汚染チェック
    bad_code_mask = price_month_end["Code"].isna() | price_month_end["Code"].isin(BAD_CODE_STRINGS)
    bad_code_rows = int(bad_code_mask.sum())
    print(f"[CHECK-A] bad Code rows: {bad_code_rows:,}")
    if bad_code_rows > 0:
        print("⚠️ bad Code sample (top20):")
        print(price_month_end.loc[bad_code_mask].head(20))
        fail("Code汚染（None/nan/空など）を検出したため停止します。")

    # (B) (Code, MonthEnd) 一意性
    dup = int(price_month_end.duplicated(["Code", "MonthEnd"]).sum())
    print(f"[CHECK-B] dup(Code,MonthEnd): {dup:,}")
    if dup != 0:
        # 参考情報を出す
        print("⚠️ duplicate sample (top50):")
        print(price_month_end[price_month_end.duplicated(["Code","MonthEnd"], keep=False)]
              .sort_values(["Code","MonthEnd","Date"])
              .head(50))
        fail("(Code,MonthEnd) が一意ではないため停止します。")

    # (C) 基本統計
    print("\n[SUMMARY]")
    print(f"rows  : {len(price_month_end):,}")
    print(f"codes : {price_month_end['Code'].nunique():,}")
    print(f"months: {price_month_end['MonthEnd'].nunique():,}")
    print(f"Date.max   : {price_month_end['Date'].max()}")
    print(f"MonthEnd.max: {price_month_end['MonthEnd'].max()}")

    # ============================================================
    # 保存
    # ============================================================
    OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    price_month_end.to_parquet(OUT_PATH, engine="pyarrow", index=False)

    print("\n" + "=" * 120)
    print("✅ saved price_month_end.parquet")
    print("=" * 120)
    print(f"path: {OUT_PATH}")
    print("columns:", list(price_month_end.columns))

if __name__ == "__main__":
    main()


[Step1 完全版] bars -> price_month_end.parquet（保存＋検査付き）
[PATH] BARS_DIR: C:\Users\yongr\Project\jquants_daily_bars_10y_parquet\daily_parquet
[PATH] OUT_PATH: C:\Users\yongr\Project\merged_data_all_stocks\factors\price_month_end.parquet
[RANGE] 2016-01-01 .. 2026-01-31
bars files: 2449  (2016-01-15 .. 2026-01-22)
  loaded 200/2449: 2016-11-07
  loaded 400/2449: 2017-08-29
  loaded 600/2449: 2018-06-22
  loaded 800/2449: 2019-04-17
  loaded 1000/2449: 2020-02-18
  loaded 1200/2449: 2020-12-11
  loaded 1400/2449: 2021-10-07
  loaded 1600/2449: 2022-08-02
  loaded 1800/2449: 2023-05-29
  loaded 2000/2449: 2024-03-21
  loaded 2200/2449: 2025-01-15
  loaded 2400/2449: 2025-11-07
  loaded 2449/2449: 2026-01-22

[検査] 保存前チェック
[CHECK-A] bad Code rows: 0
[CHECK-B] dup(Code,MonthEnd): 0

[SUMMARY]
rows  : 491,156
codes : 5,300
months: 121
Date.max   : 2026-01-22 00:00:00
MonthEnd.max: 2026-01-31 00:00:00

✅ saved price_month_end.parquet
path: C:\Users\yongr\Project\merged_data_all_stocks\factors\pric

In [54]:
import re
from pathlib import Path
import pandas as pd
import numpy as np

# 既存のfin（あなたの手元にあるディレクトリ）
FINS_SRC_DIR  = Path(r"C:\Users\yongr\Project\jquants_fins_summary_10y_parquet\daily_parquet")

# 正規化したfinの出力先（新規作成）
FINS_NORM_DIR = Path(r"C:\Users\yongr\Project\jquants_fins_summary_10y_parquet\daily_parquet_norm")

BAD_CODE_STRINGS = {"None", "nan", "", "NaN", "NULL", "null"}
date_pat = re.compile(r"date=(\d{4}-\d{2}-\d{2})\.parquet$")

# 財務諸表系を優先（必要なら追加）
DOCTYPE_PRIORITY = [
    "FYFinancialStatements_Consolidated_JP",
    "FYFinancialStatements_NonConsolidated_JP",
    "FYFinancialStatements_Consolidated_REIT",
    "FYFinancialStatements_NonConsolidated_REIT",
    # ここに IFRS/USGAAP 系を追加したければ追加
]

def hr(ch="=", n=110):
    print(ch * n)

def list_fin_files(folder: Path):
    files = []
    for p in folder.glob("date=*.parquet"):
        m = date_pat.search(str(p))
        if not m:
            continue
        d = pd.Timestamp(m.group(1))
        files.append((d, p))
    files.sort(key=lambda x: x[0])
    return files

def force_dt64ns(x):
    dt = pd.to_datetime(x, errors="coerce")
    if hasattr(dt, "astype"):
        return dt.astype("datetime64[ns]")
    if pd.isna(dt):
        return np.datetime64("NaT", "ns")
    return pd.Timestamp(dt).to_datetime64()

def normalize_fins_daily(raw: pd.DataFrame, snapshot_date: pd.Timestamp) -> pd.DataFrame:
    """
    /v2/fins/summary を日付指定で取得したraw（開示イベント一覧）を
    (Code, DiscDate)で一意にするため正規化する。
    - 財務諸表系 DocType を優先
    - 同一Code×同一DiscDateは DocType優先→DiscNoの最後 を採用
    """
    if raw is None or len(raw) == 0:
        return pd.DataFrame()

    if "Code" not in raw.columns:
        return pd.DataFrame()

    df = raw.copy()
    df["Code"] = df["Code"].astype(str)
    df = df[df["Code"].notna() & ~df["Code"].isin(BAD_CODE_STRINGS)].copy()

    # DiscDate が無い場合はファイル日付で埋める（保険）
    if "DiscDate" in df.columns:
        df["DiscDate"] = pd.to_datetime(df["DiscDate"], errors="coerce")
    else:
        df["DiscDate"] = pd.Timestamp(snapshot_date)

    # snapshot_date（asof用）
    df["snapshot_date"] = pd.Timestamp(snapshot_date).to_datetime64()
    df["snapshot_date"] = force_dt64ns(df["snapshot_date"])

    # DocType優先順位
    if "DocType" in df.columns:
        prio = {dt: i for i, dt in enumerate(DOCTYPE_PRIORITY)}
        df["doctype_rank"] = df["DocType"].map(prio).fillna(9999).astype(int)
    else:
        df["doctype_rank"] = 9999

    # 同日内の最新を優先するため DiscNo を併用
    sort_keys = ["Code", "DiscDate", "doctype_rank"]
    if "DiscNo" in df.columns:
        sort_keys.append("DiscNo")

    df = df.sort_values(sort_keys, kind="mergesort")
    df = df.drop_duplicates(["Code", "DiscDate"], keep="last")

    # 必要列に絞る（必要なら追加）
    keep = [c for c in [
        "snapshot_date", "DiscDate", "DiscTime", "Code", "DiscNo", "DocType",
        "TA", "Eq", "NP", "ShOutFY", "TrShFY", "AvgSh", "BPS", "EPS"
    ] if c in df.columns]

    out = df[keep].copy()

    # 数値化（あるものだけ）
    for c in ["TA", "Eq", "NP", "ShOutFY", "TrShFY", "AvgSh", "BPS", "EPS"]:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce")

    return out

def main():
    hr()
    print("初回: 既存ローカル fins（日次parquet）→ 正規化 fins（daily_parquet_norm）を一括生成")
    hr()
    print(f"SRC : {FINS_SRC_DIR}")
    print(f"NORM: {FINS_NORM_DIR}")
    FINS_NORM_DIR.mkdir(parents=True, exist_ok=True)

    files = list_fin_files(FINS_SRC_DIR)
    if not files:
        raise RuntimeError("FINS_SRC_DIR に date=*.parquet が見つかりません")

    print(f"files: {len(files)}  range: {files[0][0].date()} .. {files[-1][0].date()}")

    total_raw = 0
    total_norm = 0

    for i, (d, fp) in enumerate(files, 1):
        raw = pd.read_parquet(fp)
        total_raw += len(raw)

        norm = normalize_fins_daily(raw, d)
        total_norm += len(norm)

        out_fp = FINS_NORM_DIR / fp.name
        norm.to_parquet(out_fp, engine="pyarrow", index=False)

        if i % 250 == 0 or i == len(files):
            print(f"  {i}/{len(files)} saved: {out_fp.name}  norm_rows={len(norm):,}")

    hr("-", 110)
    print(f"✅ done. total_raw_rows={total_raw:,}  total_norm_rows={total_norm:,}")
    print(f"✅ output: {FINS_NORM_DIR}")

if __name__ == "__main__":
    main()


初回: 既存ローカル fins（日次parquet）→ 正規化 fins（daily_parquet_norm）を一括生成
SRC : C:\Users\yongr\Project\jquants_fins_summary_10y_parquet\daily_parquet
NORM: C:\Users\yongr\Project\jquants_fins_summary_10y_parquet\daily_parquet_norm
files: 2437  range: 2016-01-15 .. 2026-01-09
  250/2437 saved: date=2017-01-23.parquet  norm_rows=14
  500/2437 saved: date=2018-01-26.parquet  norm_rows=84
  750/2437 saved: date=2019-02-05.parquet  norm_rows=172
  1000/2437 saved: date=2020-02-20.parquet  norm_rows=7
  1250/2437 saved: date=2021-03-02.parquet  norm_rows=5
  1500/2437 saved: date=2022-03-09.parquet  norm_rows=19
  1750/2437 saved: date=2023-03-17.parquet  norm_rows=58
  2000/2437 saved: date=2024-03-26.parquet  norm_rows=20
  2250/2437 saved: date=2025-04-03.parquet  norm_rows=18
  2437/2437 saved: date=2026-01-09.parquet  norm_rows=77
--------------------------------------------------------------------------------------------------------------
✅ done. total_raw_rows=190,872  total_norm_rows=180,257
✅ o

In [57]:
import os
import re
import time
from pathlib import Path
from datetime import datetime, timedelta
from typing import Optional, Dict, List, Tuple

import pandas as pd
import numpy as np
import requests

# ============================================================
# パス（あなたの環境）
# ============================================================
BARS_DIR = Path(r"C:\Users\yongr\Project\jquants_daily_bars_10y_parquet\daily_parquet")

# ★重要：正規化済み財務を使う
FINS_DIR = Path(r"C:\Users\yongr\Project\jquants_fins_summary_10y_parquet\daily_parquet_norm")

FACTORS_DIR = Path(r"C:\Users\yongr\Project\merged_data_all_stocks\factors")
PRICE_MONTH_END_PATH = FACTORS_DIR / "price_month_end.parquet"
SNAPSHOT_PATH = FACTORS_DIR / "month_end_snapshot.parquet"

DIAG_DIR = FACTORS_DIR / "diag_update_norm"
DIAG_DIR.mkdir(parents=True, exist_ok=True)

DIAG_RIGHT_SUMMARY = DIAG_DIR / "diag_right_summary.csv"
DIAG_SNAP_COVERAGE = DIAG_DIR / "diag_snapshot_coverage_by_month.csv"
DIAG_SNAP_DUPES    = DIAG_DIR / "diag_snapshot_duplicates.csv"
DIAG_SNAP_SAMPLE   = DIAG_DIR / "diag_snapshot_latest_month_top200.csv"

# ============================================================
# 更新パラメータ
# ============================================================
RECHECK_DAYS = 7              # bars差分の巻き戻し
FIN_REFRESH_MONTHS = 12       # snapshot更新対象月（直近Nヶ月）

# ============================================================
# API（bars差分取得のみ使用）
# ============================================================
JQUANTS_BASE = "https://api.jquants.com/v2"
PLAN_DELAY_SEC = 0.5
TIMEOUT_SEC = 60

BAD_CODE_STRINGS = {"None", "nan", "", "NaN", "NULL", "null"}
date_pat = re.compile(r"date=(\d{4}-\d{2}-\d{2})\.parquet$")


# ============================================================
# util
# ============================================================
def hr(ch="=", n=120):
    print(ch * n)

def month_end(ts: pd.Series) -> pd.Series:
    return ts.dt.to_period("M").dt.to_timestamp("M")

def force_dt64ns(x):
    dt = pd.to_datetime(x, errors="coerce")
    if hasattr(dt, "astype"):
        return dt.astype("datetime64[ns]")
    if pd.isna(dt):
        return np.datetime64("NaT", "ns")
    return pd.Timestamp(dt).to_datetime64()

def list_local_dates(folder: Path) -> List[pd.Timestamp]:
    dates = []
    for p in folder.glob("date=*.parquet"):
        m = date_pat.search(str(p))
        if not m:
            continue
        dates.append(pd.Timestamp(m.group(1)))
    return sorted(dates)

def latest_local_date(folder: Path) -> Optional[pd.Timestamp]:
    ds = list_local_dates(folder)
    return ds[-1] if ds else None

def daterange(start: pd.Timestamp, end: pd.Timestamp):
    d = start
    while d <= end:
        yield d
        d += timedelta(days=1)

def safe_num(s):
    return pd.to_numeric(s, errors="coerce")


# ============================================================
# J-Quants API client (bars only)
# ============================================================
class JQuantsAPIV2:
    def __init__(self, api_key: Optional[str] = None, delay_sec: float = PLAN_DELAY_SEC, timeout_sec: int = TIMEOUT_SEC):
        self.api_key = api_key or os.getenv("JQUANTS_API_KEY")
        if not self.api_key:
            raise ValueError("環境変数 JQUANTS_API_KEY がありません。")
        self.session = requests.Session()
        self.delay_sec = delay_sec
        self.timeout_sec = timeout_sec

    def _headers(self) -> Dict[str, str]:
        return {"x-api-key": self.api_key}

    def _get(self, url: str, params: Dict, max_retries: int = 3) -> Dict:
        for attempt in range(max_retries):
            r = self.session.get(url, headers=self._headers(), params=params, timeout=self.timeout_sec)
            if r.status_code == 429:
                wait = self.delay_sec * (2 ** attempt)
                print(f"  ⚠️ rate limit: wait {wait:.1f}s")
                time.sleep(wait)
                continue
            r.raise_for_status()
            time.sleep(self.delay_sec)
            return r.json()
        raise RuntimeError(f"Max retries exceeded: {url}")

    def get_daily_bars(self, date: str) -> List[Dict]:
        url = f"{JQUANTS_BASE}/equities/bars/daily"
        js = self._get(url, {"date": date})
        return js.get("data", [])


def infer_latest_trading_day_by_bars(api: JQuantsAPIV2, base_day: pd.Timestamp, lookback_days: int = 30) -> pd.Timestamp:
    hr("-", 120)
    print("[A-0] 最新営業日推定（No-Calendar：barsが取れる直近日を探索）")
    for k in range(lookback_days + 1):
        d = base_day - timedelta(days=k)
        ds = d.strftime("%Y-%m-%d")
        try:
            data = api.get_daily_bars(ds)
            if data:
                print(f"✅ latest_trading_day inferred: {ds}")
                return d
        except Exception:
            continue
    print(f"⚠️ 推定失敗。base_day={base_day.date()} を返します")
    return base_day


# ============================================================
# Step A: bars差分取得（No-Calendar）
# ============================================================
def update_bars_no_calendar(api: JQuantsAPIV2, bars_dir: Path, end_hint: pd.Timestamp, recheck_days: int = 7):
    last_local = latest_local_date(bars_dir)
    if last_local is None:
        raise RuntimeError("bars_dir にローカルファイルがありません。初期構築が必要です。")

    start_scan = last_local - timedelta(days=recheck_days)
    if start_scan < pd.Timestamp("2000-01-01"):
        start_scan = pd.Timestamp("2000-01-01")

    hr("-", 120)
    print("[A] bars差分取得（No-Calendar：barsが空なら非営業日扱い）")
    print(f"local_last_date: {last_local.date()}")
    print(f"scan window    : {start_scan.date()} -> {end_hint.date()}  (recheck={recheck_days}d)")

    new_saved = 0
    new_empty = 0

    bars_dir.mkdir(parents=True, exist_ok=True)

    for d in daterange(start_scan, end_hint):
        ds = d.strftime("%Y-%m-%d")
        fp = bars_dir / f"date={ds}.parquet"
        if fp.exists():
            continue

        data = api.get_daily_bars(ds)
        if not data:
            new_empty += 1
            continue

        df = pd.DataFrame(data)
        if "Date" not in df.columns or "Code" not in df.columns:
            continue

        df.to_parquet(fp, engine="pyarrow", index=False)
        new_saved += 1

    print(f"\n✅ bars差分完了: new_saved={new_saved}, empty_days={new_empty}, last_saved={latest_local_date(bars_dir).date()}")


# ============================================================
# Step B: price_month_end 直近月だけ更新
# ============================================================
def build_price_month_end_for_months(bars_dir: Path, target_month_ends: List[pd.Timestamp]) -> pd.DataFrame:
    all_dates = list_local_dates(bars_dir)
    if not all_dates:
        raise RuntimeError("bars_dir is empty")

    tmes = [pd.Timestamp(x).to_period("M") for x in target_month_ends]
    tmes_set = set(tmes)

    dfs = []
    for d in all_dates:
        if d.to_period("M") not in tmes_set:
            continue
        fp = bars_dir / f"date={d.strftime('%Y-%m-%d')}.parquet"
        df = pd.read_parquet(fp)

        adj_col = None
        for c in ["AdjustedClose", "AdjustmentClose", "AdjC", "AdjClose", "AdjCl"]:
            if c in df.columns:
                adj_col = c
                break
        if adj_col is None:
            continue

        df = df[["Date", "Code", adj_col]].copy()
        df = df.rename(columns={adj_col: "AdjustedClose"})
        df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
        df["Code"] = df["Code"].astype(str)
        df = df[df["Code"].notna() & ~df["Code"].isin(BAD_CODE_STRINGS)].copy()
        df = df[df["Date"].notna()].copy()
        df["MonthEnd"] = month_end(df["Date"])
        df["AdjustedClose"] = safe_num(df["AdjustedClose"])
        df = df[df["AdjustedClose"].notna()].copy()

        dfs.append(df)

    if not dfs:
        return pd.DataFrame(columns=["Date", "Code", "AdjustedClose", "MonthEnd", "MarketCap"])

    bars_m = pd.concat(dfs, ignore_index=True)
    bars_m = bars_m.sort_values(["Code", "MonthEnd", "Date"], kind="mergesort")
    pm = bars_m.groupby(["Code", "MonthEnd"], as_index=False).tail(1)

    # MarketCapは後でfinの株数で作るのでNaNのまま
    pm["MarketCap"] = np.nan

    if (pm["Code"].isna() | pm["Code"].isin(BAD_CODE_STRINGS)).any():
        raise RuntimeError("price_month_end rebuild produced bad Code rows")
    if pm.duplicated(["Code", "MonthEnd"]).any():
        raise RuntimeError("price_month_end rebuild produced duplicate (Code,MonthEnd)")

    return pm[["Date", "Code", "AdjustedClose", "MonthEnd", "MarketCap"]].copy()


def update_price_month_end_incremental(bars_dir: Path, price_path: Path, months_back: int = 2):
    hr("-", 120)
    print("[B] price_month_end 差分更新（直近月のみ再計算）")

    if not price_path.exists():
        raise RuntimeError("price_month_end.parquet がありません。Step1を先に実行してください。")

    price = pd.read_parquet(price_path)
    price["MonthEnd"] = force_dt64ns(price["MonthEnd"])
    latest_me = pd.Timestamp(price["MonthEnd"].max())

    target_mes = [pd.Timestamp((latest_me - pd.offsets.MonthEnd(k)).to_pydatetime()) for k in range(months_back)]
    print(f"target MonthEnd(s): {[d.strftime('%Y-%m-%d') for d in target_mes]}")

    rebuilt = build_price_month_end_for_months(bars_dir, target_mes)
    print(f"rebuilt rows: {len(rebuilt):,} codes: {rebuilt['Code'].nunique():,}")

    price = price[~price["MonthEnd"].isin(target_mes)].copy()
    merged = pd.concat([price, rebuilt], ignore_index=True)
    merged = merged.sort_values(["Code", "MonthEnd"], kind="mergesort")
    if merged.duplicated(["Code", "MonthEnd"]).any():
        raise RuntimeError("price_month_end incremental update created duplicates")

    price_path.parent.mkdir(parents=True, exist_ok=True)
    merged.to_parquet(price_path, engine="pyarrow", index=False)
    print(f"✅ updated: {price_path} rows={len(merged):,} latest={merged['MonthEnd'].max()}")


# ============================================================
# Step C: month_end_snapshot（norm財務を使って直近Nヶ月だけ更新）
# ============================================================

# ===== PATCH START: load_fin_norm_recent 差し替え（右側を厳密一意化 + end_need対応） =====
def load_fin_norm_recent(fins_dir: Path, start_need: pd.Timestamp, end_need: Optional[pd.Timestamp] = None) -> pd.DataFrame:
    """
    daily_parquet_norm から必要期間のファイルだけ読み込み、right(fin) を作る。
    右側は (Code, snapshot_date) を一意化して merge_asof の前提を守る。
    """
    fin_dates = list_local_dates(fins_dir)
    if not fin_dates:
        raise RuntimeError("fins_dir (norm) is empty")

    if end_need is None:
        end_need = fin_dates[-1]

    fin_dates = [d for d in fin_dates if (d >= start_need) and (d <= end_need)]

    parts = []
    for d in fin_dates:
        fp = fins_dir / f"date={d.strftime('%Y-%m-%d')}.parquet"
        if not fp.exists():
            continue

        fin = pd.read_parquet(fp)
        if fin.empty or ("Code" not in fin.columns):
            continue

        fin = fin.copy()
        fin["Code"] = fin["Code"].astype(str)
        fin = fin[fin["Code"].notna() & ~fin["Code"].isin(BAD_CODE_STRINGS)].copy()

        # snapshot_date（asofキー）: ★スカラー代入で確実にns
        snap_dt64 = pd.Timestamp(d).to_datetime64()   # datetime64[ns] scalar
        fin["snapshot_date"] = snap_dt64
        fin["snapshot_date"] = force_dt64ns(fin["snapshot_date"])

        # 列統一：Eq/TA/NP/ShOutFY/AvgSh を numeric に
        for c in ["Eq", "TA", "NP", "ShOutFY", "AvgSh"]:
            if c in fin.columns:
                fin[c] = safe_num(fin[c])
            else:
                fin[c] = np.nan

        # SharesOut_raw 統一：ShOutFY優先、なければAvgSh
        fin["SharesOut_raw"] = fin["ShOutFY"]
        fin.loc[fin["SharesOut_raw"].isna(), "SharesOut_raw"] = fin["AvgSh"]

        keep = ["Code", "snapshot_date", "Eq", "TA", "NP", "SharesOut_raw"]
        fin = fin[keep].copy()

        parts.append(fin)

    if not parts:
        right = pd.DataFrame(columns=["Code","snapshot_date","Eq","TA","NP","SharesOut_raw"])
        # right summaryも空で出しておく
        pd.DataFrame([{
            "start_need": start_need.strftime("%Y-%m-%d"),
            "end_need": end_need.strftime("%Y-%m-%d"),
            "right_rows": 0,
            "right_codes": 0,
            "right_min_date": "",
            "right_max_date": "",
            "dedup_dropped_rows": 0,
            "cov_eq_pct": 0.0,
            "cov_ta_pct": 0.0,
            "cov_np_pct": 0.0,
            "cov_shares_pct": 0.0,
        }]).to_csv(DIAG_RIGHT_SUMMARY, index=False, encoding="utf-8-sig")
        return right

    right = pd.concat(parts, ignore_index=True)
    right["snapshot_date"] = force_dt64ns(right["snapshot_date"])
    right = right.sort_values(["Code","snapshot_date"], kind="mergesort")

    # ★ここが重要：right を (Code,snapshot_date) で厳密に一意化
    before = len(right)
    right = right.drop_duplicates(["Code","snapshot_date"], keep="last")
    dropped = before - len(right)

    summary = pd.DataFrame([{
        "start_need": start_need.strftime("%Y-%m-%d"),
        "end_need": end_need.strftime("%Y-%m-%d"),
        "right_rows": len(right),
        "right_codes": int(right["Code"].nunique()),
        "right_min_date": str(pd.Timestamp(right["snapshot_date"].min()).date()) if len(right) else "",
        "right_max_date": str(pd.Timestamp(right["snapshot_date"].max()).date()) if len(right) else "",
        "dedup_dropped_rows": int(dropped),
        "cov_eq_pct": float(right["Eq"].notna().mean()*100),
        "cov_ta_pct": float(right["TA"].notna().mean()*100),
        "cov_np_pct": float(right["NP"].notna().mean()*100),
        "cov_shares_pct": float(right["SharesOut_raw"].notna().mean()*100),
    }])
    summary.to_csv(DIAG_RIGHT_SUMMARY, index=False, encoding="utf-8-sig")

    print(f"[DEDUP right] dropped {dropped:,} rows by (Code,snapshot_date)")
    return right
# ===== PATCH END =====


# ===== PATCH START: build_snapshot_for_recent_months 差し替え（merge_asofを一発で実行） =====
def build_snapshot_for_recent_months(price_path: Path, fins_dir: Path, months_back: int) -> pd.DataFrame:
    price = pd.read_parquet(price_path)

    # 型統一・最低限のクリーニング
    price = price.copy()
    price["Date"] = force_dt64ns(price["Date"])
    price["MonthEnd"] = force_dt64ns(price["MonthEnd"])
    price["Code"] = price["Code"].astype(str)
    price["AdjustedClose"] = safe_num(price["AdjustedClose"])
    price = price[price["AdjustedClose"].notna() & price["Date"].notna() & price["MonthEnd"].notna()].copy()
    price = price[price["Code"].notna() & ~price["Code"].isin(BAD_CODE_STRINGS)].copy()

    latest_me = pd.Timestamp(price["MonthEnd"].max())
    target_mes = [pd.Timestamp((latest_me - pd.offsets.MonthEnd(k)).to_pydatetime()) for k in range(months_back)]
    target_price = price[price["MonthEnd"].isin(target_mes)].copy()

    # ★念のため：price 側が (Code,MonthEnd) 一意か確認（壊れてたらここで止める）
    dup_price = int(target_price.duplicated(["Code","MonthEnd"]).sum())
    if dup_price > 0:
        target_price[target_price.duplicated(["Code","MonthEnd"], keep=False)] \
            .sort_values(["Code","MonthEnd","Date"]) \
            .head(2000) \
            .to_csv(DIAG_DIR / "diag_price_target_duplicates.csv", index=False, encoding="utf-8-sig")
        raise RuntimeError(f"target_price has duplicates (Code,MonthEnd)={dup_price:,} (see diag_price_target_duplicates.csv)")

    # fin読み込み範囲：asofが当たるように十分前から読む（ここが浅いと coverage が死ぬ）
    start_need = (min(target_mes) - pd.Timedelta(days=370))
    end_need = latest_local_date(fins_dir) or pd.Timestamp(datetime.now().date())
    right = load_fin_norm_recent(fins_dir, start_need, end_need=end_need)

    left = target_price[["Code","MonthEnd","Date","AdjustedClose"]].copy()
    left = left.sort_values(["Code","Date"], kind="mergesort").reset_index(drop=True)
    right = right.sort_values(["Code","snapshot_date"], kind="mergesort").reset_index(drop=True)

    # ★merge_asof を一発で（by="Code"）
    asof_df = pd.merge_asof(
        left,
        right,
        left_on="Date",
        right_on="snapshot_date",
        by="Code",
        direction="backward",
        allow_exact_matches=True,
    )

    # MarketCap / BM_Ratio
    asof_df["MarketCap"] = np.where(
        asof_df["SharesOut_raw"].notna(),
        asof_df["AdjustedClose"] * asof_df["SharesOut_raw"],
        np.nan
    )
    asof_df["BM_Ratio"] = np.where(
        asof_df["Eq"].notna() & asof_df["MarketCap"].notna() & (asof_df["MarketCap"] > 0),
        asof_df["Eq"] / asof_df["MarketCap"],
        np.nan
    )

    # ROE/INV_Growth は現段階では未生成（必要なら次に計算実装）
    asof_df["ROE"] = np.nan
    asof_df["INV_Growth"] = np.nan

    snap = asof_df[["Code","MonthEnd","Date","AdjustedClose","MarketCap","BM_Ratio","ROE","INV_Growth"]].copy()

    # ★重複チェック：原因追跡用に asof_df の情報も付けて保存
    dup_mask = snap.duplicated(["Code","MonthEnd"], keep=False)
    dup_cnt = int(dup_mask.sum())
    if dup_cnt > 0:
        diag = asof_df.loc[dup_mask].copy()
        keep_diag = ["Code","MonthEnd","Date","snapshot_date","AdjustedClose","SharesOut_raw","Eq","MarketCap","BM_Ratio"]
        keep_diag = [c for c in keep_diag if c in diag.columns]
        diag = diag[keep_diag].sort_values(["Code","MonthEnd","Date"], kind="mergesort")
        diag.head(5000).to_csv(DIAG_SNAP_DUPES, index=False, encoding="utf-8-sig")
        raise RuntimeError(f"snapshot part has duplicates (Code,MonthEnd)={dup_cnt:,}. see: {DIAG_SNAP_DUPES}")

    return snap
# ===== PATCH END =====


def update_month_end_snapshot_incremental(price_path: Path, fins_dir: Path, snap_path: Path, fin_refresh_months: int):
    hr("-", 120)
    print("[C] month_end_snapshot 差分更新（norm財務で直近Nヶ月だけasof再付与）")
    print(f"refresh months: {fin_refresh_months}")
    print(f"FINS_DIR(norm): {fins_dir}")

    new_part = build_snapshot_for_recent_months(price_path, fins_dir, fin_refresh_months)
    print(f"rebuilt snapshot part rows={len(new_part):,} months={new_part['MonthEnd'].nunique():,} codes={new_part['Code'].nunique():,}")

    if snap_path.exists():
        old = pd.read_parquet(snap_path)
        old["MonthEnd"] = force_dt64ns(old["MonthEnd"])
        target_mes = new_part["MonthEnd"].unique()
        old = old[~old["MonthEnd"].isin(target_mes)].copy()
        merged = pd.concat([old, new_part], ignore_index=True)
    else:
        merged = new_part

    merged = merged.sort_values(["Code","MonthEnd"], kind="mergesort")

    # 最終重複チェック
    if merged.duplicated(["Code","MonthEnd"]).any():
        dup = merged[merged.duplicated(["Code","MonthEnd"], keep=False)].sort_values(["Code","MonthEnd","Date"]).head(1000)
        dup.to_csv(DIAG_SNAP_DUPES, index=False, encoding="utf-8-sig")
        raise RuntimeError(f"final snapshot has duplicates. see: {DIAG_SNAP_DUPES}")

    # coverage by month
    cov = (merged.groupby("MonthEnd")
           .agg(n=("Code","size"),
                mcap_notna=("MarketCap", lambda s: float(s.notna().mean())),
                bm_notna=("BM_Ratio", lambda s: float(s.notna().mean())),
                roe_notna=("ROE", lambda s: float(s.notna().mean())),
                inv_notna=("INV_Growth", lambda s: float(s.notna().mean())))
           .reset_index())
    for c in ["mcap_notna","bm_notna","roe_notna","inv_notna"]:
        cov[c] = (cov[c] * 100).round(2)
    cov.to_csv(DIAG_SNAP_COVERAGE, index=False, encoding="utf-8-sig")

    # latest month sample
    latest_me = merged["MonthEnd"].max()
    sample = merged[merged["MonthEnd"] == latest_me].copy().sort_values("MarketCap", ascending=False)
    sample.head(200).to_csv(DIAG_SNAP_SAMPLE, index=False, encoding="utf-8-sig")

    # save
    snap_path.parent.mkdir(parents=True, exist_ok=True)
    merged.to_parquet(snap_path, engine="pyarrow", index=False)

    cov_mcap = merged["MarketCap"].notna().mean() * 100
    cov_bm   = merged["BM_Ratio"].notna().mean() * 100
    print(f"✅ updated: {snap_path} rows={len(merged):,}")
    print(f"coverage(%): MarketCap={cov_mcap:.2f}  BM_Ratio={cov_bm:.2f}  (ROE/INV are NaN in this version)")
    print(f"diag saved: {DIAG_RIGHT_SUMMARY}, {DIAG_SNAP_COVERAGE}, {DIAG_SNAP_SAMPLE}")


# ============================================================
# main
# ============================================================
def main():
    hr()
    print("差分取得（API v2 / No-Calendar）→ スナップ差分更新（norm財務）")
    hr()

    FACTORS_DIR.mkdir(parents=True, exist_ok=True)

    api = JQuantsAPIV2()
    today = pd.Timestamp(datetime.now().date())

    # 1) 最新営業日推定（No-Calendar）
    latest_trading_day = infer_latest_trading_day_by_bars(api, today, lookback_days=30)

    # 2) bars差分取得→ローカル追記
    update_bars_no_calendar(api, BARS_DIR, latest_trading_day, recheck_days=RECHECK_DAYS)

    # 3) price_month_end を直近2ヶ月だけ更新
    update_price_month_end_incremental(BARS_DIR, PRICE_MONTH_END_PATH, months_back=2)

    # 4) month_end_snapshot を直近12ヶ月だけ更新（norm fin asof）
    update_month_end_snapshot_incremental(PRICE_MONTH_END_PATH, FINS_DIR, SNAPSHOT_PATH, fin_refresh_months=FIN_REFRESH_MONTHS)

    hr()
    print("✅ 完了")
    hr()
    print(f"- bars_dir last: {latest_local_date(BARS_DIR)}")
    print(f"- fins_norm last: {latest_local_date(FINS_DIR)}")
    print(f"- price_month_end: {PRICE_MONTH_END_PATH}")
    print(f"- month_end_snapshot: {SNAPSHOT_PATH}")
    print(f"- diag: {DIAG_DIR}")

if __name__ == "__main__":
    main()


差分取得（API v2 / No-Calendar）→ スナップ差分更新（norm財務）
------------------------------------------------------------------------------------------------------------------------
[A-0] 最新営業日推定（No-Calendar：barsが取れる直近日を探索）
✅ latest_trading_day inferred: 2026-01-22
------------------------------------------------------------------------------------------------------------------------
[A] bars差分取得（No-Calendar：barsが空なら非営業日扱い）
local_last_date: 2026-01-22
scan window    : 2026-01-15 -> 2026-01-22  (recheck=7d)

✅ bars差分完了: new_saved=0, empty_days=2, last_saved=2026-01-22
------------------------------------------------------------------------------------------------------------------------
[B] price_month_end 差分更新（直近月のみ再計算）
target MonthEnd(s): ['2026-01-31', '2025-12-31']
rebuilt rows: 8,581 codes: 4,306
✅ updated: C:\Users\yongr\Project\merged_data_all_stocks\factors\price_month_end.parquet rows=491,156 latest=2026-01-31 00:00:00
---------------------------------------------------------------------------

ValueError: left keys must be sorted

In [58]:
import os
import re
import time
from pathlib import Path
from datetime import datetime, timedelta
from typing import Optional, Dict, List, Tuple

import pandas as pd
import numpy as np
import requests

# ============================================================
# パス（あなたの環境）
# ============================================================
BARS_DIR = Path(r"C:\Users\yongr\Project\jquants_daily_bars_10y_parquet\daily_parquet")

# ★重要：正規化済み財務を使う
FINS_DIR = Path(r"C:\Users\yongr\Project\jquants_fins_summary_10y_parquet\daily_parquet_norm")

FACTORS_DIR = Path(r"C:\Users\yongr\Project\merged_data_all_stocks\factors")
PRICE_MONTH_END_PATH = FACTORS_DIR / "price_month_end.parquet"
SNAPSHOT_PATH = FACTORS_DIR / "month_end_snapshot.parquet"

DIAG_DIR = FACTORS_DIR / "diag_update_norm"
DIAG_DIR.mkdir(parents=True, exist_ok=True)

DIAG_RIGHT_SUMMARY = DIAG_DIR / "diag_right_summary.csv"
DIAG_SNAP_COVERAGE = DIAG_DIR / "diag_snapshot_coverage_by_month.csv"
DIAG_SNAP_DUPES    = DIAG_DIR / "diag_snapshot_duplicates.csv"
DIAG_SNAP_SAMPLE   = DIAG_DIR / "diag_snapshot_latest_month_top200.csv"

# ============================================================
# 更新パラメータ
# ============================================================
RECHECK_DAYS = 7              # bars差分の巻き戻し
FIN_REFRESH_MONTHS = 12       # snapshot更新対象月（直近Nヶ月）

# ============================================================
# API（bars差分取得のみ使用）
# ============================================================
JQUANTS_BASE = "https://api.jquants.com/v2"
PLAN_DELAY_SEC = 0.5
TIMEOUT_SEC = 60

BAD_CODE_STRINGS = {"None", "nan", "", "NaN", "NULL", "null"}
date_pat = re.compile(r"date=(\d{4}-\d{2}-\d{2})\.parquet$")


# ============================================================
# util
# ============================================================
def hr(ch="=", n=120):
    print(ch * n)

def month_end(ts: pd.Series) -> pd.Series:
    return ts.dt.to_period("M").dt.to_timestamp("M")

def force_dt64ns(x):
    dt = pd.to_datetime(x, errors="coerce")
    if hasattr(dt, "astype"):
        return dt.astype("datetime64[ns]")
    if pd.isna(dt):
        return np.datetime64("NaT", "ns")
    return pd.Timestamp(dt).to_datetime64()

def list_local_dates(folder: Path) -> List[pd.Timestamp]:
    dates = []
    for p in folder.glob("date=*.parquet"):
        m = date_pat.search(str(p))
        if not m:
            continue
        dates.append(pd.Timestamp(m.group(1)))
    return sorted(dates)

def latest_local_date(folder: Path) -> Optional[pd.Timestamp]:
    ds = list_local_dates(folder)
    return ds[-1] if ds else None

def daterange(start: pd.Timestamp, end: pd.Timestamp):
    d = start
    while d <= end:
        yield d
        d += timedelta(days=1)

def safe_num(s):
    return pd.to_numeric(s, errors="coerce")


# ============================================================
# J-Quants API client (bars only)
# ============================================================
class JQuantsAPIV2:
    def __init__(self, api_key: Optional[str] = None, delay_sec: float = PLAN_DELAY_SEC, timeout_sec: int = TIMEOUT_SEC):
        self.api_key = api_key or os.getenv("JQUANTS_API_KEY")
        if not self.api_key:
            raise ValueError("環境変数 JQUANTS_API_KEY がありません。")
        self.session = requests.Session()
        self.delay_sec = delay_sec
        self.timeout_sec = timeout_sec

    def _headers(self) -> Dict[str, str]:
        return {"x-api-key": self.api_key}

    def _get(self, url: str, params: Dict, max_retries: int = 3) -> Dict:
        for attempt in range(max_retries):
            r = self.session.get(url, headers=self._headers(), params=params, timeout=self.timeout_sec)
            if r.status_code == 429:
                wait = self.delay_sec * (2 ** attempt)
                print(f"  ⚠️ rate limit: wait {wait:.1f}s")
                time.sleep(wait)
                continue
            r.raise_for_status()
            time.sleep(self.delay_sec)
            return r.json()
        raise RuntimeError(f"Max retries exceeded: {url}")

    def get_daily_bars(self, date: str) -> List[Dict]:
        url = f"{JQUANTS_BASE}/equities/bars/daily"
        js = self._get(url, {"date": date})
        return js.get("data", [])


def infer_latest_trading_day_by_bars(api: JQuantsAPIV2, base_day: pd.Timestamp, lookback_days: int = 30) -> pd.Timestamp:
    hr("-", 120)
    print("[A-0] 最新営業日推定（No-Calendar：barsが取れる直近日を探索）")
    for k in range(lookback_days + 1):
        d = base_day - timedelta(days=k)
        ds = d.strftime("%Y-%m-%d")
        try:
            data = api.get_daily_bars(ds)
            if data:
                print(f"✅ latest_trading_day inferred: {ds}")
                return d
        except Exception:
            continue
    print(f"⚠️ 推定失敗。base_day={base_day.date()} を返します")
    return base_day


# ============================================================
# Step A: bars差分取得（No-Calendar）
# ============================================================
def update_bars_no_calendar(api: JQuantsAPIV2, bars_dir: Path, end_hint: pd.Timestamp, recheck_days: int = 7):
    last_local = latest_local_date(bars_dir)
    if last_local is None:
        raise RuntimeError("bars_dir にローカルファイルがありません。初期構築が必要です。")

    start_scan = last_local - timedelta(days=recheck_days)
    if start_scan < pd.Timestamp("2000-01-01"):
        start_scan = pd.Timestamp("2000-01-01")

    hr("-", 120)
    print("[A] bars差分取得（No-Calendar：barsが空なら非営業日扱い）")
    print(f"local_last_date: {last_local.date()}")
    print(f"scan window    : {start_scan.date()} -> {end_hint.date()}  (recheck={recheck_days}d)")

    new_saved = 0
    new_empty = 0

    bars_dir.mkdir(parents=True, exist_ok=True)

    for d in daterange(start_scan, end_hint):
        ds = d.strftime("%Y-%m-%d")
        fp = bars_dir / f"date={ds}.parquet"
        if fp.exists():
            continue

        data = api.get_daily_bars(ds)
        if not data:
            new_empty += 1
            continue

        df = pd.DataFrame(data)
        if "Date" not in df.columns or "Code" not in df.columns:
            continue

        df.to_parquet(fp, engine="pyarrow", index=False)
        new_saved += 1

    print(f"\n✅ bars差分完了: new_saved={new_saved}, empty_days={new_empty}, last_saved={latest_local_date(bars_dir).date()}")


# ============================================================
# Step B: price_month_end 直近月だけ更新
# ============================================================
def build_price_month_end_for_months(bars_dir: Path, target_month_ends: List[pd.Timestamp]) -> pd.DataFrame:
    all_dates = list_local_dates(bars_dir)
    if not all_dates:
        raise RuntimeError("bars_dir is empty")

    tmes = [pd.Timestamp(x).to_period("M") for x in target_month_ends]
    tmes_set = set(tmes)

    dfs = []
    for d in all_dates:
        if d.to_period("M") not in tmes_set:
            continue
        fp = bars_dir / f"date={d.strftime('%Y-%m-%d')}.parquet"
        df = pd.read_parquet(fp)

        adj_col = None
        for c in ["AdjustedClose", "AdjustmentClose", "AdjC", "AdjClose", "AdjCl"]:
            if c in df.columns:
                adj_col = c
                break
        if adj_col is None:
            continue

        df = df[["Date", "Code", adj_col]].copy()
        df = df.rename(columns={adj_col: "AdjustedClose"})
        df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
        df["Code"] = df["Code"].astype(str)
        df = df[df["Code"].notna() & ~df["Code"].isin(BAD_CODE_STRINGS)].copy()
        df = df[df["Date"].notna()].copy()
        df["MonthEnd"] = month_end(df["Date"])
        df["AdjustedClose"] = safe_num(df["AdjustedClose"])
        df = df[df["AdjustedClose"].notna()].copy()

        dfs.append(df)

    if not dfs:
        return pd.DataFrame(columns=["Date", "Code", "AdjustedClose", "MonthEnd", "MarketCap"])

    bars_m = pd.concat(dfs, ignore_index=True)
    bars_m = bars_m.sort_values(["Code", "MonthEnd", "Date"], kind="mergesort")
    pm = bars_m.groupby(["Code", "MonthEnd"], as_index=False).tail(1)

    # MarketCapは後でfinの株数で作るのでNaNのまま
    pm["MarketCap"] = np.nan

    if (pm["Code"].isna() | pm["Code"].isin(BAD_CODE_STRINGS)).any():
        raise RuntimeError("price_month_end rebuild produced bad Code rows")
    if pm.duplicated(["Code", "MonthEnd"]).any():
        raise RuntimeError("price_month_end rebuild produced duplicate (Code,MonthEnd)")

    return pm[["Date", "Code", "AdjustedClose", "MonthEnd", "MarketCap"]].copy()


def update_price_month_end_incremental(bars_dir: Path, price_path: Path, months_back: int = 2):
    hr("-", 120)
    print("[B] price_month_end 差分更新（直近月のみ再計算）")

    if not price_path.exists():
        raise RuntimeError("price_month_end.parquet がありません。Step1を先に実行してください。")

    price = pd.read_parquet(price_path)
    price["MonthEnd"] = force_dt64ns(price["MonthEnd"])
    latest_me = pd.Timestamp(price["MonthEnd"].max())

    target_mes = [pd.Timestamp((latest_me - pd.offsets.MonthEnd(k)).to_pydatetime()) for k in range(months_back)]
    print(f"target MonthEnd(s): {[d.strftime('%Y-%m-%d') for d in target_mes]}")

    rebuilt = build_price_month_end_for_months(bars_dir, target_mes)
    print(f"rebuilt rows: {len(rebuilt):,} codes: {rebuilt['Code'].nunique():,}")

    price = price[~price["MonthEnd"].isin(target_mes)].copy()
    merged = pd.concat([price, rebuilt], ignore_index=True)
    merged = merged.sort_values(["Code", "MonthEnd"], kind="mergesort")
    if merged.duplicated(["Code", "MonthEnd"]).any():
        raise RuntimeError("price_month_end incremental update created duplicates")

    price_path.parent.mkdir(parents=True, exist_ok=True)
    merged.to_parquet(price_path, engine="pyarrow", index=False)
    print(f"✅ updated: {price_path} rows={len(merged):,} latest={merged['MonthEnd'].max()}")


# ============================================================
# Step C: month_end_snapshot（norm財務を使って直近Nヶ月だけ更新）
# ============================================================

# ===== PATCH START: load_fin_norm_recent（前回パッチのまま） =====
def load_fin_norm_recent(fins_dir: Path, start_need: pd.Timestamp, end_need: Optional[pd.Timestamp] = None) -> pd.DataFrame:
    """
    daily_parquet_norm から必要期間のファイルだけ読み込み、right(fin) を作る。
    右側は (Code, snapshot_date) を一意化して merge_asof の前提を守る。
    """
    fin_dates = list_local_dates(fins_dir)
    if not fin_dates:
        raise RuntimeError("fins_dir (norm) is empty")

    if end_need is None:
        end_need = fin_dates[-1]

    fin_dates = [d for d in fin_dates if (d >= start_need) and (d <= end_need)]

    parts = []
    for d in fin_dates:
        fp = fins_dir / f"date={d.strftime('%Y-%m-%d')}.parquet"
        if not fp.exists():
            continue

        fin = pd.read_parquet(fp)
        if fin.empty or ("Code" not in fin.columns):
            continue

        fin = fin.copy()
        fin["Code"] = fin["Code"].astype(str)
        fin = fin[fin["Code"].notna() & ~fin["Code"].isin(BAD_CODE_STRINGS)].copy()

        # snapshot_date（asofキー）: ★スカラー代入で確実にns
        snap_dt64 = pd.Timestamp(d).to_datetime64()   # datetime64[ns] scalar
        fin["snapshot_date"] = snap_dt64
        fin["snapshot_date"] = force_dt64ns(fin["snapshot_date"])

        # 列統一：Eq/TA/NP/ShOutFY/AvgSh を numeric に
        for c in ["Eq", "TA", "NP", "ShOutFY", "AvgSh"]:
            if c in fin.columns:
                fin[c] = safe_num(fin[c])
            else:
                fin[c] = np.nan

        # SharesOut_raw 統一：ShOutFY優先、なければAvgSh
        fin["SharesOut_raw"] = fin["ShOutFY"]
        fin.loc[fin["SharesOut_raw"].isna(), "SharesOut_raw"] = fin["AvgSh"]

        keep = ["Code", "snapshot_date", "Eq", "TA", "NP", "SharesOut_raw"]
        fin = fin[keep].copy()

        parts.append(fin)

    if not parts:
        right = pd.DataFrame(columns=["Code","snapshot_date","Eq","TA","NP","SharesOut_raw"])
        # right summaryも空で出しておく
        pd.DataFrame([{
            "start_need": start_need.strftime("%Y-%m-%d"),
            "end_need": end_need.strftime("%Y-%m-%d"),
            "right_rows": 0,
            "right_codes": 0,
            "right_min_date": "",
            "right_max_date": "",
            "dedup_dropped_rows": 0,
            "cov_eq_pct": 0.0,
            "cov_ta_pct": 0.0,
            "cov_np_pct": 0.0,
            "cov_shares_pct": 0.0,
        }]).to_csv(DIAG_RIGHT_SUMMARY, index=False, encoding="utf-8-sig")
        return right

    right = pd.concat(parts, ignore_index=True)
    right["snapshot_date"] = force_dt64ns(right["snapshot_date"])
    right = right.sort_values(["Code","snapshot_date"], kind="mergesort")

    # ★ここが重要：right を (Code,snapshot_date) で厳密に一意化
    before = len(right)
    right = right.drop_duplicates(["Code","snapshot_date"], keep="last")
    dropped = before - len(right)

    summary = pd.DataFrame([{
        "start_need": start_need.strftime("%Y-%m-%d"),
        "end_need": end_need.strftime("%Y-%m-%d"),
        "right_rows": len(right),
        "right_codes": int(right["Code"].nunique()),
        "right_min_date": str(pd.Timestamp(right["snapshot_date"].min()).date()) if len(right) else "",
        "right_max_date": str(pd.Timestamp(right["snapshot_date"].max()).date()) if len(right) else "",
        "dedup_dropped_rows": int(dropped),
        "cov_eq_pct": float(right["Eq"].notna().mean()*100),
        "cov_ta_pct": float(right["TA"].notna().mean()*100),
        "cov_np_pct": float(right["NP"].notna().mean()*100),
        "cov_shares_pct": float(right["SharesOut_raw"].notna().mean()*100),
    }])
    summary.to_csv(DIAG_RIGHT_SUMMARY, index=False, encoding="utf-8-sig")

    print(f"[DEDUP right] dropped {dropped:,} rows by (Code,snapshot_date)")
    return right
# ===== PATCH END =====


# ===== PATCH START: build_snapshot_for_recent_months（ソート順をmerge_asof要件に合わせて修正） =====
def build_snapshot_for_recent_months(price_path: Path, fins_dir: Path, months_back: int) -> pd.DataFrame:
    price = pd.read_parquet(price_path)

    # 型統一・最低限のクリーニング
    price = price.copy()
    price["Date"] = force_dt64ns(price["Date"])
    price["MonthEnd"] = force_dt64ns(price["MonthEnd"])
    price["Code"] = price["Code"].astype(str)
    price["AdjustedClose"] = safe_num(price["AdjustedClose"])
    price = price[price["AdjustedClose"].notna() & price["Date"].notna() & price["MonthEnd"].notna()].copy()
    price = price[price["Code"].notna() & ~price["Code"].isin(BAD_CODE_STRINGS)].copy()

    latest_me = pd.Timestamp(price["MonthEnd"].max())
    target_mes = [pd.Timestamp((latest_me - pd.offsets.MonthEnd(k)).to_pydatetime()) for k in range(months_back)]
    target_price = price[price["MonthEnd"].isin(target_mes)].copy()

    # ★念のため：price 側が (Code,MonthEnd) 一意か確認（壊れてたらここで止める）
    dup_price = int(target_price.duplicated(["Code","MonthEnd"]).sum())
    if dup_price > 0:
        target_price[target_price.duplicated(["Code","MonthEnd"], keep=False)] \
            .sort_values(["Code","MonthEnd","Date"]) \
            .head(2000) \
            .to_csv(DIAG_DIR / "diag_price_target_duplicates.csv", index=False, encoding="utf-8-sig")
        raise RuntimeError(f"target_price has duplicates (Code,MonthEnd)={dup_price:,} (see diag_price_target_duplicates.csv)")

    # fin読み込み範囲：asofが当たるように十分前から読む（ここが浅いと coverage が死ぬ）
    start_need = (min(target_mes) - pd.Timedelta(days=370))
    end_need = latest_local_date(fins_dir) or pd.Timestamp(datetime.now().date())
    right = load_fin_norm_recent(fins_dir, start_need, end_need=end_need)

    left = target_price[["Code","MonthEnd","Date","AdjustedClose"]].copy()

    # ===== PATCH DIFF: merge_asof要件（keysが全体で単調増加）に合わせる =====
    # 変更前: left.sort_values(["Code","Date"]), right.sort_values(["Code","snapshot_date"])
    # 変更後: leftは["Date","Code"], rightは["snapshot_date","Code"]で“全体として”時系列単調増加にする
    left = left.sort_values(["Date","Code"], kind="mergesort").reset_index(drop=True)
    right = right.sort_values(["snapshot_date","Code"], kind="mergesort").reset_index(drop=True)
    # ===== PATCH DIFF END =====

    # ★merge_asof を一発で（by="Code"）
    asof_df = pd.merge_asof(
        left,
        right,
        left_on="Date",
        right_on="snapshot_date",
        by="Code",
        direction="backward",
        allow_exact_matches=True,
    )

    # MarketCap / BM_Ratio
    asof_df["MarketCap"] = np.where(
        asof_df["SharesOut_raw"].notna(),
        asof_df["AdjustedClose"] * asof_df["SharesOut_raw"],
        np.nan
    )
    asof_df["BM_Ratio"] = np.where(
        asof_df["Eq"].notna() & asof_df["MarketCap"].notna() & (asof_df["MarketCap"] > 0),
        asof_df["Eq"] / asof_df["MarketCap"],
        np.nan
    )

    # ROE/INV_Growth は現段階では未生成（必要なら次に計算実装）
    asof_df["ROE"] = np.nan
    asof_df["INV_Growth"] = np.nan

    snap = asof_df[["Code","MonthEnd","Date","AdjustedClose","MarketCap","BM_Ratio","ROE","INV_Growth"]].copy()

    # ★重複チェック：原因追跡用に asof_df の情報も付けて保存
    dup_mask = snap.duplicated(["Code","MonthEnd"], keep=False)
    dup_cnt = int(dup_mask.sum())
    if dup_cnt > 0:
        diag = asof_df.loc[dup_mask].copy()
        keep_diag = ["Code","MonthEnd","Date","snapshot_date","AdjustedClose","SharesOut_raw","Eq","MarketCap","BM_Ratio"]
        keep_diag = [c for c in keep_diag if c in diag.columns]
        diag = diag[keep_diag].sort_values(["Code","MonthEnd","Date"], kind="mergesort")
        diag.head(5000).to_csv(DIAG_SNAP_DUPES, index=False, encoding="utf-8-sig")
        raise RuntimeError(f"snapshot part has duplicates (Code,MonthEnd)={dup_cnt:,}. see: {DIAG_SNAP_DUPES}")

    return snap
# ===== PATCH END =====


def update_month_end_snapshot_incremental(price_path: Path, fins_dir: Path, snap_path: Path, fin_refresh_months: int):
    hr("-", 120)
    print("[C] month_end_snapshot 差分更新（norm財務で直近Nヶ月だけasof再付与）")
    print(f"refresh months: {fin_refresh_months}")
    print(f"FINS_DIR(norm): {fins_dir}")

    new_part = build_snapshot_for_recent_months(price_path, fins_dir, fin_refresh_months)
    print(f"rebuilt snapshot part rows={len(new_part):,} months={new_part['MonthEnd'].nunique():,} codes={new_part['Code'].nunique():,}")

    if snap_path.exists():
        old = pd.read_parquet(snap_path)
        old["MonthEnd"] = force_dt64ns(old["MonthEnd"])
        target_mes = new_part["MonthEnd"].unique()
        old = old[~old["MonthEnd"].isin(target_mes)].copy()
        merged = pd.concat([old, new_part], ignore_index=True)
    else:
        merged = new_part

    merged = merged.sort_values(["Code","MonthEnd"], kind="mergesort")

    # 最終重複チェック
    if merged.duplicated(["Code","MonthEnd"]).any():
        dup = merged[merged.duplicated(["Code","MonthEnd"], keep=False)].sort_values(["Code","MonthEnd","Date"]).head(1000)
        dup.to_csv(DIAG_SNAP_DUPES, index=False, encoding="utf-8-sig")
        raise RuntimeError(f"final snapshot has duplicates. see: {DIAG_SNAP_DUPES}")

    # coverage by month
    cov = (merged.groupby("MonthEnd")
           .agg(n=("Code","size"),
                mcap_notna=("MarketCap", lambda s: float(s.notna().mean())),
                bm_notna=("BM_Ratio", lambda s: float(s.notna().mean())),
                roe_notna=("ROE", lambda s: float(s.notna().mean())),
                inv_notna=("INV_Growth", lambda s: float(s.notna().mean())))
           .reset_index())
    for c in ["mcap_notna","bm_notna","roe_notna","inv_notna"]:
        cov[c] = (cov[c] * 100).round(2)
    cov.to_csv(DIAG_SNAP_COVERAGE, index=False, encoding="utf-8-sig")

    # latest month sample
    latest_me = merged["MonthEnd"].max()
    sample = merged[merged["MonthEnd"] == latest_me].copy().sort_values("MarketCap", ascending=False)
    sample.head(200).to_csv(DIAG_SNAP_SAMPLE, index=False, encoding="utf-8-sig")

    # save
    snap_path.parent.mkdir(parents=True, exist_ok=True)
    merged.to_parquet(snap_path, engine="pyarrow", index=False)

    cov_mcap = merged["MarketCap"].notna().mean() * 100
    cov_bm   = merged["BM_Ratio"].notna().mean() * 100
    print(f"✅ updated: {snap_path} rows={len(merged):,}")
    print(f"coverage(%): MarketCap={cov_mcap:.2f}  BM_Ratio={cov_bm:.2f}  (ROE/INV are NaN in this version)")
    print(f"diag saved: {DIAG_RIGHT_SUMMARY}, {DIAG_SNAP_COVERAGE}, {DIAG_SNAP_SAMPLE}")


# ============================================================
# main
# ============================================================
def main():
    hr()
    print("差分取得（API v2 / No-Calendar）→ スナップ差分更新（norm財務）")
    hr()

    FACTORS_DIR.mkdir(parents=True, exist_ok=True)

    api = JQuantsAPIV2()
    today = pd.Timestamp(datetime.now().date())

    # 1) 最新営業日推定（No-Calendar）
    latest_trading_day = infer_latest_trading_day_by_bars(api, today, lookback_days=30)

    # 2) bars差分取得→ローカル追記
    update_bars_no_calendar(api, BARS_DIR, latest_trading_day, recheck_days=RECHECK_DAYS)

    # 3) price_month_end を直近2ヶ月だけ更新
    update_price_month_end_incremental(BARS_DIR, PRICE_MONTH_END_PATH, months_back=2)

    # 4) month_end_snapshot を直近12ヶ月だけ更新（norm fin asof）
    update_month_end_snapshot_incremental(PRICE_MONTH_END_PATH, FINS_DIR, SNAPSHOT_PATH, fin_refresh_months=FIN_REFRESH_MONTHS)

    hr()
    print("✅ 完了")
    hr()
    print(f"- bars_dir last: {latest_local_date(BARS_DIR)}")
    print(f"- fins_norm last: {latest_local_date(FINS_DIR)}")
    print(f"- price_month_end: {PRICE_MONTH_END_PATH}")
    print(f"- month_end_snapshot: {SNAPSHOT_PATH}")
    print(f"- diag: {DIAG_DIR}")

if __name__ == "__main__":
    main()


差分取得（API v2 / No-Calendar）→ スナップ差分更新（norm財務）
------------------------------------------------------------------------------------------------------------------------
[A-0] 最新営業日推定（No-Calendar：barsが取れる直近日を探索）
✅ latest_trading_day inferred: 2026-01-22
------------------------------------------------------------------------------------------------------------------------
[A] bars差分取得（No-Calendar：barsが空なら非営業日扱い）
local_last_date: 2026-01-22
scan window    : 2026-01-15 -> 2026-01-22  (recheck=7d)

✅ bars差分完了: new_saved=0, empty_days=2, last_saved=2026-01-22
------------------------------------------------------------------------------------------------------------------------
[B] price_month_end 差分更新（直近月のみ再計算）
target MonthEnd(s): ['2026-01-31', '2025-12-31']
rebuilt rows: 8,581 codes: 4,306
✅ updated: C:\Users\yongr\Project\merged_data_all_stocks\factors\price_month_end.parquet rows=491,156 latest=2026-01-31 00:00:00
---------------------------------------------------------------------------

In [59]:
from pathlib import Path
import pandas as pd

DIAG_DIR = Path(r"C:\Users\yongr\Project\merged_data_all_stocks\factors\diag_update_norm")

fp_right = DIAG_DIR / "diag_right_summary.csv"
fp_cov   = DIAG_DIR / "diag_snapshot_coverage_by_month.csv"
fp_top   = DIAG_DIR / "diag_snapshot_latest_month_top200.csv"

print("RIGHT:", fp_right)
print("COV  :", fp_cov)
print("TOP  :", fp_top)

right = pd.read_csv(fp_right, encoding="utf-8-sig")
cov   = pd.read_csv(fp_cov,   encoding="utf-8-sig")
top   = pd.read_csv(fp_top,   encoding="utf-8-sig")

print("\n=== diag_right_summary (all) ===")
print(right)

print("\n=== diag_snapshot_coverage_by_month (tail 15) ===")
print(cov.tail(15))

# 最新月（MonthEndが文字の場合に備えて安全に変換）
cov["MonthEnd"] = pd.to_datetime(cov["MonthEnd"], errors="coerce")
latest_me = cov["MonthEnd"].max()
print("\n=== latest month coverage row ===")
print(cov[cov["MonthEnd"] == latest_me])

print("\n=== diag_snapshot_latest_month_top200 (head 10) ===")
print(top.head(10))


RIGHT: C:\Users\yongr\Project\merged_data_all_stocks\factors\diag_update_norm\diag_right_summary.csv
COV  : C:\Users\yongr\Project\merged_data_all_stocks\factors\diag_update_norm\diag_snapshot_coverage_by_month.csv
TOP  : C:\Users\yongr\Project\merged_data_all_stocks\factors\diag_update_norm\diag_snapshot_latest_month_top200.csv

=== diag_right_summary (all) ===
   start_need    end_need  right_rows  right_codes right_min_date right_max_date  dedup_dropped_rows  cov_eq_pct  cov_ta_pct  cov_np_pct  cov_shares_pct
0  2024-02-24  2026-01-09       32702         4119     2024-02-26     2026-01-09                   0   86.661366   86.667482   86.658308        86.50847

=== diag_snapshot_coverage_by_month (tail 15) ===
       MonthEnd     n  mcap_notna  bm_notna  roe_notna  inv_notna
106  2024-11-30   509        0.20      0.20       0.20       0.20
107  2024-12-31   516        0.19      0.19       0.19       0.19
108  2025-01-31   524        0.19      0.19       0.19       0.19
109  2025-02-2

In [60]:
from pathlib import Path
import pandas as pd

DIAG_DIR = Path(r"C:\Users\yongr\Project\merged_data_all_stocks\factors\diag_update_norm")

right = pd.read_csv(DIAG_DIR / "diag_right_summary.csv", encoding="utf-8-sig")
cov   = pd.read_csv(DIAG_DIR / "diag_snapshot_coverage_by_month.csv", encoding="utf-8-sig")
top   = pd.read_csv(DIAG_DIR / "diag_snapshot_latest_month_top200.csv", encoding="utf-8-sig")

# 1) right_summary（基本1行想定）
print("\n[1] diag_right_summary.csv (all columns, first row)")
print(right.head(1).to_string(index=False))

# 2) coverage 最新月行
cov["MonthEnd"] = pd.to_datetime(cov["MonthEnd"], errors="coerce")
latest_me = cov["MonthEnd"].max()
latest_row = cov.loc[cov["MonthEnd"] == latest_me].copy()
print("\n[2] diag_snapshot_coverage_by_month.csv (latest MonthEnd row)")
print(latest_row.to_string(index=False))

# 3) top200 先頭10行（列は必要なら絞る）
want_cols = [c for c in ["Code","AdjustedClose","MarketCap","BM_Ratio"] if c in top.columns]
print("\n[3] diag_snapshot_latest_month_top200.csv (head 10, selected columns)")
print(top[want_cols].head(10).to_string(index=False))



[1] diag_right_summary.csv (all columns, first row)
start_need   end_need  right_rows  right_codes right_min_date right_max_date  dedup_dropped_rows  cov_eq_pct  cov_ta_pct  cov_np_pct  cov_shares_pct
2024-02-24 2026-01-09       32702         4119     2024-02-26     2026-01-09                   0   86.661366   86.667482   86.658308        86.50847

[2] diag_snapshot_coverage_by_month.csv (latest MonthEnd row)
  MonthEnd    n  mcap_notna  bm_notna  roe_notna  inv_notna
2026-01-31 4282       83.98     83.98        0.0        0.0

[3] diag_snapshot_latest_month_top200.csv (head 10, selected columns)
 Code  AdjustedClose    MarketCap  BM_Ratio
72030         3584.0 5.660924e+13  0.679341
83060         2817.5 3.400078e+13  0.654050
65010         5264.0 2.411734e+13  0.260877
67580         3631.0 2.232996e+13  0.357997
83160         5408.0 2.086086e+13  0.733635
80350        42500.0 2.004439e+13  0.100010
99830        60850.0 1.936375e+13  0.132664
68570        22850.0 1.750633e+13  0.034864

In [62]:
from pathlib import Path
import numpy as np
import pandas as pd
from datetime import datetime

FACTORS_DIR = Path(r"C:\Users\yongr\Project\merged_data_all_stocks\factors")
DIAG_DIR = FACTORS_DIR / "diag_update_norm"

SNAP_PATH = FACTORS_DIR / "month_end_snapshot.parquet"
FINS_NORM_DIR = Path(r"C:\Users\yongr\Project\jquants_fins_summary_10y_parquet\daily_parquet_norm")

fp_right = DIAG_DIR / "diag_right_summary.csv"
fp_cov   = DIAG_DIR / "diag_snapshot_coverage_by_month.csv"
fp_top   = DIAG_DIR / "diag_snapshot_latest_month_top200.csv"

def force_dt64ns(x):
    dt = pd.to_datetime(x, errors="coerce")
    if hasattr(dt, "astype"):
        return dt.astype("datetime64[ns]")
    if pd.isna(dt):
        return np.datetime64("NaT", "ns")
    return pd.Timestamp(dt).to_datetime64()

def qstats(s, qs=(0,0.01,0.05,0.5,0.95,0.99,1.0)):
    s = pd.to_numeric(s, errors="coerce")
    return s.quantile(list(qs))

def list_fin_files(dir_path: Path, start_need: pd.Timestamp, end_need: pd.Timestamp):
    files = []
    for p in dir_path.glob("date=*.parquet"):
        ds = p.name.replace("date=","").replace(".parquet","")
        d = pd.Timestamp(ds)
        if start_need <= d <= end_need:
            files.append((d, p))
    return sorted(files, key=lambda x: x[0])

def main():
    print("=== Load diag CSVs ===")
    right = pd.read_csv(fp_right, encoding="utf-8-sig")
    cov   = pd.read_csv(fp_cov,   encoding="utf-8-sig")
    top   = pd.read_csv(fp_top,   encoding="utf-8-sig")
    print("\n[diag_right_summary]")
    print(right.to_string(index=False))
    print("\n[diag_snapshot_coverage_by_month latest]")
    cov["MonthEnd"] = pd.to_datetime(cov["MonthEnd"], errors="coerce")
    latest_me_cov = cov["MonthEnd"].max()
    print(cov[cov["MonthEnd"] == latest_me_cov].to_string(index=False))

    print("\n=== Load snapshot parquet ===")
    snap = pd.read_parquet(SNAP_PATH)
    snap["MonthEnd"] = pd.to_datetime(snap["MonthEnd"], errors="coerce")
    snap["Date"] = pd.to_datetime(snap["Date"], errors="coerce")
    snap["Code"] = snap["Code"].astype(str)

    latest_me = snap["MonthEnd"].max()
    latest = snap[snap["MonthEnd"] == latest_me].copy()

    # top200 codes
    codes = top["Code"].astype(str).unique().tolist()
    codes_set = set(codes)

    # latest month price subset for those codes
    price = latest[latest["Code"].isin(codes_set)][["Code","Date","AdjustedClose"]].copy()
    price["AdjustedClose"] = pd.to_numeric(price["AdjustedClose"], errors="coerce")
    price = price.dropna(subset=["Date","AdjustedClose"]).copy()

    # window for fin loading
    start_need = (latest_me - pd.Timedelta(days=400)).normalize()
    end_need = price["Date"].max().normalize()

    fin_files = list_fin_files(FINS_NORM_DIR, start_need, end_need)
    if not fin_files:
        raise RuntimeError("No norm fin files found in the required window.")

    cols_need = ["Code","Eq","TA","NP","ShOutFY","TrShFY","AvgSh"]
    fins_parts = []
    for d, p in fin_files:
        df = pd.read_parquet(p)
        if df.empty or "Code" not in df.columns:
            continue

        for c in cols_need:
            if c not in df.columns:
                df[c] = np.nan

        df = df[cols_need].copy()
        df["Code"] = df["Code"].astype(str)
        df = df[df["Code"].isin(codes_set)].copy()
        if df.empty:
            continue

        # ★★★ ここが修正点：snapshot_date を必ず datetime64[ns] にする ★★★
        df["snapshot_date"] = np.datetime64(pd.Timestamp(d).to_datetime64(), "ns")

        for c in ["Eq","TA","NP","ShOutFY","TrShFY","AvgSh"]:
            df[c] = pd.to_numeric(df[c], errors="coerce")

        fins_parts.append(df)

    fin_all = pd.concat(fins_parts, ignore_index=True) if fins_parts else pd.DataFrame(columns=cols_need+["snapshot_date"])
    if fin_all.empty:
        raise RuntimeError("No fin rows for top200 codes in the required window.")

    # dtype unify (extra safety)
    price["Date"] = force_dt64ns(price["Date"])
    fin_all["snapshot_date"] = force_dt64ns(fin_all["snapshot_date"])

    price_sorted = price.sort_values(["Date","Code"]).reset_index(drop=True)
    fin_sorted   = fin_all.sort_values(["snapshot_date","Code"]).reset_index(drop=True)

    merged = pd.merge_asof(
        price_sorted,
        fin_sorted,
        left_on="Date",
        right_on="snapshot_date",
        by="Code",
        direction="backward",
        allow_exact_matches=True,
    )

    print("\n=== Shares coverage (top200 codes, latest month) ===")
    for c in ["ShOutFY","TrShFY","AvgSh"]:
        print(f"{c} notna%: {merged[c].notna().mean()*100:.2f}")

    # choose shares: ShOutFY -> TrShFY -> AvgSh
    merged["Shares_pick"] = merged["ShOutFY"]
    merged.loc[merged["Shares_pick"].isna(), "Shares_pick"] = merged["TrShFY"]
    merged.loc[merged["Shares_pick"].isna(), "Shares_pick"] = merged["AvgSh"]
    print(f"\nShares_pick notna%: {merged['Shares_pick'].notna().mean()*100:.2f}")

    scales = [
        ("raw (x1)", 1.0),
        ("thousand shares (/1e3)", 1e-3),
        ("million shares (/1e6)", 1e-6),
    ]

    for name, k in scales:
        mcap = merged["AdjustedClose"] * (merged["Shares_pick"] * k)
        print(f"\n--- MarketCap scale test: {name} ---")
        print("MarketCap quantiles:")
        print(qstats(mcap).to_string())

        tmp = merged[["Code","AdjustedClose"]].copy()
        tmp["MarketCap_test"] = mcap
        tmp = tmp.sort_values("MarketCap_test", ascending=False).head(10)
        print("\nTop10 MarketCap_test:")
        print(tmp.to_string(index=False))

if __name__ == "__main__":
    main()


=== Load diag CSVs ===

[diag_right_summary]
start_need   end_need  right_rows  right_codes right_min_date right_max_date  dedup_dropped_rows  cov_eq_pct  cov_ta_pct  cov_np_pct  cov_shares_pct
2024-02-24 2026-01-09       32702         4119     2024-02-26     2026-01-09                   0   86.661366   86.667482   86.658308        86.50847

[diag_snapshot_coverage_by_month latest]
  MonthEnd    n  mcap_notna  bm_notna  roe_notna  inv_notna
2026-01-31 4282       83.98     83.98        0.0        0.0

=== Load snapshot parquet ===

=== Shares coverage (top200 codes, latest month) ===
ShOutFY notna%: 100.00
TrShFY notna%: 98.50
AvgSh notna%: 100.00

Shares_pick notna%: 100.00

--- MarketCap scale test: raw (x1) ---
MarketCap quantiles:
0.00    1.006320e+12
0.01    1.011497e+12
0.05    1.056716e+12
0.50    2.351054e+12
0.95    1.587603e+13
0.99    2.421617e+13
1.00    5.660924e+13

Top10 MarketCap_test:
 Code  AdjustedClose  MarketCap_test
72030         3584.0    5.660924e+13
83060       

In [63]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
from datetime import datetime

# =========================
# Paths
# =========================
FACTORS_DIR = Path(r"C:\Users\yongr\Project\merged_data_all_stocks\factors")
PRICE_PATH = FACTORS_DIR / "price_month_end.parquet"

FINS_NORM_DIR = Path(r"C:\Users\yongr\Project\jquants_fins_summary_10y_parquet\daily_parquet_norm")

OUT_SNAP = FACTORS_DIR / "month_end_snapshot_ff5.parquet"
DIAG_DIR = FACTORS_DIR / "diag_stepD_ff5"
DIAG_DIR.mkdir(parents=True, exist_ok=True)

BAD_CODE_STRINGS = {"None", "nan", "", "NaN", "NULL", "null"}
date_pat = re.compile(r"date=(\d{4}-\d{2}-\d{2})\.parquet$")

# =========================
# Utils
# =========================
def force_dt64ns(x):
    dt = pd.to_datetime(x, errors="coerce")
    if hasattr(dt, "astype"):
        return dt.astype("datetime64[ns]")
    if pd.isna(dt):
        return np.datetime64("NaT", "ns")
    return pd.Timestamp(dt).to_datetime64()

def safe_num(s):
    return pd.to_numeric(s, errors="coerce")

def list_local_dates(folder: Path):
    ds = []
    for p in folder.glob("date=*.parquet"):
        m = date_pat.search(p.name)
        if not m:
            continue
        ds.append(pd.Timestamp(m.group(1)))
    return sorted(ds)

def winsorize_series(s: pd.Series, lower_q=0.01, upper_q=0.99):
    x = pd.to_numeric(s, errors="coerce")
    lo = x.quantile(lower_q)
    hi = x.quantile(upper_q)
    return x.clip(lo, hi)

# =========================
# Load fins norm in window
# =========================
def load_fins_norm_window(start_need: pd.Timestamp, end_need: pd.Timestamp) -> pd.DataFrame:
    files = []
    for d in list_local_dates(FINS_NORM_DIR):
        if start_need <= d <= end_need:
            files.append((d, FINS_NORM_DIR / f"date={d.strftime('%Y-%m-%d')}.parquet"))

    cols_need = ["Code","Eq","TA","NP","ShOutFY","TrShFY","AvgSh"]
    parts = []
    for d, fp in files:
        df = pd.read_parquet(fp)
        if df.empty or "Code" not in df.columns:
            continue
        for c in cols_need:
            if c not in df.columns:
                df[c] = np.nan
        df = df[cols_need].copy()
        df["Code"] = df["Code"].astype(str)
        df = df[df["Code"].notna() & ~df["Code"].isin(BAD_CODE_STRINGS)].copy()

        df["snapshot_date"] = np.datetime64(pd.Timestamp(d).to_datetime64(), "ns")
        for c in ["Eq","TA","NP","ShOutFY","TrShFY","AvgSh"]:
            df[c] = safe_num(df[c])

        # Shares pick: ShOutFY -> TrShFY -> AvgSh
        df["SharesOut_raw"] = df["ShOutFY"]
        df.loc[df["SharesOut_raw"].isna(), "SharesOut_raw"] = df["TrShFY"]
        df.loc[df["SharesOut_raw"].isna(), "SharesOut_raw"] = df["AvgSh"]

        parts.append(df[["Code","snapshot_date","Eq","TA","NP","SharesOut_raw"]])

    if not parts:
        return pd.DataFrame(columns=["Code","snapshot_date","Eq","TA","NP","SharesOut_raw"])

    fin = pd.concat(parts, ignore_index=True)
    fin["snapshot_date"] = force_dt64ns(fin["snapshot_date"])
    fin = fin.sort_values(["snapshot_date","Code"], kind="mergesort").reset_index(drop=True)

    # right uniqueness (safety)
    fin = fin.sort_values(["Code","snapshot_date"], kind="mergesort")
    fin = fin.drop_duplicates(["Code","snapshot_date"], keep="last")
    fin = fin.sort_values(["snapshot_date","Code"], kind="mergesort").reset_index(drop=True)
    return fin

# =========================
# Main
# =========================
def main():
    price = pd.read_parquet(PRICE_PATH).copy()
    price["Date"] = force_dt64ns(price["Date"])
    price["MonthEnd"] = force_dt64ns(price["MonthEnd"])
    price["Code"] = price["Code"].astype(str)
    price["AdjustedClose"] = safe_num(price["AdjustedClose"])
    price = price[price["AdjustedClose"].notna() & price["Date"].notna() & price["MonthEnd"].notna()].copy()
    price = price[price["Code"].notna() & ~price["Code"].isin(BAD_CODE_STRINGS)].copy()

    # window for fin: need at least 13 months back to compute TA shift(12)
    latest_me = pd.Timestamp(price["MonthEnd"].max())
    start_need = (latest_me - pd.Timedelta(days=450 + 370)).normalize()  # ざっくり 2.2年分読む（余裕）
    end_need = pd.Timestamp(latest_me).normalize()

    fin = load_fins_norm_window(start_need, end_need)
    if fin.empty:
        raise RuntimeError("fin window is empty")

    # merge_asof needs global monotonic keys
    left = price[["Code","MonthEnd","Date","AdjustedClose"]].copy()
    left = left.sort_values(["Date","Code"], kind="mergesort").reset_index(drop=True)
    right = fin.sort_values(["snapshot_date","Code"], kind="mergesort").reset_index(drop=True)

    merged = pd.merge_asof(
        left,
        right,
        left_on="Date",
        right_on="snapshot_date",
        by="Code",
        direction="backward",
        allow_exact_matches=True,
    )

    # MarketCap / BM
    merged["MarketCap"] = np.where(
        merged["SharesOut_raw"].notna(),
        merged["AdjustedClose"] * merged["SharesOut_raw"],
        np.nan
    )
    merged["BM_Ratio"] = np.where(
        merged["Eq"].notna() & merged["MarketCap"].notna() & (merged["MarketCap"] > 0),
        merged["Eq"] / merged["MarketCap"],
        np.nan
    )

    # Build fundamentals_monthly base
    base = merged[["Code","MonthEnd","Date","AdjustedClose","MarketCap","BM_Ratio","Eq","TA","NP"]].copy()
    base = base.sort_values(["Code","MonthEnd"], kind="mergesort").reset_index(drop=True)

    # ROE proxy: NP/Eq (guard)
    base["ROE"] = np.where(
        base["NP"].notna() & base["Eq"].notna() & (base["Eq"] > 0),
        base["NP"] / base["Eq"],
        np.nan
    )

    # INV_Growth: YoY TA growth (12 months shift)
    base["TA_lag12"] = base.groupby("Code")["TA"].shift(12)
    base["INV_Growth"] = np.where(
        base["TA"].notna() & base["TA_lag12"].notna() & (base["TA_lag12"] > 0),
        (base["TA"] - base["TA_lag12"]) / base["TA_lag12"],
        np.nan
    )

    # winsorize (recommended to stabilize FF sorts)
    base["ROE_w"] = winsorize_series(base["ROE"], 0.01, 0.99)
    base["INV_Growth_w"] = winsorize_series(base["INV_Growth"], 0.01, 0.99)

    snap = base[["Code","MonthEnd","Date","AdjustedClose","MarketCap","BM_Ratio","ROE_w","INV_Growth_w"]].rename(
        columns={"ROE_w":"ROE", "INV_Growth_w":"INV_Growth"}
    )

    # uniqueness check
    dup = int(snap.duplicated(["Code","MonthEnd"]).sum())
    if dup:
        snap[snap.duplicated(["Code","MonthEnd"], keep=False)]\
            .sort_values(["Code","MonthEnd","Date"])\
            .head(5000)\
            .to_csv(DIAG_DIR / "diag_snapshot_duplicates.csv", index=False, encoding="utf-8-sig")
        raise RuntimeError(f"snapshot has duplicates (Code,MonthEnd)={dup}")

    # coverage diag
    cov = (snap.groupby("MonthEnd")
           .agg(n=("Code","size"),
                mcap_pct=("MarketCap", lambda s: float(s.notna().mean()*100)),
                bm_pct=("BM_Ratio", lambda s: float(s.notna().mean()*100)),
                roe_pct=("ROE", lambda s: float(s.notna().mean()*100)),
                inv_pct=("INV_Growth", lambda s: float(s.notna().mean()*100)))
           .reset_index())
    cov.to_csv(DIAG_DIR / "diag_coverage_by_month.csv", index=False, encoding="utf-8-sig")

    # latest month sample
    latest_me = snap["MonthEnd"].max()
    snap[snap["MonthEnd"] == latest_me].sort_values("MarketCap", ascending=False).head(200)\
        .to_csv(DIAG_DIR / "diag_latest_month_top200.csv", index=False, encoding="utf-8-sig")

    # quantiles
    snap["MarketCap"].dropna().quantile([0,0.01,0.05,0.5,0.95,0.99,1.0]).to_frame("MarketCap_q")\
        .to_csv(DIAG_DIR / "diag_marketcap_quantiles.csv", encoding="utf-8-sig")
    snap["BM_Ratio"].dropna().quantile([0,0.01,0.05,0.5,0.95,0.99,1.0]).to_frame("BM_q")\
        .to_csv(DIAG_DIR / "diag_bm_quantiles.csv", encoding="utf-8-sig")
    snap["ROE"].dropna().quantile([0,0.01,0.05,0.5,0.95,0.99,1.0]).to_frame("ROE_q")\
        .to_csv(DIAG_DIR / "diag_roe_quantiles.csv", encoding="utf-8-sig")
    snap["INV_Growth"].dropna().quantile([0,0.01,0.05,0.5,0.95,0.99,1.0]).to_frame("INV_q")\
        .to_csv(DIAG_DIR / "diag_inv_quantiles.csv", encoding="utf-8-sig")

    # save
    OUT_SNAP.parent.mkdir(parents=True, exist_ok=True)
    snap.to_parquet(OUT_SNAP, engine="pyarrow", index=False)

    print("✅ saved:", OUT_SNAP)
    print("coverage latest month:")
    print(cov.tail(1).to_string(index=False))

if __name__ == "__main__":
    main()


✅ saved: C:\Users\yongr\Project\merged_data_all_stocks\factors\month_end_snapshot_ff5.parquet
coverage latest month:
  MonthEnd    n  mcap_pct    bm_pct   roe_pct   inv_pct
2026-01-31 4282 83.979449 83.979449 84.002802 77.323681


In [64]:
import os
import re
import time
from pathlib import Path
from datetime import datetime, timedelta
from typing import Optional, Dict, List, Tuple

import pandas as pd
import numpy as np
import requests

# ============================================================
# パス（あなたの環境）
# ============================================================
BARS_DIR = Path(r"C:\Users\yongr\Project\jquants_daily_bars_10y_parquet\daily_parquet")

# ★重要：正規化済み財務を使う
FINS_DIR = Path(r"C:\Users\yongr\Project\jquants_fins_summary_10y_parquet\daily_parquet_norm")

FACTORS_DIR = Path(r"C:\Users\yongr\Project\merged_data_all_stocks\factors")
PRICE_MONTH_END_PATH = FACTORS_DIR / "price_month_end.parquet"
SNAPSHOT_PATH = FACTORS_DIR / "month_end_snapshot.parquet"  # ★ここへ上書き保存

DIAG_DIR = FACTORS_DIR / "diag_update_norm"
DIAG_DIR.mkdir(parents=True, exist_ok=True)

DIAG_RIGHT_SUMMARY = DIAG_DIR / "diag_right_summary.csv"
DIAG_SNAP_COVERAGE = DIAG_DIR / "diag_snapshot_coverage_by_month.csv"
DIAG_SNAP_DUPES    = DIAG_DIR / "diag_snapshot_duplicates.csv"
DIAG_SNAP_SAMPLE   = DIAG_DIR / "diag_snapshot_latest_month_top200.csv"

# StepD追加診断
DIAG_SNAP_QUANT_MC  = DIAG_DIR / "diag_marketcap_quantiles.csv"
DIAG_SNAP_QUANT_BM  = DIAG_DIR / "diag_bm_quantiles.csv"
DIAG_SNAP_QUANT_ROE = DIAG_DIR / "diag_roe_quantiles.csv"
DIAG_SNAP_QUANT_INV = DIAG_DIR / "diag_inv_quantiles.csv"

# ============================================================
# 更新パラメータ
# ============================================================
RECHECK_DAYS = 7              # bars差分の巻き戻し
FIN_REFRESH_MONTHS = 12       # snapshot更新対象月（直近Nヶ月）

# ===== PATCH START: StepD用（INV_Growth計算のための計算窓を拡張）=====
# INV_Growth = TA の12ヶ月差を作るには、少なくとも (refresh_months + 12) ヶ月分の月次データが必要。
# 余裕を見て 24ヶ月分を一時再構築し、その上で直近FIN_REFRESH_MONTHSを差し替える。
INV_LOOKBACK_MONTHS = 12
SNAP_REBUILD_MONTHS_FOR_FEATURES = FIN_REFRESH_MONTHS + INV_LOOKBACK_MONTHS  # = 24
# ===== PATCH END =====

# ============================================================
# API（bars差分取得のみ使用）
# ============================================================
JQUANTS_BASE = "https://api.jquants.com/v2"
PLAN_DELAY_SEC = 0.5
TIMEOUT_SEC = 60

BAD_CODE_STRINGS = {"None", "nan", "", "NaN", "NULL", "null"}
date_pat = re.compile(r"date=(\d{4}-\d{2}-\d{2})\.parquet$")


# ============================================================
# util
# ============================================================
def hr(ch="=", n=120):
    print(ch * n)

def month_end(ts: pd.Series) -> pd.Series:
    return ts.dt.to_period("M").dt.to_timestamp("M")

def force_dt64ns(x):
    dt = pd.to_datetime(x, errors="coerce")
    if hasattr(dt, "astype"):
        return dt.astype("datetime64[ns]")
    if pd.isna(dt):
        return np.datetime64("NaT", "ns")
    return pd.Timestamp(dt).to_datetime64()

def list_local_dates(folder: Path) -> List[pd.Timestamp]:
    dates = []
    for p in folder.glob("date=*.parquet"):
        m = date_pat.search(str(p))
        if not m:
            continue
        dates.append(pd.Timestamp(m.group(1)))
    return sorted(dates)

def latest_local_date(folder: Path) -> Optional[pd.Timestamp]:
    ds = list_local_dates(folder)
    return ds[-1] if ds else None

def daterange(start: pd.Timestamp, end: pd.Timestamp):
    d = start
    while d <= end:
        yield d
        d += timedelta(days=1)

def safe_num(s):
    return pd.to_numeric(s, errors="coerce")

def winsorize_series(s: pd.Series, lower_q=0.01, upper_q=0.99) -> pd.Series:
    x = pd.to_numeric(s, errors="coerce")
    if x.notna().sum() == 0:
        return x
    lo = x.quantile(lower_q)
    hi = x.quantile(upper_q)
    return x.clip(lo, hi)


# ============================================================
# J-Quants API client (bars only)
# ============================================================
class JQuantsAPIV2:
    def __init__(self, api_key: Optional[str] = None, delay_sec: float = PLAN_DELAY_SEC, timeout_sec: int = TIMEOUT_SEC):
        self.api_key = api_key or os.getenv("JQUANTS_API_KEY")
        if not self.api_key:
            raise ValueError("環境変数 JQUANTS_API_KEY がありません。")
        self.session = requests.Session()
        self.delay_sec = delay_sec
        self.timeout_sec = timeout_sec

    def _headers(self) -> Dict[str, str]:
        return {"x-api-key": self.api_key}

    def _get(self, url: str, params: Dict, max_retries: int = 3) -> Dict:
        for attempt in range(max_retries):
            r = self.session.get(url, headers=self._headers(), params=params, timeout=self.timeout_sec)
            if r.status_code == 429:
                wait = self.delay_sec * (2 ** attempt)
                print(f"  ⚠️ rate limit: wait {wait:.1f}s")
                time.sleep(wait)
                continue
            r.raise_for_status()
            time.sleep(self.delay_sec)
            return r.json()
        raise RuntimeError(f"Max retries exceeded: {url}")

    def get_daily_bars(self, date: str) -> List[Dict]:
        url = f"{JQUANTS_BASE}/equities/bars/daily"
        js = self._get(url, {"date": date})
        return js.get("data", [])


def infer_latest_trading_day_by_bars(api: JQuantsAPIV2, base_day: pd.Timestamp, lookback_days: int = 30) -> pd.Timestamp:
    hr("-", 120)
    print("[A-0] 最新営業日推定（No-Calendar：barsが取れる直近日を探索）")
    for k in range(lookback_days + 1):
        d = base_day - timedelta(days=k)
        ds = d.strftime("%Y-%m-%d")
        try:
            data = api.get_daily_bars(ds)
            if data:
                print(f"✅ latest_trading_day inferred: {ds}")
                return d
        except Exception:
            continue
    print(f"⚠️ 推定失敗。base_day={base_day.date()} を返します")
    return base_day


# ============================================================
# Step A: bars差分取得（No-Calendar）
# ============================================================
def update_bars_no_calendar(api: JQuantsAPIV2, bars_dir: Path, end_hint: pd.Timestamp, recheck_days: int = 7):
    last_local = latest_local_date(bars_dir)
    if last_local is None:
        raise RuntimeError("bars_dir にローカルファイルがありません。初期構築が必要です。")

    start_scan = last_local - timedelta(days=recheck_days)
    if start_scan < pd.Timestamp("2000-01-01"):
        start_scan = pd.Timestamp("2000-01-01")

    hr("-", 120)
    print("[A] bars差分取得（No-Calendar：barsが空なら非営業日扱い）")
    print(f"local_last_date: {last_local.date()}")
    print(f"scan window    : {start_scan.date()} -> {end_hint.date()}  (recheck={recheck_days}d)")

    new_saved = 0
    new_empty = 0

    bars_dir.mkdir(parents=True, exist_ok=True)

    for d in daterange(start_scan, end_hint):
        ds = d.strftime("%Y-%m-%d")
        fp = bars_dir / f"date={ds}.parquet"
        if fp.exists():
            continue

        data = api.get_daily_bars(ds)
        if not data:
            new_empty += 1
            continue

        df = pd.DataFrame(data)
        if "Date" not in df.columns or "Code" not in df.columns:
            continue

        df.to_parquet(fp, engine="pyarrow", index=False)
        new_saved += 1

    print(f"\n✅ bars差分完了: new_saved={new_saved}, empty_days={new_empty}, last_saved={latest_local_date(bars_dir).date()}")


# ============================================================
# Step B: price_month_end 直近月だけ更新
# ============================================================
def build_price_month_end_for_months(bars_dir: Path, target_month_ends: List[pd.Timestamp]) -> pd.DataFrame:
    all_dates = list_local_dates(bars_dir)
    if not all_dates:
        raise RuntimeError("bars_dir is empty")

    tmes = [pd.Timestamp(x).to_period("M") for x in target_month_ends]
    tmes_set = set(tmes)

    dfs = []
    for d in all_dates:
        if d.to_period("M") not in tmes_set:
            continue
        fp = bars_dir / f"date={d.strftime('%Y-%m-%d')}.parquet"
        df = pd.read_parquet(fp)

        adj_col = None
        for c in ["AdjustedClose", "AdjustmentClose", "AdjC", "AdjClose", "AdjCl"]:
            if c in df.columns:
                adj_col = c
                break
        if adj_col is None:
            continue

        df = df[["Date", "Code", adj_col]].copy()
        df = df.rename(columns={adj_col: "AdjustedClose"})
        df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
        df["Code"] = df["Code"].astype(str)
        df = df[df["Code"].notna() & ~df["Code"].isin(BAD_CODE_STRINGS)].copy()
        df = df[df["Date"].notna()].copy()
        df["MonthEnd"] = month_end(df["Date"])
        df["AdjustedClose"] = safe_num(df["AdjustedClose"])
        df = df[df["AdjustedClose"].notna()].copy()

        dfs.append(df)

    if not dfs:
        return pd.DataFrame(columns=["Date", "Code", "AdjustedClose", "MonthEnd", "MarketCap"])

    bars_m = pd.concat(dfs, ignore_index=True)
    bars_m = bars_m.sort_values(["Code", "MonthEnd", "Date"], kind="mergesort")
    pm = bars_m.groupby(["Code", "MonthEnd"], as_index=False).tail(1)

    # MarketCapは後でfinの株数で作るのでNaNのまま
    pm["MarketCap"] = np.nan

    if (pm["Code"].isna() | pm["Code"].isin(BAD_CODE_STRINGS)).any():
        raise RuntimeError("price_month_end rebuild produced bad Code rows")
    if pm.duplicated(["Code", "MonthEnd"]).any():
        raise RuntimeError("price_month_end rebuild produced duplicate (Code,MonthEnd)")

    return pm[["Date", "Code", "AdjustedClose", "MonthEnd", "MarketCap"]].copy()


def update_price_month_end_incremental(bars_dir: Path, price_path: Path, months_back: int = 2):
    hr("-", 120)
    print("[B] price_month_end 差分更新（直近月のみ再計算）")

    if not price_path.exists():
        raise RuntimeError("price_month_end.parquet がありません。Step1を先に実行してください。")

    price = pd.read_parquet(price_path)
    price["MonthEnd"] = force_dt64ns(price["MonthEnd"])
    latest_me = pd.Timestamp(price["MonthEnd"].max())

    target_mes = [pd.Timestamp((latest_me - pd.offsets.MonthEnd(k)).to_pydatetime()) for k in range(months_back)]
    print(f"target MonthEnd(s): {[d.strftime('%Y-%m-%d') for d in target_mes]}")

    rebuilt = build_price_month_end_for_months(bars_dir, target_mes)
    print(f"rebuilt rows: {len(rebuilt):,} codes: {rebuilt['Code'].nunique():,}")

    price = price[~price["MonthEnd"].isin(target_mes)].copy()
    merged = pd.concat([price, rebuilt], ignore_index=True)
    merged = merged.sort_values(["Code", "MonthEnd"], kind="mergesort")
    if merged.duplicated(["Code", "MonthEnd"]).any():
        raise RuntimeError("price_month_end incremental update created duplicates")

    price_path.parent.mkdir(parents=True, exist_ok=True)
    merged.to_parquet(price_path, engine="pyarrow", index=False)
    print(f"✅ updated: {price_path} rows={len(merged):,} latest={merged['MonthEnd'].max()}")


# ============================================================
# Step C(+D): month_end_snapshot（norm財務を使って直近Nヶ月だけ更新）
# ============================================================

def load_fin_norm_recent(fins_dir: Path, start_need: pd.Timestamp, end_need: Optional[pd.Timestamp] = None) -> pd.DataFrame:
    """
    daily_parquet_norm から必要期間のファイルだけ読み込み、right(fin) を作る。
    右側は (Code, snapshot_date) を一意化して merge_asof の前提を守る。
    """
    fin_dates = list_local_dates(fins_dir)
    if not fin_dates:
        raise RuntimeError("fins_dir (norm) is empty")

    if end_need is None:
        end_need = fin_dates[-1]

    fin_dates = [d for d in fin_dates if (d >= start_need) and (d <= end_need)]

    parts = []
    for d in fin_dates:
        fp = fins_dir / f"date={d.strftime('%Y-%m-%d')}.parquet"
        if not fp.exists():
            continue

        fin = pd.read_parquet(fp)
        if fin.empty or ("Code" not in fin.columns):
            continue

        fin = fin.copy()
        fin["Code"] = fin["Code"].astype(str)
        fin = fin[fin["Code"].notna() & ~fin["Code"].isin(BAD_CODE_STRINGS)].copy()

        # snapshot_date（asofキー）: ★スカラー代入で確実にns
        snap_dt64 = pd.Timestamp(d).to_datetime64()   # datetime64[ns] scalar
        fin["snapshot_date"] = snap_dt64
        fin["snapshot_date"] = force_dt64ns(fin["snapshot_date"])

        # 列統一：Eq/TA/NP/ShOutFY/TrShFY/AvgSh を numeric に
        for c in ["Eq", "TA", "NP", "ShOutFY", "TrShFY", "AvgSh"]:
            if c in fin.columns:
                fin[c] = safe_num(fin[c])
            else:
                fin[c] = np.nan

        # SharesOut_raw 統一：ShOutFY優先、なければTrShFY、最後にAvgSh
        fin["SharesOut_raw"] = fin["ShOutFY"]
        fin.loc[fin["SharesOut_raw"].isna(), "SharesOut_raw"] = fin["TrShFY"]
        fin.loc[fin["SharesOut_raw"].isna(), "SharesOut_raw"] = fin["AvgSh"]

        keep = ["Code", "snapshot_date", "Eq", "TA", "NP", "SharesOut_raw"]
        fin = fin[keep].copy()

        parts.append(fin)

    if not parts:
        right = pd.DataFrame(columns=["Code","snapshot_date","Eq","TA","NP","SharesOut_raw"])
        pd.DataFrame([{
            "start_need": start_need.strftime("%Y-%m-%d"),
            "end_need": end_need.strftime("%Y-%m-%d"),
            "right_rows": 0,
            "right_codes": 0,
            "right_min_date": "",
            "right_max_date": "",
            "dedup_dropped_rows": 0,
            "cov_eq_pct": 0.0,
            "cov_ta_pct": 0.0,
            "cov_np_pct": 0.0,
            "cov_shares_pct": 0.0,
        }]).to_csv(DIAG_RIGHT_SUMMARY, index=False, encoding="utf-8-sig")
        return right

    right = pd.concat(parts, ignore_index=True)
    right["snapshot_date"] = force_dt64ns(right["snapshot_date"])
    right = right.sort_values(["Code","snapshot_date"], kind="mergesort")

    before = len(right)
    right = right.drop_duplicates(["Code","snapshot_date"], keep="last")
    dropped = before - len(right)

    summary = pd.DataFrame([{
        "start_need": start_need.strftime("%Y-%m-%d"),
        "end_need": end_need.strftime("%Y-%m-%d"),
        "right_rows": len(right),
        "right_codes": int(right["Code"].nunique()),
        "right_min_date": str(pd.Timestamp(right["snapshot_date"].min()).date()) if len(right) else "",
        "right_max_date": str(pd.Timestamp(right["snapshot_date"].max()).date()) if len(right) else "",
        "dedup_dropped_rows": int(dropped),
        "cov_eq_pct": float(right["Eq"].notna().mean()*100),
        "cov_ta_pct": float(right["TA"].notna().mean()*100),
        "cov_np_pct": float(right["NP"].notna().mean()*100),
        "cov_shares_pct": float(right["SharesOut_raw"].notna().mean()*100),
    }])
    summary.to_csv(DIAG_RIGHT_SUMMARY, index=False, encoding="utf-8-sig")

    print(f"[DEDUP right] dropped {dropped:,} rows by (Code,snapshot_date)")
    return right


def build_snapshot_for_recent_months(price_path: Path, fins_dir: Path, months_back: int) -> pd.DataFrame:
    price = pd.read_parquet(price_path)

    price = price.copy()
    price["Date"] = force_dt64ns(price["Date"])
    price["MonthEnd"] = force_dt64ns(price["MonthEnd"])
    price["Code"] = price["Code"].astype(str)
    price["AdjustedClose"] = safe_num(price["AdjustedClose"])
    price = price[price["AdjustedClose"].notna() & price["Date"].notna() & price["MonthEnd"].notna()].copy()
    price = price[price["Code"].notna() & ~price["Code"].isin(BAD_CODE_STRINGS)].copy()

    latest_me = pd.Timestamp(price["MonthEnd"].max())
    target_mes = [pd.Timestamp((latest_me - pd.offsets.MonthEnd(k)).to_pydatetime()) for k in range(months_back)]
    target_price = price[price["MonthEnd"].isin(target_mes)].copy()

    # price 側の一意性（保険）
    dup_price = int(target_price.duplicated(["Code","MonthEnd"]).sum())
    if dup_price > 0:
        target_price[target_price.duplicated(["Code","MonthEnd"], keep=False)] \
            .sort_values(["Code","MonthEnd","Date"]) \
            .head(2000) \
            .to_csv(DIAG_DIR / "diag_price_target_duplicates.csv", index=False, encoding="utf-8-sig")
        raise RuntimeError(f"target_price has duplicates (Code,MonthEnd)={dup_price:,} (see diag_price_target_duplicates.csv)")

    # fin 読み込み：asofが当たるように十分前から読む
    # ===== PATCH START: StepD用にさらに前から読む（TAの12ヶ月ラグが必要）=====
    # months_back が FIN_REFRESH_MONTHS(12) でも、INVのためには +12ヶ月分の財務時系列が必要。
    start_need = (min(target_mes) - pd.Timedelta(days=370 + 370))  # 約2年分
    # ===== PATCH END =====
    end_need = latest_local_date(fins_dir) or pd.Timestamp(datetime.now().date())
    right = load_fin_norm_recent(fins_dir, start_need, end_need=end_need)

    left = target_price[["Code","MonthEnd","Date","AdjustedClose"]].copy()

    # merge_asof要件（keysが全体で単調増加）に合わせる
    left = left.sort_values(["Date","Code"], kind="mergesort").reset_index(drop=True)
    right = right.sort_values(["snapshot_date","Code"], kind="mergesort").reset_index(drop=True)

    # merge_asof 一発
    asof_df = pd.merge_asof(
        left,
        right,
        left_on="Date",
        right_on="snapshot_date",
        by="Code",
        direction="backward",
        allow_exact_matches=True,
    )

    # MarketCap / BM_Ratio
    asof_df["MarketCap"] = np.where(
        asof_df["SharesOut_raw"].notna(),
        asof_df["AdjustedClose"] * asof_df["SharesOut_raw"],
        np.nan
    )
    asof_df["BM_Ratio"] = np.where(
        asof_df["Eq"].notna() & asof_df["MarketCap"].notna() & (asof_df["MarketCap"] > 0),
        asof_df["Eq"] / asof_df["MarketCap"],
        np.nan
    )

    # ===== PATCH START: StepD（ROE / INV_Growth を asof_df から生成）=====
    # ROE = NP / Eq（Eq>0のみ）
    asof_df["ROE_raw"] = np.where(
        asof_df["NP"].notna() & asof_df["Eq"].notna() & (asof_df["Eq"] > 0),
        asof_df["NP"] / asof_df["Eq"],
        np.nan
    )

    # INV_Growth = (TA_t - TA_{t-12}) / TA_{t-12}
    # 月次（Code×MonthEnd）で計算するため、まず Code,MonthEnd で並べた上で TA の12ヶ月ラグを取る
    asof_df = asof_df.sort_values(["Code","MonthEnd"], kind="mergesort").reset_index(drop=True)
    asof_df["TA_lag12"] = asof_df.groupby("Code")["TA"].shift(12)
    asof_df["INV_raw"] = np.where(
        asof_df["TA"].notna() & asof_df["TA_lag12"].notna() & (asof_df["TA_lag12"] > 0),
        (asof_df["TA"] - asof_df["TA_lag12"]) / asof_df["TA_lag12"],
        np.nan
    )

    # winsorize（下流のポートフォリオ分位での暴れを抑える）
    asof_df["ROE"] = winsorize_series(asof_df["ROE_raw"], 0.01, 0.99)
    asof_df["INV_Growth"] = winsorize_series(asof_df["INV_raw"], 0.01, 0.99)
    # ===== PATCH END =====

    snap = asof_df[["Code","MonthEnd","Date","AdjustedClose","MarketCap","BM_Ratio","ROE","INV_Growth"]].copy()

    # 重複チェック
    dup_mask = snap.duplicated(["Code","MonthEnd"], keep=False)
    dup_cnt = int(dup_mask.sum())
    if dup_cnt > 0:
        diag = asof_df.loc[dup_mask].copy()
        keep_diag = ["Code","MonthEnd","Date","snapshot_date","AdjustedClose","SharesOut_raw","Eq","TA","NP","MarketCap","BM_Ratio","ROE","INV_Growth"]
        keep_diag = [c for c in keep_diag if c in diag.columns]
        diag = diag[keep_diag].sort_values(["Code","MonthEnd","Date"], kind="mergesort")
        diag.head(5000).to_csv(DIAG_SNAP_DUPES, index=False, encoding="utf-8-sig")
        raise RuntimeError(f"snapshot part has duplicates (Code,MonthEnd)={dup_cnt:,}. see: {DIAG_SNAP_DUPES}")

    return snap


def update_month_end_snapshot_incremental(price_path: Path, fins_dir: Path, snap_path: Path, fin_refresh_months: int):
    hr("-", 120)
    print("[C] month_end_snapshot 差分更新（norm財務で直近Nヶ月だけasof再付与）")
    print(f"refresh months: {fin_refresh_months}")
    print(f"FINS_DIR(norm): {fins_dir}")

    # ===== PATCH START: StepDのため、計算用は24ヶ月（12+12）を再構築してから保存は12ヶ月差し替え =====
    months_for_features = SNAP_REBUILD_MONTHS_FOR_FEATURES
    new_part_wide = build_snapshot_for_recent_months(price_path, fins_dir, months_for_features)
    # 保存対象（月差し替え）は直近 fin_refresh_months のみに絞る
    max_me = pd.Timestamp(new_part_wide["MonthEnd"].max())
    min_keep_me = (max_me - pd.offsets.MonthEnd(fin_refresh_months - 1)).normalize() + pd.offsets.MonthEnd(0)
    new_part = new_part_wide[new_part_wide["MonthEnd"] >= min_keep_me].copy()
    # ===== PATCH END =====

    print(f"rebuilt snapshot part rows={len(new_part):,} months={new_part['MonthEnd'].nunique():,} codes={new_part['Code'].nunique():,}")

    if snap_path.exists():
        old = pd.read_parquet(snap_path)
        old["MonthEnd"] = force_dt64ns(old["MonthEnd"])
        target_mes = new_part["MonthEnd"].unique()
        old = old[~old["MonthEnd"].isin(target_mes)].copy()
        merged = pd.concat([old, new_part], ignore_index=True)
    else:
        merged = new_part

    merged = merged.sort_values(["Code","MonthEnd"], kind="mergesort")

    # 最終重複チェック
    if merged.duplicated(["Code","MonthEnd"]).any():
        dup = merged[merged.duplicated(["Code","MonthEnd"], keep=False)].sort_values(["Code","MonthEnd","Date"]).head(1000)
        dup.to_csv(DIAG_SNAP_DUPES, index=False, encoding="utf-8-sig")
        raise RuntimeError(f"final snapshot has duplicates. see: {DIAG_SNAP_DUPES}")

    # coverage by month（ROE/INVも含める）
    cov = (merged.groupby("MonthEnd")
           .agg(n=("Code","size"),
                mcap_notna=("MarketCap", lambda s: float(s.notna().mean())),
                bm_notna=("BM_Ratio", lambda s: float(s.notna().mean())),
                roe_notna=("ROE", lambda s: float(s.notna().mean())),
                inv_notna=("INV_Growth", lambda s: float(s.notna().mean())))
           .reset_index())
    for c in ["mcap_notna","bm_notna","roe_notna","inv_notna"]:
        cov[c] = (cov[c] * 100).round(2)
    cov.to_csv(DIAG_SNAP_COVERAGE, index=False, encoding="utf-8-sig")

    # latest month sample
    latest_me = merged["MonthEnd"].max()
    sample = merged[merged["MonthEnd"] == latest_me].copy().sort_values("MarketCap", ascending=False)
    sample.head(200).to_csv(DIAG_SNAP_SAMPLE, index=False, encoding="utf-8-sig")

    # ===== PATCH START: StepDの分位点診断（異常値の早期発見）=====
    def _save_quantiles(series: pd.Series, out_path: Path, col_name: str):
        s = pd.to_numeric(series, errors="coerce").dropna()
        if len(s) == 0:
            pd.DataFrame({col_name: []}).to_csv(out_path, encoding="utf-8-sig")
            return
        q = s.quantile([0,0.01,0.05,0.5,0.95,0.99,1.0]).to_frame(col_name)
        q.to_csv(out_path, encoding="utf-8-sig")

    _save_quantiles(merged["MarketCap"], DIAG_SNAP_QUANT_MC, "MarketCap_q")
    _save_quantiles(merged["BM_Ratio"], DIAG_SNAP_QUANT_BM, "BM_q")
    _save_quantiles(merged["ROE"], DIAG_SNAP_QUANT_ROE, "ROE_q")
    _save_quantiles(merged["INV_Growth"], DIAG_SNAP_QUANT_INV, "INV_q")
    # ===== PATCH END =====

    # save（★month_end_snapshot.parquet に上書き保存）
    snap_path.parent.mkdir(parents=True, exist_ok=True)
    merged.to_parquet(snap_path, engine="pyarrow", index=False)

    cov_mcap = merged["MarketCap"].notna().mean() * 100
    cov_bm   = merged["BM_Ratio"].notna().mean() * 100
    cov_roe  = merged["ROE"].notna().mean() * 100
    cov_inv  = merged["INV_Growth"].notna().mean() * 100
    print(f"✅ updated: {snap_path} rows={len(merged):,}")
    print(f"coverage(%): MarketCap={cov_mcap:.2f}  BM_Ratio={cov_bm:.2f}  ROE={cov_roe:.2f}  INV_Growth={cov_inv:.2f}")
    print(f"diag saved: {DIAG_RIGHT_SUMMARY}, {DIAG_SNAP_COVERAGE}, {DIAG_SNAP_SAMPLE}")


# ============================================================
# main
# ============================================================
def main():
    hr()
    print("差分取得（API v2 / No-Calendar）→ スナップ差分更新（norm財務）")
    hr()

    FACTORS_DIR.mkdir(parents=True, exist_ok=True)

    api = JQuantsAPIV2()
    today = pd.Timestamp(datetime.now().date())

    # 1) 最新営業日推定（No-Calendar）
    latest_trading_day = infer_latest_trading_day_by_bars(api, today, lookback_days=30)

    # 2) bars差分取得→ローカル追記
    update_bars_no_calendar(api, BARS_DIR, latest_trading_day, recheck_days=RECHECK_DAYS)

    # 3) price_month_end を直近2ヶ月だけ更新
    update_price_month_end_incremental(BARS_DIR, PRICE_MONTH_END_PATH, months_back=2)

    # 4) month_end_snapshot を直近12ヶ月だけ更新（norm fin asof + StepD features）
    update_month_end_snapshot_incremental(PRICE_MONTH_END_PATH, FINS_DIR, SNAPSHOT_PATH, fin_refresh_months=FIN_REFRESH_MONTHS)

    hr()
    print("✅ 完了")
    hr()
    print(f"- bars_dir last: {latest_local_date(BARS_DIR)}")
    print(f"- fins_norm last: {latest_local_date(FINS_DIR)}")
    print(f"- price_month_end: {PRICE_MONTH_END_PATH}")
    print(f"- month_end_snapshot: {SNAPSHOT_PATH}")
    print(f"- diag: {DIAG_DIR}")

if __name__ == "__main__":
    main()


差分取得（API v2 / No-Calendar）→ スナップ差分更新（norm財務）
------------------------------------------------------------------------------------------------------------------------
[A-0] 最新営業日推定（No-Calendar：barsが取れる直近日を探索）
✅ latest_trading_day inferred: 2026-01-22
------------------------------------------------------------------------------------------------------------------------
[A] bars差分取得（No-Calendar：barsが空なら非営業日扱い）
local_last_date: 2026-01-22
scan window    : 2026-01-15 -> 2026-01-22  (recheck=7d)

✅ bars差分完了: new_saved=0, empty_days=2, last_saved=2026-01-22
------------------------------------------------------------------------------------------------------------------------
[B] price_month_end 差分更新（直近月のみ再計算）
target MonthEnd(s): ['2026-01-31', '2025-12-31']
rebuilt rows: 8,581 codes: 4,306
✅ updated: C:\Users\yongr\Project\merged_data_all_stocks\factors\price_month_end.parquet rows=491,156 latest=2026-01-31 00:00:00
---------------------------------------------------------------------------

In [65]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
from dataclasses import dataclass
from typing import Dict, Tuple, Optional, List

import numpy as np
import pandas as pd


# ============================================================
# Paths (あなたの環境)
# ============================================================
FACTORS_DIR = Path(r"C:\Users\yongr\Project\merged_data_all_stocks\factors")

PRICE_PATH = FACTORS_DIR / "price_month_end.parquet"
SNAP_PATH  = FACTORS_DIR / "month_end_snapshot.parquet"

FF5_PARQUET_PATH = FACTORS_DIR / "ff5_factors_monthly.parquet"
CURRENT_ANALYSIS_CSV = FACTORS_DIR / "market_factor_summary.csv"
RANK_ABS_CSV         = FACTORS_DIR / "market_factor_rank_abs.csv"
RANK_CONFW_CSV       = FACTORS_DIR / "market_factor_rank_conf_weighted.csv"
TREND_LAST3_CSV      = FACTORS_DIR / "market_factor_trend_last3.csv"
TREND_LAST12_CSV     = FACTORS_DIR / "market_factor_trend_last12.csv"

# diagnostics
DIAG_RET_DIST_CSV    = FACTORS_DIR / "diag_monthly_stock_return_distribution.csv"
DIAG_NUSED_BY_MONTH  = FACTORS_DIR / "diag_ff5_n_used_by_month.csv"

# ============================================================
# Params
# ============================================================
MIN_STOCKS_PER_MONTH = 500        # “信頼できる月”の下限（必要なら調整）
WINSORIZE_RET = True
RET_WINSOR_Q = (0.01, 0.99)       # ret_m_fwd のウィンザー
BM_WINSOR_Q  = (0.01, 0.99)
ROE_WINSOR_Q = (0.01, 0.99)
INV_WINSOR_Q = (0.01, 0.99)

# Size split: median
# Value/Profit/Inv split: 30/70
P30 = 0.30
P70 = 0.70

BAD_CODE_STRINGS = {"None", "nan", "", "NaN", "NULL", "null"}


# ============================================================
# Utils
# ============================================================
def force_dt64ns(x):
    dt = pd.to_datetime(x, errors="coerce")
    if hasattr(dt, "astype"):
        return dt.astype("datetime64[ns]")
    if pd.isna(dt):
        return np.datetime64("NaT", "ns")
    return pd.Timestamp(dt).to_datetime64()

def safe_num(s):
    return pd.to_numeric(s, errors="coerce")

def winsorize(s: pd.Series, q_lo=0.01, q_hi=0.99):
    x = pd.to_numeric(s, errors="coerce")
    if x.notna().sum() == 0:
        return x
    lo = x.quantile(q_lo)
    hi = x.quantile(q_hi)
    return x.clip(lo, hi)

def value_weighted_return(df: pd.DataFrame, ret_col: str, w_col: str) -> float:
    x = df[[ret_col, w_col]].dropna()
    if x.empty:
        return np.nan
    w = x[w_col].astype(float).to_numpy()
    r = x[ret_col].astype(float).to_numpy()
    wsum = w.sum()
    if not np.isfinite(wsum) or wsum <= 0:
        return np.nan
    return float(np.dot(r, w) / wsum)

def qcut_3way(x: pd.Series, p30=0.3, p70=0.7, labels=("L","M","H")) -> pd.Series:
    """30/70%で3区分"""
    a = pd.to_numeric(x, errors="coerce")
    q1 = a.quantile(p30)
    q2 = a.quantile(p70)
    out = pd.Series(index=a.index, dtype="object")
    out[a <= q1] = labels[0]
    out[(a > q1) & (a < q2)] = labels[1]
    out[a >= q2] = labels[2]
    return out

def size_split_median(mcap: pd.Series) -> pd.Series:
    a = pd.to_numeric(mcap, errors="coerce")
    med = a.quantile(0.5)
    out = pd.Series(index=a.index, dtype="object")
    out[a <= med] = "S"
    out[a > med] = "B"
    return out

def latest_evaluable_month(factors: pd.DataFrame, col="MKT") -> pd.Timestamp:
    """因子がNaNでない最終月（ret_m_fwdが存在する最終月相当）"""
    f = factors.dropna(subset=[col]).copy()
    if f.empty:
        return pd.NaT
    return pd.Timestamp(f["MonthEnd"].max())

def trend_label(vals: np.ndarray) -> str:
    """3点程度の簡易トレンド判定"""
    vals = np.array(vals, dtype=float)
    vals = vals[np.isfinite(vals)]
    if len(vals) < 2:
        return "NA"
    x = np.arange(len(vals))
    # 単回帰の傾き
    slope = np.polyfit(x, vals, 1)[0]
    if abs(slope) < 1e-6:
        return "FLAT"
    return "UP" if slope > 0 else "DOWN"

def confidence_from_n(n_used: int) -> float:
    """利用銘柄数から0-1の信頼度（単純）"""
    # 500で0.5、1000で1.0に近づくように
    return float(max(0.0, min(1.0, n_used / 1000.0)))


# ============================================================
# Step 0: Load & build ret_m_fwd
# ============================================================
def load_snapshot_and_build_forward_returns(price_path: Path, snap_path: Path) -> pd.DataFrame:
    price = pd.read_parquet(price_path).copy()
    snap = pd.read_parquet(snap_path).copy()

    # sanitize
    for df in (price, snap):
        df["Code"] = df["Code"].astype(str)
        df = df[~df["Code"].isin(BAD_CODE_STRINGS)]

    price["MonthEnd"] = force_dt64ns(price["MonthEnd"])
    snap["MonthEnd"]  = force_dt64ns(snap["MonthEnd"])
    snap["Date"]      = force_dt64ns(snap["Date"])

    # use AdjustedClose from price_month_end to compute ret_m_fwd robustly
    price["AdjustedClose"] = safe_num(price["AdjustedClose"])
    price = price.dropna(subset=["AdjustedClose", "MonthEnd", "Code"]).copy()
    price = price.sort_values(["Code", "MonthEnd"], kind="mergesort")

    price["Adj_next"] = price.groupby("Code")["AdjustedClose"].shift(-1)
    price["MonthEnd_next"] = price.groupby("Code")["MonthEnd"].shift(-1)
    price["ret_m_fwd"] = (price["Adj_next"] / price["AdjustedClose"]) - 1.0

    # keep only needed fields for join
    pr = price[["Code","MonthEnd","ret_m_fwd","MonthEnd_next"]].copy()

    # join onto snap by (Code, MonthEnd)
    snap = snap.merge(pr, on=["Code","MonthEnd"], how="left", validate="many_to_one")

    # numeric cleanup + winsor
    for c in ["MarketCap","BM_Ratio","ROE","INV_Growth","ret_m_fwd"]:
        if c in snap.columns:
            snap[c] = safe_num(snap[c])

    # basic filters
    snap = snap[(snap["MarketCap"].notna()) & (snap["MarketCap"] > 0)].copy()
    snap = snap[(snap["BM_Ratio"].notna()) & np.isfinite(snap["BM_Ratio"])].copy()

    if WINSORIZE_RET:
        snap["ret_m_fwd_w"] = winsorize(snap["ret_m_fwd"], *RET_WINSOR_Q)
    else:
        snap["ret_m_fwd_w"] = snap["ret_m_fwd"]

    snap["BM_w"]  = winsorize(snap["BM_Ratio"], *BM_WINSOR_Q)
    snap["ROE_w"] = winsorize(snap["ROE"], *ROE_WINSOR_Q) if "ROE" in snap.columns else np.nan
    snap["INV_w"] = winsorize(snap["INV_Growth"], *INV_WINSOR_Q) if "INV_Growth" in snap.columns else np.nan

    return snap


# ============================================================
# Step 1: FF5 monthly factor construction (Japan, no RF)
# ============================================================
def compute_ff5_monthly(snap: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Returns:
      ff5: MonthEnd x {MKT, SMB, HML, RMW, CMA, n_total, n_used_* ...}
      diag: monthly diagnostics
    """
    months = sorted(snap["MonthEnd"].dropna().unique())
    rows = []
    diag_rows = []

    for me in months:
        m = snap[snap["MonthEnd"] == me].copy()

        # Need forward return to define factor payoff for that month (t -> t+1)
        m = m[m["ret_m_fwd_w"].notna()].copy()
        n_total = int(m["Code"].nunique())

        # Universe sanity
        if n_total == 0:
            continue

        # Market return (value-weighted)
        mkt = value_weighted_return(m, "ret_m_fwd_w", "MarketCap")

        # Size split
        m["SZ"] = size_split_median(m["MarketCap"])

        # Value split (BM)
        m["VAL"] = qcut_3way(m["BM_w"], P30, P70, labels=("L","M","H"))

        # Profitability split (ROE)
        m["PROF"] = qcut_3way(m["ROE_w"], P30, P70, labels=("W","M","R"))

        # Investment split (INV) : Conservative (low INV) vs Aggressive (high INV)
        # lower INV -> C, mid -> M, high -> A
        m["INV"] = qcut_3way(m["INV_w"], P30, P70, labels=("C","M","A"))

        # Helpers: compute 2x3 portfolio returns
        def port_ret(sz, grp_col, grp_label):
            x = m[(m["SZ"] == sz) & (m[grp_col] == grp_label)]
            return value_weighted_return(x, "ret_m_fwd_w", "MarketCap")

        # HML components (Value: H-L)
        SH = port_ret("S","VAL","H"); SMv = port_ret("S","VAL","M"); SL = port_ret("S","VAL","L")
        BH = port_ret("B","VAL","H"); BMv = port_ret("B","VAL","M"); BL = port_ret("B","VAL","L")

        HML = np.nanmean([SH, BH]) - np.nanmean([SL, BL])

        # RMW components (Profit: R-W)
        SR = port_ret("S","PROF","R"); SMp = port_ret("S","PROF","M"); SW = port_ret("S","PROF","W")
        BR = port_ret("B","PROF","R"); BMp = port_ret("B","PROF","M"); BW = port_ret("B","PROF","W")

        RMW = np.nanmean([SR, BR]) - np.nanmean([SW, BW])

        # CMA components (Inv: C-A)
        SC = port_ret("S","INV","C"); SMa = port_ret("S","INV","M"); SA = port_ret("S","INV","A")
        BC = port_ret("B","INV","C"); BMa = port_ret("B","INV","M"); BA = port_ret("B","INV","A")

        CMA = np.nanmean([SC, BC]) - np.nanmean([SA, BA])

        # SMB: average of SMB from three sorts (FF convention)
        # SMB_HML: avg(S* - B*) across VAL buckets
        SMB_HML = np.nanmean([SH, SMv, SL]) - np.nanmean([BH, BMv, BL])
        SMB_RMW = np.nanmean([SR, SMp, SW]) - np.nanmean([BR, BMp, BW])
        SMB_CMA = np.nanmean([SC, SMa, SA]) - np.nanmean([BC, BMa, BA])
        SMB = np.nanmean([SMB_HML, SMB_RMW, SMB_CMA])

        # n_used (how many stocks effectively eligible for each factor)
        n_used_val  = int(m.dropna(subset=["SZ","VAL","MarketCap","ret_m_fwd_w"])["Code"].nunique())
        n_used_prof = int(m.dropna(subset=["SZ","PROF","MarketCap","ret_m_fwd_w"])["Code"].nunique())
        n_used_inv  = int(m.dropna(subset=["SZ","INV","MarketCap","ret_m_fwd_w"])["Code"].nunique())

        # confidence (simple)
        conf = confidence_from_n(min(n_used_val, n_used_prof, n_used_inv))

        abnormal = (min(n_used_val, n_used_prof, n_used_inv) < MIN_STOCKS_PER_MONTH)

        rows.append({
            "MonthEnd": me,
            "MKT": mkt,
            "SMB": SMB,
            "HML": HML,
            "RMW": RMW,
            "CMA": CMA,
            "n_total": n_total,
            "n_used_val": n_used_val,
            "n_used_prof": n_used_prof,
            "n_used_inv": n_used_inv,
            "confidence": conf,
            "abnormal": bool(abnormal),
        })

        # return distribution diag
        r = m["ret_m_fwd_w"]
        diag_rows.append({
            "MonthEnd": me,
            "n": int(r.notna().sum()),
            "ret_q01": float(r.quantile(0.01)),
            "ret_q05": float(r.quantile(0.05)),
            "ret_q50": float(r.quantile(0.50)),
            "ret_q95": float(r.quantile(0.95)),
            "ret_q99": float(r.quantile(0.99)),
            "ret_min": float(r.min()),
            "ret_max": float(r.max()),
            "share_abs_gt_50pct": float((r.abs() > 0.5).mean() * 100.0),
            "share_abs_gt_100pct": float((r.abs() > 1.0).mean() * 100.0),
        })

    ff5 = pd.DataFrame(rows).sort_values("MonthEnd").reset_index(drop=True)
    diag = pd.DataFrame(diag_rows).sort_values("MonthEnd").reset_index(drop=True)
    return ff5, diag


# ============================================================
# Step 2: Latest month dominant factor + trends + strategy
# ============================================================
def analyze_latest(ff5: pd.DataFrame) -> Dict:
    if ff5.empty:
        return {"error": "ff5 is empty"}

    # latest evaluable month (MKT notna)
    last_me = latest_evaluable_month(ff5, col="MKT")
    cur = ff5[ff5["MonthEnd"] == last_me].iloc[0].to_dict()

    # dominant factor by abs value (excluding n/conf fields)
    factor_cols = ["MKT","SMB","HML","RMW","CMA"]
    absvals = {c: abs(cur.get(c, np.nan)) for c in factor_cols}
    dominant = max(absvals, key=lambda k: (-np.nan_to_num(absvals[k], nan=-1), k))

    # last3/last12 trends
    ff5_sorted = ff5.sort_values("MonthEnd").reset_index(drop=True)
    last3 = ff5_sorted.tail(3)
    last12 = ff5_sorted.tail(12)

    trends3 = {c: trend_label(last3[c].to_numpy()) for c in factor_cols}
    trends12 = {c: trend_label(last12[c].to_numpy()) for c in factor_cols}

    avg3 = {f"{c}_avg3": float(np.nanmean(last3[c])) for c in factor_cols}
    avg12 = {f"{c}_avg12": float(np.nanmean(last12[c])) for c in factor_cols}

    # market regime (simple heuristic using MKT + dispersion)
    mkt = cur["MKT"]
    regime = "RANGE"
    if np.isfinite(mkt):
        if mkt > 0.01:
            regime = "BULL"
        elif mkt < -0.01:
            regime = "BEAR"

    # recommended tilts (simple mapping)
    # If dominant is SMB -> tilt small; HML -> value; RMW -> quality; CMA -> conservative; MKT -> beta
    tilt = {c: 0.0 for c in factor_cols}
    if dominant in tilt:
        tilt[dominant] = 1.0

    # confidence-aware tilt
    conf = float(cur.get("confidence", 0.0))
    tilt_conf = {k: v * conf for k, v in tilt.items()}

    out = {
        "latest_month": str(pd.Timestamp(last_me).date()),
        "dominant_factor": dominant,
        "latest": cur,
        "trends_last3": trends3,
        "trends_last12": trends12,
        "avg_last3": avg3,
        "avg_last12": avg12,
        "regime": regime,
        "tilt": tilt,
        "tilt_conf_weighted": tilt_conf,
    }
    return out


def save_outputs(ff5: pd.DataFrame, diag: pd.DataFrame, analysis: Dict):
    FACTORS_DIR.mkdir(parents=True, exist_ok=True)

    # parquet
    ff5.to_parquet(FF5_PARQUET_PATH, engine="pyarrow", index=False)

    # diagnostics
    diag.to_csv(DIAG_RET_DIST_CSV, index=False, encoding="utf-8-sig")
    ff5[["MonthEnd","n_total","n_used_val","n_used_prof","n_used_inv","confidence","abnormal"]].to_csv(
        DIAG_NUSED_BY_MONTH, index=False, encoding="utf-8-sig"
    )

    # summary tables
    latest = analysis.get("latest", {})
    factor_cols = ["MKT","SMB","HML","RMW","CMA"]

    # summary (1 row)
    summary = {
        "latest_month": analysis.get("latest_month", ""),
        "regime": analysis.get("regime", ""),
        "dominant_factor": analysis.get("dominant_factor", ""),
        "confidence": float(latest.get("confidence", np.nan)),
        "abnormal": bool(latest.get("abnormal", True)),
        "n_total": int(latest.get("n_total", 0) or 0),
        "n_used_val": int(latest.get("n_used_val", 0) or 0),
        "n_used_prof": int(latest.get("n_used_prof", 0) or 0),
        "n_used_inv": int(latest.get("n_used_inv", 0) or 0),
    }
    for c in factor_cols:
        summary[c] = float(latest.get(c, np.nan))

    pd.DataFrame([summary]).to_csv(CURRENT_ANALYSIS_CSV, index=False, encoding="utf-8-sig")

    # ranks
    abs_rank = pd.DataFrame([{
        "factor": c,
        "value": float(latest.get(c, np.nan)),
        "abs_value": abs(float(latest.get(c, np.nan))) if np.isfinite(latest.get(c, np.nan)) else np.nan
    } for c in factor_cols]).sort_values("abs_value", ascending=False)
    abs_rank.to_csv(RANK_ABS_CSV, index=False, encoding="utf-8-sig")

    conf = float(latest.get("confidence", 0.0))
    confw_rank = abs_rank.copy()
    confw_rank["conf_weighted_abs"] = confw_rank["abs_value"] * conf
    confw_rank.to_csv(RANK_CONFW_CSV, index=False, encoding="utf-8-sig")

    # trends
    # last3/last12 tables
    ff5_sorted = ff5.sort_values("MonthEnd").reset_index(drop=True)
    last3 = ff5_sorted.tail(3)[["MonthEnd"] + factor_cols].copy()
    last12 = ff5_sorted.tail(12)[["MonthEnd"] + factor_cols].copy()
    last3.to_csv(TREND_LAST3_CSV, index=False, encoding="utf-8-sig")
    last12.to_csv(TREND_LAST12_CSV, index=False, encoding="utf-8-sig")


def main():
    print("="*120)
    print("FF5月次計算 → 直近評価（月次リターンが存在する最終月）→ トレンド → 戦略示唆")
    print("="*120)
    print(f"- PRICE_PATH: {PRICE_PATH}")
    print(f"- SNAP_PATH : {SNAP_PATH}")

    snap = load_snapshot_and_build_forward_returns(PRICE_PATH, SNAP_PATH)
    print(f"[LOAD] snap rows(after basic filters) = {len(snap):,}, codes={snap['Code'].nunique():,}, months={snap['MonthEnd'].nunique():,}")

    ff5, diag = compute_ff5_monthly(snap)
    print(f"[FF5] months computed = {len(ff5):,}")
    if ff5.empty:
        raise RuntimeError("FF5 result is empty. Check n_used thresholds / data coverage.")

    analysis = analyze_latest(ff5)

    # print key outputs
    last_me = analysis.get("latest_month", "")
    dom = analysis.get("dominant_factor", "")
    latest = analysis.get("latest", {})
    print("-"*120)
    print(f"[LATEST EVALUABLE MONTH] {last_me}")
    print(f"[DOMINANT FACTOR] {dom}")
    print(f"[REGIME] {analysis.get('regime')}")
    print(f"[CONFIDENCE] {latest.get('confidence'):.3f} | abnormal={latest.get('abnormal')} | "
          f"n_total={latest.get('n_total')} used(val/prof/inv)={latest.get('n_used_val')}/{latest.get('n_used_prof')}/{latest.get('n_used_inv')}")
    print("Factor returns:")
    for c in ["MKT","SMB","HML","RMW","CMA"]:
        print(f"  {c}: {latest.get(c)}")

    save_outputs(ff5, diag, analysis)
    print("-"*120)
    print("✅ saved:")
    print(f"  - {FF5_PARQUET_PATH}")
    print(f"  - {CURRENT_ANALYSIS_CSV}")
    print(f"  - {RANK_ABS_CSV}")
    print(f"  - {RANK_CONFW_CSV}")
    print(f"  - {TREND_LAST3_CSV}")
    print(f"  - {TREND_LAST12_CSV}")
    print(f"  - {DIAG_RET_DIST_CSV}")
    print(f"  - {DIAG_NUSED_BY_MONTH}")


if __name__ == "__main__":
    main()


FF5月次計算 → 直近評価（月次リターンが存在する最終月）→ トレンド → 戦略示唆
- PRICE_PATH: C:\Users\yongr\Project\merged_data_all_stocks\factors\price_month_end.parquet
- SNAP_PATH : C:\Users\yongr\Project\merged_data_all_stocks\factors\month_end_snapshot.parquet
[LOAD] snap rows(after basic filters) = 43,340, codes=3,929, months=121
[FF5] months computed = 11
------------------------------------------------------------------------------------------------------------------------
[LATEST EVALUABLE MONTH] 2025-12-31
[DOMINANT FACTOR] HML
[REGIME] BULL
[CONFIDENCE] 1.000 | abnormal=False | n_total=3597 used(val/prof/inv)=3597/3592/3361
Factor returns:
  MKT: 0.058203414293272845
  SMB: -0.013373316268626893
  HML: 0.012826770706282083
  RMW: -0.019694675094994403
  CMA: 0.01890257041128761
------------------------------------------------------------------------------------------------------------------------
✅ saved:
  - C:\Users\yongr\Project\merged_data_all_stocks\factors\ff5_factors_monthly.parquet
  - C:\Users\yongr\

In [66]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
from typing import Dict, Tuple
import numpy as np
import pandas as pd


# ============================================================
# Paths (あなたの環境)
# ============================================================
FACTORS_DIR = Path(r"C:\Users\yongr\Project\merged_data_all_stocks\factors")

PRICE_PATH = FACTORS_DIR / "price_month_end.parquet"
SNAP_PATH  = FACTORS_DIR / "month_end_snapshot.parquet"

OUT_PARQUET = FACTORS_DIR / "ff5_mom_factors_monthly.parquet"

OUT_SUMMARY = FACTORS_DIR / "market_factor_summary_ff5_mom.csv"
OUT_RANK_ABS = FACTORS_DIR / "market_factor_rank_abs_ff5_mom.csv"
OUT_RANK_CONFW = FACTORS_DIR / "market_factor_rank_conf_weighted_ff5_mom.csv"
OUT_TREND_LAST3 = FACTORS_DIR / "market_factor_trend_last3_ff5_mom.csv"
OUT_TREND_LAST12 = FACTORS_DIR / "market_factor_trend_last12_ff5_mom.csv"

# diagnostics
DIAG_RET_DIST = FACTORS_DIR / "diag_monthly_stock_return_distribution_ff5_mom.csv"
DIAG_NUSED = FACTORS_DIR / "diag_ff5_mom_n_used_by_month.csv"
DIAG_MOM_COVERAGE = FACTORS_DIR / "diag_mom_coverage_by_month.csv"

# ============================================================
# Params
# ============================================================
MIN_STOCKS_PER_MONTH = 500        # abnormal判定の下限
WINSORIZE_RET = True
RET_WINSOR_Q = (0.01, 0.99)

BM_WINSOR_Q  = (0.01, 0.99)
ROE_WINSOR_Q = (0.01, 0.99)
INV_WINSOR_Q = (0.01, 0.99)
MOM_WINSOR_Q = (0.01, 0.99)

# 2x3 split
P30 = 0.30
P70 = 0.70

BAD_CODE_STRINGS = {"None", "nan", "", "NaN", "NULL", "null"}


# ============================================================
# Utils
# ============================================================
def force_dt64ns(x):
    dt = pd.to_datetime(x, errors="coerce")
    if hasattr(dt, "astype"):
        return dt.astype("datetime64[ns]")
    if pd.isna(dt):
        return np.datetime64("NaT", "ns")
    return pd.Timestamp(dt).to_datetime64()

def safe_num(s):
    return pd.to_numeric(s, errors="coerce")

def winsorize(s: pd.Series, q_lo=0.01, q_hi=0.99):
    x = pd.to_numeric(s, errors="coerce")
    if x.notna().sum() == 0:
        return x
    lo = x.quantile(q_lo)
    hi = x.quantile(q_hi)
    return x.clip(lo, hi)

def value_weighted_return(df: pd.DataFrame, ret_col: str, w_col: str) -> float:
    x = df[[ret_col, w_col]].dropna()
    if x.empty:
        return np.nan
    w = x[w_col].astype(float).to_numpy()
    r = x[ret_col].astype(float).to_numpy()
    wsum = w.sum()
    if not np.isfinite(wsum) or wsum <= 0:
        return np.nan
    return float(np.dot(r, w) / wsum)

def qcut_3way(x: pd.Series, p30=0.3, p70=0.7, labels=("L","M","H")) -> pd.Series:
    a = pd.to_numeric(x, errors="coerce")
    q1 = a.quantile(p30)
    q2 = a.quantile(p70)
    out = pd.Series(index=a.index, dtype="object")
    out[a <= q1] = labels[0]
    out[(a > q1) & (a < q2)] = labels[1]
    out[a >= q2] = labels[2]
    return out

def size_split_median(mcap: pd.Series) -> pd.Series:
    a = pd.to_numeric(mcap, errors="coerce")
    med = a.quantile(0.5)
    out = pd.Series(index=a.index, dtype="object")
    out[a <= med] = "S"
    out[a > med] = "B"
    return out

def confidence_from_n(n_used: int) -> float:
    # 500で0.5、1000で1.0
    return float(max(0.0, min(1.0, n_used / 1000.0)))

def trend_label(vals: np.ndarray) -> str:
    vals = np.array(vals, dtype=float)
    vals = vals[np.isfinite(vals)]
    if len(vals) < 2:
        return "NA"
    x = np.arange(len(vals))
    slope = np.polyfit(x, vals, 1)[0]
    if abs(slope) < 1e-6:
        return "FLAT"
    return "UP" if slope > 0 else "DOWN"

def latest_evaluable_month(factors: pd.DataFrame, col="MKT") -> pd.Timestamp:
    f = factors.dropna(subset=[col]).copy()
    if f.empty:
        return pd.NaT
    return pd.Timestamp(f["MonthEnd"].max())


# ============================================================
# Step 0: Load snapshot + build forward returns + MOM(12-1)
# ============================================================
def load_data_and_build_features(price_path: Path, snap_path: Path) -> pd.DataFrame:
    price = pd.read_parquet(price_path).copy()
    snap = pd.read_parquet(snap_path).copy()

    # sanitize
    for df in (price, snap):
        df["Code"] = df["Code"].astype(str)
        df = df[~df["Code"].isin(BAD_CODE_STRINGS)]

    price["MonthEnd"] = force_dt64ns(price["MonthEnd"])
    snap["MonthEnd"]  = force_dt64ns(snap["MonthEnd"])

    price["AdjustedClose"] = safe_num(price["AdjustedClose"])
    price = price.dropna(subset=["Code","MonthEnd","AdjustedClose"]).copy()
    price = price.sort_values(["Code","MonthEnd"], kind="mergesort")

    # --------------------------------------------------------
    # Forward return (t -> t+1)
    # --------------------------------------------------------
    price["Adj_next"] = price.groupby("Code")["AdjustedClose"].shift(-1)
    price["ret_m_fwd"] = (price["Adj_next"] / price["AdjustedClose"]) - 1.0

    # --------------------------------------------------------
    # Momentum 12-1:
    # MOM_12_1(t) = P_{t-1}/P_{t-12} - 1
    # where P_{t-1} = lag1, P_{t-12} = lag12
    # --------------------------------------------------------
    price["Adj_lag1"] = price.groupby("Code")["AdjustedClose"].shift(1)
    price["Adj_lag12"] = price.groupby("Code")["AdjustedClose"].shift(12)
    price["MOM_12_1"] = (price["Adj_lag1"] / price["Adj_lag12"]) - 1.0

    pr = price[["Code","MonthEnd","ret_m_fwd","MOM_12_1"]].copy()

    # join to snapshot by (Code, MonthEnd)
    # snapshotには MarketCap/BM/ROE/INV が入っている前提
    snap = snap.merge(pr, on=["Code","MonthEnd"], how="left", validate="many_to_one")

    # numeric cleanup
    for c in ["MarketCap","BM_Ratio","ROE","INV_Growth","ret_m_fwd","MOM_12_1"]:
        if c in snap.columns:
            snap[c] = safe_num(snap[c])

    # Basic filters (marketcap + bm)
    snap = snap[(snap["MarketCap"].notna()) & (snap["MarketCap"] > 0)].copy()
    snap = snap[(snap["BM_Ratio"].notna()) & np.isfinite(snap["BM_Ratio"])].copy()

    # winsorize
    snap["ret_m_fwd_w"] = winsorize(snap["ret_m_fwd"], *RET_WINSOR_Q) if WINSORIZE_RET else snap["ret_m_fwd"]
    snap["BM_w"]  = winsorize(snap["BM_Ratio"], *BM_WINSOR_Q)
    snap["ROE_w"] = winsorize(snap["ROE"], *ROE_WINSOR_Q)
    snap["INV_w"] = winsorize(snap["INV_Growth"], *INV_WINSOR_Q)
    snap["MOM_w"] = winsorize(snap["MOM_12_1"], *MOM_WINSOR_Q)

    # MOM coverage diag by month (not needed for factor calc but useful)
    cov = (snap.groupby("MonthEnd")
              .agg(n=("Code","nunique"),
                   mom_notna=("MOM_12_1", lambda s: float(s.notna().mean()*100)),
                   ret_notna=("ret_m_fwd", lambda s: float(s.notna().mean()*100)))
              .reset_index())
    cov.to_csv(DIAG_MOM_COVERAGE, index=False, encoding="utf-8-sig")

    return snap


# ============================================================
# Step 1: Compute FF5 + MOM (WML)
# ============================================================
def compute_ff5_mom_monthly(snap: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    months = sorted(snap["MonthEnd"].dropna().unique())
    rows = []
    diag_ret_rows = []
    diag_n_rows = []

    for me in months:
        m = snap[snap["MonthEnd"] == me].copy()

        # Need forward return for factor payoff
        m = m[m["ret_m_fwd_w"].notna()].copy()
        n_total = int(m["Code"].nunique())
        if n_total == 0:
            continue

        # Market return
        MKT = value_weighted_return(m, "ret_m_fwd_w", "MarketCap")

        # common splits
        m["SZ"] = size_split_median(m["MarketCap"])
        m["VAL"] = qcut_3way(m["BM_w"], P30, P70, labels=("L","M","H"))
        m["PROF"] = qcut_3way(m["ROE_w"], P30, P70, labels=("W","M","R"))
        m["INV"] = qcut_3way(m["INV_w"], P30, P70, labels=("C","M","A"))
        m["MOM"] = qcut_3way(m["MOM_w"], P30, P70, labels=("L","M","W"))  # Loser/Mid/Winner

        def port_ret(sz, col, lab):
            x = m[(m["SZ"] == sz) & (m[col] == lab)]
            return value_weighted_return(x, "ret_m_fwd_w", "MarketCap")

        # --- HML ---
        SH = port_ret("S","VAL","H"); SMv = port_ret("S","VAL","M"); SL = port_ret("S","VAL","L")
        BH = port_ret("B","VAL","H"); BMv = port_ret("B","VAL","M"); BL = port_ret("B","VAL","L")
        HML = np.nanmean([SH, BH]) - np.nanmean([SL, BL])

        # --- RMW ---
        SR = port_ret("S","PROF","R"); SMp = port_ret("S","PROF","M"); SW = port_ret("S","PROF","W")
        BR = port_ret("B","PROF","R"); BMp = port_ret("B","PROF","M"); BW = port_ret("B","PROF","W")
        RMW = np.nanmean([SR, BR]) - np.nanmean([SW, BW])

        # --- CMA ---
        SC = port_ret("S","INV","C"); SMa = port_ret("S","INV","M"); SA = port_ret("S","INV","A")
        BC = port_ret("B","INV","C"); BMa = port_ret("B","INV","M"); BA = port_ret("B","INV","A")
        CMA = np.nanmean([SC, BC]) - np.nanmean([SA, BA])

        # --- SMB (FF convention) ---
        SMB_HML = np.nanmean([SH, SMv, SL]) - np.nanmean([BH, BMv, BL])
        SMB_RMW = np.nanmean([SR, SMp, SW]) - np.nanmean([BR, BMp, BW])
        SMB_CMA = np.nanmean([SC, SMa, SA]) - np.nanmean([BC, BMa, BA])
        SMB = np.nanmean([SMB_HML, SMB_RMW, SMB_CMA])

        # --- WML (Momentum) ---
        # Winner - Loser
        SWin = port_ret("S","MOM","W"); SMomM = port_ret("S","MOM","M"); SLos = port_ret("S","MOM","L")
        BWin = port_ret("B","MOM","W"); BMomM = port_ret("B","MOM","M"); BLos = port_ret("B","MOM","L")
        WML = np.nanmean([SWin, BWin]) - np.nanmean([SLos, BLos])

        # n_used by factor (how many stocks have required fields)
        n_used_val  = int(m.dropna(subset=["SZ","VAL","MarketCap","ret_m_fwd_w"])["Code"].nunique())
        n_used_prof = int(m.dropna(subset=["SZ","PROF","MarketCap","ret_m_fwd_w"])["Code"].nunique())
        n_used_inv  = int(m.dropna(subset=["SZ","INV","MarketCap","ret_m_fwd_w"])["Code"].nunique())
        n_used_mom  = int(m.dropna(subset=["SZ","MOM","MarketCap","ret_m_fwd_w"])["Code"].nunique())

        n_used_all = min(n_used_val, n_used_prof, n_used_inv, n_used_mom)
        conf = confidence_from_n(n_used_all)
        abnormal = (n_used_all < MIN_STOCKS_PER_MONTH)

        rows.append({
            "MonthEnd": me,
            "MKT": MKT,
            "SMB": SMB,
            "HML": HML,
            "RMW": RMW,
            "CMA": CMA,
            "WML": WML,
            "n_total": n_total,
            "n_used_val": n_used_val,
            "n_used_prof": n_used_prof,
            "n_used_inv": n_used_inv,
            "n_used_mom": n_used_mom,
            "confidence": conf,
            "abnormal": bool(abnormal),
        })

        # return distribution diag
        r = m["ret_m_fwd_w"]
        diag_ret_rows.append({
            "MonthEnd": me,
            "n": int(r.notna().sum()),
            "ret_q01": float(r.quantile(0.01)),
            "ret_q05": float(r.quantile(0.05)),
            "ret_q50": float(r.quantile(0.50)),
            "ret_q95": float(r.quantile(0.95)),
            "ret_q99": float(r.quantile(0.99)),
            "ret_min": float(r.min()),
            "ret_max": float(r.max()),
            "share_abs_gt_50pct": float((r.abs() > 0.5).mean() * 100.0),
            "share_abs_gt_100pct": float((r.abs() > 1.0).mean() * 100.0),
        })

        diag_n_rows.append({
            "MonthEnd": me,
            "n_total": n_total,
            "n_used_val": n_used_val,
            "n_used_prof": n_used_prof,
            "n_used_inv": n_used_inv,
            "n_used_mom": n_used_mom,
            "confidence": conf,
            "abnormal": bool(abnormal),
        })

    ff = pd.DataFrame(rows).sort_values("MonthEnd").reset_index(drop=True)
    diag_ret = pd.DataFrame(diag_ret_rows).sort_values("MonthEnd").reset_index(drop=True)
    diag_n = pd.DataFrame(diag_n_rows).sort_values("MonthEnd").reset_index(drop=True)
    return ff, diag_ret, diag_n


# ============================================================
# Step 2: Latest dominant + trends + strategy
# ============================================================
def analyze_latest(ff: pd.DataFrame) -> Dict:
    if ff.empty:
        return {"error": "factor table is empty"}

    last_me = latest_evaluable_month(ff, col="MKT")
    cur = ff[ff["MonthEnd"] == last_me].iloc[0].to_dict()

    factor_cols = ["MKT","SMB","HML","RMW","CMA","WML"]
    absvals = {c: abs(cur.get(c, np.nan)) for c in factor_cols}
    dominant = max(absvals, key=lambda k: (-np.nan_to_num(absvals[k], nan=-1), k))

    ff_sorted = ff.sort_values("MonthEnd").reset_index(drop=True)
    last3 = ff_sorted.tail(3)
    last12 = ff_sorted.tail(12)

    trends3 = {c: trend_label(last3[c].to_numpy()) for c in factor_cols}
    trends12 = {c: trend_label(last12[c].to_numpy()) for c in factor_cols}

    avg3 = {f"{c}_avg3": float(np.nanmean(last3[c])) for c in factor_cols}
    avg12 = {f"{c}_avg12": float(np.nanmean(last12[c])) for c in factor_cols}

    # regime (simple)
    mkt = cur.get("MKT", np.nan)
    regime = "RANGE"
    if np.isfinite(mkt):
        if mkt > 0.01:
            regime = "BULL"
        elif mkt < -0.01:
            regime = "BEAR"

    # suggested tilts (directional)
    # + means "tilt long" exposure to that factor, - means avoid/underweight
    # We'll base on last3 averages (more stable than single month), confidence-adjusted.
    conf = float(cur.get("confidence", 0.0))
    tilt_raw = {}
    for c in factor_cols:
        v = float(avg3.get(f"{c}_avg3", np.nan))
        if not np.isfinite(v):
            tilt_raw[c] = 0.0
        else:
            tilt_raw[c] = float(np.sign(v))  # +1 / -1
    tilt_conf = {k: v * conf for k, v in tilt_raw.items()}

    return {
        "latest_month": str(pd.Timestamp(last_me).date()),
        "dominant_factor": dominant,
        "latest": cur,
        "regime": regime,
        "trends_last3": trends3,
        "trends_last12": trends12,
        "avg_last3": avg3,
        "avg_last12": avg12,
        "tilt": tilt_raw,
        "tilt_conf_weighted": tilt_conf,
    }


def save_outputs(ff: pd.DataFrame, diag_ret: pd.DataFrame, diag_n: pd.DataFrame, analysis: Dict):
    FACTORS_DIR.mkdir(parents=True, exist_ok=True)

    ff.to_parquet(OUT_PARQUET, engine="pyarrow", index=False)

    diag_ret.to_csv(DIAG_RET_DIST, index=False, encoding="utf-8-sig")
    diag_n.to_csv(DIAG_NUSED, index=False, encoding="utf-8-sig")

    factor_cols = ["MKT","SMB","HML","RMW","CMA","WML"]
    latest = analysis.get("latest", {})

    summary = {
        "latest_month": analysis.get("latest_month", ""),
        "regime": analysis.get("regime", ""),
        "dominant_factor": analysis.get("dominant_factor", ""),
        "confidence": float(latest.get("confidence", np.nan)),
        "abnormal": bool(latest.get("abnormal", True)),
        "n_total": int(latest.get("n_total", 0) or 0),
        "n_used_val": int(latest.get("n_used_val", 0) or 0),
        "n_used_prof": int(latest.get("n_used_prof", 0) or 0),
        "n_used_inv": int(latest.get("n_used_inv", 0) or 0),
        "n_used_mom": int(latest.get("n_used_mom", 0) or 0),
    }
    for c in factor_cols:
        summary[c] = float(latest.get(c, np.nan))
    pd.DataFrame([summary]).to_csv(OUT_SUMMARY, index=False, encoding="utf-8-sig")

    # ranking by abs
    abs_rank = pd.DataFrame([{
        "factor": c,
        "value": float(latest.get(c, np.nan)),
        "abs_value": abs(float(latest.get(c, np.nan))) if np.isfinite(latest.get(c, np.nan)) else np.nan
    } for c in factor_cols]).sort_values("abs_value", ascending=False)
    abs_rank.to_csv(OUT_RANK_ABS, index=False, encoding="utf-8-sig")

    conf = float(latest.get("confidence", 0.0))
    confw_rank = abs_rank.copy()
    confw_rank["conf_weighted_abs"] = confw_rank["abs_value"] * conf
    confw_rank.to_csv(OUT_RANK_CONFW, index=False, encoding="utf-8-sig")

    # trend tables
    ff_sorted = ff.sort_values("MonthEnd").reset_index(drop=True)
    ff_sorted.tail(3)[["MonthEnd"]+factor_cols].to_csv(OUT_TREND_LAST3, index=False, encoding="utf-8-sig")
    ff_sorted.tail(12)[["MonthEnd"]+factor_cols].to_csv(OUT_TREND_LAST12, index=False, encoding="utf-8-sig")


def main():
    print("="*120)
    print("FF5 + MOM(12-1) 月次因子計算 → 直近評価 → 3ヶ月/12ヶ月トレンド → 戦略示唆")
    print("="*120)
    print(f"- PRICE_PATH: {PRICE_PATH}")
    print(f"- SNAP_PATH : {SNAP_PATH}")

    snap = load_data_and_build_features(PRICE_PATH, SNAP_PATH)
    print(f"[LOAD] snap rows(after filters) = {len(snap):,}, codes={snap['Code'].nunique():,}, months={snap['MonthEnd'].nunique():,}")

    ff, diag_ret, diag_n = compute_ff5_mom_monthly(snap)
    print(f"[FACTORS] months computed = {len(ff):,}")
    if ff.empty:
        raise RuntimeError("Factor result is empty. Check coverage / thresholds.")

    analysis = analyze_latest(ff)

    # console output (minimum)
    last_me = analysis.get("latest_month", "")
    dom = analysis.get("dominant_factor", "")
    latest = analysis.get("latest", {})
    print("-"*120)
    print(f"[LATEST EVALUABLE MONTH] {last_me}")
    print(f"[DOMINANT FACTOR] {dom}")
    print(f"[REGIME] {analysis.get('regime')}")
    print(f"[CONFIDENCE] {latest.get('confidence'):.3f} | abnormal={latest.get('abnormal')} | "
          f"n_total={latest.get('n_total')} used(val/prof/inv/mom)={latest.get('n_used_val')}/{latest.get('n_used_prof')}/{latest.get('n_used_inv')}/{latest.get('n_used_mom')}")
    print("Factor returns:")
    for c in ["MKT","SMB","HML","RMW","CMA","WML"]:
        print(f"  {c}: {latest.get(c)}")

    # strategy note (simple)
    print("-"*120)
    print("[STRATEGY HINT] (based on last3 avg direction, confidence-weighted)")
    for c, sign in analysis.get("tilt_conf_weighted", {}).items():
        if sign > 0:
            s = "OVERWEIGHT (long tilt)"
        elif sign < 0:
            s = "UNDERWEIGHT (avoid tilt)"
        else:
            s = "NEUTRAL"
        print(f"  {c}: {s}  (score={sign:.2f})")

    save_outputs(ff, diag_ret, diag_n, analysis)
    print("-"*120)
    print("✅ saved:")
    print(f"  - {OUT_PARQUET}")
    print(f"  - {OUT_SUMMARY}")
    print(f"  - {OUT_RANK_ABS}")
    print(f"  - {OUT_RANK_CONFW}")
    print(f"  - {OUT_TREND_LAST3}")
    print(f"  - {OUT_TREND_LAST12}")
    print(f"  - {DIAG_RET_DIST}")
    print(f"  - {DIAG_NUSED}")
    print(f"  - {DIAG_MOM_COVERAGE}")


if __name__ == "__main__":
    main()


FF5 + MOM(12-1) 月次因子計算 → 直近評価 → 3ヶ月/12ヶ月トレンド → 戦略示唆
- PRICE_PATH: C:\Users\yongr\Project\merged_data_all_stocks\factors\price_month_end.parquet
- SNAP_PATH : C:\Users\yongr\Project\merged_data_all_stocks\factors\month_end_snapshot.parquet
[LOAD] snap rows(after filters) = 43,340, codes=3,929, months=121
[FACTORS] months computed = 11
------------------------------------------------------------------------------------------------------------------------
[LATEST EVALUABLE MONTH] 2025-12-31
[DOMINANT FACTOR] HML
[REGIME] BULL
[CONFIDENCE] 1.000 | abnormal=False | n_total=3597 used(val/prof/inv/mom)=3597/3592/3361/3553
Factor returns:
  MKT: 0.058203414293272845
  SMB: -0.013373316268626893
  HML: 0.012826770706282083
  RMW: -0.019694675094994403
  CMA: 0.01890257041128761
  WML: 0.025980346280246036
------------------------------------------------------------------------------------------------------------------------
[STRATEGY HINT] (based on last3 avg direction, confidence-weighted)
  M

In [67]:
import numpy as np
import pandas as pd
from pathlib import Path

FACTORS_DIR = Path(r"C:\Users\yongr\Project\merged_data_all_stocks\factors")
FF_PATH = FACTORS_DIR / "ff5_mom_factors_monthly.parquet"

BT_PATH = Path("backtest_results_october_unit.csv")  # あなたの出力

OUT_REG = FACTORS_DIR / "eval_value_quality_ff5mom_regression.csv"
OUT_CONTRIB = FACTORS_DIR / "eval_value_quality_ff5mom_contribution_by_year.csv"

FACTOR_COLS = ["MKT","SMB","HML","RMW","CMA","WML"]

def to_monthend(ts):
    x = pd.to_datetime(ts, errors="coerce")
    return x.dt.to_period("M").dt.to_timestamp("M")

def compound_return(x):
    x = pd.to_numeric(x, errors="coerce").dropna()
    if x.empty:
        return np.nan
    return float((1.0 + x).prod() - 1.0)

def main():
    # --- load backtest ---
    bt = pd.read_csv(BT_PATH, encoding="utf-8-sig")
    bt["date"] = pd.to_datetime(bt["date"], errors="coerce")  # このdateは「次のリバランス日（10/1）」想定
    bt = bt.dropna(subset=["date"]).copy()
    bt = bt.sort_values("date").reset_index(drop=True)

    # strategy return (税引前/後どちらで回帰するか選べる)
    # まずは税引前（gross）で評価するのが因子との整合が良い
    bt["R_strat"] = pd.to_numeric(bt["strategy_return_gross"], errors="coerce")

    # 期間ラベル（2016-10→2017-10）
    bt["start_date"] = bt["date"] - pd.DateOffset(years=1)
    bt["label"] = bt["start_date"].dt.year.astype(str) + "-10 to " + bt["date"].dt.year.astype(str) + "-10"

    # --- load factors monthly ---
    ff = pd.read_parquet(FF_PATH).copy()
    ff["MonthEnd"] = pd.to_datetime(ff["MonthEnd"], errors="coerce")
    ff = ff.sort_values("MonthEnd").reset_index(drop=True)

    # 年次区間ごとに因子を合成（10/1→翌10/1）
    rows = []
    for _, r in bt.iterrows():
        start = pd.Timestamp(r["start_date"])
        end = pd.Timestamp(r["date"])

        # 対象月：startの当月末～endの前月末まで（10/1→翌10/1なら、2016-10末〜2017-09末の12ヶ月）
        start_me = (start.to_period("M").to_timestamp("M"))
        end_me = ((end - pd.DateOffset(days=1)).to_period("M").to_timestamp("M"))

        sub = ff[(ff["MonthEnd"] >= start_me) & (ff["MonthEnd"] <= end_me)].copy()
        if len(sub) < 6:
            # 月数が少なすぎる場合はスキップ
            continue

        fac_ann = {f: compound_return(sub[f]) for f in FACTOR_COLS}
        fac_ann["n_months"] = int(len(sub))
        fac_ann["label"] = r["label"]
        fac_ann["end_date"] = end
        fac_ann["R_strat"] = float(r["R_strat"]) if pd.notna(r["R_strat"]) else np.nan
        fac_ann["topix_return"] = float(r.get("topix_return", np.nan))
        fac_ann["investment_ratio"] = float(r.get("investment_ratio", np.nan))

        rows.append(fac_ann)

    ann = pd.DataFrame(rows).dropna(subset=["R_strat"]).copy()
    if ann.empty:
        raise RuntimeError("Annual alignment produced empty dataset. Check dates and factor availability.")

    # --- regression (OLS) ---
    # y = alpha + sum(beta_i * factor_i)
    Y = ann["R_strat"].to_numpy(dtype=float)
    X = ann[FACTOR_COLS].to_numpy(dtype=float)
    X = np.column_stack([np.ones(len(X)), X])  # intercept
    cols = ["alpha"] + FACTOR_COLS

    # solve (beta) via least squares
    beta, *_ = np.linalg.lstsq(X, Y, rcond=None)
    y_hat = X @ beta
    resid = Y - y_hat

    # R^2
    ss_res = np.sum(resid**2)
    ss_tot = np.sum((Y - Y.mean())**2) if len(Y) > 1 else np.nan
    r2 = 1 - ss_res/ss_tot if (ss_tot is not None and ss_tot > 0) else np.nan

    reg = pd.DataFrame([{
        "n_years": len(ann),
        "R2": r2,
        **{cols[i]: float(beta[i]) for i in range(len(cols))}
    }])
    reg.to_csv(OUT_REG, index=False, encoding="utf-8-sig")

    # --- contribution by year ---
    # contribution_i = beta_i * factor_i (alpha separately)
    contrib = ann[["label","end_date","R_strat","topix_return","investment_ratio"]].copy()
    contrib["alpha_contrib"] = float(beta[0])
    for i, f in enumerate(FACTOR_COLS, start=1):
        contrib[f"contrib_{f}"] = ann[f].astype(float) * float(beta[i])
    contrib["predicted"] = contrib["alpha_contrib"] + contrib[[f"contrib_{f}" for f in FACTOR_COLS]].sum(axis=1)
    contrib["residual"] = contrib["R_strat"] - contrib["predicted"]
    contrib.to_csv(OUT_CONTRIB, index=False, encoding="utf-8-sig")

    print("✅ saved:")
    print(" -", OUT_REG)
    print(" -", OUT_CONTRIB)
    print("\nRegression summary:")
    print(reg.to_string(index=False))
    print("\nContribution (head):")
    print(contrib.head(5).to_string(index=False))

if __name__ == "__main__":
    main()


✅ saved:
 - C:\Users\yongr\Project\merged_data_all_stocks\factors\eval_value_quality_ff5mom_regression.csv
 - C:\Users\yongr\Project\merged_data_all_stocks\factors\eval_value_quality_ff5mom_contribution_by_year.csv

Regression summary:
 n_years  R2    alpha      MKT       SMB      HML      RMW       CMA      WML
       1 NaN 0.425328 0.086788 -0.018201 0.038708 0.046854 -0.012444 0.016352

Contribution (head):
             label   end_date  R_strat  topix_return  investment_ratio  alpha_contrib  contrib_MKT  contrib_SMB  contrib_HML  contrib_RMW  contrib_CMA  contrib_WML  predicted  residual
2024-10 to 2025-10 2025-10-01 0.453493      0.136349          0.868307       0.425328     0.017709     0.000779     0.003523     0.005161     0.000364     0.000629   0.453493       0.0


In [77]:
# -*- coding: utf-8 -*-
"""
FULL PIPELINE (Conservative Range):
- Rebuild month_end_snapshot.parquet for 2015-10 .. 2025-09 (overwrite)
- Recompute FF5+MOM monthly factors for 2016-10 .. 2025-09 (save)
- Evaluate Oct-1 annual rebalance strategy (NET returns) from 2017-10 .. 2025-10
  via FF5+MOM regression using compounded monthly factors over each Oct-year.

Data sources (LOCAL ONLY):
- price_month_end.parquet
- fins_norm daily parquet directory (daily_parquet_norm)
- backtest_results_october_unit.csv

Outputs (under FACTORS_DIR):
- month_end_snapshot.parquet (OVERWRITE)
- ff5_mom_factors_monthly.parquet
- eval_value_quality_ff5mom_regression_net.csv
- eval_value_quality_ff5mom_contribution_by_year_net.csv
- diag_* CSVs

Run:
  python full_rebuild_snapshot_and_eval_october_ff5mom_net.py
"""

from __future__ import annotations

from pathlib import Path
import re
import numpy as np
import pandas as pd


# ===================== PATHS (keep your structure) =====================
FACTORS_DIR = Path(r"C:\Users\yongr\Project\merged_data_all_stocks\factors")
PRICE_PATH = FACTORS_DIR / "price_month_end.parquet"
SNAP_PATH  = FACTORS_DIR / "month_end_snapshot.parquet"  # overwrite
DIAG_DIR   = FACTORS_DIR / "diag_rebuild_eval_ff5mom"
DIAG_DIR.mkdir(parents=True, exist_ok=True)

FINS_NORM_DIR = Path(r"C:\Users\yongr\Project\jquants_fins_summary_10y_parquet\daily_parquet_norm")

BACKTEST_CSV = Path(r"backtest_results_october_unit.csv")  # adjust if needed

FF5MOM_OUT = FACTORS_DIR / "ff5_mom_factors_monthly.parquet"
OUT_REG = FACTORS_DIR / "eval_value_quality_ff5mom_regression_net.csv"
OUT_CONTRIB = FACTORS_DIR / "eval_value_quality_ff5mom_contribution_by_year_net.csv"

# ===================== GLOBAL SETTINGS =====================
BAD_CODE_STRINGS = {"None", "nan", "", "NaN", "NULL", "null"}

# ===== PATCH START =====
# (案1) 2015-10〜2025-09 を保守レンジとして固定
REBUILD_START = pd.Timestamp("2015-10-01")
REBUILD_END   = pd.Timestamp("2025-09-30")
# ===== PATCH END =====

# Factor windows
P30, P70 = 0.30, 0.70
MIN_STOCKS_PER_MONTH = 500

# Winsorize quantiles (light)
WINSOR_Q = (0.01, 0.99)

# backtest uses NET return
RETURN_COL = "strategy_return_net"
TOPIX_COL = "topix_return"

FACTOR_COLS = ["MKT", "SMB", "HML", "RMW", "CMA", "WML"]


# ===================== UTIL =====================
def _safe_to_datetime_ns(s: pd.Series) -> pd.Series:
    return pd.to_datetime(s, errors="coerce").astype("datetime64[ns]")


def winsorize_series(x: pd.Series, q=(0.01, 0.99)) -> pd.Series:
    x = pd.to_numeric(x, errors="coerce")
    if x.notna().any():
        lo = x.quantile(q[0])
        hi = x.quantile(q[1])
        return x.clip(lower=lo, upper=hi)
    return x


def vwap_return(ret: pd.Series, w: pd.Series) -> float:
    ret = pd.to_numeric(ret, errors="coerce")
    w = pd.to_numeric(w, errors="coerce")
    m = ret.notna() & w.notna() & (w > 0)
    if m.sum() == 0:
        return np.nan
    return float((ret[m] * w[m]).sum() / w[m].sum())


def compound_returns(r: pd.Series) -> float:
    r = pd.to_numeric(r, errors="coerce").dropna()
    if len(r) == 0:
        return np.nan
    return float((1.0 + r).prod() - 1.0)


def month_end_range_for_october_year(start_date: pd.Timestamp) -> tuple[pd.Timestamp, pd.Timestamp]:
    start_date = pd.Timestamp(start_date).normalize()
    start_me = (start_date + pd.offsets.MonthEnd(0)).normalize()
    end_me = (start_date + pd.DateOffset(years=1) - pd.offsets.MonthEnd(1)).normalize()
    return start_me, end_me


# ===================== STEP 1: Load fins_norm in range =====================
# ===== PATCH START =====
# date=YYYY-MM-DD.parquet に対応したファイル列挙
def _list_parquet_files_in_range(fins_dir: Path, start_date: pd.Timestamp, end_date: pd.Timestamp) -> list[Path]:
    """
    Supports naming like:
      date=2016-01-15.parquet
    """
    files = []
    pat = re.compile(r"date=(\d{4}-\d{2}-\d{2})\.parquet$", re.I)

    for p in fins_dir.glob("*.parquet"):
        m = pat.search(p.name)
        if not m:
            continue
        d = pd.Timestamp(m.group(1))
        if start_date <= d <= end_date:
            files.append(p)

    files.sort()
    return files
# ===== PATCH END =====


# ===== PATCH START =====
# load_fin_norm_range を差し替え（重要）:
# - 必須列: Code + snapshot_date（← Dateではない！）
# - 0件でも diag_right_summary_rebuild.csv を必ず出す
# - Code は英数字混在を許容
# - right は ["Code","snapshot_date","Eq","TA","NP","SharesOut_raw"] を返す
def load_fin_norm_range(fins_dir: Path, start_need: pd.Timestamp, end_need: pd.Timestamp) -> pd.DataFrame:
    files = _list_parquet_files_in_range(fins_dir, start_need, end_need)

    if len(files) == 0:
        diag = {
            "start_need": str(start_need.date()),
            "end_need": str(end_need.date()),
            "right_rows": 0,
            "right_codes": 0,
            "right_min_date": "",
            "right_max_date": "",
            "dedup_dropped_rows": 0,
            "cov_eq_pct": 0.0,
            "cov_ta_pct": 0.0,
            "cov_np_pct": 0.0,
            "cov_shares_pct": 0.0,
            "snapshot_date_notna_pct": 0.0,
            "n_files": 0,
            "note": f"NO FILES MATCHED. Expect 'date=YYYY-MM-DD.parquet' under {fins_dir}",
        }
        pd.DataFrame([diag]).to_csv(DIAG_DIR / "diag_right_summary_rebuild.csv", index=False, encoding="utf-8-sig")
        return pd.DataFrame(columns=["Code", "snapshot_date", "Eq", "TA", "NP", "SharesOut_raw"])

    chunks = []
    n_read_ok = 0

    for fp in files:
        try:
            df = pd.read_parquet(fp)
        except Exception as e:
            print(f"[WARN] read parquet failed: {fp} ({e})")
            continue

        # --- ここが今回の根本原因対応: Date ではなく snapshot_date ---
        if not {"Code", "snapshot_date"}.issubset(df.columns):
            continue

        cols = ["Code", "snapshot_date"]
        for c in ["Eq", "TA", "NP", "ShOutFY", "TrShFY", "AvgSh"]:
            if c in df.columns:
                cols.append(c)
        df = df[cols].copy()

        # Code: keep alphanumeric (TSE new codes)
        df["Code"] = df["Code"].astype(str).str.strip()
        df = df[~df["Code"].isin(BAD_CODE_STRINGS)].copy()

        # snapshot_date normalize
        df["snapshot_date"] = pd.to_datetime(df["snapshot_date"], errors="coerce").astype("datetime64[ns]").dt.floor("D")

        # numeric columns
        for c in ["Eq", "TA", "NP", "ShOutFY", "TrShFY", "AvgSh"]:
            if c in df.columns:
                df[c] = pd.to_numeric(df[c], errors="coerce")

        # shares pick priority: TrShFY -> ShOutFY -> AvgSh
        if "TrShFY" in df.columns:
            df["SharesOut_raw"] = df["TrShFY"]
        elif "ShOutFY" in df.columns:
            df["SharesOut_raw"] = df["ShOutFY"]
        elif "AvgSh" in df.columns:
            df["SharesOut_raw"] = df["AvgSh"]
        else:
            df["SharesOut_raw"] = np.nan

        df = df[["Code", "snapshot_date", "Eq", "TA", "NP", "SharesOut_raw"]].copy()
        chunks.append(df)
        n_read_ok += 1

    if len(chunks) == 0:
        diag = {
            "start_need": str(start_need.date()),
            "end_need": str(end_need.date()),
            "right_rows": 0,
            "right_codes": 0,
            "right_min_date": "",
            "right_max_date": "",
            "dedup_dropped_rows": 0,
            "cov_eq_pct": 0.0,
            "cov_ta_pct": 0.0,
            "cov_np_pct": 0.0,
            "cov_shares_pct": 0.0,
            "snapshot_date_notna_pct": 0.0,
            "n_files": int(len(files)),
            "n_files_read_ok": int(n_read_ok),
            "note": "FILES FOUND BUT NO VALID ROWS (missing required columns or read failures).",
        }
        pd.DataFrame([diag]).to_csv(DIAG_DIR / "diag_right_summary_rebuild.csv", index=False, encoding="utf-8-sig")
        return pd.DataFrame(columns=["Code", "snapshot_date", "Eq", "TA", "NP", "SharesOut_raw"])

    right = pd.concat(chunks, ignore_index=True)

    # drop rows with NaT snapshot_date early (important for merge_asof)
    right = right[right["snapshot_date"].notna()].copy()

    before = len(right)
    right = right.sort_values(["snapshot_date", "Code"], kind="mergesort")
    right = right.drop_duplicates(["Code", "snapshot_date"], keep="last").reset_index(drop=True)
    dedup_drop = before - len(right)

    diag = {
        "start_need": str(start_need.date()),
        "end_need": str(end_need.date()),
        "right_rows": int(len(right)),
        "right_codes": int(right["Code"].nunique()) if len(right) else 0,
        "right_min_date": str(right["snapshot_date"].min()) if len(right) else "",
        "right_max_date": str(right["snapshot_date"].max()) if len(right) else "",
        "dedup_dropped_rows": int(dedup_drop),
        "cov_eq_pct": float(right["Eq"].notna().mean() * 100) if len(right) else 0.0,
        "cov_ta_pct": float(right["TA"].notna().mean() * 100) if len(right) else 0.0,
        "cov_np_pct": float(right["NP"].notna().mean() * 100) if len(right) else 0.0,
        "cov_shares_pct": float(right["SharesOut_raw"].notna().mean() * 100) if len(right) else 0.0,
        "snapshot_date_notna_pct": float(right["snapshot_date"].notna().mean() * 100) if len(right) else 0.0,
        "n_files": int(len(files)),
        "n_files_read_ok": int(n_read_ok),
        "note": "OK",
    }
    pd.DataFrame([diag]).to_csv(DIAG_DIR / "diag_right_summary_rebuild.csv", index=False, encoding="utf-8-sig")

    return right
# ===== PATCH END =====


# ===================== STEP 2: Build snapshot (full range) =====================
def rebuild_month_end_snapshot(price_path: Path,
                               fins_dir: Path,
                               out_snapshot_path: Path,
                               rebuild_start: pd.Timestamp,
                               rebuild_end: pd.Timestamp) -> pd.DataFrame:
    """
    Build month_end_snapshot for MonthEnd in [rebuild_start, rebuild_end] (month-end dates),
    using merge_asof (left Date=MonthEnd) to attach latest financial snapshot <= MonthEnd.
    Also computes MarketCap/BM_Ratio/ROE/INV_Growth.
    """
    price = pd.read_parquet(price_path)

    for c in ["Code", "MonthEnd", "Date", "AdjustedClose"]:
        if c not in price.columns:
            raise ValueError(f"price_month_end missing required column: {c}")

    price["Code"] = price["Code"].astype(str).str.strip()
    price = price[~price["Code"].isin(BAD_CODE_STRINGS)].copy()

    price["MonthEnd"] = _safe_to_datetime_ns(price["MonthEnd"]).dt.normalize()
    price["Date"] = _safe_to_datetime_ns(price["MonthEnd"]).dt.normalize()
    price["AdjustedClose"] = pd.to_numeric(price["AdjustedClose"], errors="coerce")

    mask = (price["MonthEnd"] >= rebuild_start.normalize()) & (price["MonthEnd"] <= rebuild_end.normalize())
    left = price.loc[mask, ["Code", "MonthEnd", "Date", "AdjustedClose"]].copy()

    dup = left.duplicated(["Code", "MonthEnd"], keep=False)
    if dup.any():
        left.loc[dup].sort_values(["MonthEnd", "Code"]).to_csv(
            DIAG_DIR / "diag_price_target_duplicates.csv", index=False, encoding="utf-8-sig"
        )
        left = left.sort_values(["MonthEnd", "Code"], kind="mergesort").drop_duplicates(["Code", "MonthEnd"], keep="last")

    start_need = rebuild_start - pd.Timedelta(days=500)
    end_need = rebuild_end

    print(f"[SNAP] rebuild MonthEnd range: {rebuild_start.date()}..{rebuild_end.date()}")
    print(f"[SNAP] fins read range: {start_need.date()}..{end_need.date()}")

    right = load_fin_norm_range(fins_dir, start_need, end_need)

    # merge_asof requirements: keys sorted globally and dtype aligned
    left["Date"] = _safe_to_datetime_ns(left["Date"]).dt.floor("D")
    right["snapshot_date"] = _safe_to_datetime_ns(right["snapshot_date"]).dt.floor("D")

    left = left.sort_values(["Date", "Code"], kind="mergesort").reset_index(drop=True)
    right = right.sort_values(["snapshot_date", "Code"], kind="mergesort").reset_index(drop=True)

    if not left["Date"].is_monotonic_increasing:
        left[["Date", "Code"]].head(2000).to_csv(DIAG_DIR / "diag_left_not_sorted.csv", index=False, encoding="utf-8-sig")
        raise RuntimeError("left['Date'] not monotonic increasing. see diag_left_not_sorted.csv")

    if len(right) and (not right["snapshot_date"].is_monotonic_increasing):
        right[["snapshot_date", "Code"]].head(2000).to_csv(DIAG_DIR / "diag_right_not_sorted.csv", index=False, encoding="utf-8-sig")
        raise RuntimeError("right['snapshot_date'] not monotonic increasing. see diag_right_not_sorted.csv")

    merged = pd.merge_asof(
        left,
        right,
        left_on="Date",
        right_on="snapshot_date",
        by="Code",
        direction="backward",
        allow_exact_matches=True
    )

    merged["MarketCap"] = pd.to_numeric(merged["AdjustedClose"], errors="coerce") * pd.to_numeric(merged["SharesOut_raw"], errors="coerce")
    merged.loc[merged["MarketCap"] <= 0, "MarketCap"] = np.nan

    merged["BM_Ratio"] = pd.to_numeric(merged["Eq"], errors="coerce") / merged["MarketCap"]
    merged.loc[(merged["BM_Ratio"] <= 0) | (merged["BM_Ratio"] > 10), "BM_Ratio"] = np.nan

    merged["ROE"] = pd.to_numeric(merged["NP"], errors="coerce") / pd.to_numeric(merged["Eq"], errors="coerce")
    merged.loc[(pd.to_numeric(merged["Eq"], errors="coerce") <= 0), "ROE"] = np.nan

    merged["TA"] = pd.to_numeric(merged["TA"], errors="coerce")
    merged = merged.sort_values(["Code", "MonthEnd"], kind="mergesort").reset_index(drop=True)
    merged["TA_lag12"] = merged.groupby("Code")["TA"].shift(12)
    merged["INV_Growth"] = (merged["TA"] - merged["TA_lag12"]) / merged["TA_lag12"]
    merged.loc[(merged["TA_lag12"] <= 0), "INV_Growth"] = np.nan

    d2 = merged.duplicated(["Code", "MonthEnd"], keep=False)
    if d2.any():
        merged.loc[d2].sort_values(["MonthEnd", "Code"]).to_csv(
            DIAG_DIR / "diag_snapshot_duplicates_fullrebuild.csv", index=False, encoding="utf-8-sig"
        )
        raise RuntimeError("snapshot has (Code,MonthEnd) duplicates. see diag_snapshot_duplicates_fullrebuild.csv")

    cov = merged.groupby("MonthEnd").agg(
        n=("Code", "size"),
        mcap_notna=("MarketCap", lambda x: float(x.notna().mean() * 100)),
        bm_notna=("BM_Ratio", lambda x: float(x.notna().mean() * 100)),
        roe_notna=("ROE", lambda x: float(x.notna().mean() * 100)),
        inv_notna=("INV_Growth", lambda x: float(x.notna().mean() * 100)),
    ).reset_index()
    cov.to_csv(DIAG_DIR / "diag_snapshot_coverage_by_month_fullrebuild.csv", index=False, encoding="utf-8-sig")

    # ===== PATCH START =====
    # 保存列に Eq/TA/NP/SharesOut_raw を残す（デバッグ・再利用性のため）
    keep_cols = [
        "Code", "MonthEnd", "Date", "AdjustedClose",
        "Eq", "TA", "NP", "SharesOut_raw",
        "MarketCap", "BM_Ratio", "ROE", "INV_Growth"
    ]
    merged_out = merged[[c for c in keep_cols if c in merged.columns]].copy()
    # ===== PATCH END =====

    merged_out.to_parquet(out_snapshot_path, index=False)

    latest_me = merged_out["MonthEnd"].max()
    latest_row = cov.sort_values("MonthEnd").tail(1)
    print(f"[SNAP SAVED] {out_snapshot_path} rows={len(merged_out)} MonthEnd.max={latest_me}")
    if len(latest_row):
        r = latest_row.iloc[0]
        print(f"[SNAP COVERAGE latest] MonthEnd={pd.Timestamp(r['MonthEnd']).date()} n={int(r['n'])} "
              f"mcap={r['mcap_notna']:.2f}% bm={r['bm_notna']:.2f}% roe={r['roe_notna']:.2f}% inv={r['inv_notna']:.2f}%")

    return merged_out


# ===================== STEP 3: Build monthly factors (FF5+MOM) =====================
def build_monthly_forward_returns_and_mom(price_path: Path) -> pd.DataFrame:
    """
    From price_month_end:
      - ret_m_fwd = P_{t+1} / P_t - 1
      - MOM_12_1  = P_{t-1}/P_{t-12} - 1
    """
    p = pd.read_parquet(price_path)
    for c in ["Code", "MonthEnd", "AdjustedClose"]:
        if c not in p.columns:
            raise ValueError(f"price_month_end missing required column: {c}")

    p["Code"] = p["Code"].astype(str).str.strip()
    p = p[~p["Code"].isin(BAD_CODE_STRINGS)].copy()
    p["MonthEnd"] = _safe_to_datetime_ns(p["MonthEnd"]).dt.normalize()
    p["AdjustedClose"] = pd.to_numeric(p["AdjustedClose"], errors="coerce")

    p = p.sort_values(["Code", "MonthEnd"], kind="mergesort").reset_index(drop=True)

    p["Adj_lag1"] = p.groupby("Code")["AdjustedClose"].shift(1)
    p["Adj_lag12"] = p.groupby("Code")["AdjustedClose"].shift(12)
    p["Adj_fwd1"] = p.groupby("Code")["AdjustedClose"].shift(-1)

    p["ret_m_fwd"] = (p["Adj_fwd1"] / p["AdjustedClose"]) - 1.0
    p.loc[(p["AdjustedClose"] <= 0) | (p["Adj_fwd1"] <= 0), "ret_m_fwd"] = np.nan

    p["MOM_12_1"] = (p["Adj_lag1"] / p["Adj_lag12"]) - 1.0
    p.loc[(p["Adj_lag1"] <= 0) | (p["Adj_lag12"] <= 0), "MOM_12_1"] = np.nan

    return p[["Code", "MonthEnd", "ret_m_fwd", "MOM_12_1"]].copy()


def compute_ff5_mom_factors(snapshot_path: Path,
                            price_path: Path,
                            out_path: Path,
                            factor_start: pd.Timestamp,
                            factor_end: pd.Timestamp) -> pd.DataFrame:
    """
    Use snapshot (MarketCap, BM_Ratio, ROE, INV_Growth) + price-derived ret_m_fwd + MOM_12_1
    to compute monthly factors for MonthEnd in [factor_start, factor_end].
    """
    snap = pd.read_parquet(snapshot_path)
    for c in ["Code", "MonthEnd", "AdjustedClose", "MarketCap", "BM_Ratio", "ROE", "INV_Growth"]:
        if c not in snap.columns:
            raise ValueError(f"snapshot missing required column: {c}")

    snap["Code"] = snap["Code"].astype(str).str.strip()
    snap = snap[~snap["Code"].isin(BAD_CODE_STRINGS)].copy()
    snap["MonthEnd"] = _safe_to_datetime_ns(snap["MonthEnd"]).dt.normalize()

    pm = build_monthly_forward_returns_and_mom(price_path)
    pm["MonthEnd"] = _safe_to_datetime_ns(pm["MonthEnd"]).dt.normalize()

    df = snap.merge(pm, on=["Code", "MonthEnd"], how="left")

    for c in ["MarketCap", "BM_Ratio", "ROE", "INV_Growth", "ret_m_fwd", "MOM_12_1"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    for c in ["BM_Ratio", "ROE", "INV_Growth", "ret_m_fwd", "MOM_12_1"]:
        df[c] = winsorize_series(df[c], WINSOR_Q)

    df = df[(df["MonthEnd"] >= factor_start.normalize()) & (df["MonthEnd"] <= factor_end.normalize())].copy()

    mom_cov = df.groupby("MonthEnd").agg(
        n=("Code", "size"),
        mom_notna=("MOM_12_1", lambda x: float(x.notna().mean() * 100))
    ).reset_index()
    mom_cov.to_csv(DIAG_DIR / "diag_mom_coverage_by_month_fullrebuild.csv", index=False, encoding="utf-8-sig")

    months = sorted(df["MonthEnd"].unique())
    rows = []

    for me in months:
        d = df[df["MonthEnd"] == me].copy()
        n_total = int(d["Code"].nunique())

        mkt = vwap_return(d["ret_m_fwd"], d["MarketCap"])

        mc = d["MarketCap"]
        if mc.notna().sum() < MIN_STOCKS_PER_MONTH:
            continue

        size_cut = mc.median()
        d["SB"] = np.where(d["MarketCap"] <= size_cut, "S", "B")

        def qcut30_70(x: pd.Series):
            x = pd.to_numeric(x, errors="coerce")
            q30 = x.quantile(P30)
            q70 = x.quantile(P70)
            return q30, q70

        q30, q70 = qcut30_70(d["BM_Ratio"].dropna())
        d["VAL"] = np.where(d["BM_Ratio"] <= q30, "L", np.where(d["BM_Ratio"] >= q70, "H", "M"))

        q30p, q70p = qcut30_70(d["ROE"].dropna())
        d["PROF"] = np.where(d["ROE"] <= q30p, "W", np.where(d["ROE"] >= q70p, "R", "M"))

        q30i, q70i = qcut30_70(d["INV_Growth"].dropna())
        d["INV"] = np.where(d["INV_Growth"] <= q30i, "C", np.where(d["INV_Growth"] >= q70i, "A", "M"))

        q30m, q70m = qcut30_70(d["MOM_12_1"].dropna())
        d["MOM"] = np.where(d["MOM_12_1"] <= q30m, "L", np.where(d["MOM_12_1"] >= q70m, "W", "M"))

        def bucket_ret(mask):
            dd = d[mask].copy()
            return vwap_return(dd["ret_m_fwd"], dd["MarketCap"])

        r_SH = bucket_ret((d["SB"] == "S") & (d["VAL"] == "H"))
        r_BH = bucket_ret((d["SB"] == "B") & (d["VAL"] == "H"))
        r_SL = bucket_ret((d["SB"] == "S") & (d["VAL"] == "L"))
        r_BL = bucket_ret((d["SB"] == "B") & (d["VAL"] == "L"))
        hml = np.nanmean([r_SH, r_BH]) - np.nanmean([r_SL, r_BL])

        r_SR = bucket_ret((d["SB"] == "S") & (d["PROF"] == "R"))
        r_BR = bucket_ret((d["SB"] == "B") & (d["PROF"] == "R"))
        r_SW = bucket_ret((d["SB"] == "S") & (d["PROF"] == "W"))
        r_BW = bucket_ret((d["SB"] == "B") & (d["PROF"] == "W"))
        rmw = np.nanmean([r_SR, r_BR]) - np.nanmean([r_SW, r_BW])

        r_SC = bucket_ret((d["SB"] == "S") & (d["INV"] == "C"))
        r_BC = bucket_ret((d["SB"] == "B") & (d["INV"] == "C"))
        r_SA = bucket_ret((d["SB"] == "S") & (d["INV"] == "A"))
        r_BA = bucket_ret((d["SB"] == "B") & (d["INV"] == "A"))
        cma = np.nanmean([r_SC, r_BC]) - np.nanmean([r_SA, r_BA])

        r_SWm = bucket_ret((d["SB"] == "S") & (d["MOM"] == "W"))
        r_BWm = bucket_ret((d["SB"] == "B") & (d["MOM"] == "W"))
        r_SLm = bucket_ret((d["SB"] == "S") & (d["MOM"] == "L"))
        r_BLm = bucket_ret((d["SB"] == "B") & (d["MOM"] == "L"))
        wml = np.nanmean([r_SWm, r_BWm]) - np.nanmean([r_SLm, r_BLm])

        s_val = np.nanmean([bucket_ret((d["SB"] == "S") & (d["VAL"] == x)) for x in ["H", "M", "L"]])
        b_val = np.nanmean([bucket_ret((d["SB"] == "B") & (d["VAL"] == x)) for x in ["H", "M", "L"]])
        smb_val = s_val - b_val

        s_prof = np.nanmean([bucket_ret((d["SB"] == "S") & (d["PROF"] == x)) for x in ["R", "M", "W"]])
        b_prof = np.nanmean([bucket_ret((d["SB"] == "B") & (d["PROF"] == x)) for x in ["R", "M", "W"]])
        smb_prof = s_prof - b_prof

        s_inv = np.nanmean([bucket_ret((d["SB"] == "S") & (d["INV"] == x)) for x in ["C", "M", "A"]])
        b_inv = np.nanmean([bucket_ret((d["SB"] == "B") & (d["INV"] == x)) for x in ["C", "M", "A"]])
        smb_inv = s_inv - b_inv

        s_mom = np.nanmean([bucket_ret((d["SB"] == "S") & (d["MOM"] == x)) for x in ["W", "M", "L"]])
        b_mom = np.nanmean([bucket_ret((d["SB"] == "B") & (d["MOM"] == x)) for x in ["W", "M", "L"]])
        smb_mom = s_mom - b_mom

        smb = np.nanmean([smb_val, smb_prof, smb_inv, smb_mom])

        used_val = int(d["BM_Ratio"].notna().sum())
        used_prof = int(d["ROE"].notna().sum())
        used_inv = int(d["INV_Growth"].notna().sum())
        used_mom = int(d["MOM_12_1"].notna().sum())

        abnormal = (min(used_val, used_prof, used_inv, used_mom) < MIN_STOCKS_PER_MONTH)
        confidence = 1.0 if not abnormal else max(0.0, min(used_val, used_prof, used_inv, used_mom) / MIN_STOCKS_PER_MONTH)

        rows.append({
            "MonthEnd": me,
            "MKT": mkt,
            "SMB": smb,
            "HML": hml,
            "RMW": rmw,
            "CMA": cma,
            "WML": wml,
            "n_total": n_total,
            "used_val": used_val,
            "used_prof": used_prof,
            "used_inv": used_inv,
            "used_mom": used_mom,
            "abnormal": bool(abnormal),
            "confidence": float(confidence),
        })

    fac = pd.DataFrame(rows)

    # 明示停止（次に迷子にならない）
    if fac.empty:
        (DIAG_DIR / "diag_factor_rows_empty.txt").write_text(
            "No factor months computed. Check snapshot coverage + diag_right_summary_rebuild.csv\n",
            encoding="utf-8"
        )
        raise RuntimeError("No factor months computed (rows empty).")

    fac = fac.sort_values("MonthEnd").reset_index(drop=True)
    fac.to_parquet(out_path, index=False)
    fac.to_csv(DIAG_DIR / "diag_ff5mom_n_used_by_month_fullrebuild.csv", index=False, encoding="utf-8-sig")

    print(f"[FACTORS SAVED] {out_path} months={len(fac)} min={fac['MonthEnd'].min()} max={fac['MonthEnd'].max()}")

    return fac


# ===================== STEP 4: Evaluate annual Oct strategy =====================
def load_backtest(csv_path: Path) -> pd.DataFrame:
    bt = pd.read_csv(csv_path)
    bt["date"] = pd.to_datetime(bt["date"]).dt.normalize()
    for c in bt.columns:
        if c != "date":
            bt[c] = pd.to_numeric(bt[c], errors="coerce")
    bt = bt.sort_values("date").reset_index(drop=True)
    return bt


def run_ols_with_intercept(y: np.ndarray, X: np.ndarray) -> tuple[np.ndarray, float]:
    y = y.reshape(-1, 1)
    X = np.asarray(X)
    X1 = np.concatenate([np.ones((X.shape[0], 1)), X], axis=1)
    beta = np.linalg.pinv(X1) @ y
    y_hat = X1 @ beta
    resid = y - y_hat
    ss_res = float((resid ** 2).sum())
    ss_tot = float(((y - y.mean()) ** 2).sum()) if y.shape[0] > 1 else np.nan
    r2 = 1.0 - ss_res / ss_tot if (ss_tot is not np.nan and ss_tot > 0) else np.nan
    return beta.flatten(), r2


def eval_strategy_october_net(ff5mom_path: Path, backtest_csv: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    fac = pd.read_parquet(ff5mom_path)
    fac["MonthEnd"] = _safe_to_datetime_ns(fac["MonthEnd"]).dt.normalize()

    bt = load_backtest(backtest_csv)
    bt = bt[bt["date"] >= pd.Timestamp("2017-10-01")].copy().reset_index(drop=True)

    rows = []
    for _, r in bt.iterrows():
        start = pd.Timestamp(r["date"])
        end = start + pd.DateOffset(years=1)
        start_me, end_me = month_end_range_for_october_year(start)

        mask = (fac["MonthEnd"] >= start_me) & (fac["MonthEnd"] <= end_me)
        f = fac.loc[mask].copy()

        fac_year = {c: compound_returns(f[c]) for c in FACTOR_COLS}
        n_months = int(f["MonthEnd"].nunique())

        rows.append({
            "label": f"{start.date()} to {end.date()}",
            "start_date": start,
            "end_date": end,
            "start_me": start_me,
            "end_me": end_me,
            "n_months_factors": n_months,
            "strategy_return_net": float(r[RETURN_COL]),
            "topix_return": float(r[TOPIX_COL]) if TOPIX_COL in bt.columns else np.nan,
            "investment_ratio": float(r["investment_ratio"]) if "investment_ratio" in bt.columns else np.nan,
            **fac_year
        })

    yearly = pd.DataFrame(rows)
    yearly["ok"] = (yearly["n_months_factors"] >= 10)
    yearly.to_csv(DIAG_DIR / "diag_eval_alignment_after_fullrebuild.csv", index=False, encoding="utf-8-sig")

    usable = yearly[yearly["ok"]].copy()

    y = usable["strategy_return_net"].to_numpy(dtype=float)
    X = usable[FACTOR_COLS].to_numpy(dtype=float)
    good = np.isfinite(y) & np.isfinite(X).all(axis=1)
    usable = usable.loc[good].copy()
    y = usable["strategy_return_net"].to_numpy(dtype=float)
    X = usable[FACTOR_COLS].to_numpy(dtype=float)

    if len(usable) >= 2:
        params, r2 = run_ols_with_intercept(y, X)
        alpha = float(params[0])
        betas = params[1:]
    else:
        alpha = np.nan
        betas = np.array([np.nan] * len(FACTOR_COLS))
        r2 = np.nan

    reg = pd.DataFrame([{
        "n_years": int(len(usable)),
        "R2": float(r2) if np.isfinite(r2) else np.nan,
        "alpha": float(alpha) if np.isfinite(alpha) else np.nan,
        **{f"beta_{c}": float(b) if np.isfinite(b) else np.nan for c, b in zip(FACTOR_COLS, betas)}
    }])

    contrib_rows = []
    for _, row in usable.iterrows():
        pred = alpha
        parts = {"alpha_contrib": alpha}
        for c, b in zip(FACTOR_COLS, betas):
            v = float(row[c]) if pd.notna(row[c]) else np.nan
            parts[f"contrib_{c}"] = float(b) * v if np.isfinite(b) and np.isfinite(v) else np.nan
            pred += parts[f"contrib_{c}"] if np.isfinite(parts[f"contrib_{c}"]) else 0.0

        r_strat = float(row["strategy_return_net"])
        resid = r_strat - pred if np.isfinite(r_strat) and np.isfinite(pred) else np.nan

        contrib_rows.append({
            "label": row["label"],
            "start_date": row["start_date"],
            "end_date": row["end_date"],
            "n_months_factors": int(row["n_months_factors"]),
            "R_strat_net": r_strat,
            "topix_return": float(row["topix_return"]) if pd.notna(row["topix_return"]) else np.nan,
            "investment_ratio": float(row["investment_ratio"]) if pd.notna(row["investment_ratio"]) else np.nan,
            **parts,
            "predicted": float(pred) if np.isfinite(pred) else np.nan,
            "residual": float(resid) if np.isfinite(resid) else np.nan,
        })

    contrib = pd.DataFrame(contrib_rows)
    return reg, contrib


# ===================== MAIN =====================
def main():
    print("=== FULL REBUILD (snapshot) → FACTORS → EVAL (NET) ===")

    # 1) rebuild snapshot for conservative range
    rebuild_month_end_snapshot(
        price_path=PRICE_PATH,
        fins_dir=FINS_NORM_DIR,
        out_snapshot_path=SNAP_PATH,
        rebuild_start=REBUILD_START,
        rebuild_end=REBUILD_END,
    )

    # 2) compute monthly factors in desired window
    factor_start = pd.Timestamp("2016-10-31")
    factor_end = (pd.Timestamp("2025-09-30") + pd.offsets.MonthEnd(0)).normalize()

    compute_ff5_mom_factors(
        snapshot_path=SNAP_PATH,
        price_path=PRICE_PATH,
        out_path=FF5MOM_OUT,
        factor_start=factor_start,
        factor_end=factor_end,
    )

    # 3) evaluate annual strategy (net)
    reg, contrib = eval_strategy_october_net(FF5MOM_OUT, BACKTEST_CSV)
    reg.to_csv(OUT_REG, index=False, encoding="utf-8-sig")
    contrib.to_csv(OUT_CONTRIB, index=False, encoding="utf-8-sig")

    print(f"[SAVE] regression: {OUT_REG}")
    print(f"[SAVE] contribution: {OUT_CONTRIB}")

    print("\n=== REGRESSION SUMMARY ===")
    print(reg.to_string(index=False))

    if len(contrib):
        print("\n=== CONTRIBUTION (head) ===")
        cols2 = ["label", "R_strat_net", "predicted", "residual", "alpha_contrib"] + [f"contrib_{c}" for c in FACTOR_COLS]
        cols2 = [c for c in cols2 if c in contrib.columns]
        print(contrib[cols2].head(10).to_string(index=False))

    print("\n[NEXT] Check diags:")
    print(f" - {DIAG_DIR / 'diag_right_summary_rebuild.csv'}")
    print(f" - {DIAG_DIR / 'diag_snapshot_coverage_by_month_fullrebuild.csv'}")
    print(f" - {DIAG_DIR / 'diag_ff5mom_n_used_by_month_fullrebuild.csv'}")
    print(f" - {DIAG_DIR / 'diag_eval_alignment_after_fullrebuild.csv'}")


if __name__ == "__main__":
    main()


=== FULL REBUILD (snapshot) → FACTORS → EVAL (NET) ===
[SNAP] rebuild MonthEnd range: 2015-10-01..2025-09-30
[SNAP] fins read range: 2014-05-19..2025-09-30
[SNAP SAVED] C:\Users\yongr\Project\merged_data_all_stocks\factors\month_end_snapshot.parquet rows=474003 MonthEnd.max=2025-09-30 00:00:00
[SNAP COVERAGE latest] MonthEnd=2025-09-30 n=4282 mcap=80.06% bm=15.90% roe=84.56% inv=79.61%
[FACTORS SAVED] C:\Users\yongr\Project\merged_data_all_stocks\factors\ff5_mom_factors_monthly.parquet months=108 min=2016-10-31 00:00:00 max=2025-09-30 00:00:00
[SAVE] regression: C:\Users\yongr\Project\merged_data_all_stocks\factors\eval_value_quality_ff5mom_regression_net.csv
[SAVE] contribution: C:\Users\yongr\Project\merged_data_all_stocks\factors\eval_value_quality_ff5mom_contribution_by_year_net.csv

=== REGRESSION SUMMARY ===
 n_years       R2    alpha  beta_MKT  beta_SMB  beta_HML  beta_RMW  beta_CMA  beta_WML
       8 0.966757 0.345931 -0.380664 -1.178092 -1.141833  1.357676  2.847794  0.994328


In [78]:
from pathlib import Path
import pandas as pd

FINS_NORM_DIR = Path(r"C:\Users\yongr\Project\jquants_fins_summary_10y_parquet\daily_parquet_norm")
fp = sorted(FINS_NORM_DIR.glob("date=*.parquet"))[-1]
df = pd.read_parquet(fp)
print(fp.name)
print([c for c in df.columns if "Div" in c or "div" in c or "DPS" in c or "Dividend" in c])
print("all cols(head 60):", df.columns.tolist()[:60])


date=2026-01-09.parquet
[]
all cols(head 60): ['snapshot_date', 'DiscDate', 'DiscTime', 'Code', 'DiscNo', 'DocType', 'TA', 'Eq', 'NP', 'ShOutFY', 'TrShFY', 'AvgSh', 'BPS', 'EPS']


In [1]:
# -*- coding: utf-8 -*-
"""
PoC: yfinanceで日本株（TOPIX想定ユニバース上位200）について配当(dividends)取得可否を検証する。

【目的】
- J-Quants Standard では「配当金情報 API」が使えないため、外部ソース(yfinance/Yahoo Finance)で代替可能かをPoCで確認する。
- 上位200銘柄について、取得成功率、取得不能コード一覧、エラー理由、配当データの期間・件数をCSV出力する。

【ユニバース定義（PoC）】
- month_end_snapshot.parquet の最新 MonthEnd における MarketCap 上位200を「TOPIX上位200の近似」として使用。
  * 厳密なTOPIX構成銘柄リストが別ファイルにある場合は、そこから読み込むよう拡張可能。

【想定シンボル変換ルール（Yahoo Finance / yfinance）】
- 東証コードが「5桁で末尾0」の場合が多い（例: 72030）ので、
  Yahoo Finance側で一般的な「4桁.T」を最優先で試す：
    - 第1候補: <CodeWithoutLast0>.T 例: 72030 -> 7203.T
    - 第2候補: <Code>.T            例: 72030 -> 72030.T（保険）
- 英数字コード（例: 130A0）の場合は、末尾0落としが危険なので、
    - <Code>.T のみ試す（例: 130A0.T）

【出力】
- diag_yfinance_dividend_poc/ 配下に以下を保存:
  - diag_top200_universe.csv              : 対象200銘柄一覧（Code, MarketCap, MonthEnd等）
  - diag_yf_symbol_attempts.csv           : 各Codeの試行シンボル一覧と結果
  - diag_yf_dividend_fetch_results.csv    : 最終結果（成功/失敗、件数、期間、エラー）
  - diag_yf_dividend_success_samples.csv  : 成功した銘柄の配当データ直近サンプル
  - diag_yf_dividend_fail_codes.csv       : 取得できないCode一覧
  - diag_yf_dividend_summary.csv          : 成功率などの集計
  - diag_yf_runtime_log.txt               : 実行ログ簡易版

【依存】
- pip install yfinance pandas pyarrow

実行:
  python poc_yfinance_dividends_top200.py
"""

import time
from pathlib import Path
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd

# yfinanceは外部依存（Yahoo Financeの仕様変更で壊れる可能性あり）
import yfinance as yf


# ============================================================
# Paths (あなたの環境)
# ============================================================
FACTORS_DIR = Path(r"C:\Users\yongr\Project\merged_data_all_stocks\factors")
SNAP_PATH = FACTORS_DIR / "month_end_snapshot.parquet"

DIAG_DIR = FACTORS_DIR / "diag_yfinance_dividend_poc"
DIAG_DIR.mkdir(parents=True, exist_ok=True)

OUT_UNIVERSE = DIAG_DIR / "diag_top200_universe.csv"
OUT_ATTEMPTS = DIAG_DIR / "diag_yf_symbol_attempts.csv"
OUT_RESULTS  = DIAG_DIR / "diag_yf_dividend_fetch_results.csv"
OUT_SUCCESS_SAMPLE = DIAG_DIR / "diag_yf_dividend_success_samples.csv"
OUT_FAIL_CODES = DIAG_DIR / "diag_yf_dividend_fail_codes.csv"
OUT_SUMMARY = DIAG_DIR / "diag_yf_dividend_summary.csv"
OUT_LOG = DIAG_DIR / "diag_yf_runtime_log.txt"


# ============================================================
# Params
# ============================================================
TOP_N = 200

# Yahoo側に負荷をかけすぎない
SLEEP_SEC = 0.35

# 成功サンプルとして保存する配当レコード数（直近）
MAX_SAMPLE_ROWS = 20

BAD_CODE_STRINGS = {"None", "nan", "", "NaN", "NULL", "null"}


# ============================================================
# Helpers
# ============================================================
def log_line(s: str):
    print(s)
    with open(OUT_LOG, "a", encoding="utf-8") as f:
        f.write(s + "\n")


def normalize_code(code: str) -> str:
    """コードを文字列化してトリム。英数字は保持（東証新仕様対応）。"""
    if code is None:
        return ""
    s = str(code).strip()
    if s in BAD_CODE_STRINGS:
        return ""
    return s


def make_symbol_candidates(code: str) -> List[str]:
    """
    Yahoo Finance想定シンボル候補（優先順を修正版）

    - 数字5桁 & 末尾0: 4桁.T を最優先（例: 72030 -> 7203.T）
                       次に 5桁.T（例: 72030.T）
    - それ以外（英数字含む等）: <Code>.T のみ
    """
    code = normalize_code(code)
    if not code:
        return []

    cands = []

    # ★ここが今回の変更点：5桁末尾0は4桁.Tを最優先
    if code.isdigit() and len(code) == 5 and code.endswith("0"):
        c4 = code[:-1]
        cands.append(f"{c4}.T")    # 例: 72030 -> 7203.T（優先）
        cands.append(f"{code}.T")  # 例: 72030.T（保険）
    else:
        # 英数字コード/末尾0以外/5桁以外はそのまま
        cands.append(f"{code}.T")

    # 重複排除（順序保持）
    seen = set()
    out = []
    for x in cands:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out


def pick_top200_universe_from_snapshot(snap_path: Path, top_n: int) -> pd.DataFrame:
    """
    month_end_snapshot.parquet から最新MonthEndのMarketCap上位Nを抽出。
    snapshotが更新済みである前提。
    """
    snap = pd.read_parquet(snap_path)

    if "Code" not in snap.columns or "MonthEnd" not in snap.columns or "MarketCap" not in snap.columns:
        raise ValueError("month_end_snapshot.parquet must have Code, MonthEnd, MarketCap")

    snap["Code"] = snap["Code"].astype(str).map(normalize_code)
    snap = snap[snap["Code"] != ""].copy()
    snap["MonthEnd"] = pd.to_datetime(snap["MonthEnd"], errors="coerce").dt.normalize()
    snap["MarketCap"] = pd.to_numeric(snap["MarketCap"], errors="coerce")

    latest_me = snap["MonthEnd"].max()
    if pd.isna(latest_me):
        raise ValueError("Cannot find latest MonthEnd in snapshot.")

    s = snap[snap["MonthEnd"] == latest_me].copy()
    s = s.dropna(subset=["MarketCap"]).copy()
    s = s.sort_values("MarketCap", ascending=False).drop_duplicates(["Code"], keep="first")

    uni = s.head(top_n).copy()
    uni["Rank_MarketCap"] = np.arange(1, len(uni) + 1)
    uni = uni[["MonthEnd", "Code", "MarketCap", "Rank_MarketCap"]].copy()

    log_line(f"[UNIVERSE] latest MonthEnd={latest_me.date()} | candidates={len(s):,} | top{top_n}={len(uni):,}")
    return uni


def fetch_dividends_for_symbol(symbol: str) -> Tuple[bool, Optional[pd.Series], str]:
    """
    yfinanceでdividends取得。
    戻り: (success, dividends_series, error_reason)

    注: Yahoo側に配当が存在しない場合は空Seriesになることがあります。
    """
    try:
        t = yf.Ticker(symbol)
        div = t.dividends  # pandas Series indexed by date
        if div is None:
            return False, None, "dividends is None"
        if not isinstance(div, pd.Series):
            return False, None, f"dividends type unexpected: {type(div)}"
        return True, div, ""
    except Exception as e:
        return False, None, f"{type(e).__name__}: {e}"


def summarize_dividends(div: pd.Series) -> Dict:
    """配当Seriesを要約して返す（件数、期間、合計、直近等）"""
    div = div.dropna()
    if len(div) == 0:
        return {
            "div_rows": 0,
            "div_min_date": "",
            "div_max_date": "",
            "div_sum": np.nan,
            "div_last": np.nan,
        }
    idx = pd.to_datetime(div.index, errors="coerce")
    return {
        "div_rows": int(len(div)),
        "div_min_date": str(pd.Timestamp(idx.min()).date()) if pd.notna(idx.min()) else "",
        "div_max_date": str(pd.Timestamp(idx.max()).date()) if pd.notna(idx.max()) else "",
        "div_sum": float(div.sum()),
        "div_last": float(div.iloc[-1]),
    }


# ============================================================
# Main PoC
# ============================================================
def main():
    # reset log
    with open(OUT_LOG, "w", encoding="utf-8") as f:
        f.write("")

    log_line("=" * 90)
    log_line("PoC: yfinance dividends coverage (TOPIX-like top200 by MarketCap from snapshot)")
    log_line("=" * 90)
    log_line(f"SNAP_PATH: {SNAP_PATH}")
    log_line(f"DIAG_DIR : {DIAG_DIR}")

    # 1) Universe
    uni = pick_top200_universe_from_snapshot(SNAP_PATH, TOP_N)
    uni.to_csv(OUT_UNIVERSE, index=False, encoding="utf-8-sig")
    log_line(f"[SAVE] universe: {OUT_UNIVERSE}")

    # 2) For each Code, try symbol candidates
    attempt_rows = []
    result_rows = []
    success_samples = []

    for _, row in uni.iterrows():
        code = row["Code"]
        cands = make_symbol_candidates(code)

        best = {
            "ok": False,
            "symbol": "",
            "div_rows": 0,
            "div_min_date": "",
            "div_max_date": "",
            "div_sum": np.nan,
            "div_last": np.nan,
            "error": "",
            "attempted": "|".join(cands),
        }

        if not cands:
            best["error"] = "no_symbol_candidates"
            result_rows.append({"Code": code, **best})
            continue

        last_err = ""
        for sym in cands:
            ok, div, err = fetch_dividends_for_symbol(sym)

            attempt_rows.append({
                "Code": code,
                "symbol": sym,
                "success_series": bool(ok),
                "error": err,
            })

            if not ok:
                last_err = err
                time.sleep(SLEEP_SEC)
                continue

            summ = summarize_dividends(div)
            best.update({
                "ok": True,
                "symbol": sym,
                **summ,
                "error": "",
            })

            div2 = div.dropna().copy()
            if len(div2) > 0:
                s2 = div2.tail(MAX_SAMPLE_ROWS)
                for dt, val in s2.items():
                    success_samples.append({
                        "Code": code,
                        "symbol": sym,
                        "date": str(pd.Timestamp(dt).date()),
                        "dividend": float(val),
                    })

            time.sleep(SLEEP_SEC)
            break

        if not best["ok"]:
            best["error"] = last_err or "all_candidates_failed"

        result_rows.append({
            "MonthEnd": str(pd.Timestamp(row["MonthEnd"]).date()),
            "Rank_MarketCap": int(row["Rank_MarketCap"]),
            "MarketCap": float(row["MarketCap"]),
            "Code": code,
            **best
        })

    # 3) Save outputs
    attempts_df = pd.DataFrame(attempt_rows)
    attempts_df.to_csv(OUT_ATTEMPTS, index=False, encoding="utf-8-sig")
    log_line(f"[SAVE] attempts: {OUT_ATTEMPTS}")

    res = pd.DataFrame(result_rows)
    res.to_csv(OUT_RESULTS, index=False, encoding="utf-8-sig")
    log_line(f"[SAVE] results: {OUT_RESULTS}")

    if success_samples:
        pd.DataFrame(success_samples).to_csv(OUT_SUCCESS_SAMPLE, index=False, encoding="utf-8-sig")
        log_line(f"[SAVE] success samples: {OUT_SUCCESS_SAMPLE}")

    fail = res[~res["ok"]].copy()
    fail[["Code", "attempted", "error"]].to_csv(OUT_FAIL_CODES, index=False, encoding="utf-8-sig")
    log_line(f"[SAVE] fail codes: {OUT_FAIL_CODES}")

    # 4) Summary
    total = len(res)
    ok_any = int(res["ok"].sum()) if total else 0
    ok_nonempty = int((res["ok"] & (res["div_rows"] > 0)).sum()) if total else 0
    rate_any = ok_any / total * 100.0 if total else 0.0
    rate_nonempty = ok_nonempty / total * 100.0 if total else 0.0

    summary = pd.DataFrame([{
        "top_n": TOP_N,
        "success_series_cnt": ok_any,
        "success_series_pct": rate_any,
        "success_nonempty_cnt": ok_nonempty,
        "success_nonempty_pct": rate_nonempty,
        "note_symbol_rule": "Primary(digit5&endswith0): <CodeWithoutLast0>.T ; Secondary: <Code>.T ; Else: <Code>.T",
        "sleep_sec": SLEEP_SEC,
    }])
    summary.to_csv(OUT_SUMMARY, index=False, encoding="utf-8-sig")
    log_line(f"[SAVE] summary: {OUT_SUMMARY}")

    log_line("-" * 90)
    log_line(f"[RESULT] TOP{TOP_N} dividends fetch:")
    log_line(f"  success (series obtained) : {ok_any}/{total} ({rate_any:.2f}%)")
    log_line(f"  success (non-empty divs)  : {ok_nonempty}/{total} ({rate_nonempty:.2f}%)")
    log_line(f"  failed                    : {total - ok_any}/{total}")
    log_line(f"  symbol rule: {summary.loc[0,'note_symbol_rule']}")
    log_line("-" * 90)
    log_line("DONE.")


if __name__ == "__main__":
    main()


PoC: yfinance dividends coverage (TOPIX-like top200 by MarketCap from snapshot)
SNAP_PATH: C:\Users\yongr\Project\merged_data_all_stocks\factors\month_end_snapshot.parquet
DIAG_DIR : C:\Users\yongr\Project\merged_data_all_stocks\factors\diag_yfinance_dividend_poc
[UNIVERSE] latest MonthEnd=2025-09-30 | candidates=3,428 | top200=200
[SAVE] universe: C:\Users\yongr\Project\merged_data_all_stocks\factors\diag_yfinance_dividend_poc\diag_top200_universe.csv


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: 167A0.T"}}}
$167A0.T: possibly delisted; no timezone found


[SAVE] attempts: C:\Users\yongr\Project\merged_data_all_stocks\factors\diag_yfinance_dividend_poc\diag_yf_symbol_attempts.csv
[SAVE] results: C:\Users\yongr\Project\merged_data_all_stocks\factors\diag_yfinance_dividend_poc\diag_yf_dividend_fetch_results.csv
[SAVE] success samples: C:\Users\yongr\Project\merged_data_all_stocks\factors\diag_yfinance_dividend_poc\diag_yf_dividend_success_samples.csv
[SAVE] fail codes: C:\Users\yongr\Project\merged_data_all_stocks\factors\diag_yfinance_dividend_poc\diag_yf_dividend_fail_codes.csv
[SAVE] summary: C:\Users\yongr\Project\merged_data_all_stocks\factors\diag_yfinance_dividend_poc\diag_yf_dividend_summary.csv
------------------------------------------------------------------------------------------
[RESULT] TOP200 dividends fetch:
  success (series obtained) : 200/200 (100.00%)
  success (non-empty divs)  : 198/200 (99.00%)
  failed                    : 0/200
  symbol rule: Primary(digit5&endswith0): <CodeWithoutLast0>.T ; Secondary: <Code>.T ; 

In [2]:
# -*- coding: utf-8 -*-
"""
FF5 + MOM(12-1) + DY_TTM（月次）:
- 既存データ: price_month_end.parquet / month_end_snapshot.parquet
- 追加データ: yfinance dividends（Yahoo Finance由来）
- 対象: 最新MonthEndのMarketCap上位200銘柄（TOPIX上位近似）
- DY定義: DY_TTM = (過去12ヶ月の配当合計) / 月末株価(AdjustedClose)
- DY因子: HDYMLDY = High DY - Low DY（Sizeニュートラル：Small/Big平均）

出力:
- factors/ff5_mom_dy_factors_monthly_top200.parquet
- factors/market_factor_summary_ff5_mom_dy_top200.csv
- factors/market_factor_trend_last3_ff5_mom_dy_top200.csv
- factors/market_factor_trend_last12_ff5_mom_dy_top200.csv
- factors/diag_yfinance_dividend_poc_dy/ 以下の診断CSV

注意:
- yfinanceは非公式データソースで、Yahoo側の仕様変更で壊れる可能性があります。
- yfinance dividends の日付は配当の支払日ではなく権利落ち日等であることが多い点に注意。[Source](https://github.com/ranaroussi/yfinance/issues/568)

依存:
  pip install yfinance pandas pyarrow
"""

import warnings
warnings.filterwarnings("ignore")

import time
from pathlib import Path
from typing import Dict, Tuple, List, Optional

import numpy as np
import pandas as pd
import yfinance as yf


# ============================================================
# Paths (あなたの環境)
# ============================================================
FACTORS_DIR = Path(r"C:\Users\yongr\Project\merged_data_all_stocks\factors")
PRICE_PATH = FACTORS_DIR / "price_month_end.parquet"
SNAP_PATH  = FACTORS_DIR / "month_end_snapshot.parquet"

# 出力（既存ファイルは上書きしない：別名で保存）
OUT_PARQUET = FACTORS_DIR / "ff5_mom_dy_factors_monthly_top200.parquet"
OUT_SUMMARY = FACTORS_DIR / "market_factor_summary_ff5_mom_dy_top200.csv"
OUT_TREND_LAST3 = FACTORS_DIR / "market_factor_trend_last3_ff5_mom_dy_top200.csv"
OUT_TREND_LAST12 = FACTORS_DIR / "market_factor_trend_last12_ff5_mom_dy_top200.csv"

# diagnostics
DIAG_DIR = FACTORS_DIR / "diag_yfinance_dividend_poc_dy"
DIAG_DIR.mkdir(parents=True, exist_ok=True)

DIAG_UNIVERSE = DIAG_DIR / "diag_universe_top200.csv"
DIAG_ATTEMPTS = DIAG_DIR / "diag_yf_symbol_attempts.csv"
DIAG_FETCH_RESULTS = DIAG_DIR / "diag_yf_dividend_fetch_results.csv"
DIAG_DIV_RAW_SAMPLE = DIAG_DIR / "diag_yf_dividend_success_samples.csv"
DIAG_DY_TTM_BY_MONTH = DIAG_DIR / "diag_dy_ttm_coverage_by_month.csv"
DIAG_DY_FACTOR_NUSED = DIAG_DIR / "diag_dy_n_used_by_month.csv"
DIAG_LOG = DIAG_DIR / "diag_runtime_log.txt"


# ============================================================
# Params
# ============================================================
TOP_N = 200

MIN_STOCKS_PER_MONTH = 200  # top200前提なので500は無理。診断用途も兼ねて200に下げる
P30 = 0.30
P70 = 0.70

# winsorize quantiles
Q_RET = (0.01, 0.99)
Q_BM  = (0.01, 0.99)
Q_ROE = (0.01, 0.99)
Q_INV = (0.01, 0.99)
Q_MOM = (0.01, 0.99)
Q_DY  = (0.01, 0.99)

BAD_CODE_STRINGS = {"None", "nan", "", "NaN", "NULL", "null"}

# yfinance polite access
SLEEP_SEC = 0.20
MAX_DIV_SAMPLE = 15


# ============================================================
# Utils
# ============================================================
def log_line(s: str):
    print(s)
    with open(DIAG_LOG, "a", encoding="utf-8") as f:
        f.write(s + "\n")


def normalize_code(code: str) -> str:
    if code is None:
        return ""
    s = str(code).strip()
    if s in BAD_CODE_STRINGS:
        return ""
    return s


def winsorize(s: pd.Series, q=(0.01, 0.99)) -> pd.Series:
    x = pd.to_numeric(s, errors="coerce")
    if x.notna().sum() == 0:
        return x
    lo = x.quantile(q[0])
    hi = x.quantile(q[1])
    return x.clip(lo, hi)


def value_weighted_return(df: pd.DataFrame, ret_col: str, w_col: str) -> float:
    x = df[[ret_col, w_col]].dropna()
    if x.empty:
        return np.nan
    w = x[w_col].astype(float).to_numpy()
    r = x[ret_col].astype(float).to_numpy()
    wsum = w.sum()
    if not np.isfinite(wsum) or wsum <= 0:
        return np.nan
    return float(np.dot(r, w) / wsum)


def qcut_3way(x: pd.Series, p30=0.3, p70=0.7, labels=("L", "M", "H")) -> pd.Series:
    a = pd.to_numeric(x, errors="coerce")
    q1 = a.quantile(p30)
    q2 = a.quantile(p70)
    out = pd.Series(index=a.index, dtype="object")
    out[a <= q1] = labels[0]
    out[(a > q1) & (a < q2)] = labels[1]
    out[a >= q2] = labels[2]
    return out


def size_split_median(mcap: pd.Series) -> pd.Series:
    a = pd.to_numeric(mcap, errors="coerce")
    med = a.quantile(0.5)
    out = pd.Series(index=a.index, dtype="object")
    out[a <= med] = "S"
    out[a > med] = "B"
    return out


def trend_label(vals: np.ndarray) -> str:
    vals = np.array(vals, dtype=float)
    vals = vals[np.isfinite(vals)]
    if len(vals) < 2:
        return "NA"
    x = np.arange(len(vals))
    slope = np.polyfit(x, vals, 1)[0]
    if abs(slope) < 1e-6:
        return "FLAT"
    return "UP" if slope > 0 else "DOWN"


def latest_evaluable_month(factors: pd.DataFrame, col="MKT") -> pd.Timestamp:
    f = factors.dropna(subset=[col]).copy()
    if f.empty:
        return pd.NaT
    return pd.Timestamp(f["MonthEnd"].max())


# ============================================================
# Universe: latest MonthEnd top200 by MarketCap
# ============================================================
def pick_top200_universe() -> pd.DataFrame:
    snap = pd.read_parquet(SNAP_PATH)
    snap["Code"] = snap["Code"].astype(str).map(normalize_code)
    snap = snap[snap["Code"] != ""].copy()
    snap["MonthEnd"] = pd.to_datetime(snap["MonthEnd"], errors="coerce").dt.normalize()
    snap["MarketCap"] = pd.to_numeric(snap["MarketCap"], errors="coerce")

    latest_me = snap["MonthEnd"].max()
    s = snap[snap["MonthEnd"] == latest_me].dropna(subset=["MarketCap"]).copy()
    s = s.sort_values("MarketCap", ascending=False).drop_duplicates(["Code"], keep="first")

    uni = s.head(TOP_N).copy()
    uni["Rank_MarketCap"] = np.arange(1, len(uni) + 1)
    uni = uni[["MonthEnd", "Code", "MarketCap", "Rank_MarketCap"]].copy()

    log_line(f"[UNIVERSE] latest MonthEnd={latest_me.date()} candidates={len(s):,} top{TOP_N}={len(uni):,}")
    uni.to_csv(DIAG_UNIVERSE, index=False, encoding="utf-8-sig")
    return uni


# ============================================================
# yfinance dividends fetch (symbol rule: 4-digit.T first for digit5 ending 0)
# ============================================================
def make_symbol_candidates(code: str) -> List[str]:
    code = normalize_code(code)
    if not code:
        return []
    cands = []
    if code.isdigit() and len(code) == 5 and code.endswith("0"):
        c4 = code[:-1]
        cands.append(f"{c4}.T")     # primary
        cands.append(f"{code}.T")   # secondary
    else:
        cands.append(f"{code}.T")   # alphanumeric or other cases
    # dedup keep order
    out, seen = [], set()
    for x in cands:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out


def fetch_dividends(symbol: str) -> Tuple[bool, Optional[pd.Series], str]:
    try:
        t = yf.Ticker(symbol)
        div = t.dividends
        if div is None or not isinstance(div, pd.Series):
            return False, None, "dividends is None or not Series"
        return True, div, ""
    except Exception as e:
        return False, None, f"{type(e).__name__}: {e}"


def build_dividend_table_for_top200(uni: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Returns:
      - div_events_df: columns [Code, symbol, date, dividend] (event-level)
      - fetch_results_df: per-code summary
    """
    attempt_rows = []
    results_rows = []
    div_samples = []

    div_events = []

    for _, r in uni.iterrows():
        code = r["Code"]
        cands = make_symbol_candidates(code)

        best_ok = False
        best_symbol = ""
        best_div = None
        last_err = ""

        for sym in cands:
            ok, div, err = fetch_dividends(sym)
            attempt_rows.append({"Code": code, "symbol": sym, "success_series": bool(ok), "error": err})
            if not ok:
                last_err = err
                time.sleep(SLEEP_SEC)
                continue

            best_ok = True
            best_symbol = sym
            best_div = div.dropna().copy()
            time.sleep(SLEEP_SEC)
            break

        if not best_ok:
            results_rows.append({
                "Code": code,
                "ok": False,
                "symbol": "",
                "div_rows": 0,
                "div_min_date": "",
                "div_max_date": "",
                "div_sum": np.nan,
                "div_last": np.nan,
                "attempted": "|".join(cands),
                "error": last_err or "all_candidates_failed",
            })
            continue

        # summarize
        if best_div is None or len(best_div) == 0:
            results_rows.append({
                "Code": code,
                "ok": True,
                "symbol": best_symbol,
                "div_rows": 0,
                "div_min_date": "",
                "div_max_date": "",
                "div_sum": np.nan,
                "div_last": np.nan,
                "attempted": "|".join(cands),
                "error": "",
            })
            continue

        idx = pd.to_datetime(best_div.index, errors="coerce")
        results_rows.append({
            "Code": code,
            "ok": True,
            "symbol": best_symbol,
            "div_rows": int(len(best_div)),
            "div_min_date": str(pd.Timestamp(idx.min()).date()) if pd.notna(idx.min()) else "",
            "div_max_date": str(pd.Timestamp(idx.max()).date()) if pd.notna(idx.max()) else "",
            "div_sum": float(best_div.sum()),
            "div_last": float(best_div.iloc[-1]),
            "attempted": "|".join(cands),
            "error": "",
        })

        # keep samples
        tail = best_div.tail(MAX_DIV_SAMPLE)
        for dt, val in tail.items():
            div_samples.append({
                "Code": code,
                "symbol": best_symbol,
                "date": str(pd.Timestamp(dt).date()),
                "dividend": float(val),
            })

        # store all events
        for dt, val in best_div.items():
            div_events.append({
                "Code": code,
                "symbol": best_symbol,
                "date": pd.Timestamp(dt).normalize(),
                "dividend": float(val),
            })

    attempts_df = pd.DataFrame(attempt_rows)
    results_df = pd.DataFrame(results_rows)
    div_events_df = pd.DataFrame(div_events)
    div_samples_df = pd.DataFrame(div_samples)

    attempts_df.to_csv(DIAG_ATTEMPTS, index=False, encoding="utf-8-sig")
    results_df.to_csv(DIAG_FETCH_RESULTS, index=False, encoding="utf-8-sig")
    if len(div_samples_df):
        div_samples_df.to_csv(DIAG_DIV_RAW_SAMPLE, index=False, encoding="utf-8-sig")

    log_line(f"[SAVE] attempts={DIAG_ATTEMPTS}")
    log_line(f"[SAVE] fetch_results={DIAG_FETCH_RESULTS}")
    if len(div_samples_df):
        log_line(f"[SAVE] div_samples={DIAG_DIV_RAW_SAMPLE}")

    return div_events_df, results_df


# ============================================================
# Build DY_TTM on MonthEnd
# ============================================================
def compute_dy_ttm_for_monthends(uni: pd.DataFrame, div_events_df: pd.DataFrame) -> pd.DataFrame:
    """
    For each (Code, MonthEnd) in the analysis window (we will use all MonthEnd available in price data),
    compute Div_TTM = sum(dividends in (MonthEnd-365d, MonthEnd]) and DY_TTM = Div_TTM / AdjustedClose.
    """
    price = pd.read_parquet(PRICE_PATH)[["Code", "MonthEnd", "AdjustedClose"]].copy()
    price["Code"] = price["Code"].astype(str).map(normalize_code)
    price = price[price["Code"] != ""].copy()
    price["MonthEnd"] = pd.to_datetime(price["MonthEnd"], errors="coerce").dt.normalize()
    price["AdjustedClose"] = pd.to_numeric(price["AdjustedClose"], errors="coerce")

    # restrict to top200 codes only
    codes = set(uni["Code"].unique())
    price = price[price["Code"].isin(codes)].copy()

    if div_events_df.empty:
        price["Div_TTM"] = np.nan
        price["DY_TTM"] = np.nan
        return price

    div_events_df = div_events_df.copy()
    div_events_df["Code"] = div_events_df["Code"].astype(str).map(normalize_code)
    div_events_df["date"] = pd.to_datetime(div_events_df["date"], errors="coerce").dt.normalize()
    div_events_df["dividend"] = pd.to_numeric(div_events_df["dividend"], errors="coerce")
    div_events_df = div_events_df.dropna(subset=["Code", "date", "dividend"]).copy()

    # pre-sort
    div_events_df = div_events_df.sort_values(["Code", "date"]).reset_index(drop=True)
    price = price.sort_values(["Code", "MonthEnd"]).reset_index(drop=True)

    # compute Div_TTM per row by rolling window
    # For speed, do per-code loop (top200なら十分速い)
    out_rows = []
    for code, g in price.groupby("Code", sort=False):
        gg = g.copy()
        ev = div_events_df[div_events_df["Code"] == code][["date", "dividend"]].copy()
        ev = ev.sort_values("date")
        if ev.empty:
            gg["Div_TTM"] = np.nan
            gg["DY_TTM"] = np.nan
            out_rows.append(gg)
            continue

        # convert to arrays for quick slicing
        ev_dates = ev["date"].to_numpy(dtype="datetime64[ns]")
        ev_vals = ev["dividend"].to_numpy(dtype=float)

        div_ttm_list = []
        for me in gg["MonthEnd"].to_numpy(dtype="datetime64[ns]"):
            start = me - np.timedelta64(365, "D")
            # find events in (start, me]
            # numpy searchsorted
            left = np.searchsorted(ev_dates, start, side="right")
            right = np.searchsorted(ev_dates, me, side="right")
            s = ev_vals[left:right].sum() if right > left else 0.0
            div_ttm_list.append(float(s))

        gg["Div_TTM"] = div_ttm_list
        gg["DY_TTM"] = gg["Div_TTM"] / gg["AdjustedClose"]
        out_rows.append(gg)

    dy = pd.concat(out_rows, ignore_index=True)
    return dy


# ============================================================
# Build FF5+MOM+DY factors monthly (top200 universe)
# ============================================================
def load_snapshot_features_top200(uni: pd.DataFrame, dy_df: pd.DataFrame) -> pd.DataFrame:
    """
    Load month_end_snapshot, restrict to top200 codes, merge:
      - MarketCap, BM_Ratio, ROE, INV_Growth
      - ret_m_fwd, MOM_12_1 from price
      - DY_TTM from yfinance
    """
    snap = pd.read_parquet(SNAP_PATH).copy()
    snap["Code"] = snap["Code"].astype(str).map(normalize_code)
    snap = snap[snap["Code"] != ""].copy()
    snap["MonthEnd"] = pd.to_datetime(snap["MonthEnd"], errors="coerce").dt.normalize()

    codes = set(uni["Code"].unique())
    snap = snap[snap["Code"].isin(codes)].copy()

    # price returns and MOM
    price = pd.read_parquet(PRICE_PATH)[["Code", "MonthEnd", "AdjustedClose"]].copy()
    price["Code"] = price["Code"].astype(str).map(normalize_code)
    price = price[price["Code"] != ""].copy()
    price["MonthEnd"] = pd.to_datetime(price["MonthEnd"], errors="coerce").dt.normalize()
    price["AdjustedClose"] = pd.to_numeric(price["AdjustedClose"], errors="coerce")
    price = price.sort_values(["Code", "MonthEnd"], kind="mergesort")

    price["Adj_next"] = price.groupby("Code")["AdjustedClose"].shift(-1)
    price["ret_m_fwd"] = (price["Adj_next"] / price["AdjustedClose"]) - 1.0
    price["Adj_lag1"] = price.groupby("Code")["AdjustedClose"].shift(1)
    price["Adj_lag12"] = price.groupby("Code")["AdjustedClose"].shift(12)
    price["MOM_12_1"] = (price["Adj_lag1"] / price["Adj_lag12"]) - 1.0

    pr = price[["Code", "MonthEnd", "ret_m_fwd", "MOM_12_1", "AdjustedClose"]].copy()

    # merge snap + pr
    df = snap.merge(pr, on=["Code", "MonthEnd"], how="left", validate="many_to_one")

    # merge DY
    dy_df2 = dy_df[["Code", "MonthEnd", "Div_TTM", "DY_TTM"]].copy()
    dy_df2["MonthEnd"] = pd.to_datetime(dy_df2["MonthEnd"], errors="coerce").dt.normalize()
    df = df.merge(dy_df2, on=["Code", "MonthEnd"], how="left", validate="many_to_one")

    # numeric
    for c in ["MarketCap", "BM_Ratio", "ROE", "INV_Growth", "ret_m_fwd", "MOM_12_1", "DY_TTM"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    # basic filters: need MarketCap + ret for factor payoffs; value axis needs BM or later
    df = df[(df["MarketCap"].notna()) & (df["MarketCap"] > 0)].copy()
    df = df[df["ret_m_fwd"].notna()].copy()

    # winsorize (global simple; PoCとしては十分)
    df["ret_w"] = winsorize(df["ret_m_fwd"], Q_RET)
    df["BM_w"]  = winsorize(df["BM_Ratio"], Q_BM)
    df["ROE_w"] = winsorize(df["ROE"], Q_ROE)
    df["INV_w"] = winsorize(df["INV_Growth"], Q_INV)
    df["MOM_w"] = winsorize(df["MOM_12_1"], Q_MOM)
    df["DY_w"]  = winsorize(df["DY_TTM"], Q_DY)

    # DY coverage diag
    cov = (df.groupby("MonthEnd")
             .agg(n=("Code", "nunique"),
                  dy_notna=("DY_TTM", lambda s: float(s.notna().mean()*100)),
                  bm_notna=("BM_Ratio", lambda s: float(s.notna().mean()*100)),
                  mcap_notna=("MarketCap", lambda s: float(s.notna().mean()*100)))
             .reset_index())
    cov.to_csv(DIAG_DY_TTM_BY_MONTH, index=False, encoding="utf-8-sig")

    return df


def compute_ff5_mom_dy_monthly(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    months = sorted(df["MonthEnd"].dropna().unique())
    rows = []
    diag_n = []

    for me in months:
        m = df[df["MonthEnd"] == me].copy()
        n_total = int(m["Code"].nunique())
        if n_total < 30:
            continue

        # Market
        MKT = value_weighted_return(m, "ret_w", "MarketCap")

        # splits
        m["SZ"] = size_split_median(m["MarketCap"])

        m["VAL"]  = qcut_3way(m["BM_w"],  P30, P70, labels=("L","M","H"))
        m["PROF"] = qcut_3way(m["ROE_w"], P30, P70, labels=("W","M","R"))
        m["INV"]  = qcut_3way(m["INV_w"], P30, P70, labels=("C","M","A"))
        m["MOM"]  = qcut_3way(m["MOM_w"], P30, P70, labels=("L","M","W"))
        m["DY"]   = qcut_3way(m["DY_w"],  P30, P70, labels=("L","M","H"))

        def port_ret(sz, col, lab):
            x = m[(m["SZ"] == sz) & (m[col] == lab)]
            return value_weighted_return(x, "ret_w", "MarketCap")

        # HML
        SH = port_ret("S","VAL","H"); SL = port_ret("S","VAL","L")
        BH = port_ret("B","VAL","H"); BL = port_ret("B","VAL","L")
        HML = np.nanmean([SH, BH]) - np.nanmean([SL, BL])

        # RMW
        SR = port_ret("S","PROF","R"); SW = port_ret("S","PROF","W")
        BR = port_ret("B","PROF","R"); BW = port_ret("B","PROF","W")
        RMW = np.nanmean([SR, BR]) - np.nanmean([SW, BW])

        # CMA
        SC = port_ret("S","INV","C"); SA = port_ret("S","INV","A")
        BC = port_ret("B","INV","C"); BA = port_ret("B","INV","A")
        CMA = np.nanmean([SC, BC]) - np.nanmean([SA, BA])

        # SMB (3 legs avg; top200なので簡略)
        SMv = port_ret("S","VAL","M"); BMv = port_ret("B","VAL","M")
        SMB_HML = np.nanmean([SH, SMv, SL]) - np.nanmean([BH, BMv, BL])
        SMp = port_ret("S","PROF","M"); BMp = port_ret("B","PROF","M")
        SMB_RMW = np.nanmean([SR, SMp, SW]) - np.nanmean([BR, BMp, BW])
        SMa = port_ret("S","INV","M"); BMa = port_ret("B","INV","M")
        SMB_CMA = np.nanmean([SC, SMa, SA]) - np.nanmean([BC, BMa, BA])
        SMB = np.nanmean([SMB_HML, SMB_RMW, SMB_CMA])

        # WML
        SWin = port_ret("S","MOM","W"); SLos = port_ret("S","MOM","L")
        BWin = port_ret("B","MOM","W"); BLos = port_ret("B","MOM","L")
        WML = np.nanmean([SWin, BWin]) - np.nanmean([SLos, BLos])

        # DY factor (HDYMLDY)
        SHD = port_ret("S","DY","H"); SLD = port_ret("S","DY","L")
        BHD = port_ret("B","DY","H"); BLD = port_ret("B","DY","L")
        HDYMLDY = np.nanmean([SHD, BHD]) - np.nanmean([SLD, BLD])

        # n_used
        n_used_dy = int(m.dropna(subset=["DY_w","ret_w","MarketCap"])["Code"].nunique())
        conf = min(1.0, n_used_dy / TOP_N)
        abnormal = (n_used_dy < MIN_STOCKS_PER_MONTH)

        rows.append({
            "MonthEnd": me,
            "MKT": MKT,
            "SMB": SMB,
            "HML": HML,
            "RMW": RMW,
            "CMA": CMA,
            "WML": WML,
            "HDYMLDY": HDYMLDY,
            "n_total": n_total,
            "n_used_dy": n_used_dy,
            "confidence": float(conf),
            "abnormal": bool(abnormal),
        })

        diag_n.append({
            "MonthEnd": me,
            "n_total": n_total,
            "n_used_dy": n_used_dy,
            "confidence": float(conf),
            "abnormal": bool(abnormal),
        })

    fac = pd.DataFrame(rows).sort_values("MonthEnd").reset_index(drop=True)
    diag_n_df = pd.DataFrame(diag_n).sort_values("MonthEnd").reset_index(drop=True)
    diag_n_df.to_csv(DIAG_DY_FACTOR_NUSED, index=False, encoding="utf-8-sig")
    return fac, diag_n_df


def analyze_latest(fac: pd.DataFrame) -> Dict:
    if fac.empty:
        return {"error": "factor table is empty"}
    last_me = latest_evaluable_month(fac, col="MKT")
    cur = fac[fac["MonthEnd"] == last_me].iloc[0].to_dict()

    factor_cols = ["MKT","SMB","HML","RMW","CMA","WML","HDYMLDY"]
    absvals = {c: abs(cur.get(c, np.nan)) for c in factor_cols}
    dominant = max(absvals, key=lambda k: (-np.nan_to_num(absvals[k], nan=-1), k))

    fs = fac.sort_values("MonthEnd").reset_index(drop=True)
    last3 = fs.tail(3)
    last12 = fs.tail(12)

    trends3 = {c: trend_label(last3[c].to_numpy()) for c in factor_cols}
    trends12 = {c: trend_label(last12[c].to_numpy()) for c in factor_cols}

    # regime from MKT
    mkt = cur.get("MKT", np.nan)
    regime = "RANGE"
    if np.isfinite(mkt):
        if mkt > 0.01:
            regime = "BULL"
        elif mkt < -0.01:
            regime = "BEAR"

    # tilt from last3 avg sign (confidence weighted)
    conf = float(cur.get("confidence", 0.0))
    tilt = {}
    for c in factor_cols:
        v = float(np.nanmean(last3[c]))
        tilt[c] = float(np.sign(v)) if np.isfinite(v) else 0.0
    tilt_conf = {k: v * conf for k, v in tilt.items()}

    return {
        "latest_month": str(pd.Timestamp(last_me).date()),
        "dominant_factor": dominant,
        "regime": regime,
        "latest": cur,
        "trends_last3": trends3,
        "trends_last12": trends12,
        "tilt_conf_weighted": tilt_conf,
    }


def save_outputs(fac: pd.DataFrame, analysis: Dict):
    fac.to_parquet(OUT_PARQUET, index=False)

    # summary
    latest = analysis.get("latest", {})
    summary = {
        "latest_month": analysis.get("latest_month", ""),
        "regime": analysis.get("regime", ""),
        "dominant_factor": analysis.get("dominant_factor", ""),
        "confidence": float(latest.get("confidence", np.nan)),
        "abnormal": bool(latest.get("abnormal", True)),
        "n_total": int(latest.get("n_total", 0) or 0),
        "n_used_dy": int(latest.get("n_used_dy", 0) or 0),
    }
    for c in ["MKT","SMB","HML","RMW","CMA","WML","HDYMLDY"]:
        summary[c] = float(latest.get(c, np.nan))
    pd.DataFrame([summary]).to_csv(OUT_SUMMARY, index=False, encoding="utf-8-sig")

    # trends
    fac_sorted = fac.sort_values("MonthEnd").reset_index(drop=True)
    cols = ["MonthEnd","MKT","SMB","HML","RMW","CMA","WML","HDYMLDY"]
    fac_sorted.tail(3)[cols].to_csv(OUT_TREND_LAST3, index=False, encoding="utf-8-sig")
    fac_sorted.tail(12)[cols].to_csv(OUT_TREND_LAST12, index=False, encoding="utf-8-sig")


def main():
    with open(DIAG_LOG, "w", encoding="utf-8") as f:
        f.write("")

    log_line("="*110)
    log_line("FF5 + MOM(12-1) + DY_TTM (TTM dividend yield) on Top200 universe")
    log_line("="*110)
    log_line(f"PRICE_PATH: {PRICE_PATH}")
    log_line(f"SNAP_PATH : {SNAP_PATH}")
    log_line(f"DIAG_DIR  : {DIAG_DIR}")

    # 1) universe
    uni = pick_top200_universe()

    # 2) yfinance dividends
    div_events_df, fetch_results_df = build_dividend_table_for_top200(uni)

    ok_any = int(fetch_results_df["ok"].sum()) if len(fetch_results_df) else 0
    ok_nonempty = int((fetch_results_df["ok"] & (fetch_results_df["div_rows"] > 0)).sum()) if len(fetch_results_df) else 0
    log_line(f"[YF] ok(series)={ok_any}/{len(fetch_results_df)} | nonempty={ok_nonempty}/{len(fetch_results_df)}")

    # 3) DY_TTM
    dy_df = compute_dy_ttm_for_monthends(uni, div_events_df)
    # merge DY to check coverage quickly
    cov_latest = (dy_df[dy_df["MonthEnd"] == dy_df["MonthEnd"].max()]
                  .assign(dy_notna=lambda x: x["DY_TTM"].notna())
                  ["dy_notna"].mean() * 100.0)
    log_line(f"[DY] latest MonthEnd={dy_df['MonthEnd'].max().date()} DY_TTM notna%={cov_latest:.2f}")

    # 4) load features and compute factors
    feat = load_snapshot_features_top200(uni, dy_df)
    log_line(f"[FEATURE] rows={len(feat):,} codes={feat['Code'].nunique():,} months={feat['MonthEnd'].nunique():,}")

    fac, diag_n = compute_ff5_mom_dy_monthly(feat)
    log_line(f"[FACTORS] months computed={len(fac)} min={fac['MonthEnd'].min()} max={fac['MonthEnd'].max()}")

    if fac.empty:
        raise RuntimeError("Factor result is empty. Check DY/BM coverage and thresholds.")

    analysis = analyze_latest(fac)
    last = analysis.get("latest", {})
    log_line("-"*110)
    log_line(f"[LATEST EVALUABLE MONTH] {analysis.get('latest_month')}")
    log_line(f"[DOMINANT FACTOR] {analysis.get('dominant_factor')}")
    log_line(f"[REGIME] {analysis.get('regime')}")
    log_line(f"[CONFIDENCE] {last.get('confidence'):.3f} | abnormal={last.get('abnormal')} | "
             f"n_total={last.get('n_total')} n_used_dy={last.get('n_used_dy')}")
    log_line("Factor returns:")
    for c in ["MKT","SMB","HML","RMW","CMA","WML","HDYMLDY"]:
        log_line(f"  {c}: {last.get(c)}")

    log_line("-"*110)
    log_line("[STRATEGY HINT] (last3 avg direction, confidence-weighted)")
    for c, score in analysis.get("tilt_conf_weighted", {}).items():
        if score > 0:
            s = "OVERWEIGHT"
        elif score < 0:
            s = "UNDERWEIGHT"
        else:
            s = "NEUTRAL"
        log_line(f"  {c}: {s} (score={score:.2f})")

    # 5) save
    save_outputs(fac, analysis)
    log_line("-"*110)
    log_line("✅ saved:")
    log_line(f"  - {OUT_PARQUET}")
    log_line(f"  - {OUT_SUMMARY}")
    log_line(f"  - {OUT_TREND_LAST3}")
    log_line(f"  - {OUT_TREND_LAST12}")
    log_line(f"  - {DIAG_UNIVERSE}")
    log_line(f"  - {DIAG_ATTEMPTS}")
    log_line(f"  - {DIAG_FETCH_RESULTS}")
    log_line(f"  - {DIAG_DY_TTM_BY_MONTH}")
    log_line(f"  - {DIAG_DY_FACTOR_NUSED}")
    log_line(f"  - {DIAG_LOG}")


if __name__ == "__main__":
    main()


FF5 + MOM(12-1) + DY_TTM (TTM dividend yield) on Top200 universe
PRICE_PATH: C:\Users\yongr\Project\merged_data_all_stocks\factors\price_month_end.parquet
SNAP_PATH : C:\Users\yongr\Project\merged_data_all_stocks\factors\month_end_snapshot.parquet
DIAG_DIR  : C:\Users\yongr\Project\merged_data_all_stocks\factors\diag_yfinance_dividend_poc_dy
[UNIVERSE] latest MonthEnd=2025-09-30 candidates=3,428 top200=200


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: 167A0.T"}}}
$167A0.T: possibly delisted; no timezone found


[SAVE] attempts=C:\Users\yongr\Project\merged_data_all_stocks\factors\diag_yfinance_dividend_poc_dy\diag_yf_symbol_attempts.csv
[SAVE] fetch_results=C:\Users\yongr\Project\merged_data_all_stocks\factors\diag_yfinance_dividend_poc_dy\diag_yf_dividend_fetch_results.csv
[SAVE] div_samples=C:\Users\yongr\Project\merged_data_all_stocks\factors\diag_yfinance_dividend_poc_dy\diag_yf_dividend_success_samples.csv
[YF] ok(series)=200/200 | nonempty=198/200
[DY] latest MonthEnd=2026-01-31 DY_TTM notna%=99.00
[FEATURE] rows=21,261 codes=200 months=117
[FACTORS] months computed=117 min=2016-01-31 00:00:00 max=2025-09-30 00:00:00
--------------------------------------------------------------------------------------------------------------
[LATEST EVALUABLE MONTH] 2025-09-30
[DOMINANT FACTOR] HDYMLDY
[REGIME] BULL
[CONFIDENCE] 0.990 | abnormal=True | n_total=200 n_used_dy=198
Factor returns:
  MKT: 0.04527692742714705
  SMB: -0.04833973863909095
  HML: 0.029368950744556546
  RMW: 0.017767223339891492

In [3]:
# -*- coding: utf-8 -*-
"""
FF5 + MOM(12-1) + DY_TTM（TTM dividend yield） on Top200 universe
+ yfinance dividends cache per symbol (parquet) with incremental update

目的:
- J-Quants Standardでは配当APIが使えないため、yfinance（Yahoo Finance）から配当を取得
- 毎回取り直すのを避けるため、銘柄別に配当履歴をキャッシュし、必要時のみ更新する
- DY_TTM = 過去12ヶ月配当合計 / 月末株価(AdjustedClose)
- 因子: HDYMLDY = High DY - Low DY（Sizeニュートラル：Small/Big平均）
- 出力: top200向けのFF5+MOM+DY因子と診断CSV

注意:
- yfinanceは非公式で、Yahoo側の仕様変更で壊れる可能性があります
- dividendsの日付は支払日ではなく権利落ち日中心のことがあるため、TTM合計を採用。[Source](https://github.com/ranaroussi/yfinance/issues/568)

依存:
  pip install yfinance pandas pyarrow
"""

import warnings
warnings.filterwarnings("ignore")

import time
from pathlib import Path
from typing import Dict, Tuple, List, Optional

import numpy as np
import pandas as pd
import yfinance as yf


# ============================================================
# Paths (あなたの環境)
# ============================================================
FACTORS_DIR = Path(r"C:\Users\yongr\Project\merged_data_all_stocks\factors")
PRICE_PATH = FACTORS_DIR / "price_month_end.parquet"
SNAP_PATH  = FACTORS_DIR / "month_end_snapshot.parquet"

# 出力（既存ファイルは上書きしない：別名で保存）
OUT_PARQUET = FACTORS_DIR / "ff5_mom_dy_factors_monthly_top200.parquet"
OUT_SUMMARY = FACTORS_DIR / "market_factor_summary_ff5_mom_dy_top200.csv"
OUT_TREND_LAST3 = FACTORS_DIR / "market_factor_trend_last3_ff5_mom_dy_top200.csv"
OUT_TREND_LAST12 = FACTORS_DIR / "market_factor_trend_last12_ff5_mom_dy_top200.csv"

# diagnostics + cache
DIAG_DIR = FACTORS_DIR / "diag_yfinance_dividend_poc_dy"
DIAG_DIR.mkdir(parents=True, exist_ok=True)

CACHE_DIR = FACTORS_DIR / "diag_yfinance_dividend_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_EVENTS_DIR = CACHE_DIR / "events_by_symbol"
CACHE_EVENTS_DIR.mkdir(parents=True, exist_ok=True)

DIAG_UNIVERSE = DIAG_DIR / "diag_universe_top200.csv"
DIAG_ATTEMPTS = DIAG_DIR / "diag_yf_symbol_attempts.csv"
DIAG_FETCH_RESULTS = DIAG_DIR / "diag_yf_dividend_fetch_results.csv"
DIAG_DIV_RAW_SAMPLE = DIAG_DIR / "diag_yf_dividend_success_samples.csv"
DIAG_DY_TTM_BY_MONTH = DIAG_DIR / "diag_dy_ttm_coverage_by_month.csv"
DIAG_DY_FACTOR_NUSED = DIAG_DIR / "diag_dy_n_used_by_month.csv"
DIAG_CACHE_SUMMARY = DIAG_DIR / "diag_yf_cache_summary.csv"
DIAG_LOG = DIAG_DIR / "diag_runtime_log.txt"


# ============================================================
# Params
# ============================================================
TOP_N = 200

# top200前提なので500は無理。欠損で少し落ちても abnormal になりにくい閾値
MIN_STOCKS_PER_MONTH = 150
P30 = 0.30
P70 = 0.70

# winsorize quantiles
Q_RET = (0.01, 0.99)
Q_BM  = (0.01, 0.99)
Q_ROE = (0.01, 0.99)
Q_INV = (0.01, 0.99)
Q_MOM = (0.01, 0.99)
Q_DY  = (0.01, 0.99)

BAD_CODE_STRINGS = {"None", "nan", "", "NaN", "NULL", "null"}

# yfinance polite access
SLEEP_SEC = 0.15
MAX_DIV_SAMPLE = 15

# ===== キャッシュ更新ポリシー =====
# TTL_DAYS 以内に取得したキャッシュは再取得しない（差分更新の代わりに「必要時のみ再取得」方式）
# まずはPoCとして安定性優先。運用時に短くしてもOK。
TTL_DAYS = 14


# ============================================================
# Utils
# ============================================================
def log_line(s: str):
    print(s)
    with open(DIAG_LOG, "a", encoding="utf-8") as f:
        f.write(s + "\n")


def normalize_code(code: str) -> str:
    if code is None:
        return ""
    s = str(code).strip()
    if s in BAD_CODE_STRINGS:
        return ""
    return s


def winsorize(s: pd.Series, q=(0.01, 0.99)) -> pd.Series:
    x = pd.to_numeric(s, errors="coerce")
    if x.notna().sum() == 0:
        return x
    lo = x.quantile(q[0])
    hi = x.quantile(q[1])
    return x.clip(lo, hi)


def value_weighted_return(df: pd.DataFrame, ret_col: str, w_col: str) -> float:
    x = df[[ret_col, w_col]].dropna()
    if x.empty:
        return np.nan
    w = x[w_col].astype(float).to_numpy()
    r = x[ret_col].astype(float).to_numpy()
    wsum = w.sum()
    if not np.isfinite(wsum) or wsum <= 0:
        return np.nan
    return float(np.dot(r, w) / wsum)


def qcut_3way(x: pd.Series, p30=0.3, p70=0.7, labels=("L", "M", "H")) -> pd.Series:
    a = pd.to_numeric(x, errors="coerce")
    q1 = a.quantile(p30)
    q2 = a.quantile(p70)
    out = pd.Series(index=a.index, dtype="object")
    out[a <= q1] = labels[0]
    out[(a > q1) & (a < q2)] = labels[1]
    out[a >= q2] = labels[2]
    return out


def size_split_median(mcap: pd.Series) -> pd.Series:
    a = pd.to_numeric(mcap, errors="coerce")
    med = a.quantile(0.5)
    out = pd.Series(index=a.index, dtype="object")
    out[a <= med] = "S"
    out[a > med] = "B"
    return out


def trend_label(vals: np.ndarray) -> str:
    vals = np.array(vals, dtype=float)
    vals = vals[np.isfinite(vals)]
    if len(vals) < 2:
        return "NA"
    x = np.arange(len(vals))
    slope = np.polyfit(x, vals, 1)[0]
    if abs(slope) < 1e-6:
        return "FLAT"
    return "UP" if slope > 0 else "DOWN"


def latest_evaluable_month(factors: pd.DataFrame, col="MKT") -> pd.Timestamp:
    f = factors.dropna(subset=[col]).copy()
    if f.empty:
        return pd.NaT
    return pd.Timestamp(f["MonthEnd"].max())


# ============================================================
# Universe: latest MonthEnd top200 by MarketCap
# ============================================================
def pick_top200_universe() -> pd.DataFrame:
    snap = pd.read_parquet(SNAP_PATH)
    snap["Code"] = snap["Code"].astype(str).map(normalize_code)
    snap = snap[snap["Code"] != ""].copy()
    snap["MonthEnd"] = pd.to_datetime(snap["MonthEnd"], errors="coerce").dt.normalize()
    snap["MarketCap"] = pd.to_numeric(snap["MarketCap"], errors="coerce")

    latest_me = snap["MonthEnd"].max()
    s = snap[snap["MonthEnd"] == latest_me].dropna(subset=["MarketCap"]).copy()
    s = s.sort_values("MarketCap", ascending=False).drop_duplicates(["Code"], keep="first")

    uni = s.head(TOP_N).copy()
    uni["Rank_MarketCap"] = np.arange(1, len(uni) + 1)
    uni = uni[["MonthEnd", "Code", "MarketCap", "Rank_MarketCap"]].copy()

    log_line(f"[UNIVERSE] latest MonthEnd={latest_me.date()} candidates={len(s):,} top{TOP_N}={len(uni):,}")
    uni.to_csv(DIAG_UNIVERSE, index=False, encoding="utf-8-sig")
    return uni


# ============================================================
# yfinance symbols
# ============================================================
def make_symbol_candidates(code: str) -> List[str]:
    code = normalize_code(code)
    if not code:
        return []
    cands = []
    # 5桁数字末尾0 -> 4桁.T を優先
    if code.isdigit() and len(code) == 5 and code.endswith("0"):
        c4 = code[:-1]
        cands.append(f"{c4}.T")     # primary
        cands.append(f"{code}.T")   # secondary
    else:
        cands.append(f"{code}.T")   # alphanumeric or other cases

    # dedup keep order
    out, seen = [], set()
    for x in cands:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out


def fetch_dividends(symbol: str) -> Tuple[bool, Optional[pd.Series], str]:
    try:
        t = yf.Ticker(symbol)
        div = t.dividends
        if div is None or not isinstance(div, pd.Series):
            return False, None, "dividends is None or not Series"
        return True, div, ""
    except Exception as e:
        return False, None, f"{type(e).__name__}: {e}"


# ============================================================
# Cache I/O
# ============================================================
def _cache_path_for_symbol(symbol: str) -> Path:
    # ファイル名に使えない文字を避ける（. は OK）
    safe = symbol.replace("/", "_")
    return CACHE_EVENTS_DIR / f"{safe}.parquet"


def load_div_cache(symbol: str) -> Optional[pd.DataFrame]:
    fp = _cache_path_for_symbol(symbol)
    if not fp.exists():
        return None
    try:
        df = pd.read_parquet(fp)
    except Exception:
        return None
    if df.empty:
        return df
    # normalize
    df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.normalize()
    df["dividend"] = pd.to_numeric(df["dividend"], errors="coerce")
    df = df.dropna(subset=["date", "dividend"]).drop_duplicates(["date"], keep="last").sort_values("date")
    return df.reset_index(drop=True)


def save_div_cache(symbol: str, df: pd.DataFrame):
    fp = _cache_path_for_symbol(symbol)
    fp.parent.mkdir(parents=True, exist_ok=True)
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.normalize()
    df["dividend"] = pd.to_numeric(df["dividend"], errors="coerce")
    df = df.dropna(subset=["date", "dividend"]).drop_duplicates(["date"], keep="last").sort_values("date")
    df.to_parquet(fp, index=False)


def cache_is_fresh(symbol: str) -> bool:
    """
    TTL方式。ファイル更新日時でfresh判定。
    """
    fp = _cache_path_for_symbol(symbol)
    if not fp.exists():
        return False
    mtime = pd.Timestamp(fp.stat().st_mtime, unit="s")
    age_days = (pd.Timestamp.now() - mtime).total_seconds() / 86400.0
    return age_days <= TTL_DAYS


def merge_cache_and_new(cache_df: Optional[pd.DataFrame], new_series: pd.Series) -> pd.DataFrame:
    """
    cache_df: columns [date, dividend]
    new_series: pandas Series indexed by date
    """
    new_df = pd.DataFrame({
        "date": pd.to_datetime(new_series.index, errors="coerce").normalize(),
        "dividend": pd.to_numeric(new_series.values, errors="coerce"),
    }).dropna(subset=["date", "dividend"]).drop_duplicates(["date"], keep="last")

    if cache_df is None or cache_df.empty:
        out = new_df
    else:
        out = pd.concat([cache_df, new_df], ignore_index=True)
        out["date"] = pd.to_datetime(out["date"], errors="coerce").dt.normalize()
        out["dividend"] = pd.to_numeric(out["dividend"], errors="coerce")
        out = out.dropna(subset=["date", "dividend"]).drop_duplicates(["date"], keep="last")

    out = out.sort_values("date").reset_index(drop=True)
    return out


# ============================================================
# Fetch dividends with cache
# ============================================================
def build_dividend_table_for_top200_with_cache(uni: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Returns:
      - div_events_df: [Code, symbol, date, dividend] (event-level)
      - fetch_results_df: per-code summary

    Cache policy:
      - Try symbol candidates in order.
      - If primary symbol cache exists and is fresh -> use cache without hitting network.
      - Else fetch from yfinance, merge with cache, save back.
    """
    attempt_rows = []
    results_rows = []
    div_samples = []
    div_events = []

    cache_hit = 0
    cache_miss = 0
    updated_cnt = 0

    for _, r in uni.iterrows():
        code = r["Code"]
        cands = make_symbol_candidates(code)

        best_ok = False
        best_symbol = ""
        best_df = None
        last_err = ""

        for sym in cands:
            used_cache = False

            # 1) if cache exists and fresh, use it
            if cache_is_fresh(sym):
                cdf = load_div_cache(sym)
                if cdf is not None:
                    best_ok = True
                    best_symbol = sym
                    best_df = cdf
                    used_cache = True
                    cache_hit += 1
                    attempt_rows.append({"Code": code, "symbol": sym, "success_series": True, "error": "", "used_cache": True})
                    break

            # 2) else fetch from yfinance and update cache
            ok, div, err = fetch_dividends(sym)
            attempt_rows.append({"Code": code, "symbol": sym, "success_series": bool(ok), "error": err, "used_cache": False})

            if not ok:
                last_err = err
                time.sleep(SLEEP_SEC)
                continue

            cache_miss += 1
            cache_df = load_div_cache(sym)
            merged_df = merge_cache_and_new(cache_df, div.dropna())
            save_div_cache(sym, merged_df)
            updated_cnt += 1

            best_ok = True
            best_symbol = sym
            best_df = merged_df

            time.sleep(SLEEP_SEC)
            break

        if not best_ok:
            results_rows.append({
                "Code": code,
                "ok": False,
                "symbol": "",
                "div_rows": 0,
                "div_min_date": "",
                "div_max_date": "",
                "div_sum": np.nan,
                "div_last": np.nan,
                "attempted": "|".join(cands),
                "error": last_err or "all_candidates_failed",
            })
            continue

        # summarize from best_df
        best_df = best_df if best_df is not None else pd.DataFrame(columns=["date", "dividend"])
        if best_df.empty:
            results_rows.append({
                "Code": code,
                "ok": True,
                "symbol": best_symbol,
                "div_rows": 0,
                "div_min_date": "",
                "div_max_date": "",
                "div_sum": np.nan,
                "div_last": np.nan,
                "attempted": "|".join(cands),
                "error": "",
            })
            continue

        idx = pd.to_datetime(best_df["date"], errors="coerce")
        results_rows.append({
            "Code": code,
            "ok": True,
            "symbol": best_symbol,
            "div_rows": int(len(best_df)),
            "div_min_date": str(pd.Timestamp(idx.min()).date()) if pd.notna(idx.min()) else "",
            "div_max_date": str(pd.Timestamp(idx.max()).date()) if pd.notna(idx.max()) else "",
            "div_sum": float(pd.to_numeric(best_df["dividend"], errors="coerce").sum()),
            "div_last": float(pd.to_numeric(best_df["dividend"], errors="coerce").iloc[-1]),
            "attempted": "|".join(cands),
            "error": "",
        })

        # sample (tail)
        tail = best_df.tail(MAX_DIV_SAMPLE)
        for _, rr in tail.iterrows():
            div_samples.append({
                "Code": code,
                "symbol": best_symbol,
                "date": str(pd.Timestamp(rr["date"]).date()),
                "dividend": float(rr["dividend"]),
            })

        # all events
        for _, rr in best_df.iterrows():
            div_events.append({
                "Code": code,
                "symbol": best_symbol,
                "date": pd.Timestamp(rr["date"]).normalize(),
                "dividend": float(rr["dividend"]),
            })

    attempts_df = pd.DataFrame(attempt_rows)
    results_df = pd.DataFrame(results_rows)
    div_events_df = pd.DataFrame(div_events)
    div_samples_df = pd.DataFrame(div_samples)

    attempts_df.to_csv(DIAG_ATTEMPTS, index=False, encoding="utf-8-sig")
    results_df.to_csv(DIAG_FETCH_RESULTS, index=False, encoding="utf-8-sig")
    if len(div_samples_df):
        div_samples_df.to_csv(DIAG_DIV_RAW_SAMPLE, index=False, encoding="utf-8-sig")

    # cache summary
    cache_summary = pd.DataFrame([{
        "top_n": TOP_N,
        "ttl_days": TTL_DAYS,
        "cache_hit": cache_hit,
        "cache_miss_fetch": cache_miss,
        "cache_updated_symbols": updated_cnt,
        "sleep_sec": SLEEP_SEC,
        "cache_dir": str(CACHE_EVENTS_DIR),
    }])
    cache_summary.to_csv(DIAG_CACHE_SUMMARY, index=False, encoding="utf-8-sig")

    log_line(f"[SAVE] attempts={DIAG_ATTEMPTS}")
    log_line(f"[SAVE] fetch_results={DIAG_FETCH_RESULTS}")
    log_line(f"[SAVE] cache_summary={DIAG_CACHE_SUMMARY}")
    if len(div_samples_df):
        log_line(f"[SAVE] div_samples={DIAG_DIV_RAW_SAMPLE}")

    return div_events_df, results_df


# ============================================================
# Build DY_TTM on MonthEnd
# ============================================================
def compute_dy_ttm_for_monthends(uni: pd.DataFrame, div_events_df: pd.DataFrame) -> pd.DataFrame:
    price = pd.read_parquet(PRICE_PATH)[["Code", "MonthEnd", "AdjustedClose"]].copy()
    price["Code"] = price["Code"].astype(str).map(normalize_code)
    price = price[price["Code"] != ""].copy()
    price["MonthEnd"] = pd.to_datetime(price["MonthEnd"], errors="coerce").dt.normalize()
    price["AdjustedClose"] = pd.to_numeric(price["AdjustedClose"], errors="coerce")

    codes = set(uni["Code"].unique())
    price = price[price["Code"].isin(codes)].copy()
    price = price.sort_values(["Code", "MonthEnd"]).reset_index(drop=True)

    if div_events_df.empty:
        price["Div_TTM"] = np.nan
        price["DY_TTM"] = np.nan
        return price

    ev = div_events_df.copy()
    ev["Code"] = ev["Code"].astype(str).map(normalize_code)
    ev["date"] = pd.to_datetime(ev["date"], errors="coerce").dt.normalize()
    ev["dividend"] = pd.to_numeric(ev["dividend"], errors="coerce")
    ev = ev.dropna(subset=["Code", "date", "dividend"]).copy()
    ev = ev.sort_values(["Code", "date"]).reset_index(drop=True)

    out_rows = []
    for code, g in price.groupby("Code", sort=False):
        gg = g.copy()
        e = ev[ev["Code"] == code][["date", "dividend"]].copy()
        if e.empty:
            gg["Div_TTM"] = np.nan
            gg["DY_TTM"] = np.nan
            out_rows.append(gg)
            continue

        e = e.sort_values("date")
        e_dates = e["date"].to_numpy(dtype="datetime64[ns]")
        e_vals = e["dividend"].to_numpy(dtype=float)

        div_ttm_list = []
        for me in gg["MonthEnd"].to_numpy(dtype="datetime64[ns]"):
            start = me - np.timedelta64(365, "D")
            left = np.searchsorted(e_dates, start, side="right")
            right = np.searchsorted(e_dates, me, side="right")
            s = e_vals[left:right].sum() if right > left else 0.0
            div_ttm_list.append(float(s))

        gg["Div_TTM"] = div_ttm_list
        gg["DY_TTM"] = gg["Div_TTM"] / gg["AdjustedClose"]
        out_rows.append(gg)

    dy = pd.concat(out_rows, ignore_index=True)
    return dy


# ============================================================
# Load snapshot features
# ============================================================
def load_snapshot_features_top200(uni: pd.DataFrame, dy_df: pd.DataFrame) -> pd.DataFrame:
    snap = pd.read_parquet(SNAP_PATH).copy()
    snap["Code"] = snap["Code"].astype(str).map(normalize_code)
    snap = snap[snap["Code"] != ""].copy()
    snap["MonthEnd"] = pd.to_datetime(snap["MonthEnd"], errors="coerce").dt.normalize()

    codes = set(uni["Code"].unique())
    snap = snap[snap["Code"].isin(codes)].copy()

    # price returns and MOM
    price = pd.read_parquet(PRICE_PATH)[["Code", "MonthEnd", "AdjustedClose"]].copy()
    price["Code"] = price["Code"].astype(str).map(normalize_code)
    price = price[price["Code"] != ""].copy()
    price["MonthEnd"] = pd.to_datetime(price["MonthEnd"], errors="coerce").dt.normalize()
    price["AdjustedClose"] = pd.to_numeric(price["AdjustedClose"], errors="coerce")
    price = price.sort_values(["Code", "MonthEnd"], kind="mergesort")

    price["Adj_next"] = price.groupby("Code")["AdjustedClose"].shift(-1)
    price["ret_m_fwd"] = (price["Adj_next"] / price["AdjustedClose"]) - 1.0
    price["Adj_lag1"] = price.groupby("Code")["AdjustedClose"].shift(1)
    price["Adj_lag12"] = price.groupby("Code")["AdjustedClose"].shift(12)
    price["MOM_12_1"] = (price["Adj_lag1"] / price["Adj_lag12"]) - 1.0

    pr = price[["Code", "MonthEnd", "ret_m_fwd", "MOM_12_1", "AdjustedClose"]].copy()
    df = snap.merge(pr, on=["Code", "MonthEnd"], how="left", validate="many_to_one")

    dy2 = dy_df[["Code", "MonthEnd", "Div_TTM", "DY_TTM"]].copy()
    dy2["MonthEnd"] = pd.to_datetime(dy2["MonthEnd"], errors="coerce").dt.normalize()
    df = df.merge(dy2, on=["Code", "MonthEnd"], how="left", validate="many_to_one")

    for c in ["MarketCap", "BM_Ratio", "ROE", "INV_Growth", "ret_m_fwd", "MOM_12_1", "DY_TTM"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    df = df[(df["MarketCap"].notna()) & (df["MarketCap"] > 0)].copy()
    df = df[df["ret_m_fwd"].notna()].copy()

    df["ret_w"] = winsorize(df["ret_m_fwd"], Q_RET)
    df["BM_w"]  = winsorize(df["BM_Ratio"], Q_BM)
    df["ROE_w"] = winsorize(df["ROE"], Q_ROE)
    df["INV_w"] = winsorize(df["INV_Growth"], Q_INV)
    df["MOM_w"] = winsorize(df["MOM_12_1"], Q_MOM)
    df["DY_w"]  = winsorize(df["DY_TTM"], Q_DY)

    cov = (df.groupby("MonthEnd")
             .agg(n=("Code", "nunique"),
                  dy_notna=("DY_TTM", lambda s: float(s.notna().mean()*100)),
                  bm_notna=("BM_Ratio", lambda s: float(s.notna().mean()*100)),
                  mcap_notna=("MarketCap", lambda s: float(s.notna().mean()*100)))
             .reset_index())
    cov.to_csv(DIAG_DY_TTM_BY_MONTH, index=False, encoding="utf-8-sig")

    return df


# ============================================================
# Compute FF5 + MOM + DY factor
# ============================================================
def compute_ff5_mom_dy_monthly(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    months = sorted(df["MonthEnd"].dropna().unique())
    rows = []
    diag_n = []

    for me in months:
        m = df[df["MonthEnd"] == me].copy()
        n_total = int(m["Code"].nunique())
        if n_total < 30:
            continue

        MKT = value_weighted_return(m, "ret_w", "MarketCap")

        m["SZ"] = size_split_median(m["MarketCap"])
        m["VAL"]  = qcut_3way(m["BM_w"],  P30, P70, labels=("L","M","H"))
        m["PROF"] = qcut_3way(m["ROE_w"], P30, P70, labels=("W","M","R"))
        m["INV"]  = qcut_3way(m["INV_w"], P30, P70, labels=("C","M","A"))
        m["MOM"]  = qcut_3way(m["MOM_w"], P30, P70, labels=("L","M","W"))
        m["DY"]   = qcut_3way(m["DY_w"],  P30, P70, labels=("L","M","H"))

        def port_ret(sz, col, lab):
            x = m[(m["SZ"] == sz) & (m[col] == lab)]
            return value_weighted_return(x, "ret_w", "MarketCap")

        # HML
        SH = port_ret("S","VAL","H"); SL = port_ret("S","VAL","L")
        BH = port_ret("B","VAL","H"); BL = port_ret("B","VAL","L")
        HML = np.nanmean([SH, BH]) - np.nanmean([SL, BL])

        # RMW
        SR = port_ret("S","PROF","R"); SW = port_ret("S","PROF","W")
        BR = port_ret("B","PROF","R"); BW = port_ret("B","PROF","W")
        RMW = np.nanmean([SR, BR]) - np.nanmean([SW, BW])

        # CMA
        SC = port_ret("S","INV","C"); SA = port_ret("S","INV","A")
        BC = port_ret("B","INV","C"); BA = port_ret("B","INV","A")
        CMA = np.nanmean([SC, BC]) - np.nanmean([SA, BA])

        # SMB
        SMv = port_ret("S","VAL","M"); BMv = port_ret("B","VAL","M")
        SMB_HML = np.nanmean([SH, SMv, SL]) - np.nanmean([BH, BMv, BL])
        SMp = port_ret("S","PROF","M"); BMp = port_ret("B","PROF","M")
        SMB_RMW = np.nanmean([SR, SMp, SW]) - np.nanmean([BR, BMp, BW])
        SMa = port_ret("S","INV","M"); BMa = port_ret("B","INV","M")
        SMB_CMA = np.nanmean([SC, SMa, SA]) - np.nanmean([BC, BMa, BA])
        SMB = np.nanmean([SMB_HML, SMB_RMW, SMB_CMA])

        # WML
        SWin = port_ret("S","MOM","W"); SLos = port_ret("S","MOM","L")
        BWin = port_ret("B","MOM","W"); BLos = port_ret("B","MOM","L")
        WML = np.nanmean([SWin, BWin]) - np.nanmean([SLos, BLos])

        # HDYMLDY
        SHD = port_ret("S","DY","H"); SLD = port_ret("S","DY","L")
        BHD = port_ret("B","DY","H"); BLD = port_ret("B","DY","L")
        HDYMLDY = np.nanmean([SHD, BHD]) - np.nanmean([SLD, BLD])

        n_used_dy = int(m.dropna(subset=["DY_w","ret_w","MarketCap"])["Code"].nunique())
        confidence = float(min(1.0, n_used_dy / TOP_N))
        abnormal = (n_used_dy < MIN_STOCKS_PER_MONTH)

        rows.append({
            "MonthEnd": me,
            "MKT": MKT,
            "SMB": SMB,
            "HML": HML,
            "RMW": RMW,
            "CMA": CMA,
            "WML": WML,
            "HDYMLDY": HDYMLDY,
            "n_total": n_total,
            "n_used_dy": n_used_dy,
            "confidence": confidence,
            "abnormal": bool(abnormal),
        })

        diag_n.append({
            "MonthEnd": me,
            "n_total": n_total,
            "n_used_dy": n_used_dy,
            "confidence": confidence,
            "abnormal": bool(abnormal),
        })

    fac = pd.DataFrame(rows).sort_values("MonthEnd").reset_index(drop=True)
    diag_n_df = pd.DataFrame(diag_n).sort_values("MonthEnd").reset_index(drop=True)
    diag_n_df.to_csv(DIAG_DY_FACTOR_NUSED, index=False, encoding="utf-8-sig")
    return fac, diag_n_df


# ============================================================
# Latest analysis + save
# ============================================================
def analyze_latest(fac: pd.DataFrame) -> Dict:
    if fac.empty:
        return {"error": "factor table is empty"}
    last_me = latest_evaluable_month(fac, col="MKT")
    cur = fac[fac["MonthEnd"] == last_me].iloc[0].to_dict()

    factor_cols = ["MKT","SMB","HML","RMW","CMA","WML","HDYMLDY"]
    absvals = {c: abs(cur.get(c, np.nan)) for c in factor_cols}
    dominant = max(absvals, key=lambda k: (-np.nan_to_num(absvals[k], nan=-1), k))

    fs = fac.sort_values("MonthEnd").reset_index(drop=True)
    last3 = fs.tail(3)
    last12 = fs.tail(12)

    trends3 = {c: trend_label(last3[c].to_numpy()) for c in factor_cols}
    trends12 = {c: trend_label(last12[c].to_numpy()) for c in factor_cols}

    mkt = cur.get("MKT", np.nan)
    regime = "RANGE"
    if np.isfinite(mkt):
        if mkt > 0.01:
            regime = "BULL"
        elif mkt < -0.01:
            regime = "BEAR"

    conf = float(cur.get("confidence", 0.0))
    tilt = {}
    for c in factor_cols:
        v = float(np.nanmean(last3[c]))
        tilt[c] = float(np.sign(v)) if np.isfinite(v) else 0.0
    tilt_conf = {k: v * conf for k, v in tilt.items()}

    return {
        "latest_month": str(pd.Timestamp(last_me).date()),
        "dominant_factor": dominant,
        "regime": regime,
        "latest": cur,
        "trends_last3": trends3,
        "trends_last12": trends12,
        "tilt_conf_weighted": tilt_conf,
    }


def save_outputs(fac: pd.DataFrame, analysis: Dict):
    fac.to_parquet(OUT_PARQUET, index=False)

    latest = analysis.get("latest", {})
    summary = {
        "latest_month": analysis.get("latest_month", ""),
        "regime": analysis.get("regime", ""),
        "dominant_factor": analysis.get("dominant_factor", ""),
        "confidence": float(latest.get("confidence", np.nan)),
        "abnormal": bool(latest.get("abnormal", True)),
        "n_total": int(latest.get("n_total", 0) or 0),
        "n_used_dy": int(latest.get("n_used_dy", 0) or 0),
    }
    for c in ["MKT","SMB","HML","RMW","CMA","WML","HDYMLDY"]:
        summary[c] = float(latest.get(c, np.nan))
    pd.DataFrame([summary]).to_csv(OUT_SUMMARY, index=False, encoding="utf-8-sig")

    fac_sorted = fac.sort_values("MonthEnd").reset_index(drop=True)
    cols = ["MonthEnd","MKT","SMB","HML","RMW","CMA","WML","HDYMLDY"]
    fac_sorted.tail(3)[cols].to_csv(OUT_TREND_LAST3, index=False, encoding="utf-8-sig")
    fac_sorted.tail(12)[cols].to_csv(OUT_TREND_LAST12, index=False, encoding="utf-8-sig")


# ============================================================
# Main
# ============================================================
def main():
    with open(DIAG_LOG, "w", encoding="utf-8") as f:
        f.write("")

    log_line("="*110)
    log_line("FF5 + MOM(12-1) + DY_TTM + yfinance cache (top200)")
    log_line("="*110)
    log_line(f"PRICE_PATH: {PRICE_PATH}")
    log_line(f"SNAP_PATH : {SNAP_PATH}")
    log_line(f"DIAG_DIR  : {DIAG_DIR}")
    log_line(f"CACHE_DIR : {CACHE_EVENTS_DIR}")
    log_line(f"TTL_DAYS  : {TTL_DAYS}")

    # 1) universe
    uni = pick_top200_universe()

    # 2) dividends with cache
    div_events_df, fetch_results_df = build_dividend_table_for_top200_with_cache(uni)
    ok_any = int(fetch_results_df["ok"].sum()) if len(fetch_results_df) else 0
    ok_nonempty = int((fetch_results_df["ok"] & (fetch_results_df["div_rows"] > 0)).sum()) if len(fetch_results_df) else 0
    log_line(f"[YF] ok(series)={ok_any}/{len(fetch_results_df)} | nonempty={ok_nonempty}/{len(fetch_results_df)}")

    # 3) DY_TTM
    dy_df = compute_dy_ttm_for_monthends(uni, div_events_df)
    cov_latest = (dy_df[dy_df["MonthEnd"] == dy_df["MonthEnd"].max()]
                  .assign(dy_notna=lambda x: x["DY_TTM"].notna())
                  ["dy_notna"].mean() * 100.0)
    log_line(f"[DY] latest MonthEnd={dy_df['MonthEnd'].max().date()} DY_TTM notna%={cov_latest:.2f}")

    # 4) features + factors
    feat = load_snapshot_features_top200(uni, dy_df)
    log_line(f"[FEATURE] rows={len(feat):,} codes={feat['Code'].nunique():,} months={feat['MonthEnd'].nunique():,}")

    fac, _ = compute_ff5_mom_dy_monthly(feat)
    log_line(f"[FACTORS] months computed={len(fac)} min={fac['MonthEnd'].min()} max={fac['MonthEnd'].max()}")

    if fac.empty:
        raise RuntimeError("Factor result is empty. Check DY/BM coverage and thresholds.")

    analysis = analyze_latest(fac)
    last = analysis.get("latest", {})
    log_line("-"*110)
    log_line(f"[LATEST EVALUABLE MONTH] {analysis.get('latest_month')}")
    log_line(f"[DOMINANT FACTOR] {analysis.get('dominant_factor')}")
    log_line(f"[REGIME] {analysis.get('regime')}")
    log_line(f"[CONFIDENCE] {last.get('confidence'):.3f} | abnormal={last.get('abnormal')} | "
             f"n_total={last.get('n_total')} n_used_dy={last.get('n_used_dy')}")
    log_line("Factor returns:")
    for c in ["MKT","SMB","HML","RMW","CMA","WML","HDYMLDY"]:
        log_line(f"  {c}: {last.get(c)}")

    log_line("-"*110)
    log_line("[STRATEGY HINT] (last3 avg direction, confidence-weighted)")
    for c, score in analysis.get("tilt_conf_weighted", {}).items():
        if score > 0:
            s = "OVERWEIGHT"
        elif score < 0:
            s = "UNDERWEIGHT"
        else:
            s = "NEUTRAL"
        log_line(f"  {c}: {s} (score={score:.2f})")

    # 5) save
    save_outputs(fac, analysis)
    log_line("-"*110)
    log_line("✅ saved:")
    log_line(f"  - {OUT_PARQUET}")
    log_line(f"  - {OUT_SUMMARY}")
    log_line(f"  - {OUT_TREND_LAST3}")
    log_line(f"  - {OUT_TREND_LAST12}")
    log_line(f"  - {DIAG_UNIVERSE}")
    log_line(f"  - {DIAG_ATTEMPTS}")
    log_line(f"  - {DIAG_FETCH_RESULTS}")
    log_line(f"  - {DIAG_CACHE_SUMMARY}")
    log_line(f"  - {DIAG_DY_TTM_BY_MONTH}")
    log_line(f"  - {DIAG_DY_FACTOR_NUSED}")
    log_line(f"  - {DIAG_LOG}")


if __name__ == "__main__":
    main()


FF5 + MOM(12-1) + DY_TTM + yfinance cache (top200)
PRICE_PATH: C:\Users\yongr\Project\merged_data_all_stocks\factors\price_month_end.parquet
SNAP_PATH : C:\Users\yongr\Project\merged_data_all_stocks\factors\month_end_snapshot.parquet
DIAG_DIR  : C:\Users\yongr\Project\merged_data_all_stocks\factors\diag_yfinance_dividend_poc_dy
CACHE_DIR : C:\Users\yongr\Project\merged_data_all_stocks\factors\diag_yfinance_dividend_cache\events_by_symbol
TTL_DAYS  : 14
[UNIVERSE] latest MonthEnd=2025-09-30 candidates=3,428 top200=200


$167A0.T: possibly delisted; no timezone found


[SAVE] attempts=C:\Users\yongr\Project\merged_data_all_stocks\factors\diag_yfinance_dividend_poc_dy\diag_yf_symbol_attempts.csv
[SAVE] fetch_results=C:\Users\yongr\Project\merged_data_all_stocks\factors\diag_yfinance_dividend_poc_dy\diag_yf_dividend_fetch_results.csv
[SAVE] cache_summary=C:\Users\yongr\Project\merged_data_all_stocks\factors\diag_yfinance_dividend_poc_dy\diag_yf_cache_summary.csv
[SAVE] div_samples=C:\Users\yongr\Project\merged_data_all_stocks\factors\diag_yfinance_dividend_poc_dy\diag_yf_dividend_success_samples.csv
[YF] ok(series)=200/200 | nonempty=198/200
[DY] latest MonthEnd=2026-01-31 DY_TTM notna%=99.00
[FEATURE] rows=21,261 codes=200 months=117
[FACTORS] months computed=117 min=2016-01-31 00:00:00 max=2025-09-30 00:00:00
--------------------------------------------------------------------------------------------------------------
[LATEST EVALUABLE MONTH] 2025-09-30
[DOMINANT FACTOR] HDYMLDY
[REGIME] BULL
[CONFIDENCE] 0.990 | abnormal=False | n_total=200 n_used_d

In [4]:
# -*- coding: utf-8 -*-
"""
FF5 + MOM(12-1) + DY_TTM（TTM dividend yield）
+ yfinance dividends per-symbol cache (parquet)
+ incremental update: append only events after last_cached_date (no TTL)

対象ユニバース:
- 既定: 最新MonthEndのMarketCap上位200（TOPIX上位近似）
- 拡張しやすい設計: UNIVERSE_MODE を "top200" / "all_snapshot_latest" に切替可能

DY定義:
- Div_TTM = sum(dividends in (MonthEnd-365D, MonthEnd])
- DY_TTM = Div_TTM / AdjustedClose

DY因子:
- HDYMLDY = High DY - Low DY（サイズニュートラル：Small/Big平均）

dominant因子判定:
- dominant = argmax(|factor| * confidence)  ※要求仕様

注意:
- yfinanceは非公式でYahoo側仕様変更で壊れる可能性あり
- dividendsのイベント日付は権利落ち日中心のことがある → TTM合計で吸収
  [Source] https://github.com/ranaroussi/yfinance/issues/568

依存:
  pip install yfinance pandas pyarrow
"""

import warnings
warnings.filterwarnings("ignore")

import time
from pathlib import Path
from typing import Dict, Tuple, List, Optional

import numpy as np
import pandas as pd
import yfinance as yf


# ============================================================
# Paths (あなたの環境)
# ============================================================
FACTORS_DIR = Path(r"C:\Users\yongr\Project\merged_data_all_stocks\factors")
PRICE_PATH = FACTORS_DIR / "price_month_end.parquet"
SNAP_PATH  = FACTORS_DIR / "month_end_snapshot.parquet"

# 出力（既存FF5+MOMの出力は壊さない：別名）
OUT_PARQUET = FACTORS_DIR / "ff5_mom_dy_factors_monthly.parquet"
OUT_SUMMARY = FACTORS_DIR / "market_factor_summary_ff5_mom_dy.csv"
OUT_TREND_LAST3 = FACTORS_DIR / "market_factor_trend_last3_ff5_mom_dy.csv"
OUT_TREND_LAST12 = FACTORS_DIR / "market_factor_trend_last12_ff5_mom_dy.csv"

# diagnostics + cache
DIAG_DIR = FACTORS_DIR / "diag_yfinance_dividend_incremental"
DIAG_DIR.mkdir(parents=True, exist_ok=True)

CACHE_DIR = FACTORS_DIR / "yf_dividend_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_EVENTS_DIR = CACHE_DIR / "events_by_symbol"
CACHE_EVENTS_DIR.mkdir(parents=True, exist_ok=True)

# diagnostics
DIAG_UNIVERSE = DIAG_DIR / "diag_universe.csv"
DIAG_ATTEMPTS = DIAG_DIR / "diag_yf_symbol_attempts.csv"
DIAG_FETCH_RESULTS = DIAG_DIR / "diag_yf_fetch_results.csv"
DIAG_CACHE_UPDATE = DIAG_DIR / "diag_yf_cache_update.csv"
DIAG_DIV_SAMPLE = DIAG_DIR / "diag_yf_dividend_samples.csv"
DIAG_DY_COV_BY_MONTH = DIAG_DIR / "diag_dy_ttm_coverage_by_month.csv"
DIAG_FACTOR_NUSED = DIAG_DIR / "diag_factor_n_used_by_month.csv"
DIAG_LOG = DIAG_DIR / "diag_runtime_log.txt"


# ============================================================
# Params
# ============================================================
# ユニバース設定：拡張しやすいようにモード化
# - "top200": 最新MonthEndのMarketCap上位200
# - "all_snapshot_latest": 最新MonthEndに存在する全銘柄（全銘柄へ拡張したい場合の入口）
UNIVERSE_MODE = "top200"  # <-- 必要に応じて変更

TOP_N = 200

# yfinance polite access
SLEEP_SEC = 0.10
MAX_DIV_SAMPLE = 10

# factor split
P30 = 0.30
P70 = 0.70

# winsorize quantiles
Q_RET = (0.01, 0.99)
Q_BM  = (0.01, 0.99)
Q_ROE = (0.01, 0.99)
Q_INV = (0.01, 0.99)
Q_MOM = (0.01, 0.99)
Q_DY  = (0.01, 0.99)

BAD_CODE_STRINGS = {"None", "nan", "", "NaN", "NULL", "null"}

# top200前提のabnormal閾値（全銘柄に拡張したら上げる）
MIN_STOCKS_PER_MONTH = 150


# ============================================================
# Utils
# ============================================================
def log_line(s: str):
    print(s)
    with open(DIAG_LOG, "a", encoding="utf-8") as f:
        f.write(s + "\n")


def normalize_code(code: str) -> str:
    if code is None:
        return ""
    s = str(code).strip()
    if s in BAD_CODE_STRINGS:
        return ""
    return s


def winsorize(s: pd.Series, q=(0.01, 0.99)) -> pd.Series:
    x = pd.to_numeric(s, errors="coerce")
    if x.notna().sum() == 0:
        return x
    lo = x.quantile(q[0])
    hi = x.quantile(q[1])
    return x.clip(lo, hi)


def value_weighted_return(df: pd.DataFrame, ret_col: str, w_col: str) -> float:
    x = df[[ret_col, w_col]].dropna()
    if x.empty:
        return np.nan
    w = x[w_col].astype(float).to_numpy()
    r = x[ret_col].astype(float).to_numpy()
    wsum = w.sum()
    if not np.isfinite(wsum) or wsum <= 0:
        return np.nan
    return float(np.dot(r, w) / wsum)


def qcut_3way(x: pd.Series, p30=0.3, p70=0.7, labels=("L","M","H")) -> pd.Series:
    a = pd.to_numeric(x, errors="coerce")
    q1 = a.quantile(p30)
    q2 = a.quantile(p70)
    out = pd.Series(index=a.index, dtype="object")
    out[a <= q1] = labels[0]
    out[(a > q1) & (a < q2)] = labels[1]
    out[a >= q2] = labels[2]
    return out


def size_split_median(mcap: pd.Series) -> pd.Series:
    a = pd.to_numeric(mcap, errors="coerce")
    med = a.quantile(0.5)
    out = pd.Series(index=a.index, dtype="object")
    out[a <= med] = "S"
    out[a > med] = "B"
    return out


def trend_label(vals: np.ndarray) -> str:
    vals = np.array(vals, dtype=float)
    vals = vals[np.isfinite(vals)]
    if len(vals) < 2:
        return "NA"
    x = np.arange(len(vals))
    slope = np.polyfit(x, vals, 1)[0]
    if abs(slope) < 1e-6:
        return "FLAT"
    return "UP" if slope > 0 else "DOWN"


def latest_evaluable_month(factors: pd.DataFrame, col="MKT") -> pd.Timestamp:
    f = factors.dropna(subset=[col]).copy()
    if f.empty:
        return pd.NaT
    return pd.Timestamp(f["MonthEnd"].max())


# ============================================================
# Universe selection
# ============================================================
def load_snapshot_latest() -> pd.DataFrame:
    snap = pd.read_parquet(SNAP_PATH)
    snap["Code"] = snap["Code"].astype(str).map(normalize_code)
    snap = snap[snap["Code"] != ""].copy()
    snap["MonthEnd"] = pd.to_datetime(snap["MonthEnd"], errors="coerce").dt.normalize()
    snap["MarketCap"] = pd.to_numeric(snap["MarketCap"], errors="coerce")
    latest_me = snap["MonthEnd"].max()
    s = snap[snap["MonthEnd"] == latest_me].copy()
    return s


def pick_universe() -> pd.DataFrame:
    s = load_snapshot_latest().copy()
    s = s.dropna(subset=["MarketCap"]).copy()
    s = s.sort_values("MarketCap", ascending=False).drop_duplicates(["Code"], keep="first")

    if UNIVERSE_MODE == "top200":
        uni = s.head(TOP_N).copy()
        uni["Rank"] = np.arange(1, len(uni) + 1)
    elif UNIVERSE_MODE == "all_snapshot_latest":
        uni = s.copy()
        uni["Rank"] = np.arange(1, len(uni) + 1)
    else:
        raise ValueError(f"Unknown UNIVERSE_MODE={UNIVERSE_MODE}")

    uni = uni[["MonthEnd", "Code", "MarketCap", "Rank"]].copy()
    uni.to_csv(DIAG_UNIVERSE, index=False, encoding="utf-8-sig")
    log_line(f"[UNIVERSE] mode={UNIVERSE_MODE} latest MonthEnd={pd.Timestamp(uni['MonthEnd'].iloc[0]).date()} n={len(uni):,}")
    return uni


# ============================================================
# yfinance symbols
# ============================================================
def make_symbol_candidates(code: str) -> List[str]:
    code = normalize_code(code)
    if not code:
        return []
    cands = []
    # 5桁数字末尾0 -> 4桁.T 優先
    if code.isdigit() and len(code) == 5 and code.endswith("0"):
        c4 = code[:-1]
        cands.append(f"{c4}.T")
        cands.append(f"{code}.T")
    else:
        cands.append(f"{code}.T")

    out, seen = [], set()
    for x in cands:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out


def fetch_dividends(symbol: str) -> Tuple[bool, Optional[pd.Series], str]:
    try:
        t = yf.Ticker(symbol)
        div = t.dividends
        if div is None or not isinstance(div, pd.Series):
            return False, None, "dividends is None or not Series"
        return True, div.dropna(), ""
    except Exception as e:
        return False, None, f"{type(e).__name__}: {e}"


# ============================================================
# Cache I/O (parquet) + incremental append
# ============================================================
def _cache_path(symbol: str) -> Path:
    safe = symbol.replace("/", "_")
    return CACHE_EVENTS_DIR / f"{safe}.parquet"


def load_cache(symbol: str) -> pd.DataFrame:
    fp = _cache_path(symbol)
    if not fp.exists():
        return pd.DataFrame(columns=["date", "dividend"])
    df = pd.read_parquet(fp)
    if df is None or df.empty:
        return pd.DataFrame(columns=["date", "dividend"])
    df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.normalize()
    df["dividend"] = pd.to_numeric(df["dividend"], errors="coerce")
    df = df.dropna(subset=["date", "dividend"]).drop_duplicates(["date"], keep="last").sort_values("date")
    return df.reset_index(drop=True)


def save_cache(symbol: str, df: pd.DataFrame):
    fp = _cache_path(symbol)
    fp.parent.mkdir(parents=True, exist_ok=True)
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.normalize()
    df["dividend"] = pd.to_numeric(df["dividend"], errors="coerce")
    df = df.dropna(subset=["date", "dividend"]).drop_duplicates(["date"], keep="last").sort_values("date")
    df.to_parquet(fp, index=False)


def cache_last_date(cache_df: pd.DataFrame) -> pd.Timestamp:
    if cache_df is None or cache_df.empty:
        return pd.NaT
    return pd.Timestamp(cache_df["date"].max())


def series_to_df(div: pd.Series) -> pd.DataFrame:
    return pd.DataFrame({
        "date": pd.to_datetime(div.index, errors="coerce").normalize(),
        "dividend": pd.to_numeric(div.values, errors="coerce"),
    }).dropna(subset=["date", "dividend"]).drop_duplicates(["date"], keep="last").sort_values("date").reset_index(drop=True)


def incremental_update_cache(symbol: str, cache_df: pd.DataFrame) -> Tuple[pd.DataFrame, Dict]:
    """
    差分更新（last_date以降のみ追記）:
    - yfinanceからdividends Seriesを取得（Yahoo側の都合で全履歴が返ることがある）
    - cache_last_date を境に「新しいイベントだけ」を抽出して追記
    - 既存キャッシュを壊さず、append & dedup

    戻り:
      (updated_cache_df, stats)
    """
    last = cache_last_date(cache_df)
    ok, div, err = fetch_dividends(symbol)
    if not ok:
        return cache_df, {"fetched": False, "error": err, "added_rows": 0, "last_cached": str(last) if pd.notna(last) else ""}

    new_df = series_to_df(div)
    if pd.isna(last):
        # 初回: 全部をキャッシュ
        updated = new_df
        added = int(len(updated))
    else:
        # last_date より後だけ
        delta = new_df[new_df["date"] > last].copy()
        added = int(len(delta))
        if added == 0:
            updated = cache_df
        else:
            updated = pd.concat([cache_df, delta], ignore_index=True)
            updated = updated.drop_duplicates(["date"], keep="last").sort_values("date").reset_index(drop=True)

    return updated, {"fetched": True, "error": "", "added_rows": added, "last_cached": str(last.date()) if pd.notna(last) else ""}


# ============================================================
# Build dividends table (for DY calc) using cache
# ============================================================
def build_div_events_with_incremental_cache(uni: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Returns:
      - div_events_df: [Code, symbol, date, dividend]
      - fetch_results_df: per code summary
      - cache_update_df: per symbol update stats
    """
    attempt_rows = []
    results_rows = []
    cache_update_rows = []
    div_samples = []
    div_events = []

    for _, rr in uni.iterrows():
        code = rr["Code"]
        cands = make_symbol_candidates(code)

        best_ok = False
        best_symbol = ""
        best_cache = None
        last_err = ""

        # candidate loop
        for sym in cands:
            cache_df = load_cache(sym)

            # 差分更新（毎回 TTL ではなく update を試みる）
            updated_cache, stat = incremental_update_cache(sym, cache_df)
            cache_update_rows.append({"Code": code, "symbol": sym, **stat})

            if stat.get("fetched") is False:
                last_err = stat.get("error", "")
                attempt_rows.append({"Code": code, "symbol": sym, "success_series": False, "error": last_err})
                time.sleep(SLEEP_SEC)
                continue

            # fetch succeeded (even if added_rows=0)
            attempt_rows.append({"Code": code, "symbol": sym, "success_series": True, "error": ""})

            # persist cache only if changed (or first time)
            if stat.get("added_rows", 0) > 0 or (cache_df is None) or (cache_df.empty and len(updated_cache) > 0):
                save_cache(sym, updated_cache)

            best_ok = True
            best_symbol = sym
            best_cache = updated_cache
            time.sleep(SLEEP_SEC)
            break

        if not best_ok:
            results_rows.append({
                "Code": code,
                "ok": False,
                "symbol": "",
                "div_rows": 0,
                "div_min_date": "",
                "div_max_date": "",
                "attempted": "|".join(cands),
                "error": last_err or "all_candidates_failed",
            })
            continue

        # summarize best_cache
        if best_cache is None or best_cache.empty:
            results_rows.append({
                "Code": code,
                "ok": True,
                "symbol": best_symbol,
                "div_rows": 0,
                "div_min_date": "",
                "div_max_date": "",
                "attempted": "|".join(cands),
                "error": "",
            })
            continue

        dmin = pd.Timestamp(best_cache["date"].min()).date()
        dmax = pd.Timestamp(best_cache["date"].max()).date()
        results_rows.append({
            "Code": code,
            "ok": True,
            "symbol": best_symbol,
            "div_rows": int(len(best_cache)),
            "div_min_date": str(dmin),
            "div_max_date": str(dmax),
            "attempted": "|".join(cands),
            "error": "",
        })

        # samples
        tail = best_cache.tail(MAX_DIV_SAMPLE)
        for _, r2 in tail.iterrows():
            div_samples.append({
                "Code": code,
                "symbol": best_symbol,
                "date": str(pd.Timestamp(r2["date"]).date()),
                "dividend": float(r2["dividend"]),
            })

        # events
        for _, r2 in best_cache.iterrows():
            div_events.append({
                "Code": code,
                "symbol": best_symbol,
                "date": pd.Timestamp(r2["date"]).normalize(),
                "dividend": float(r2["dividend"]),
            })

    attempts_df = pd.DataFrame(attempt_rows)
    results_df = pd.DataFrame(results_rows)
    cache_update_df = pd.DataFrame(cache_update_rows)
    div_events_df = pd.DataFrame(div_events)
    div_samples_df = pd.DataFrame(div_samples)

    attempts_df.to_csv(DIAG_ATTEMPTS, index=False, encoding="utf-8-sig")
    results_df.to_csv(DIAG_FETCH_RESULTS, index=False, encoding="utf-8-sig")
    cache_update_df.to_csv(DIAG_CACHE_UPDATE, index=False, encoding="utf-8-sig")
    if len(div_samples_df):
        div_samples_df.to_csv(DIAG_DIV_SAMPLE, index=False, encoding="utf-8-sig")

    log_line(f"[SAVE] attempts={DIAG_ATTEMPTS}")
    log_line(f"[SAVE] fetch_results={DIAG_FETCH_RESULTS}")
    log_line(f"[SAVE] cache_update={DIAG_CACHE_UPDATE}")
    if len(div_samples_df):
        log_line(f"[SAVE] dividend_samples={DIAG_DIV_SAMPLE}")

    return div_events_df, results_df, cache_update_df


# ============================================================
# DY_TTM
# ============================================================
def compute_dy_ttm_for_monthends(uni: pd.DataFrame, div_events_df: pd.DataFrame) -> pd.DataFrame:
    price = pd.read_parquet(PRICE_PATH)[["Code", "MonthEnd", "AdjustedClose"]].copy()
    price["Code"] = price["Code"].astype(str).map(normalize_code)
    price = price[price["Code"] != ""].copy()
    price["MonthEnd"] = pd.to_datetime(price["MonthEnd"], errors="coerce").dt.normalize()
    price["AdjustedClose"] = pd.to_numeric(price["AdjustedClose"], errors="coerce")

    codes = set(uni["Code"].unique())
    price = price[price["Code"].isin(codes)].copy()
    price = price.sort_values(["Code", "MonthEnd"]).reset_index(drop=True)

    if div_events_df.empty:
        price["Div_TTM"] = np.nan
        price["DY_TTM"] = np.nan
        return price

    ev = div_events_df.copy()
    ev["Code"] = ev["Code"].astype(str).map(normalize_code)
    ev["date"] = pd.to_datetime(ev["date"], errors="coerce").dt.normalize()
    ev["dividend"] = pd.to_numeric(ev["dividend"], errors="coerce")
    ev = ev.dropna(subset=["Code", "date", "dividend"]).copy()
    ev = ev.sort_values(["Code", "date"]).reset_index(drop=True)

    out_rows = []
    for code, g in price.groupby("Code", sort=False):
        gg = g.copy()
        e = ev[ev["Code"] == code][["date", "dividend"]].copy()
        if e.empty:
            gg["Div_TTM"] = np.nan
            gg["DY_TTM"] = np.nan
            out_rows.append(gg)
            continue

        e = e.sort_values("date")
        e_dates = e["date"].to_numpy(dtype="datetime64[ns]")
        e_vals = e["dividend"].to_numpy(dtype=float)

        div_ttm_list = []
        for me in gg["MonthEnd"].to_numpy(dtype="datetime64[ns]"):
            start = me - np.timedelta64(365, "D")
            left = np.searchsorted(e_dates, start, side="right")
            right = np.searchsorted(e_dates, me, side="right")
            s = e_vals[left:right].sum() if right > left else 0.0
            div_ttm_list.append(float(s))

        gg["Div_TTM"] = div_ttm_list
        gg["DY_TTM"] = gg["Div_TTM"] / gg["AdjustedClose"]
        out_rows.append(gg)

    dy = pd.concat(out_rows, ignore_index=True)
    return dy


# ============================================================
# Features + Factors
# ============================================================
def load_features(uni: pd.DataFrame, dy_df: pd.DataFrame) -> pd.DataFrame:
    snap = pd.read_parquet(SNAP_PATH).copy()
    snap["Code"] = snap["Code"].astype(str).map(normalize_code)
    snap = snap[snap["Code"] != ""].copy()
    snap["MonthEnd"] = pd.to_datetime(snap["MonthEnd"], errors="coerce").dt.normalize()

    codes = set(uni["Code"].unique())
    snap = snap[snap["Code"].isin(codes)].copy()

    price = pd.read_parquet(PRICE_PATH)[["Code", "MonthEnd", "AdjustedClose"]].copy()
    price["Code"] = price["Code"].astype(str).map(normalize_code)
    price = price[price["Code"] != ""].copy()
    price["MonthEnd"] = pd.to_datetime(price["MonthEnd"], errors="coerce").dt.normalize()
    price["AdjustedClose"] = pd.to_numeric(price["AdjustedClose"], errors="coerce")
    price = price.sort_values(["Code", "MonthEnd"], kind="mergesort")

    price["Adj_next"] = price.groupby("Code")["AdjustedClose"].shift(-1)
    price["ret_m_fwd"] = (price["Adj_next"] / price["AdjustedClose"]) - 1.0
    price["Adj_lag1"] = price.groupby("Code")["AdjustedClose"].shift(1)
    price["Adj_lag12"] = price.groupby("Code")["AdjustedClose"].shift(12)
    price["MOM_12_1"] = (price["Adj_lag1"] / price["Adj_lag12"]) - 1.0

    pr = price[["Code", "MonthEnd", "ret_m_fwd", "MOM_12_1"]].copy()
    df = snap.merge(pr, on=["Code", "MonthEnd"], how="left", validate="many_to_one")

    dy2 = dy_df[["Code", "MonthEnd", "Div_TTM", "DY_TTM"]].copy()
    dy2["MonthEnd"] = pd.to_datetime(dy2["MonthEnd"], errors="coerce").dt.normalize()
    df = df.merge(dy2, on=["Code", "MonthEnd"], how="left", validate="many_to_one")

    for c in ["MarketCap","BM_Ratio","ROE","INV_Growth","ret_m_fwd","MOM_12_1","DY_TTM"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    df = df[(df["MarketCap"].notna()) & (df["MarketCap"] > 0)].copy()
    df = df[df["ret_m_fwd"].notna()].copy()

    df["ret_w"] = winsorize(df["ret_m_fwd"], Q_RET)
    df["BM_w"]  = winsorize(df["BM_Ratio"], Q_BM)
    df["ROE_w"] = winsorize(df["ROE"], Q_ROE)
    df["INV_w"] = winsorize(df["INV_Growth"], Q_INV)
    df["MOM_w"] = winsorize(df["MOM_12_1"], Q_MOM)
    df["DY_w"]  = winsorize(df["DY_TTM"], Q_DY)

    cov = (df.groupby("MonthEnd")
             .agg(n=("Code","nunique"),
                  dy_notna=("DY_TTM", lambda s: float(s.notna().mean()*100)),
                  bm_notna=("BM_Ratio", lambda s: float(s.notna().mean()*100)))
             .reset_index())
    cov.to_csv(DIAG_DY_COV_BY_MONTH, index=False, encoding="utf-8-sig")

    return df


def compute_factors(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    months = sorted(df["MonthEnd"].dropna().unique())
    rows = []
    diag_n = []

    for me in months:
        m = df[df["MonthEnd"] == me].copy()
        n_total = int(m["Code"].nunique())
        if n_total < 30:
            continue

        MKT = value_weighted_return(m, "ret_w", "MarketCap")

        m["SZ"] = size_split_median(m["MarketCap"])
        m["VAL"]  = qcut_3way(m["BM_w"],  P30, P70, labels=("L","M","H"))
        m["PROF"] = qcut_3way(m["ROE_w"], P30, P70, labels=("W","M","R"))
        m["INV"]  = qcut_3way(m["INV_w"], P30, P70, labels=("C","M","A"))
        m["MOM"]  = qcut_3way(m["MOM_w"], P30, P70, labels=("L","M","W"))
        m["DY"]   = qcut_3way(m["DY_w"],  P30, P70, labels=("L","M","H"))

        def port_ret(sz, col, lab):
            x = m[(m["SZ"] == sz) & (m[col] == lab)]
            return value_weighted_return(x, "ret_w", "MarketCap")

        # HML
        SH = port_ret("S","VAL","H"); SL = port_ret("S","VAL","L")
        BH = port_ret("B","VAL","H"); BL = port_ret("B","VAL","L")
        HML = np.nanmean([SH, BH]) - np.nanmean([SL, BL])

        # RMW
        SR = port_ret("S","PROF","R"); SW = port_ret("S","PROF","W")
        BR = port_ret("B","PROF","R"); BW = port_ret("B","PROF","W")
        RMW = np.nanmean([SR, BR]) - np.nanmean([SW, BW])

        # CMA
        SC = port_ret("S","INV","C"); SA = port_ret("S","INV","A")
        BC = port_ret("B","INV","C"); BA = port_ret("B","INV","A")
        CMA = np.nanmean([SC, BC]) - np.nanmean([SA, BA])

        # SMB
        SMv = port_ret("S","VAL","M"); BMv = port_ret("B","VAL","M")
        SMB_HML = np.nanmean([SH, SMv, SL]) - np.nanmean([BH, BMv, BL])
        SMp = port_ret("S","PROF","M"); BMp = port_ret("B","PROF","M")
        SMB_RMW = np.nanmean([SR, SMp, SW]) - np.nanmean([BR, BMp, BW])
        SMa = port_ret("S","INV","M"); BMa = port_ret("B","INV","M")
        SMB_CMA = np.nanmean([SC, SMa, SA]) - np.nanmean([BC, BMa, BA])
        SMB = np.nanmean([SMB_HML, SMB_RMW, SMB_CMA])

        # WML
        SWin = port_ret("S","MOM","W"); SLos = port_ret("S","MOM","L")
        BWin = port_ret("B","MOM","W"); BLos = port_ret("B","MOM","L")
        WML = np.nanmean([SWin, BWin]) - np.nanmean([SLos, BLos])

        # HDYMLDY
        SHD = port_ret("S","DY","H"); SLD = port_ret("S","DY","L")
        BHD = port_ret("B","DY","H"); BLD = port_ret("B","DY","L")
        HDYMLDY = np.nanmean([SHD, BHD]) - np.nanmean([SLD, BLD])

        n_used_dy = int(m.dropna(subset=["DY_w","ret_w","MarketCap"])["Code"].nunique())
        confidence = float(min(1.0, n_used_dy / max(1, n_total)))
        abnormal = (n_used_dy < MIN_STOCKS_PER_MONTH)

        rows.append({
            "MonthEnd": me,
            "MKT": MKT, "SMB": SMB, "HML": HML, "RMW": RMW, "CMA": CMA, "WML": WML, "HDYMLDY": HDYMLDY,
            "n_total": n_total,
            "n_used_dy": n_used_dy,
            "confidence": confidence,
            "abnormal": bool(abnormal),
        })

        diag_n.append({
            "MonthEnd": me,
            "n_total": n_total,
            "n_used_dy": n_used_dy,
            "confidence": confidence,
            "abnormal": bool(abnormal),
        })

    fac = pd.DataFrame(rows).sort_values("MonthEnd").reset_index(drop=True)
    diag_n_df = pd.DataFrame(diag_n).sort_values("MonthEnd").reset_index(drop=True)
    diag_n_df.to_csv(DIAG_FACTOR_NUSED, index=False, encoding="utf-8-sig")
    return fac, diag_n_df


# ============================================================
# dominant = abs * confidence (requested)
# ============================================================
def analyze_latest(fac: pd.DataFrame) -> Dict:
    if fac.empty:
        return {"error": "factor table is empty"}

    last_me = latest_evaluable_month(fac, col="MKT")
    cur = fac[fac["MonthEnd"] == last_me].iloc[0].to_dict()

    factor_cols = ["MKT","SMB","HML","RMW","CMA","WML","HDYMLDY"]
    conf = float(cur.get("confidence", 0.0))

    # ===== requested dominant definition =====
    score = {c: abs(float(cur.get(c, np.nan))) * conf for c in factor_cols}
    dominant = max(score, key=lambda k: (-np.nan_to_num(score[k], nan=-1), k))

    fs = fac.sort_values("MonthEnd").reset_index(drop=True)
    last3 = fs.tail(3)
    last12 = fs.tail(12)

    trends3 = {c: trend_label(last3[c].to_numpy()) for c in factor_cols}
    trends12 = {c: trend_label(last12[c].to_numpy()) for c in factor_cols}

    mkt = cur.get("MKT", np.nan)
    regime = "RANGE"
    if np.isfinite(mkt):
        if mkt > 0.01:
            regime = "BULL"
        elif mkt < -0.01:
            regime = "BEAR"

    tilt = {}
    for c in factor_cols:
        v = float(np.nanmean(last3[c]))
        tilt[c] = float(np.sign(v)) if np.isfinite(v) else 0.0
    tilt_conf = {k: v * conf for k, v in tilt.items()}

    return {
        "latest_month": str(pd.Timestamp(last_me).date()),
        "dominant_factor": dominant,
        "dominant_scores": score,
        "regime": regime,
        "latest": cur,
        "trends_last3": trends3,
        "trends_last12": trends12,
        "tilt_conf_weighted": tilt_conf,
    }


def save_outputs(fac: pd.DataFrame, analysis: Dict):
    fac.to_parquet(OUT_PARQUET, index=False)

    latest = analysis.get("latest", {})
    summary = {
        "latest_month": analysis.get("latest_month", ""),
        "regime": analysis.get("regime", ""),
        "dominant_factor": analysis.get("dominant_factor", ""),
        "confidence": float(latest.get("confidence", np.nan)),
        "abnormal": bool(latest.get("abnormal", True)),
        "n_total": int(latest.get("n_total", 0) or 0),
        "n_used_dy": int(latest.get("n_used_dy", 0) or 0),
    }
    for c in ["MKT","SMB","HML","RMW","CMA","WML","HDYMLDY"]:
        summary[c] = float(latest.get(c, np.nan))
        summary[f"score_{c}"] = float(analysis.get("dominant_scores", {}).get(c, np.nan))
    pd.DataFrame([summary]).to_csv(OUT_SUMMARY, index=False, encoding="utf-8-sig")

    fac_sorted = fac.sort_values("MonthEnd").reset_index(drop=True)
    cols = ["MonthEnd","MKT","SMB","HML","RMW","CMA","WML","HDYMLDY","confidence","abnormal"]
    fac_sorted.tail(3)[cols].to_csv(OUT_TREND_LAST3, index=False, encoding="utf-8-sig")
    fac_sorted.tail(12)[cols].to_csv(OUT_TREND_LAST12, index=False, encoding="utf-8-sig")


# ============================================================
# Main
# ============================================================
def main():
    with open(DIAG_LOG, "w", encoding="utf-8") as f:
        f.write("")

    log_line("="*110)
    log_line("FF5 + MOM(12-1) + DY_TTM + yfinance cache (incremental append)")
    log_line("="*110)
    log_line(f"PRICE_PATH: {PRICE_PATH}")
    log_line(f"SNAP_PATH : {SNAP_PATH}")
    log_line(f"UNIVERSE_MODE: {UNIVERSE_MODE} (TOP_N={TOP_N})")
    log_line(f"CACHE_DIR : {CACHE_EVENTS_DIR}")

    # 1) universe
    uni = pick_universe()

    # 2) dividends (incremental cache update)
    div_events_df, fetch_results_df, cache_update_df = build_div_events_with_incremental_cache(uni)
    ok_any = int(fetch_results_df["ok"].sum()) if len(fetch_results_df) else 0
    ok_nonempty = int((fetch_results_df["ok"] & (fetch_results_df["div_rows"] > 0)).sum()) if len(fetch_results_df) else 0
    log_line(f"[YF] ok(symbol found)= {ok_any}/{len(fetch_results_df)} | nonempty(div_rows>0)= {ok_nonempty}/{len(fetch_results_df)}")

    # cache update summary
    if len(cache_update_df):
        added_total = int(pd.to_numeric(cache_update_df["added_rows"], errors="coerce").fillna(0).sum())
        fetched_fail = int((cache_update_df.get("fetched", True) == False).sum()) if "fetched" in cache_update_df.columns else 0
        log_line(f"[CACHE] symbols tried={len(cache_update_df)} total_added_rows={added_total} fetched_fail={fetched_fail}")

    # 3) DY_TTM
    dy_df = compute_dy_ttm_for_monthends(uni, div_events_df)
    cov_latest = (dy_df[dy_df["MonthEnd"] == dy_df["MonthEnd"].max()]
                  .assign(dy_notna=lambda x: x["DY_TTM"].notna())
                  ["dy_notna"].mean() * 100.0)
    log_line(f"[DY] latest MonthEnd={dy_df['MonthEnd'].max().date()} DY_TTM notna%={cov_latest:.2f}")

    # 4) features + factors
    feat = load_features(uni, dy_df)
    log_line(f"[FEATURE] rows={len(feat):,} codes={feat['Code'].nunique():,} months={feat['MonthEnd'].nunique():,}")

    fac, _ = compute_factors(feat)
    log_line(f"[FACTORS] months computed={len(fac)} min={fac['MonthEnd'].min()} max={fac['MonthEnd'].max()}")

    if fac.empty:
        raise RuntimeError("Factor result is empty. Check DY/BM coverage and thresholds.")

    analysis = analyze_latest(fac)
    last = analysis.get("latest", {})
    log_line("-"*110)
    log_line(f"[LATEST EVALUABLE MONTH] {analysis.get('latest_month')}")
    log_line(f"[DOMINANT FACTOR (abs*confidence)] {analysis.get('dominant_factor')}")
    log_line(f"[REGIME] {analysis.get('regime')}")
    log_line(f"[CONFIDENCE] {last.get('confidence'):.3f} | abnormal={last.get('abnormal')} | "
             f"n_total={last.get('n_total')} n_used_dy={last.get('n_used_dy')}")
    log_line("Factor returns:")
    for c in ["MKT","SMB","HML","RMW","CMA","WML","HDYMLDY"]:
        log_line(f"  {c}: {last.get(c)}")
    log_line("Dominant scores (abs*confidence):")
    for c, sc in sorted(analysis.get("dominant_scores", {}).items(), key=lambda kv: -np.nan_to_num(kv[1], nan=-1)):
        log_line(f"  score_{c}: {sc}")

    log_line("-"*110)
    log_line("[STRATEGY HINT] (last3 avg direction, confidence-weighted)")
    for c, score in analysis.get("tilt_conf_weighted", {}).items():
        if score > 0:
            s = "OVERWEIGHT"
        elif score < 0:
            s = "UNDERWEIGHT"
        else:
            s = "NEUTRAL"
        log_line(f"  {c}: {s} (score={score:.2f})")

    # 5) save outputs
    save_outputs(fac, analysis)
    log_line("-"*110)
    log_line("✅ saved:")
    log_line(f"  - {OUT_PARQUET}")
    log_line(f"  - {OUT_SUMMARY}")
    log_line(f"  - {OUT_TREND_LAST3}")
    log_line(f"  - {OUT_TREND_LAST12}")
    log_line(f"  - {DIAG_UNIVERSE}")
    log_line(f"  - {DIAG_ATTEMPTS}")
    log_line(f"  - {DIAG_FETCH_RESULTS}")
    log_line(f"  - {DIAG_CACHE_UPDATE}")
    log_line(f"  - {DIAG_DY_COV_BY_MONTH}")
    log_line(f"  - {DIAG_FACTOR_NUSED}")
    log_line(f"  - {DIAG_LOG}")


if __name__ == "__main__":
    main()


FF5 + MOM(12-1) + DY_TTM + yfinance cache (incremental append)
PRICE_PATH: C:\Users\yongr\Project\merged_data_all_stocks\factors\price_month_end.parquet
SNAP_PATH : C:\Users\yongr\Project\merged_data_all_stocks\factors\month_end_snapshot.parquet
UNIVERSE_MODE: top200 (TOP_N=200)
CACHE_DIR : C:\Users\yongr\Project\merged_data_all_stocks\factors\yf_dividend_cache\events_by_symbol
[UNIVERSE] mode=top200 latest MonthEnd=2025-09-30 n=200


$167A0.T: possibly delisted; no timezone found


[SAVE] attempts=C:\Users\yongr\Project\merged_data_all_stocks\factors\diag_yfinance_dividend_incremental\diag_yf_symbol_attempts.csv
[SAVE] fetch_results=C:\Users\yongr\Project\merged_data_all_stocks\factors\diag_yfinance_dividend_incremental\diag_yf_fetch_results.csv
[SAVE] cache_update=C:\Users\yongr\Project\merged_data_all_stocks\factors\diag_yfinance_dividend_incremental\diag_yf_cache_update.csv
[SAVE] dividend_samples=C:\Users\yongr\Project\merged_data_all_stocks\factors\diag_yfinance_dividend_incremental\diag_yf_dividend_samples.csv
[YF] ok(symbol found)= 200/200 | nonempty(div_rows>0)= 198/200
[CACHE] symbols tried=200 total_added_rows=8398 fetched_fail=0
[DY] latest MonthEnd=2026-01-31 DY_TTM notna%=99.00
[FEATURE] rows=21,261 codes=200 months=117
[FACTORS] months computed=117 min=2016-01-31 00:00:00 max=2025-09-30 00:00:00
--------------------------------------------------------------------------------------------------------------
[LATEST EVALUABLE MONTH] 2025-09-30
[DOMINANT

In [5]:
# -*- coding: utf-8 -*-
"""
FF5 + MOM(12-1) + DY_TTM（TTM dividend yield）
+ yfinance dividends per-symbol cache (parquet)
+ incremental update: append only events after last_cached_date (no TTL)

対象ユニバース:
- 既定: 最新MonthEndのMarketCap上位200（TOPIX上位近似）
- 拡張しやすい設計: UNIVERSE_MODE を "top200" / "all_snapshot_latest" に切替可能

DY定義:
- Div_TTM = sum(dividends in (MonthEnd-365D, MonthEnd])
- DY_TTM = Div_TTM / AdjustedClose

DY因子:
- HDYMLDY = High DY - Low DY（サイズニュートラル：Small/Big平均）

dominant因子判定:
- dominant = argmax(|factor| * confidence)  ※要求仕様

注意:
- yfinanceは非公式でYahoo側仕様変更で壊れる可能性あり
- dividendsのイベント日付は権利落ち日中心のことがある → TTM合計で吸収
  [Source] https://github.com/ranaroussi/yfinance/issues/568

依存:
  pip install yfinance pandas pyarrow
"""

import warnings
warnings.filterwarnings("ignore")

import time
from pathlib import Path
from typing import Dict, Tuple, List, Optional

import numpy as np
import pandas as pd
import yfinance as yf


# ============================================================
# Paths (あなたの環境)
# ============================================================
FACTORS_DIR = Path(r"C:\Users\yongr\Project\merged_data_all_stocks\factors")
PRICE_PATH = FACTORS_DIR / "price_month_end.parquet"
SNAP_PATH  = FACTORS_DIR / "month_end_snapshot.parquet"

# 出力（既存FF5+MOMの出力は壊さない：別名）
OUT_PARQUET = FACTORS_DIR / "ff5_mom_dy_factors_monthly.parquet"
OUT_SUMMARY = FACTORS_DIR / "market_factor_summary_ff5_mom_dy.csv"
OUT_TREND_LAST3 = FACTORS_DIR / "market_factor_trend_last3_ff5_mom_dy.csv"
OUT_TREND_LAST12 = FACTORS_DIR / "market_factor_trend_last12_ff5_mom_dy.csv"

# diagnostics + cache
DIAG_DIR = FACTORS_DIR / "diag_yfinance_dividend_incremental"
DIAG_DIR.mkdir(parents=True, exist_ok=True)

CACHE_DIR = FACTORS_DIR / "yf_dividend_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_EVENTS_DIR = CACHE_DIR / "events_by_symbol"
CACHE_EVENTS_DIR.mkdir(parents=True, exist_ok=True)

# diagnostics
DIAG_UNIVERSE = DIAG_DIR / "diag_universe.csv"
DIAG_ATTEMPTS = DIAG_DIR / "diag_yf_symbol_attempts.csv"
DIAG_FETCH_RESULTS = DIAG_DIR / "diag_yf_fetch_results.csv"
DIAG_CACHE_UPDATE = DIAG_DIR / "diag_yf_cache_update.csv"
DIAG_DIV_SAMPLE = DIAG_DIR / "diag_yf_dividend_samples.csv"
DIAG_DY_COV_BY_MONTH = DIAG_DIR / "diag_dy_ttm_coverage_by_month.csv"
DIAG_FACTOR_NUSED = DIAG_DIR / "diag_factor_n_used_by_month.csv"
DIAG_LOG = DIAG_DIR / "diag_runtime_log.txt"


# ============================================================
# Params
# ============================================================
# ユニバース設定：拡張しやすいようにモード化
# - "top200": 最新MonthEndのMarketCap上位200
# - "all_snapshot_latest": 最新MonthEndに存在する全銘柄（全銘柄へ拡張したい場合の入口）
UNIVERSE_MODE = "top200"  # <-- 必要に応じて変更

TOP_N = 200

# yfinance polite access
SLEEP_SEC = 0.10
MAX_DIV_SAMPLE = 10

# factor split
P30 = 0.30
P70 = 0.70

# winsorize quantiles
Q_RET = (0.01, 0.99)
Q_BM  = (0.01, 0.99)
Q_ROE = (0.01, 0.99)
Q_INV = (0.01, 0.99)
Q_MOM = (0.01, 0.99)
Q_DY  = (0.01, 0.99)

BAD_CODE_STRINGS = {"None", "nan", "", "NaN", "NULL", "null"}

# top200前提のabnormal閾値（全銘柄に拡張したら上げる）
MIN_STOCKS_PER_MONTH = 150


# ============================================================
# Utils
# ============================================================
def log_line(s: str):
    print(s)
    with open(DIAG_LOG, "a", encoding="utf-8") as f:
        f.write(s + "\n")


def normalize_code(code: str) -> str:
    if code is None:
        return ""
    s = str(code).strip()
    if s in BAD_CODE_STRINGS:
        return ""
    return s


def winsorize(s: pd.Series, q=(0.01, 0.99)) -> pd.Series:
    x = pd.to_numeric(s, errors="coerce")
    if x.notna().sum() == 0:
        return x
    lo = x.quantile(q[0])
    hi = x.quantile(q[1])
    return x.clip(lo, hi)


def value_weighted_return(df: pd.DataFrame, ret_col: str, w_col: str) -> float:
    x = df[[ret_col, w_col]].dropna()
    if x.empty:
        return np.nan
    w = x[w_col].astype(float).to_numpy()
    r = x[ret_col].astype(float).to_numpy()
    wsum = w.sum()
    if not np.isfinite(wsum) or wsum <= 0:
        return np.nan
    return float(np.dot(r, w) / wsum)


def qcut_3way(x: pd.Series, p30=0.3, p70=0.7, labels=("L","M","H")) -> pd.Series:
    a = pd.to_numeric(x, errors="coerce")
    q1 = a.quantile(p30)
    q2 = a.quantile(p70)
    out = pd.Series(index=a.index, dtype="object")
    out[a <= q1] = labels[0]
    out[(a > q1) & (a < q2)] = labels[1]
    out[a >= q2] = labels[2]
    return out


def size_split_median(mcap: pd.Series) -> pd.Series:
    a = pd.to_numeric(mcap, errors="coerce")
    med = a.quantile(0.5)
    out = pd.Series(index=a.index, dtype="object")
    out[a <= med] = "S"
    out[a > med] = "B"
    return out


def trend_label(vals: np.ndarray) -> str:
    vals = np.array(vals, dtype=float)
    vals = vals[np.isfinite(vals)]
    if len(vals) < 2:
        return "NA"
    x = np.arange(len(vals))
    slope = np.polyfit(x, vals, 1)[0]
    if abs(slope) < 1e-6:
        return "FLAT"
    return "UP" if slope > 0 else "DOWN"


def latest_evaluable_month(factors: pd.DataFrame, col="MKT") -> pd.Timestamp:
    f = factors.dropna(subset=[col]).copy()
    if f.empty:
        return pd.NaT
    return pd.Timestamp(f["MonthEnd"].max())


# ============================================================
# Universe selection
# ============================================================
def load_snapshot_latest() -> pd.DataFrame:
    snap = pd.read_parquet(SNAP_PATH)
    snap["Code"] = snap["Code"].astype(str).map(normalize_code)
    snap = snap[snap["Code"] != ""].copy()
    snap["MonthEnd"] = pd.to_datetime(snap["MonthEnd"], errors="coerce").dt.normalize()
    snap["MarketCap"] = pd.to_numeric(snap["MarketCap"], errors="coerce")
    latest_me = snap["MonthEnd"].max()
    s = snap[snap["MonthEnd"] == latest_me].copy()
    return s


def pick_universe() -> pd.DataFrame:
    s = load_snapshot_latest().copy()
    s = s.dropna(subset=["MarketCap"]).copy()
    s = s.sort_values("MarketCap", ascending=False).drop_duplicates(["Code"], keep="first")

    if UNIVERSE_MODE == "top200":
        uni = s.head(TOP_N).copy()
        uni["Rank"] = np.arange(1, len(uni) + 1)
    elif UNIVERSE_MODE == "all_snapshot_latest":
        uni = s.copy()
        uni["Rank"] = np.arange(1, len(uni) + 1)
    else:
        raise ValueError(f"Unknown UNIVERSE_MODE={UNIVERSE_MODE}")

    uni = uni[["MonthEnd", "Code", "MarketCap", "Rank"]].copy()
    uni.to_csv(DIAG_UNIVERSE, index=False, encoding="utf-8-sig")
    log_line(f"[UNIVERSE] mode={UNIVERSE_MODE} latest MonthEnd={pd.Timestamp(uni['MonthEnd'].iloc[0]).date()} n={len(uni):,}")
    return uni


# ============================================================
# yfinance symbols
# ============================================================
def make_symbol_candidates(code: str) -> List[str]:
    code = normalize_code(code)
    if not code:
        return []
    cands = []
    # 5桁数字末尾0 -> 4桁.T 優先
    if code.isdigit() and len(code) == 5 and code.endswith("0"):
        c4 = code[:-1]
        cands.append(f"{c4}.T")
        cands.append(f"{code}.T")
    else:
        cands.append(f"{code}.T")

    out, seen = [], set()
    for x in cands:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out


def fetch_dividends(symbol: str) -> Tuple[bool, Optional[pd.Series], str]:
    try:
        t = yf.Ticker(symbol)
        div = t.dividends
        if div is None or not isinstance(div, pd.Series):
            return False, None, "dividends is None or not Series"
        return True, div.dropna(), ""
    except Exception as e:
        return False, None, f"{type(e).__name__}: {e}"


# ============================================================
# Cache I/O (parquet) + incremental append
# ============================================================
def _cache_path(symbol: str) -> Path:
    safe = symbol.replace("/", "_")
    return CACHE_EVENTS_DIR / f"{safe}.parquet"


def load_cache(symbol: str) -> pd.DataFrame:
    fp = _cache_path(symbol)
    if not fp.exists():
        return pd.DataFrame(columns=["date", "dividend"])
    df = pd.read_parquet(fp)
    if df is None or df.empty:
        return pd.DataFrame(columns=["date", "dividend"])
    df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.normalize()
    df["dividend"] = pd.to_numeric(df["dividend"], errors="coerce")
    df = df.dropna(subset=["date", "dividend"]).drop_duplicates(["date"], keep="last").sort_values("date")
    return df.reset_index(drop=True)


def save_cache(symbol: str, df: pd.DataFrame):
    fp = _cache_path(symbol)
    fp.parent.mkdir(parents=True, exist_ok=True)
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.normalize()
    df["dividend"] = pd.to_numeric(df["dividend"], errors="coerce")
    df = df.dropna(subset=["date", "dividend"]).drop_duplicates(["date"], keep="last").sort_values("date")
    df.to_parquet(fp, index=False)


def cache_last_date(cache_df: pd.DataFrame) -> pd.Timestamp:
    if cache_df is None or cache_df.empty:
        return pd.NaT
    return pd.Timestamp(cache_df["date"].max())


def series_to_df(div: pd.Series) -> pd.DataFrame:
    return pd.DataFrame({
        "date": pd.to_datetime(div.index, errors="coerce").normalize(),
        "dividend": pd.to_numeric(div.values, errors="coerce"),
    }).dropna(subset=["date", "dividend"]).drop_duplicates(["date"], keep="last").sort_values("date").reset_index(drop=True)


def incremental_update_cache(symbol: str, cache_df: pd.DataFrame) -> Tuple[pd.DataFrame, Dict]:
    """
    差分更新（last_date以降のみ追記）:
    - yfinanceからdividends Seriesを取得（Yahoo側の都合で全履歴が返ることがある）
    - cache_last_date を境に「新しいイベントだけ」を抽出して追記
    - 既存キャッシュを壊さず、append & dedup

    戻り:
      (updated_cache_df, stats)
    """
    last = cache_last_date(cache_df)
    ok, div, err = fetch_dividends(symbol)
    if not ok:
        return cache_df, {"fetched": False, "error": err, "added_rows": 0, "last_cached": str(last) if pd.notna(last) else ""}

    new_df = series_to_df(div)
    if pd.isna(last):
        # 初回: 全部をキャッシュ
        updated = new_df
        added = int(len(updated))
    else:
        # last_date より後だけ
        delta = new_df[new_df["date"] > last].copy()
        added = int(len(delta))
        if added == 0:
            updated = cache_df
        else:
            updated = pd.concat([cache_df, delta], ignore_index=True)
            updated = updated.drop_duplicates(["date"], keep="last").sort_values("date").reset_index(drop=True)

    return updated, {"fetched": True, "error": "", "added_rows": added, "last_cached": str(last.date()) if pd.notna(last) else ""}


# ============================================================
# Build dividends table (for DY calc) using cache
# ============================================================
def build_div_events_with_incremental_cache(uni: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Returns:
      - div_events_df: [Code, symbol, date, dividend]
      - fetch_results_df: per code summary
      - cache_update_df: per symbol update stats
    """
    attempt_rows = []
    results_rows = []
    cache_update_rows = []
    div_samples = []
    div_events = []

    for _, rr in uni.iterrows():
        code = rr["Code"]
        cands = make_symbol_candidates(code)

        best_ok = False
        best_symbol = ""
        best_cache = None
        last_err = ""

        # candidate loop
        for sym in cands:
            cache_df = load_cache(sym)

            # 差分更新（毎回 TTL ではなく update を試みる）
            updated_cache, stat = incremental_update_cache(sym, cache_df)
            cache_update_rows.append({"Code": code, "symbol": sym, **stat})

            if stat.get("fetched") is False:
                last_err = stat.get("error", "")
                attempt_rows.append({"Code": code, "symbol": sym, "success_series": False, "error": last_err})
                time.sleep(SLEEP_SEC)
                continue

            # fetch succeeded (even if added_rows=0)
            attempt_rows.append({"Code": code, "symbol": sym, "success_series": True, "error": ""})

            # persist cache only if changed (or first time)
            if stat.get("added_rows", 0) > 0 or (cache_df is None) or (cache_df.empty and len(updated_cache) > 0):
                save_cache(sym, updated_cache)

            best_ok = True
            best_symbol = sym
            best_cache = updated_cache
            time.sleep(SLEEP_SEC)
            break

        if not best_ok:
            results_rows.append({
                "Code": code,
                "ok": False,
                "symbol": "",
                "div_rows": 0,
                "div_min_date": "",
                "div_max_date": "",
                "attempted": "|".join(cands),
                "error": last_err or "all_candidates_failed",
            })
            continue

        # summarize best_cache
        if best_cache is None or best_cache.empty:
            results_rows.append({
                "Code": code,
                "ok": True,
                "symbol": best_symbol,
                "div_rows": 0,
                "div_min_date": "",
                "div_max_date": "",
                "attempted": "|".join(cands),
                "error": "",
            })
            continue

        dmin = pd.Timestamp(best_cache["date"].min()).date()
        dmax = pd.Timestamp(best_cache["date"].max()).date()
        results_rows.append({
            "Code": code,
            "ok": True,
            "symbol": best_symbol,
            "div_rows": int(len(best_cache)),
            "div_min_date": str(dmin),
            "div_max_date": str(dmax),
            "attempted": "|".join(cands),
            "error": "",
        })

        # samples
        tail = best_cache.tail(MAX_DIV_SAMPLE)
        for _, r2 in tail.iterrows():
            div_samples.append({
                "Code": code,
                "symbol": best_symbol,
                "date": str(pd.Timestamp(r2["date"]).date()),
                "dividend": float(r2["dividend"]),
            })

        # events
        for _, r2 in best_cache.iterrows():
            div_events.append({
                "Code": code,
                "symbol": best_symbol,
                "date": pd.Timestamp(r2["date"]).normalize(),
                "dividend": float(r2["dividend"]),
            })

    attempts_df = pd.DataFrame(attempt_rows)
    results_df = pd.DataFrame(results_rows)
    cache_update_df = pd.DataFrame(cache_update_rows)
    div_events_df = pd.DataFrame(div_events)
    div_samples_df = pd.DataFrame(div_samples)

    attempts_df.to_csv(DIAG_ATTEMPTS, index=False, encoding="utf-8-sig")
    results_df.to_csv(DIAG_FETCH_RESULTS, index=False, encoding="utf-8-sig")
    cache_update_df.to_csv(DIAG_CACHE_UPDATE, index=False, encoding="utf-8-sig")
    if len(div_samples_df):
        div_samples_df.to_csv(DIAG_DIV_SAMPLE, index=False, encoding="utf-8-sig")

    log_line(f"[SAVE] attempts={DIAG_ATTEMPTS}")
    log_line(f"[SAVE] fetch_results={DIAG_FETCH_RESULTS}")
    log_line(f"[SAVE] cache_update={DIAG_CACHE_UPDATE}")
    if len(div_samples_df):
        log_line(f"[SAVE] dividend_samples={DIAG_DIV_SAMPLE}")

    return div_events_df, results_df, cache_update_df


# ============================================================
# DY_TTM
# ============================================================
def compute_dy_ttm_for_monthends(uni: pd.DataFrame, div_events_df: pd.DataFrame) -> pd.DataFrame:
    price = pd.read_parquet(PRICE_PATH)[["Code", "MonthEnd", "AdjustedClose"]].copy()
    price["Code"] = price["Code"].astype(str).map(normalize_code)
    price = price[price["Code"] != ""].copy()
    price["MonthEnd"] = pd.to_datetime(price["MonthEnd"], errors="coerce").dt.normalize()
    price["AdjustedClose"] = pd.to_numeric(price["AdjustedClose"], errors="coerce")

    codes = set(uni["Code"].unique())
    price = price[price["Code"].isin(codes)].copy()
    price = price.sort_values(["Code", "MonthEnd"]).reset_index(drop=True)

    if div_events_df.empty:
        price["Div_TTM"] = np.nan
        price["DY_TTM"] = np.nan
        return price

    ev = div_events_df.copy()
    ev["Code"] = ev["Code"].astype(str).map(normalize_code)
    ev["date"] = pd.to_datetime(ev["date"], errors="coerce").dt.normalize()
    ev["dividend"] = pd.to_numeric(ev["dividend"], errors="coerce")
    ev = ev.dropna(subset=["Code", "date", "dividend"]).copy()
    ev = ev.sort_values(["Code", "date"]).reset_index(drop=True)

    out_rows = []
    for code, g in price.groupby("Code", sort=False):
        gg = g.copy()
        e = ev[ev["Code"] == code][["date", "dividend"]].copy()
        if e.empty:
            gg["Div_TTM"] = np.nan
            gg["DY_TTM"] = np.nan
            out_rows.append(gg)
            continue

        e = e.sort_values("date")
        e_dates = e["date"].to_numpy(dtype="datetime64[ns]")
        e_vals = e["dividend"].to_numpy(dtype=float)

        div_ttm_list = []
        for me in gg["MonthEnd"].to_numpy(dtype="datetime64[ns]"):
            start = me - np.timedelta64(365, "D")
            left = np.searchsorted(e_dates, start, side="right")
            right = np.searchsorted(e_dates, me, side="right")
            s = e_vals[left:right].sum() if right > left else 0.0
            div_ttm_list.append(float(s))

        gg["Div_TTM"] = div_ttm_list
        gg["DY_TTM"] = gg["Div_TTM"] / gg["AdjustedClose"]
        out_rows.append(gg)

    dy = pd.concat(out_rows, ignore_index=True)
    return dy


# ============================================================
# Features + Factors
# ============================================================
def load_features(uni: pd.DataFrame, dy_df: pd.DataFrame) -> pd.DataFrame:
    snap = pd.read_parquet(SNAP_PATH).copy()
    snap["Code"] = snap["Code"].astype(str).map(normalize_code)
    snap = snap[snap["Code"] != ""].copy()
    snap["MonthEnd"] = pd.to_datetime(snap["MonthEnd"], errors="coerce").dt.normalize()

    codes = set(uni["Code"].unique())
    snap = snap[snap["Code"].isin(codes)].copy()

    price = pd.read_parquet(PRICE_PATH)[["Code", "MonthEnd", "AdjustedClose"]].copy()
    price["Code"] = price["Code"].astype(str).map(normalize_code)
    price = price[price["Code"] != ""].copy()
    price["MonthEnd"] = pd.to_datetime(price["MonthEnd"], errors="coerce").dt.normalize()
    price["AdjustedClose"] = pd.to_numeric(price["AdjustedClose"], errors="coerce")
    price = price.sort_values(["Code", "MonthEnd"], kind="mergesort")

    price["Adj_next"] = price.groupby("Code")["AdjustedClose"].shift(-1)
    price["ret_m_fwd"] = (price["Adj_next"] / price["AdjustedClose"]) - 1.0
    price["Adj_lag1"] = price.groupby("Code")["AdjustedClose"].shift(1)
    price["Adj_lag12"] = price.groupby("Code")["AdjustedClose"].shift(12)
    price["MOM_12_1"] = (price["Adj_lag1"] / price["Adj_lag12"]) - 1.0

    pr = price[["Code", "MonthEnd", "ret_m_fwd", "MOM_12_1"]].copy()
    df = snap.merge(pr, on=["Code", "MonthEnd"], how="left", validate="many_to_one")

    dy2 = dy_df[["Code", "MonthEnd", "Div_TTM", "DY_TTM"]].copy()
    dy2["MonthEnd"] = pd.to_datetime(dy2["MonthEnd"], errors="coerce").dt.normalize()
    df = df.merge(dy2, on=["Code", "MonthEnd"], how="left", validate="many_to_one")

    for c in ["MarketCap","BM_Ratio","ROE","INV_Growth","ret_m_fwd","MOM_12_1","DY_TTM"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    df = df[(df["MarketCap"].notna()) & (df["MarketCap"] > 0)].copy()
    df = df[df["ret_m_fwd"].notna()].copy()

    df["ret_w"] = winsorize(df["ret_m_fwd"], Q_RET)
    df["BM_w"]  = winsorize(df["BM_Ratio"], Q_BM)
    df["ROE_w"] = winsorize(df["ROE"], Q_ROE)
    df["INV_w"] = winsorize(df["INV_Growth"], Q_INV)
    df["MOM_w"] = winsorize(df["MOM_12_1"], Q_MOM)
    df["DY_w"]  = winsorize(df["DY_TTM"], Q_DY)

    cov = (df.groupby("MonthEnd")
             .agg(n=("Code","nunique"),
                  dy_notna=("DY_TTM", lambda s: float(s.notna().mean()*100)),
                  bm_notna=("BM_Ratio", lambda s: float(s.notna().mean()*100)))
             .reset_index())
    cov.to_csv(DIAG_DY_COV_BY_MONTH, index=False, encoding="utf-8-sig")

    return df


def compute_factors(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    months = sorted(df["MonthEnd"].dropna().unique())
    rows = []
    diag_n = []

    for me in months:
        m = df[df["MonthEnd"] == me].copy()
        n_total = int(m["Code"].nunique())
        if n_total < 30:
            continue

        MKT = value_weighted_return(m, "ret_w", "MarketCap")

        m["SZ"] = size_split_median(m["MarketCap"])
        m["VAL"]  = qcut_3way(m["BM_w"],  P30, P70, labels=("L","M","H"))
        m["PROF"] = qcut_3way(m["ROE_w"], P30, P70, labels=("W","M","R"))
        m["INV"]  = qcut_3way(m["INV_w"], P30, P70, labels=("C","M","A"))
        m["MOM"]  = qcut_3way(m["MOM_w"], P30, P70, labels=("L","M","W"))
        m["DY"]   = qcut_3way(m["DY_w"],  P30, P70, labels=("L","M","H"))

        def port_ret(sz, col, lab):
            x = m[(m["SZ"] == sz) & (m[col] == lab)]
            return value_weighted_return(x, "ret_w", "MarketCap")

        # HML
        SH = port_ret("S","VAL","H"); SL = port_ret("S","VAL","L")
        BH = port_ret("B","VAL","H"); BL = port_ret("B","VAL","L")
        HML = np.nanmean([SH, BH]) - np.nanmean([SL, BL])

        # RMW
        SR = port_ret("S","PROF","R"); SW = port_ret("S","PROF","W")
        BR = port_ret("B","PROF","R"); BW = port_ret("B","PROF","W")
        RMW = np.nanmean([SR, BR]) - np.nanmean([SW, BW])

        # CMA
        SC = port_ret("S","INV","C"); SA = port_ret("S","INV","A")
        BC = port_ret("B","INV","C"); BA = port_ret("B","INV","A")
        CMA = np.nanmean([SC, BC]) - np.nanmean([SA, BA])

        # SMB
        SMv = port_ret("S","VAL","M"); BMv = port_ret("B","VAL","M")
        SMB_HML = np.nanmean([SH, SMv, SL]) - np.nanmean([BH, BMv, BL])
        SMp = port_ret("S","PROF","M"); BMp = port_ret("B","PROF","M")
        SMB_RMW = np.nanmean([SR, SMp, SW]) - np.nanmean([BR, BMp, BW])
        SMa = port_ret("S","INV","M"); BMa = port_ret("B","INV","M")
        SMB_CMA = np.nanmean([SC, SMa, SA]) - np.nanmean([BC, BMa, BA])
        SMB = np.nanmean([SMB_HML, SMB_RMW, SMB_CMA])

        # WML
        SWin = port_ret("S","MOM","W"); SLos = port_ret("S","MOM","L")
        BWin = port_ret("B","MOM","W"); BLos = port_ret("B","MOM","L")
        WML = np.nanmean([SWin, BWin]) - np.nanmean([SLos, BLos])

        # HDYMLDY
        SHD = port_ret("S","DY","H"); SLD = port_ret("S","DY","L")
        BHD = port_ret("B","DY","H"); BLD = port_ret("B","DY","L")
        HDYMLDY = np.nanmean([SHD, BHD]) - np.nanmean([SLD, BLD])

        n_used_dy = int(m.dropna(subset=["DY_w","ret_w","MarketCap"])["Code"].nunique())
        confidence = float(min(1.0, n_used_dy / max(1, n_total)))
        abnormal = (n_used_dy < MIN_STOCKS_PER_MONTH)

        rows.append({
            "MonthEnd": me,
            "MKT": MKT, "SMB": SMB, "HML": HML, "RMW": RMW, "CMA": CMA, "WML": WML, "HDYMLDY": HDYMLDY,
            "n_total": n_total,
            "n_used_dy": n_used_dy,
            "confidence": confidence,
            "abnormal": bool(abnormal),
        })

        diag_n.append({
            "MonthEnd": me,
            "n_total": n_total,
            "n_used_dy": n_used_dy,
            "confidence": confidence,
            "abnormal": bool(abnormal),
        })

    fac = pd.DataFrame(rows).sort_values("MonthEnd").reset_index(drop=True)
    diag_n_df = pd.DataFrame(diag_n).sort_values("MonthEnd").reset_index(drop=True)
    diag_n_df.to_csv(DIAG_FACTOR_NUSED, index=False, encoding="utf-8-sig")
    return fac, diag_n_df


# ============================================================
# dominant = abs * confidence (requested)
# ============================================================
def analyze_latest(fac: pd.DataFrame) -> Dict:
    if fac.empty:
        return {"error": "factor table is empty"}

    last_me = latest_evaluable_month(fac, col="MKT")
    cur = fac[fac["MonthEnd"] == last_me].iloc[0].to_dict()

    factor_cols = ["MKT","SMB","HML","RMW","CMA","WML","HDYMLDY"]
    conf = float(cur.get("confidence", 0.0))

    # ===== PATCH START: dominant definition fix (abs*confidence 最大を正しく選ぶ) =====
    score = {c: abs(float(cur.get(c, np.nan))) * conf for c in factor_cols}
    dominant = max(score, key=lambda k: np.nan_to_num(score[k], nan=-1))
    # ===== PATCH END =====

    fs = fac.sort_values("MonthEnd").reset_index(drop=True)
    last3 = fs.tail(3)
    last12 = fs.tail(12)

    trends3 = {c: trend_label(last3[c].to_numpy()) for c in factor_cols}
    trends12 = {c: trend_label(last12[c].to_numpy()) for c in factor_cols}

    mkt = cur.get("MKT", np.nan)
    regime = "RANGE"
    if np.isfinite(mkt):
        if mkt > 0.01:
            regime = "BULL"
        elif mkt < -0.01:
            regime = "BEAR"

    tilt = {}
    for c in factor_cols:
        v = float(np.nanmean(last3[c]))
        tilt[c] = float(np.sign(v)) if np.isfinite(v) else 0.0
    tilt_conf = {k: v * conf for k, v in tilt.items()}

    return {
        "latest_month": str(pd.Timestamp(last_me).date()),
        "dominant_factor": dominant,
        "dominant_scores": score,
        "regime": regime,
        "latest": cur,
        "trends_last3": trends3,
        "trends_last12": trends12,
        "tilt_conf_weighted": tilt_conf,
    }


def save_outputs(fac: pd.DataFrame, analysis: Dict):
    fac.to_parquet(OUT_PARQUET, index=False)

    latest = analysis.get("latest", {})
    summary = {
        "latest_month": analysis.get("latest_month", ""),
        "regime": analysis.get("regime", ""),
        "dominant_factor": analysis.get("dominant_factor", ""),
        "confidence": float(latest.get("confidence", np.nan)),
        "abnormal": bool(latest.get("abnormal", True)),
        "n_total": int(latest.get("n_total", 0) or 0),
        "n_used_dy": int(latest.get("n_used_dy", 0) or 0),
    }
    for c in ["MKT","SMB","HML","RMW","CMA","WML","HDYMLDY"]:
        summary[c] = float(latest.get(c, np.nan))
        summary[f"score_{c}"] = float(analysis.get("dominant_scores", {}).get(c, np.nan))
    pd.DataFrame([summary]).to_csv(OUT_SUMMARY, index=False, encoding="utf-8-sig")

    fac_sorted = fac.sort_values("MonthEnd").reset_index(drop=True)
    cols = ["MonthEnd","MKT","SMB","HML","RMW","CMA","WML","HDYMLDY","confidence","abnormal"]
    fac_sorted.tail(3)[cols].to_csv(OUT_TREND_LAST3, index=False, encoding="utf-8-sig")
    fac_sorted.tail(12)[cols].to_csv(OUT_TREND_LAST12, index=False, encoding="utf-8-sig")


# ============================================================
# Main
# ============================================================
def main():
    with open(DIAG_LOG, "w", encoding="utf-8") as f:
        f.write("")

    log_line("="*110)
    log_line("FF5 + MOM(12-1) + DY_TTM + yfinance cache (incremental append)")
    log_line("="*110)
    log_line(f"PRICE_PATH: {PRICE_PATH}")
    log_line(f"SNAP_PATH : {SNAP_PATH}")
    log_line(f"UNIVERSE_MODE: {UNIVERSE_MODE} (TOP_N={TOP_N})")
    log_line(f"CACHE_DIR : {CACHE_EVENTS_DIR}")

    # 1) universe
    uni = pick_universe()

    # 2) dividends (incremental cache update)
    div_events_df, fetch_results_df, cache_update_df = build_div_events_with_incremental_cache(uni)
    ok_any = int(fetch_results_df["ok"].sum()) if len(fetch_results_df) else 0
    ok_nonempty = int((fetch_results_df["ok"] & (fetch_results_df["div_rows"] > 0)).sum()) if len(fetch_results_df) else 0
    log_line(f"[YF] ok(symbol found)= {ok_any}/{len(fetch_results_df)} | nonempty(div_rows>0)= {ok_nonempty}/{len(fetch_results_df)}")

    # cache update summary
    if len(cache_update_df):
        added_total = int(pd.to_numeric(cache_update_df["added_rows"], errors="coerce").fillna(0).sum())
        fetched_fail = int((cache_update_df.get("fetched", True) == False).sum()) if "fetched" in cache_update_df.columns else 0
        log_line(f"[CACHE] symbols tried={len(cache_update_df)} total_added_rows={added_total} fetched_fail={fetched_fail}")

    # 3) DY_TTM
    dy_df = compute_dy_ttm_for_monthends(uni, div_events_df)
    cov_latest = (dy_df[dy_df["MonthEnd"] == dy_df["MonthEnd"].max()]
                  .assign(dy_notna=lambda x: x["DY_TTM"].notna())
                  ["dy_notna"].mean() * 100.0)
    log_line(f"[DY] latest MonthEnd={dy_df['MonthEnd'].max().date()} DY_TTM notna%={cov_latest:.2f}")

    # 4) features + factors
    feat = load_features(uni, dy_df)
    log_line(f"[FEATURE] rows={len(feat):,} codes={feat['Code'].nunique():,} months={feat['MonthEnd'].nunique():,}")

    fac, _ = compute_factors(feat)
    log_line(f"[FACTORS] months computed={len(fac)} min={fac['MonthEnd'].min()} max={fac['MonthEnd'].max()}")

    if fac.empty:
        raise RuntimeError("Factor result is empty. Check DY/BM coverage and thresholds.")

    analysis = analyze_latest(fac)
    last = analysis.get("latest", {})
    log_line("-"*110)
    log_line(f"[LATEST EVALUABLE MONTH] {analysis.get('latest_month')}")
    log_line(f"[DOMINANT FACTOR (abs*confidence)] {analysis.get('dominant_factor')}")
    log_line(f"[REGIME] {analysis.get('regime')}")
    log_line(f"[CONFIDENCE] {last.get('confidence'):.3f} | abnormal={last.get('abnormal')} | "
             f"n_total={last.get('n_total')} n_used_dy={last.get('n_used_dy')}")
    log_line("Factor returns:")
    for c in ["MKT","SMB","HML","RMW","CMA","WML","HDYMLDY"]:
        log_line(f"  {c}: {last.get(c)}")
    log_line("Dominant scores (abs*confidence):")
    for c, sc in sorted(analysis.get("dominant_scores", {}).items(), key=lambda kv: -np.nan_to_num(kv[1], nan=-1)):
        log_line(f"  score_{c}: {sc}")

    log_line("-"*110)
    log_line("[STRATEGY HINT] (last3 avg direction, confidence-weighted)")
    for c, score in analysis.get("tilt_conf_weighted", {}).items():
        if score > 0:
            s = "OVERWEIGHT"
        elif score < 0:
            s = "UNDERWEIGHT"
        else:
            s = "NEUTRAL"
        log_line(f"  {c}: {s} (score={score:.2f})")

    # 5) save outputs
    save_outputs(fac, analysis)
    log_line("-"*110)
    log_line("✅ saved:")
    log_line(f"  - {OUT_PARQUET}")
    log_line(f"  - {OUT_SUMMARY}")
    log_line(f"  - {OUT_TREND_LAST3}")
    log_line(f"  - {OUT_TREND_LAST12}")
    log_line(f"  - {DIAG_UNIVERSE}")
    log_line(f"  - {DIAG_ATTEMPTS}")
    log_line(f"  - {DIAG_FETCH_RESULTS}")
    log_line(f"  - {DIAG_CACHE_UPDATE}")
    log_line(f"  - {DIAG_DY_COV_BY_MONTH}")
    log_line(f"  - {DIAG_FACTOR_NUSED}")
    log_line(f"  - {DIAG_LOG}")


if __name__ == "__main__":
    main()


FF5 + MOM(12-1) + DY_TTM + yfinance cache (incremental append)
PRICE_PATH: C:\Users\yongr\Project\merged_data_all_stocks\factors\price_month_end.parquet
SNAP_PATH : C:\Users\yongr\Project\merged_data_all_stocks\factors\month_end_snapshot.parquet
UNIVERSE_MODE: top200 (TOP_N=200)
CACHE_DIR : C:\Users\yongr\Project\merged_data_all_stocks\factors\yf_dividend_cache\events_by_symbol
[UNIVERSE] mode=top200 latest MonthEnd=2025-09-30 n=200


$167A0.T: possibly delisted; no timezone found


[SAVE] attempts=C:\Users\yongr\Project\merged_data_all_stocks\factors\diag_yfinance_dividend_incremental\diag_yf_symbol_attempts.csv
[SAVE] fetch_results=C:\Users\yongr\Project\merged_data_all_stocks\factors\diag_yfinance_dividend_incremental\diag_yf_fetch_results.csv
[SAVE] cache_update=C:\Users\yongr\Project\merged_data_all_stocks\factors\diag_yfinance_dividend_incremental\diag_yf_cache_update.csv
[SAVE] dividend_samples=C:\Users\yongr\Project\merged_data_all_stocks\factors\diag_yfinance_dividend_incremental\diag_yf_dividend_samples.csv
[YF] ok(symbol found)= 200/200 | nonempty(div_rows>0)= 198/200
[CACHE] symbols tried=200 total_added_rows=0 fetched_fail=0
[DY] latest MonthEnd=2026-01-31 DY_TTM notna%=99.00
[FEATURE] rows=21,261 codes=200 months=117
[FACTORS] months computed=117 min=2016-01-31 00:00:00 max=2025-09-30 00:00:00
--------------------------------------------------------------------------------------------------------------
[LATEST EVALUABLE MONTH] 2025-09-30
[DOMINANT FA

In [ ]:
# -*- coding: utf-8 -*-
"""
FF5 + MOM(12-1) + DY_TTM（TTM dividend yield）
+ yfinance dividends per-symbol cache (parquet)
+ incremental update: append only events after last_cached_date (no TTL)

[A2運用]
- 初回は MarketCap上位 N_START（例:1000）でキャッシュ構築＆傾向確認
- 問題がなければ all_snapshot_latest（全銘柄）へ拡張

dominant因子判定:
- dominant = argmax(|factor| * confidence)

注意:
- yfinanceは非公式でYahoo側仕様変更で壊れる可能性あり
- dividendsのイベント日付は権利落ち日中心のことがある → TTM合計で吸収
  [Source] https://github.com/ranaroussi/yfinance/issues/568

依存:
  pip install yfinance pandas pyarrow
"""

import warnings
warnings.filterwarnings("ignore")

import time
from pathlib import Path
from typing import Dict, Tuple, List, Optional

import numpy as np
import pandas as pd
import yfinance as yf


# ============================================================
# Paths (あなたの環境)
# ============================================================
FACTORS_DIR = Path(r"C:\Users\yongr\Project\merged_data_all_stocks\factors")
PRICE_PATH = FACTORS_DIR / "price_month_end.parquet"
SNAP_PATH  = FACTORS_DIR / "month_end_snapshot.parquet"

# 出力（既存FF5+MOMの出力は壊さない：別名）
OUT_PARQUET = FACTORS_DIR / "ff5_mom_dy_factors_monthly.parquet"
OUT_SUMMARY = FACTORS_DIR / "market_factor_summary_ff5_mom_dy.csv"
OUT_TREND_LAST3 = FACTORS_DIR / "market_factor_trend_last3_ff5_mom_dy.csv"
OUT_TREND_LAST12 = FACTORS_DIR / "market_factor_trend_last12_ff5_mom_dy.csv"

# diagnostics + cache
DIAG_DIR = FACTORS_DIR / "diag_yfinance_dividend_incremental"
DIAG_DIR.mkdir(parents=True, exist_ok=True)

CACHE_DIR = FACTORS_DIR / "yf_dividend_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_EVENTS_DIR = CACHE_DIR / "events_by_symbol"
CACHE_EVENTS_DIR.mkdir(parents=True, exist_ok=True)

# diagnostics
DIAG_UNIVERSE = DIAG_DIR / "diag_universe.csv"
DIAG_ATTEMPTS = DIAG_DIR / "diag_yf_symbol_attempts.csv"
DIAG_FETCH_RESULTS = DIAG_DIR / "diag_yf_fetch_results.csv"
DIAG_CACHE_UPDATE = DIAG_DIR / "diag_yf_cache_update.csv"
DIAG_DIV_SAMPLE = DIAG_DIR / "diag_yf_dividend_samples.csv"
DIAG_DY_COV_BY_MONTH = DIAG_DIR / "diag_dy_ttm_coverage_by_month.csv"
DIAG_FACTOR_NUSED = DIAG_DIR / "diag_factor_n_used_by_month.csv"
DIAG_LOG = DIAG_DIR / "diag_runtime_log.txt"

# ===== PATCH START: A2向けの追加診断CSV =====
DIAG_YF_FAIL_CODES = DIAG_DIR / "diag_yf_fail_codes.csv"
DIAG_YF_RULE_STATS = DIAG_DIR / "diag_yf_symbol_rule_stats.csv"
DIAG_CACHE_INVENTORY = DIAG_DIR / "diag_cache_inventory.csv"
# ===== PATCH END =====


# ============================================================
# Params
# ============================================================
# ユニバース設定：
# - "topN": 最新MonthEndのMarketCap上位N（A2の初回検証で使う）
# - "all_snapshot_latest": 最新MonthEndに存在する全銘柄（全銘柄へ拡張）
UNIVERSE_MODE = "topN"   # <-- A2: まず topN 推奨

# ===== PATCH START: A2向け (Top200固定ではなくTopNへ) =====
TOP_N = 1000            # A2初回：まずは1000
# ===== PATCH END =====

# yfinance polite access（全銘柄時は少し重め推奨）
SLEEP_SEC = 0.10 if UNIVERSE_MODE in ("topN", "top200") else 0.30
MAX_DIV_SAMPLE = 10

# factor split
P30 = 0.30
P70 = 0.70

# winsorize quantiles
Q_RET = (0.01, 0.99)
Q_BM  = (0.01, 0.99)
Q_ROE = (0.01, 0.99)
Q_INV = (0.01, 0.99)
Q_MOM = (0.01, 0.99)
Q_DY  = (0.01, 0.99)

BAD_CODE_STRINGS = {"None", "nan", "", "NaN", "NULL", "null"}

# ===== PATCH START: abnormal閾値を「比率ベース」に（全銘柄拡張で自然にスケール） =====
ABNORMAL_MIN_RATIO = 0.70   # n_used_dy < 0.7*n_total を abnormal
MIN_STOCKS_PER_MONTH_FALLBACK = 150  # 互換用（topNが小さい時の保険）
# ===== PATCH END =====


# ============================================================
# Utils
# ============================================================
def log_line(s: str):
    print(s)
    with open(DIAG_LOG, "a", encoding="utf-8") as f:
        f.write(s + "\n")


def normalize_code(code: str) -> str:
    if code is None:
        return ""
    s = str(code).strip()
    if s in BAD_CODE_STRINGS:
        return ""
    return s


def winsorize(s: pd.Series, q=(0.01, 0.99)) -> pd.Series:
    x = pd.to_numeric(s, errors="coerce")
    if x.notna().sum() == 0:
        return x
    lo = x.quantile(q[0])
    hi = x.quantile(q[1])
    return x.clip(lo, hi)


def value_weighted_return(df: pd.DataFrame, ret_col: str, w_col: str) -> float:
    x = df[[ret_col, w_col]].dropna()
    if x.empty:
        return np.nan
    w = x[w_col].astype(float).to_numpy()
    r = x[ret_col].astype(float).to_numpy()
    wsum = w.sum()
    if not np.isfinite(wsum) or wsum <= 0:
        return np.nan
    return float(np.dot(r, w) / wsum)


def qcut_3way(x: pd.Series, p30=0.3, p70=0.7, labels=("L","M","H")) -> pd.Series:
    a = pd.to_numeric(x, errors="coerce")
    q1 = a.quantile(p30)
    q2 = a.quantile(p70)
    out = pd.Series(index=a.index, dtype="object")
    out[a <= q1] = labels[0]
    out[(a > q1) & (a < q2)] = labels[1]
    out[a >= q2] = labels[2]
    return out


def size_split_median(mcap: pd.Series) -> pd.Series:
    a = pd.to_numeric(mcap, errors="coerce")
    med = a.quantile(0.5)
    out = pd.Series(index=a.index, dtype="object")
    out[a <= med] = "S"
    out[a > med] = "B"
    return out


def trend_label(vals: np.ndarray) -> str:
    vals = np.array(vals, dtype=float)
    vals = vals[np.isfinite(vals)]
    if len(vals) < 2:
        return "NA"
    x = np.arange(len(vals))
    slope = np.polyfit(x, vals, 1)[0]
    if abs(slope) < 1e-6:
        return "FLAT"
    return "UP" if slope > 0 else "DOWN"


def latest_evaluable_month(factors: pd.DataFrame, col="MKT") -> pd.Timestamp:
    f = factors.dropna(subset=[col]).copy()
    if f.empty:
        return pd.NaT
    return pd.Timestamp(f["MonthEnd"].max())


# ============================================================
# Universe selection
# ============================================================
def load_snapshot_latest() -> pd.DataFrame:
    snap = pd.read_parquet(SNAP_PATH)
    snap["Code"] = snap["Code"].astype(str).map(normalize_code)
    snap = snap[snap["Code"] != ""].copy()
    snap["MonthEnd"] = pd.to_datetime(snap["MonthEnd"], errors="coerce").dt.normalize()
    snap["MarketCap"] = pd.to_numeric(snap["MarketCap"], errors="coerce")
    latest_me = snap["MonthEnd"].max()
    s = snap[snap["MonthEnd"] == latest_me].copy()
    return s


def pick_universe() -> pd.DataFrame:
    s = load_snapshot_latest().copy()
    s = s.dropna(subset=["MarketCap"]).copy()
    s = s.sort_values("MarketCap", ascending=False).drop_duplicates(["Code"], keep="first")

    # ===== PATCH START: UNIVERSE_MODE拡張（topN/all） =====
    if UNIVERSE_MODE in ("topN", "top200"):
        n = TOP_N if UNIVERSE_MODE == "topN" else 200
        uni = s.head(n).copy()
        uni["Rank"] = np.arange(1, len(uni) + 1)
    elif UNIVERSE_MODE == "all_snapshot_latest":
        uni = s.copy()
        uni["Rank"] = np.arange(1, len(uni) + 1)
    else:
        raise ValueError(f"Unknown UNIVERSE_MODE={UNIVERSE_MODE}")
    # ===== PATCH END =====

    uni = uni[["MonthEnd", "Code", "MarketCap", "Rank"]].copy()
    uni.to_csv(DIAG_UNIVERSE, index=False, encoding="utf-8-sig")
    log_line(f"[UNIVERSE] mode={UNIVERSE_MODE} latest MonthEnd={pd.Timestamp(uni['MonthEnd'].iloc[0]).date()} n={len(uni):,}")
    return uni


# ============================================================
# yfinance symbols
# ============================================================
def make_symbol_candidates(code: str) -> List[str]:
    code = normalize_code(code)
    if not code:
        return []
    cands = []
    # 5桁数字末尾0 -> 4桁.T 優先
    if code.isdigit() and len(code) == 5 and code.endswith("0"):
        c4 = code[:-1]
        cands.append(f"{c4}.T")
        cands.append(f"{code}.T")
    else:
        cands.append(f"{code}.T")

    out, seen = [], set()
    for x in cands:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out


def classify_symbol_rule(symbol: str) -> str:
    """
    ルール別成功率の診断用
    """
    if symbol.endswith(".T"):
        base = symbol[:-2]
        if base.isdigit() and len(base) == 4:
            return "digit4_dotT"
        if base.isdigit() and len(base) == 5:
            return "digit5_dotT"
        if any(ch.isalpha() for ch in base):
            return "alnum_dotT"
    return "other"


def fetch_dividends(symbol: str) -> Tuple[bool, Optional[pd.Series], str]:
    try:
        t = yf.Ticker(symbol)
        div = t.dividends
        if div is None or not isinstance(div, pd.Series):
            return False, None, "dividends is None or not Series"
        return True, div.dropna(), ""
    except Exception as e:
        return False, None, f"{type(e).__name__}: {e}"


# ============================================================
# Cache I/O (parquet) + incremental append
# ============================================================
def _cache_path(symbol: str) -> Path:
    safe = symbol.replace("/", "_")
    return CACHE_EVENTS_DIR / f"{safe}.parquet"


def load_cache(symbol: str) -> pd.DataFrame:
    fp = _cache_path(symbol)
    if not fp.exists():
        return pd.DataFrame(columns=["date", "dividend"])
    df = pd.read_parquet(fp)
    if df is None or df.empty:
        return pd.DataFrame(columns=["date", "dividend"])
    df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.normalize()
    df["dividend"] = pd.to_numeric(df["dividend"], errors="coerce")
    df = df.dropna(subset=["date", "dividend"]).drop_duplicates(["date"], keep="last").sort_values("date")
    return df.reset_index(drop=True)


def save_cache(symbol: str, df: pd.DataFrame):
    fp = _cache_path(symbol)
    fp.parent.mkdir(parents=True, exist_ok=True)
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.normalize()
    df["dividend"] = pd.to_numeric(df["dividend"], errors="coerce")
    df = df.dropna(subset=["date", "dividend"]).drop_duplicates(["date"], keep="last").sort_values("date")
    df.to_parquet(fp, index=False)


def cache_last_date(cache_df: pd.DataFrame) -> pd.Timestamp:
    if cache_df is None or cache_df.empty:
        return pd.NaT
    return pd.Timestamp(cache_df["date"].max())


def series_to_df(div: pd.Series) -> pd.DataFrame:
    return pd.DataFrame({
        "date": pd.to_datetime(div.index, errors="coerce").normalize(),
        "dividend": pd.to_numeric(div.values, errors="coerce"),
    }).dropna(subset=["date", "dividend"]).drop_duplicates(["date"], keep="last").sort_values("date").reset_index(drop=True)


def incremental_update_cache(symbol: str, cache_df: pd.DataFrame) -> Tuple[pd.DataFrame, Dict]:
    last = cache_last_date(cache_df)
    ok, div, err = fetch_dividends(symbol)
    if not ok:
        return cache_df, {"fetched": False, "error": err, "added_rows": 0, "last_cached": str(last) if pd.notna(last) else ""}

    new_df = series_to_df(div)
    if pd.isna(last):
        updated = new_df
        added = int(len(updated))
    else:
        delta = new_df[new_df["date"] > last].copy()
        added = int(len(delta))
        if added == 0:
            updated = cache_df
        else:
            updated = pd.concat([cache_df, delta], ignore_index=True)
            updated = updated.drop_duplicates(["date"], keep="last").sort_values("date").reset_index(drop=True)

    return updated, {"fetched": True, "error": "", "added_rows": added, "last_cached": str(last.date()) if pd.notna(last) else ""}


# ============================================================
# Build dividends table (for DY calc) using cache
# ============================================================
def build_div_events_with_incremental_cache(uni: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    attempt_rows = []
    results_rows = []
    cache_update_rows = []
    div_samples = []
    div_events = []

    for _, rr in uni.iterrows():
        code = rr["Code"]
        cands = make_symbol_candidates(code)

        best_ok = False
        best_symbol = ""
        best_cache = None
        last_err = ""

        for sym in cands:
            cache_df = load_cache(sym)
            updated_cache, stat = incremental_update_cache(sym, cache_df)
            cache_update_rows.append({"Code": code, "symbol": sym, "rule": classify_symbol_rule(sym), **stat})

            if stat.get("fetched") is False:
                last_err = stat.get("error", "")
                attempt_rows.append({"Code": code, "symbol": sym, "rule": classify_symbol_rule(sym),
                                     "success_series": False, "error": last_err})
                time.sleep(SLEEP_SEC)
                continue

            attempt_rows.append({"Code": code, "symbol": sym, "rule": classify_symbol_rule(sym),
                                 "success_series": True, "error": ""})

            if stat.get("added_rows", 0) > 0 or (cache_df is None) or (cache_df.empty and len(updated_cache) > 0):
                save_cache(sym, updated_cache)

            best_ok = True
            best_symbol = sym
            best_cache = updated_cache
            time.sleep(SLEEP_SEC)
            break

        if not best_ok:
            results_rows.append({
                "Code": code, "ok": False, "symbol": "", "div_rows": 0,
                "div_min_date": "", "div_max_date": "",
                "attempted": "|".join(cands), "error": last_err or "all_candidates_failed",
            })
            continue

        if best_cache is None or best_cache.empty:
            results_rows.append({
                "Code": code, "ok": True, "symbol": best_symbol, "div_rows": 0,
                "div_min_date": "", "div_max_date": "", "attempted": "|".join(cands), "error": "",
            })
            continue

        dmin = pd.Timestamp(best_cache["date"].min()).date()
        dmax = pd.Timestamp(best_cache["date"].max()).date()
        results_rows.append({
            "Code": code, "ok": True, "symbol": best_symbol, "div_rows": int(len(best_cache)),
            "div_min_date": str(dmin), "div_max_date": str(dmax),
            "attempted": "|".join(cands), "error": "",
        })

        tail = best_cache.tail(MAX_DIV_SAMPLE)
        for _, r2 in tail.iterrows():
            div_samples.append({
                "Code": code, "symbol": best_symbol,
                "date": str(pd.Timestamp(r2["date"]).date()),
                "dividend": float(r2["dividend"]),
            })

        for _, r2 in best_cache.iterrows():
            div_events.append({
                "Code": code, "symbol": best_symbol,
                "date": pd.Timestamp(r2["date"]).normalize(),
                "dividend": float(r2["dividend"]),
            })

    attempts_df = pd.DataFrame(attempt_rows)
    results_df = pd.DataFrame(results_rows)
    cache_update_df = pd.DataFrame(cache_update_rows)
    div_events_df = pd.DataFrame(div_events)
    div_samples_df = pd.DataFrame(div_samples)

    attempts_df.to_csv(DIAG_ATTEMPTS, index=False, encoding="utf-8-sig")
    results_df.to_csv(DIAG_FETCH_RESULTS, index=False, encoding="utf-8-sig")
    cache_update_df.to_csv(DIAG_CACHE_UPDATE, index=False, encoding="utf-8-sig")
    if len(div_samples_df):
        div_samples_df.to_csv(DIAG_DIV_SAMPLE, index=False, encoding="utf-8-sig")

    # ===== PATCH START: fail一覧 + ルール別成功率 =====
    if len(results_df):
        fail = results_df[~results_df["ok"]].copy()
        fail.to_csv(DIAG_YF_FAIL_CODES, index=False, encoding="utf-8-sig")

    if len(cache_update_df):
        # ルール別（試行ベース）
        g = cache_update_df.groupby(["rule"], dropna=False).agg(
            tried=("symbol","count"),
            fetched_ok=("fetched", lambda s: int(pd.Series(s).astype(bool).sum())),
            added_rows=("added_rows", lambda s: float(pd.to_numeric(s, errors="coerce").fillna(0).sum())),
        ).reset_index()
        g["fetched_ok_rate"] = g["fetched_ok"] / g["tried"]
        g.to_csv(DIAG_YF_RULE_STATS, index=False, encoding="utf-8-sig")
    # ===== PATCH END =====

    log_line(f"[SAVE] attempts={DIAG_ATTEMPTS}")
    log_line(f"[SAVE] fetch_results={DIAG_FETCH_RESULTS}")
    log_line(f"[SAVE] cache_update={DIAG_CACHE_UPDATE}")
    if len(div_samples_df):
        log_line(f"[SAVE] dividend_samples={DIAG_DIV_SAMPLE}")
    log_line(f"[SAVE] fail_codes={DIAG_YF_FAIL_CODES}")
    log_line(f"[SAVE] rule_stats={DIAG_YF_RULE_STATS}")

    return div_events_df, results_df, cache_update_df


# ============================================================
# DY_TTM
# ============================================================
def compute_dy_ttm_for_monthends(uni: pd.DataFrame, div_events_df: pd.DataFrame) -> pd.DataFrame:
    price = pd.read_parquet(PRICE_PATH)[["Code", "MonthEnd", "AdjustedClose"]].copy()
    price["Code"] = price["Code"].astype(str).map(normalize_code)
    price = price[price["Code"] != ""].copy()
    price["MonthEnd"] = pd.to_datetime(price["MonthEnd"], errors="coerce").dt.normalize()
    price["AdjustedClose"] = pd.to_numeric(price["AdjustedClose"], errors="coerce")

    codes = set(uni["Code"].unique())
    price = price[price["Code"].isin(codes)].copy()
    price = price.sort_values(["Code", "MonthEnd"]).reset_index(drop=True)

    if div_events_df.empty:
        price["Div_TTM"] = np.nan
        price["DY_TTM"] = np.nan
        return price

    ev = div_events_df.copy()
    ev["Code"] = ev["Code"].astype(str).map(normalize_code)
    ev["date"] = pd.to_datetime(ev["date"], errors="coerce").dt.normalize()
    ev["dividend"] = pd.to_numeric(ev["dividend"], errors="coerce")
    ev = ev.dropna(subset=["Code", "date", "dividend"]).copy()
    ev = ev.sort_values(["Code", "date"]).reset_index(drop=True)

    out_rows = []
    for code, g in price.groupby("Code", sort=False):
        gg = g.copy()
        e = ev[ev["Code"] == code][["date", "dividend"]].copy()
        if e.empty:
            gg["Div_TTM"] = np.nan
            gg["DY_TTM"] = np.nan
            out_rows.append(gg)
            continue

        e = e.sort_values("date")
        e_dates = e["date"].to_numpy(dtype="datetime64[ns]")
        e_vals = e["dividend"].to_numpy(dtype=float)

        div_ttm_list = []
        for me in gg["MonthEnd"].to_numpy(dtype="datetime64[ns]"):
            start = me - np.timedelta64(365, "D")
            left = np.searchsorted(e_dates, start, side="right")
            right = np.searchsorted(e_dates, me, side="right")
            s = e_vals[left:right].sum() if right > left else 0.0
            div_ttm_list.append(float(s))

        gg["Div_TTM"] = div_ttm_list
        gg["DY_TTM"] = gg["Div_TTM"] / gg["AdjustedClose"]
        out_rows.append(gg)

    dy = pd.concat(out_rows, ignore_index=True)
    return dy


# ============================================================
# Features + Factors
# ============================================================
def load_features(uni: pd.DataFrame, dy_df: pd.DataFrame) -> pd.DataFrame:
    snap = pd.read_parquet(SNAP_PATH).copy()
    snap["Code"] = snap["Code"].astype(str).map(normalize_code)
    snap = snap[snap["Code"] != ""].copy()
    snap["MonthEnd"] = pd.to_datetime(snap["MonthEnd"], errors="coerce").dt.normalize()

    codes = set(uni["Code"].unique())
    snap = snap[snap["Code"].isin(codes)].copy()

    price = pd.read_parquet(PRICE_PATH)[["Code", "MonthEnd", "AdjustedClose"]].copy()
    price["Code"] = price["Code"].astype(str).map(normalize_code)
    price = price[price["Code"] != ""].copy()
    price["MonthEnd"] = pd.to_datetime(price["MonthEnd"], errors="coerce").dt.normalize()
    price["AdjustedClose"] = pd.to_numeric(price["AdjustedClose"], errors="coerce")
    price = price.sort_values(["Code", "MonthEnd"], kind="mergesort")

    price["Adj_next"] = price.groupby("Code")["AdjustedClose"].shift(-1)
    price["ret_m_fwd"] = (price["Adj_next"] / price["AdjustedClose"]) - 1.0
    price["Adj_lag1"] = price.groupby("Code")["AdjustedClose"].shift(1)
    price["Adj_lag12"] = price.groupby("Code")["AdjustedClose"].shift(12)
    price["MOM_12_1"] = (price["Adj_lag1"] / price["Adj_lag12"]) - 1.0

    pr = price[["Code", "MonthEnd", "ret_m_fwd", "MOM_12_1"]].copy()
    df = snap.merge(pr, on=["Code", "MonthEnd"], how="left", validate="many_to_one")

    dy2 = dy_df[["Code", "MonthEnd", "Div_TTM", "DY_TTM"]].copy()
    dy2["MonthEnd"] = pd.to_datetime(dy2["MonthEnd"], errors="coerce").dt.normalize()
    df = df.merge(dy2, on=["Code", "MonthEnd"], how="left", validate="many_to_one")

    for c in ["MarketCap","BM_Ratio","ROE","INV_Growth","ret_m_fwd","MOM_12_1","DY_TTM"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    df = df[(df["MarketCap"].notna()) & (df["MarketCap"] > 0)].copy()
    df = df[df["ret_m_fwd"].notna()].copy()

    df["ret_w"] = winsorize(df["ret_m_fwd"], Q_RET)
    df["BM_w"]  = winsorize(df["BM_Ratio"], Q_BM)
    df["ROE_w"] = winsorize(df["ROE"], Q_ROE)
    df["INV_w"] = winsorize(df["INV_Growth"], Q_INV)
    df["MOM_w"] = winsorize(df["MOM_12_1"], Q_MOM)
    df["DY_w"]  = winsorize(df["DY_TTM"], Q_DY)

    cov = (df.groupby("MonthEnd")
             .agg(n=("Code","nunique"),
                  dy_notna=("DY_TTM", lambda s: float(s.notna().mean()*100)),
                  bm_notna=("BM_Ratio", lambda s: float(s.notna().mean()*100)))
             .reset_index())
    cov.to_csv(DIAG_DY_COV_BY_MONTH, index=False, encoding="utf-8-sig")

    return df


def compute_factors(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    months = sorted(df["MonthEnd"].dropna().unique())
    rows = []
    diag_n = []

    for me in months:
        m = df[df["MonthEnd"] == me].copy()
        n_total = int(m["Code"].nunique())
        if n_total < 30:
            continue

        MKT = value_weighted_return(m, "ret_w", "MarketCap")

        m["SZ"] = size_split_median(m["MarketCap"])
        m["VAL"]  = qcut_3way(m["BM_w"],  P30, P70, labels=("L","M","H"))
        m["PROF"] = qcut_3way(m["ROE_w"], P30, P70, labels=("W","M","R"))
        m["INV"]  = qcut_3way(m["INV_w"], P30, P70, labels=("C","M","A"))
        m["MOM"]  = qcut_3way(m["MOM_w"], P30, P70, labels=("L","M","W"))
        m["DY"]   = qcut_3way(m["DY_w"],  P30, P70, labels=("L","M","H"))

        def port_ret(sz, col, lab):
            x = m[(m["SZ"] == sz) & (m[col] == lab)]
            return value_weighted_return(x, "ret_w", "MarketCap")

        SH = port_ret("S","VAL","H"); SL = port_ret("S","VAL","L")
        BH = port_ret("B","VAL","H"); BL = port_ret("B","VAL","L")
        HML = np.nanmean([SH, BH]) - np.nanmean([SL, BL])

        SR = port_ret("S","PROF","R"); SW = port_ret("S","PROF","W")
        BR = port_ret("B","PROF","R"); BW = port_ret("B","PROF","W")
        RMW = np.nanmean([SR, BR]) - np.nanmean([SW, BW])

        SC = port_ret("S","INV","C"); SA = port_ret("S","INV","A")
        BC = port_ret("B","INV","C"); BA = port_ret("B","INV","A")
        CMA = np.nanmean([SC, BC]) - np.nanmean([SA, BA])

        SMv = port_ret("S","VAL","M"); BMv = port_ret("B","VAL","M")
        SMB_HML = np.nanmean([SH, SMv, SL]) - np.nanmean([BH, BMv, BL])
        SMp = port_ret("S","PROF","M"); BMp = port_ret("B","PROF","M")
        SMB_RMW = np.nanmean([SR, SMp, SW]) - np.nanmean([BR, BMp, BW])
        SMa = port_ret("S","INV","M"); BMa = port_ret("B","INV","M")
        SMB_CMA = np.nanmean([SC, SMa, SA]) - np.nanmean([BC, BMa, BA])
        SMB = np.nanmean([SMB_HML, SMB_RMW, SMB_CMA])

        SWin = port_ret("S","MOM","W"); SLos = port_ret("S","MOM","L")
        BWin = port_ret("B","MOM","W"); BLos = port_ret("B","MOM","L")
        WML = np.nanmean([SWin, BWin]) - np.nanmean([SLos, BLos])

        SHD = port_ret("S","DY","H"); SLD = port_ret("S","DY","L")
        BHD = port_ret("B","DY","H"); BLD = port_ret("B","DY","L")
        HDYMLDY = np.nanmean([SHD, BHD]) - np.nanmean([SLD, BLD])

        n_used_dy = int(m.dropna(subset=["DY_w","ret_w","MarketCap"])["Code"].nunique())
        confidence = float(min(1.0, n_used_dy / max(1, n_total)))

        # ===== PATCH START: abnormal比率判定（全銘柄でスケール） =====
        abnormal = (n_used_dy < ABNORMAL_MIN_RATIO * n_total) or (n_used_dy < MIN_STOCKS_PER_MONTH_FALLBACK)
        # ===== PATCH END =====

        rows.append({
            "MonthEnd": me,
            "MKT": MKT, "SMB": SMB, "HML": HML, "RMW": RMW, "CMA": CMA, "WML": WML, "HDYMLDY": HDYMLDY,
            "n_total": n_total,
            "n_used_dy": n_used_dy,
            "confidence": confidence,
            "abnormal": bool(abnormal),
        })

        diag_n.append({
            "MonthEnd": me,
            "n_total": n_total,
            "n_used_dy": n_used_dy,
            "confidence": confidence,
            "abnormal": bool(abnormal),
        })

    fac = pd.DataFrame(rows).sort_values("MonthEnd").reset_index(drop=True)
    diag_n_df = pd.DataFrame(diag_n).sort_values("MonthEnd").reset_index(drop=True)
    diag_n_df.to_csv(DIAG_FACTOR_NUSED, index=False, encoding="utf-8-sig")
    return fac, diag_n_df


def analyze_latest(fac: pd.DataFrame) -> Dict:
    if fac.empty:
        return {"error": "factor table is empty"}

    last_me = latest_evaluable_month(fac, col="MKT")
    cur = fac[fac["MonthEnd"] == last_me].iloc[0].to_dict()

    factor_cols = ["MKT","SMB","HML","RMW","CMA","WML","HDYMLDY"]
    conf = float(cur.get("confidence", 0.0))

    score = {c: abs(float(cur.get(c, np.nan))) * conf for c in factor_cols}
    dominant = max(score, key=lambda k: np.nan_to_num(score[k], nan=-1))

    fs = fac.sort_values("MonthEnd").reset_index(drop=True)
    last3 = fs.tail(3)
    last12 = fs.tail(12)

    trends3 = {c: trend_label(last3[c].to_numpy()) for c in factor_cols}
    trends12 = {c: trend_label(last12[c].to_numpy()) for c in factor_cols}

    mkt = cur.get("MKT", np.nan)
    regime = "RANGE"
    if np.isfinite(mkt):
        if mkt > 0.01:
            regime = "BULL"
        elif mkt < -0.01:
            regime = "BEAR"

    tilt = {}
    for c in factor_cols:
        v = float(np.nanmean(last3[c]))
        tilt[c] = float(np.sign(v)) if np.isfinite(v) else 0.0
    tilt_conf = {k: v * conf for k, v in tilt.items()}

    return {
        "latest_month": str(pd.Timestamp(last_me).date()),
        "dominant_factor": dominant,
        "dominant_scores": score,
        "regime": regime,
        "latest": cur,
        "trends_last3": trends3,
        "trends_last12": trends12,
        "tilt_conf_weighted": tilt_conf,
    }


def save_outputs(fac: pd.DataFrame, analysis: Dict):
    fac.to_parquet(OUT_PARQUET, index=False)

    latest = analysis.get("latest", {})
    summary = {
        "latest_month": analysis.get("latest_month", ""),
        "regime": analysis.get("regime", ""),
        "dominant_factor": analysis.get("dominant_factor", ""),
        "confidence": float(latest.get("confidence", np.nan)),
        "abnormal": bool(latest.get("abnormal", True)),
        "n_total": int(latest.get("n_total", 0) or 0),
        "n_used_dy": int(latest.get("n_used_dy", 0) or 0),
    }
    for c in ["MKT","SMB","HML","RMW","CMA","WML","HDYMLDY"]:
        summary[c] = float(latest.get(c, np.nan))
        summary[f"score_{c}"] = float(analysis.get("dominant_scores", {}).get(c, np.nan))
    pd.DataFrame([summary]).to_csv(OUT_SUMMARY, index=False, encoding="utf-8-sig")

    fac_sorted = fac.sort_values("MonthEnd").reset_index(drop=True)
    cols = ["MonthEnd","MKT","SMB","HML","RMW","CMA","WML","HDYMLDY","confidence","abnormal"]
    fac_sorted.tail(3)[cols].to_csv(OUT_TREND_LAST3, index=False, encoding="utf-8-sig")
    fac_sorted.tail(12)[cols].to_csv(OUT_TREND_LAST12, index=False, encoding="utf-8-sig")

    # ===== PATCH START: cache在庫診断（全銘柄化に備える） =====
    inv_rows = []
    for fp in sorted(CACHE_EVENTS_DIR.glob("*.parquet")):
        try:
            df = pd.read_parquet(fp)
            if df is None or len(df) == 0:
                inv_rows.append({"file": fp.name, "rows": 0, "min_date": "", "max_date": ""})
                continue
            d = pd.to_datetime(df["date"], errors="coerce")
            inv_rows.append({
                "file": fp.name,
                "rows": int(len(df)),
                "min_date": str(pd.Timestamp(d.min()).date()) if d.notna().any() else "",
                "max_date": str(pd.Timestamp(d.max()).date()) if d.notna().any() else "",
            })
        except Exception:
            inv_rows.append({"file": fp.name, "rows": -1, "min_date": "", "max_date": ""})

    pd.DataFrame(inv_rows).to_csv(DIAG_CACHE_INVENTORY, index=False, encoding="utf-8-sig")
    # ===== PATCH END =====


def main():
    with open(DIAG_LOG, "w", encoding="utf-8") as f:
        f.write("")

    log_line("="*110)
    log_line("FF5 + MOM(12-1) + DY_TTM + yfinance cache (incremental append) [A2 topN->all]")
    log_line("="*110)
    log_line(f"PRICE_PATH: {PRICE_PATH}")
    log_line(f"SNAP_PATH : {SNAP_PATH}")
    log_line(f"UNIVERSE_MODE: {UNIVERSE_MODE} (TOP_N={TOP_N})")
    log_line(f"CACHE_DIR : {CACHE_EVENTS_DIR}")
    log_line(f"SLEEP_SEC : {SLEEP_SEC}")

    uni = pick_universe()

    div_events_df, fetch_results_df, cache_update_df = build_div_events_with_incremental_cache(uni)
    ok_any = int(fetch_results_df["ok"].sum()) if len(fetch_results_df) else 0
    ok_nonempty = int((fetch_results_df["ok"] & (fetch_results_df["div_rows"] > 0)).sum()) if len(fetch_results_df) else 0
    log_line(f"[YF] ok(symbol found)= {ok_any}/{len(fetch_results_df)} | nonempty(div_rows>0)= {ok_nonempty}/{len(fetch_results_df)}")

    if len(cache_update_df):
        added_total = int(pd.to_numeric(cache_update_df["added_rows"], errors="coerce").fillna(0).sum())
        fetched_fail = int((cache_update_df.get("fetched", True) == False).sum()) if "fetched" in cache_update_df.columns else 0
        log_line(f"[CACHE] symbols tried={len(cache_update_df)} total_added_rows={added_total} fetched_fail={fetched_fail}")

    dy_df = compute_dy_ttm_for_monthends(uni, div_events_df)
    cov_latest = (dy_df[dy_df["MonthEnd"] == dy_df["MonthEnd"].max()]
                  .assign(dy_notna=lambda x: x["DY_TTM"].notna())
                  ["dy_notna"].mean() * 100.0)
    log_line(f"[DY] latest MonthEnd={dy_df['MonthEnd'].max().date()} DY_TTM notna%={cov_latest:.2f}")

    feat = load_features(uni, dy_df)
    log_line(f"[FEATURE] rows={len(feat):,} codes={feat['Code'].nunique():,} months={feat['MonthEnd'].nunique():,}")

    fac, _ = compute_factors(feat)
    log_line(f"[FACTORS] months computed={len(fac)} min={fac['MonthEnd'].min()} max={fac['MonthEnd'].max()}")

    if fac.empty:
        raise RuntimeError("Factor result is empty. Check DY/BM coverage and thresholds.")

    analysis = analyze_latest(fac)
    last = analysis.get("latest", {})
    log_line("-"*110)
    log_line(f"[LATEST EVALUABLE MONTH] {analysis.get('latest_month')}")
    log_line(f"[DOMINANT FACTOR (abs*confidence)] {analysis.get('dominant_factor')}")
    log_line(f"[REGIME] {analysis.get('regime')}")
    log_line(f"[CONFIDENCE] {last.get('confidence'):.3f} | abnormal={last.get('abnormal')} | "
             f"n_total={last.get('n_total')} n_used_dy={last.get('n_used_dy')}")

    log_line("Factor returns:")
    for c in ["MKT","SMB","HML","RMW","CMA","WML","HDYMLDY"]:
        log_line(f"  {c}: {last.get(c)}")
    log_line("Dominant scores (abs*confidence):")
    for c, sc in sorted(analysis.get("dominant_scores", {}).items(), key=lambda kv: -np.nan_to_num(kv[1], nan=-1)):
        log_line(f"  score_{c}: {sc}")

    save_outputs(fac, analysis)
    log_line("-"*110)
    log_line("✅ saved:")
    log_line(f"  - {OUT_PARQUET}")
    log_line(f"  - {OUT_SUMMARY}")
    log_line(f"  - {OUT_TREND_LAST3}")
    log_line(f"  - {OUT_TREND_LAST12}")
    log_line(f"  - {DIAG_UNIVERSE}")
    log_line(f"  - {DIAG_ATTEMPTS}")
    log_line(f"  - {DIAG_FETCH_RESULTS}")
    log_line(f"  - {DIAG_CACHE_UPDATE}")
    log_line(f"  - {DIAG_YF_FAIL_CODES}")
    log_line(f"  - {DIAG_YF_RULE_STATS}")
    log_line(f"  - {DIAG_CACHE_INVENTORY}")
    log_line(f"  - {DIAG_DY_COV_BY_MONTH}")
    log_line(f"  - {DIAG_FACTOR_NUSED}")
    log_line(f"  - {DIAG_LOG}")


if __name__ == "__main__":
    main()


FF5 + MOM(12-1) + DY_TTM + yfinance cache (incremental append) [A2 topN->all]
PRICE_PATH: C:\Users\yongr\Project\merged_data_all_stocks\factors\price_month_end.parquet
SNAP_PATH : C:\Users\yongr\Project\merged_data_all_stocks\factors\month_end_snapshot.parquet
UNIVERSE_MODE: topN (TOP_N=1000)
CACHE_DIR : C:\Users\yongr\Project\merged_data_all_stocks\factors\yf_dividend_cache\events_by_symbol
SLEEP_SEC : 0.1
[UNIVERSE] mode=topN latest MonthEnd=2025-09-30 n=1,000


$167A0.T: possibly delisted; no timezone found
$8279.T: possibly delisted; no timezone found
$9066.T: possibly delisted; no timezone found
Exception ignored from cffi callback <function buffer_callback at 0x000001DC7F9FFF60>:
Traceback (most recent call last):
  File "C:\Users\yongr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\curl_cffi\curl.py", line 100, in buffer_callback
    @ffi.def_extern()
KeyboardInterrupt: 
Failed to get ticker '5541.T' reason: Failed to perform, curl: (23) Failure writing output to destination, passed 13 returned 0. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.
$5541.T: possibly delisted; no timezone found


In [2]:
# -*- coding: utf-8 -*-
"""
FF5+MOM だけで
「各月末に、直近 LOOKBACK_M ヶ月の銘柄別FF5+MOM回帰alpha（rolling）」上位20銘柄を等金額で買い、
翌月リターンで評価する月次バックテスト（2016-10〜2025-09）

Lookback:
- LOOKBACK_M = 24 または 36 に切替可能（今回の要求）

Outputs（前回と同等）:
- monthly returns CSV
- cumulative curve CSV
- performance summary CSV (CAGR, vol, sharpe(0), maxDD etc.)
- portfolio regression vs FF5+MOM CSV
- diag_selected_top20_by_month.csv
- diag_coverage_by_month.csv

依存:
  pip install pandas numpy pyarrow
（回帰は numpy でOLS実装 / statsmodels不要）
"""

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
from typing import Tuple, Dict, Optional, List

import numpy as np
import pandas as pd


# ============================================================
# Paths (あなたの環境)
# ============================================================
FACTORS_DIR = Path(r"C:\Users\yongr\Project\merged_data_all_stocks\factors")
PRICE_PATH = FACTORS_DIR / "price_month_end.parquet"
SNAP_PATH  = FACTORS_DIR / "month_end_snapshot.parquet"
FACTOR_PATH = FACTORS_DIR / "ff5_mom_factors_monthly.parquet"  # 既存のFF5+MOM(月次因子)

# ============================================================
# Backtest params
# ============================================================
START_MONTHEND = pd.Timestamp("2016-10-31")
END_MONTHEND   = pd.Timestamp("2025-09-30")

# ===== PATCH START: lookbackを 24/36 に切替可能 =====
LOOKBACK_M = 24     # 24 or 36 に変更して再実行
# ===== PATCH END =====

TOP_K = 20
MIN_OBS = 18 if LOOKBACK_M >= 24 else 10   # 欠損対策：24なら18、36なら24でも良い（必要なら調整）
EQUAL_WEIGHT = True

# 速度/負荷対策（全銘柄が重い場合ここを 1000 などに）
MAX_CODES: Optional[int] = None

# 回帰で使う因子列（MOMはWMLとして扱う）
FACTOR_COLS = ["MKT", "SMB", "HML", "RMW", "CMA", "WML"]

# ============================================================
# Outputs (lookback別に保存して混ざらないようにする)
# ============================================================
OUT_DIR = FACTORS_DIR / f"bt_top20_rolling_alpha_ff5mom_lb{LOOKBACK_M}_201610_202509"
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_MONTHLY_RET = OUT_DIR / "bt_top20_monthly_returns.csv"
OUT_CUMCURVE    = OUT_DIR / "bt_top20_cumulative_curve.csv"
OUT_PERF        = OUT_DIR / "bt_top20_performance_summary.csv"
OUT_REG         = OUT_DIR / "bt_top20_portfolio_regression_ff5mom.csv"
OUT_DIAG_SEL    = OUT_DIR / "diag_selected_top20_by_month.csv"
OUT_DIAG_COV    = OUT_DIR / "diag_coverage_by_month.csv"


# ============================================================
# Utils
# ============================================================
BAD_CODE_STRINGS = {"None", "nan", "", "NaN", "NULL", "null"}

def normalize_code(code) -> str:
    if code is None:
        return ""
    s = str(code).strip()
    if s in BAD_CODE_STRINGS:
        return ""
    return s

def log(s: str):
    print(s)

def safe_to_datetime(s):
    return pd.to_datetime(s, errors="coerce")

def ols_alpha_beta(y: np.ndarray, X: np.ndarray) -> Tuple[float, np.ndarray, float, float]:
    """
    OLS with intercept.
    Returns alpha, betas, R2, stderr_alpha (簡易; homoskedastic)
    - y: (n,)
    - X: (n, k)  (factors)
    """
    n = y.shape[0]
    if n < 3:
        return np.nan, np.full(X.shape[1], np.nan), np.nan, np.nan

    X1 = np.column_stack([np.ones(n), X])  # add intercept
    XtX = X1.T @ X1
    try:
        inv = np.linalg.inv(XtX)
    except np.linalg.LinAlgError:
        inv = np.linalg.pinv(XtX)
    b = inv @ (X1.T @ y)
    alpha = float(b[0])
    betas = b[1:].astype(float)

    yhat = X1 @ b
    resid = y - yhat
    sse = float(np.sum(resid**2))
    sst = float(np.sum((y - y.mean())**2))
    r2 = np.nan if sst <= 0 else (1.0 - sse / sst)

    k = X1.shape[1]
    dof = max(1, n - k)
    sigma2 = sse / dof
    var_b = sigma2 * inv
    se_alpha = float(np.sqrt(max(0.0, var_b[0, 0])))

    return alpha, betas, r2, se_alpha

def compute_drawdown(cum: pd.Series) -> pd.Series:
    peak = cum.cummax()
    dd = cum / peak - 1.0
    return dd

def perf_stats(monthly_ret: pd.Series) -> Dict:
    r = monthly_ret.dropna().astype(float)
    if r.empty:
        return {
            "n_months": 0, "CAGR": np.nan, "ann_vol": np.nan, "ann_mean": np.nan,
            "sharpe0": np.nan, "maxDD": np.nan, "cum_end": np.nan
        }
    n = len(r)
    cum = (1.0 + r).cumprod()
    years = n / 12.0
    cagr = float(cum.iloc[-1] ** (1/years) - 1.0) if years > 0 else np.nan
    ann_mean = float(r.mean() * 12.0)
    ann_vol = float(r.std(ddof=1) * np.sqrt(12.0)) if n >= 2 else np.nan
    sharpe0 = float(ann_mean / ann_vol) if ann_vol and ann_vol > 0 else np.nan
    dd = compute_drawdown(cum)
    maxdd = float(dd.min())
    return {
        "n_months": int(n),
        "CAGR": cagr,
        "ann_mean": ann_mean,
        "ann_vol": ann_vol,
        "sharpe0": sharpe0,
        "maxDD": maxdd,
        "cum_end": float(cum.iloc[-1]),
    }


# ============================================================
# Load data
# ============================================================
def load_factors() -> pd.DataFrame:
    fac = pd.read_parquet(FACTOR_PATH).copy()
    fac["MonthEnd"] = safe_to_datetime(fac["MonthEnd"]).dt.normalize()
    fac = fac.sort_values("MonthEnd").reset_index(drop=True)
    need = ["MonthEnd"] + FACTOR_COLS
    missing = [c for c in need if c not in fac.columns]
    if missing:
        raise KeyError(f"Factor file missing columns: {missing} in {FACTOR_PATH}")
    return fac[need].copy()

def load_prices() -> pd.DataFrame:
    pr = pd.read_parquet(PRICE_PATH)[["Code", "MonthEnd", "AdjustedClose"]].copy()
    pr["Code"] = pr["Code"].astype(str).map(normalize_code)
    pr = pr[pr["Code"] != ""].copy()
    pr["MonthEnd"] = safe_to_datetime(pr["MonthEnd"]).dt.normalize()
    pr["AdjustedClose"] = pd.to_numeric(pr["AdjustedClose"], errors="coerce")
    pr = pr.dropna(subset=["MonthEnd", "AdjustedClose"]).copy()
    pr = pr.sort_values(["Code", "MonthEnd"], kind="mergesort").reset_index(drop=True)

    pr["Adj_next"] = pr.groupby("Code")["AdjustedClose"].shift(-1)
    pr["ret_fwd_1m"] = pr["Adj_next"] / pr["AdjustedClose"] - 1.0
    return pr[["Code", "MonthEnd", "ret_fwd_1m", "AdjustedClose"]].copy()

def pick_universe_codes() -> List[str]:
    """
    Universe: 最新MonthEndのsnapshotにいる銘柄（MarketCapで並べ、MAX_CODESで切る）
    """
    snap = pd.read_parquet(SNAP_PATH)[["Code", "MonthEnd", "MarketCap"]].copy()
    snap["Code"] = snap["Code"].astype(str).map(normalize_code)
    snap = snap[snap["Code"] != ""].copy()
    snap["MonthEnd"] = safe_to_datetime(snap["MonthEnd"]).dt.normalize()
    snap["MarketCap"] = pd.to_numeric(snap["MarketCap"], errors="coerce")
    latest = snap["MonthEnd"].max()
    s = snap[snap["MonthEnd"] == latest].dropna(subset=["MarketCap"]).copy()
    s = s.sort_values("MarketCap", ascending=False).drop_duplicates(["Code"], keep="first")
    codes = s["Code"].tolist()
    if MAX_CODES is not None:
        codes = codes[:MAX_CODES]
    log(f"[UNIVERSE] latest MonthEnd={latest.date()} codes={len(codes):,} (MAX_CODES={MAX_CODES})")
    return codes


# ============================================================
# Rolling alpha ranking + backtest
# ============================================================
def build_panel(pr: pd.DataFrame, fac: pd.DataFrame, codes: List[str]) -> pd.DataFrame:
    """
    Panel with [Code, MonthEnd, ret_fwd_1m] + factor columns aligned on MonthEnd
    """
    p = pr[pr["Code"].isin(codes)].copy()
    p = p.merge(fac, on="MonthEnd", how="left", validate="many_to_one")
    p = p.sort_values(["Code", "MonthEnd"]).reset_index(drop=True)
    return p

def rolling_alpha_by_month(panel: pd.DataFrame,
                           start_me: pd.Timestamp,
                           end_me: pd.Timestamp,
                           lookback_m: int = 24,
                           min_obs: int = 18) -> pd.DataFrame:
    """
    Selection at MonthEnd t uses realized returns in months (t-lookback_m .. t-1).
    realized month return at MonthEnd m is ret_fwd_1m at MonthEnd m (m->m+1).
    """
    months_all = sorted(panel["MonthEnd"].dropna().unique())
    months_all = [pd.Timestamp(x) for x in months_all]

    # selection months restricted
    sel_months = [m for m in months_all if (m >= start_me) and (m <= end_me)]

    out_rows = []
    diag_cov = []

    by_code = {c: g.reset_index(drop=True) for c, g in panel.groupby("Code", sort=False)}

    # factors by month cache
    fac_month = (panel[["MonthEnd"] + FACTOR_COLS]
                 .drop_duplicates("MonthEnd")
                 .set_index("MonthEnd")
                 .sort_index())

    for t in sel_months:
        # window ends at t-1
        prior = [m for m in months_all if m < t]
        if len(prior) < lookback_m:
            continue
        win_months = prior[-lookback_m:]

        fac_win = fac_month.reindex(win_months)
        # drop months where any factor missing
        fac_win = fac_win.dropna()
        if len(fac_win) < min_obs:
            continue

        months_aligned = fac_win.index.to_list()
        X_by_month = fac_win[FACTOR_COLS].to_numpy(dtype=float)

        n_alpha = 0

        for code, g in by_code.items():
            gg = g[g["MonthEnd"].isin(months_aligned)][["MonthEnd", "ret_fwd_1m"]].set_index("MonthEnd").sort_index()
            if gg.empty:
                continue
            y = gg.reindex(months_aligned)["ret_fwd_1m"].to_numpy(dtype=float)

            mask = np.isfinite(y)
            if mask.sum() < min_obs:
                continue

            y2 = y[mask]
            X2 = X_by_month[mask, :]

            alpha, betas, r2, se_a = ols_alpha_beta(y2, X2)
            if np.isfinite(alpha):
                out_rows.append({
                    "MonthEnd": t,
                    "Code": code,
                    "alpha": alpha,
                    "n_obs": int(mask.sum()),
                    "r2": r2,
                    "se_alpha": se_a,
                })
                n_alpha += 1

        diag_cov.append({
            "MonthEnd": t,
            "n_codes_in_universe": int(len(by_code)),
            "n_codes_with_alpha": int(n_alpha),
            "lookback_months": int(lookback_m),
            "min_obs": int(min_obs),
        })

    alpha_df = pd.DataFrame(out_rows)
    cov_df = pd.DataFrame(diag_cov).sort_values("MonthEnd")
    cov_df.to_csv(OUT_DIAG_COV, index=False, encoding="utf-8-sig")
    return alpha_df

def backtest_topk_equal_weight(panel: pd.DataFrame,
                               alpha_df: pd.DataFrame,
                               top_k: int = 20) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    For each MonthEnd t:
      - select top_k codes by alpha at t
      - realize next-month return = average of ret_fwd_1m at t for selected codes
    """
    if alpha_df.empty:
        raise RuntimeError("alpha_df is empty; check coverage/period/inputs")

    ret_map = panel.set_index(["MonthEnd", "Code"])["ret_fwd_1m"]

    months = sorted(alpha_df["MonthEnd"].unique())
    bt_rows = []
    sel_rows = []

    for t in months:
        a = alpha_df[alpha_df["MonthEnd"] == t].copy()
        a = a.dropna(subset=["alpha"]).sort_values("alpha", ascending=False).head(top_k)
        if a.empty:
            continue

        codes = a["Code"].tolist()

        rets = []
        for c in codes:
            try:
                r = float(ret_map.loc[(t, c)])
            except KeyError:
                r = np.nan
            rets.append(r)

        rets = np.array(rets, dtype=float)
        used = np.isfinite(rets)

        port_ret = float(np.nanmean(rets)) if used.sum() > 0 else np.nan

        bt_rows.append({
            "MonthEnd": t,
            "portfolio_ret": port_ret,
            "n_selected": int(len(codes)),
            "n_used_ret": int(used.sum()),
            "avg_alpha_selected": float(a["alpha"].mean()),
        })

        for _, r in a.iterrows():
            sel_rows.append({
                "MonthEnd": t,
                "Code": r["Code"],
                "alpha": float(r["alpha"]),
                "n_obs": int(r["n_obs"]),
                "r2": float(r["r2"]) if pd.notna(r["r2"]) else np.nan,
            })

    bt = pd.DataFrame(bt_rows).sort_values("MonthEnd").reset_index(drop=True)
    sel = pd.DataFrame(sel_rows).sort_values(["MonthEnd", "alpha"], ascending=[True, False]).reset_index(drop=True)
    sel.to_csv(OUT_DIAG_SEL, index=False, encoding="utf-8-sig")
    return bt, sel


# ============================================================
# Portfolio regression vs FF5+MOM
# ============================================================
def regress_portfolio_vs_factors(bt: pd.DataFrame, fac: pd.DataFrame) -> pd.DataFrame:
    df = bt.merge(fac, on="MonthEnd", how="left", validate="many_to_one").copy()
    df = df.dropna(subset=["portfolio_ret"] + FACTOR_COLS).copy()

    y = df["portfolio_ret"].to_numpy(dtype=float)
    X = df[FACTOR_COLS].to_numpy(dtype=float)

    alpha, betas, r2, se_a = ols_alpha_beta(y, X)

    out = {
        "lookback_months_for_selection": int(LOOKBACK_M),
        "n_months": int(len(df)),
        "R2": float(r2),
        "alpha_monthly": float(alpha),
        "alpha_annualized_approx": float(alpha * 12.0),
        "se_alpha": float(se_a),
    }
    for c, b in zip(FACTOR_COLS, betas):
        out[f"beta_{c}"] = float(b)

    return pd.DataFrame([out])


# ============================================================
# Main
# ============================================================
def main():
    log("=" * 110)
    log("Backtest: Top20 by rolling alpha (FF5+MOM) -> next-month equal-weight returns")
    log("=" * 110)
    log(f"PRICE_PATH : {PRICE_PATH}")
    log(f"SNAP_PATH  : {SNAP_PATH}")
    log(f"FACTOR_PATH: {FACTOR_PATH}")
    log(f"PERIOD     : {START_MONTHEND.date()} .. {END_MONTHEND.date()}")
    log(f"LOOKBACK   : {LOOKBACK_M} months | TOP_K={TOP_K} | MIN_OBS={MIN_OBS}")
    log(f"OUT_DIR    : {OUT_DIR}")

    fac = load_factors()
    pr = load_prices()
    codes = pick_universe_codes()

    panel = build_panel(pr, fac, codes)

    # Keep earlier months for lookback
    min_needed = START_MONTHEND - pd.offsets.MonthEnd(LOOKBACK_M + 2)
    panel = panel[(panel["MonthEnd"] >= min_needed) & (panel["MonthEnd"] <= END_MONTHEND)].copy()

    log(f"[PANEL] rows={len(panel):,} codes={panel['Code'].nunique():,} months={panel['MonthEnd'].nunique():,}")

    alpha_df = rolling_alpha_by_month(panel, START_MONTHEND, END_MONTHEND, LOOKBACK_M, MIN_OBS)
    log(f"[ALPHA] rows={len(alpha_df):,} months={alpha_df['MonthEnd'].nunique() if len(alpha_df) else 0:,}")

    bt, sel = backtest_topk_equal_weight(panel, alpha_df, TOP_K)
    log(f"[BT] months={len(bt):,} start={bt['MonthEnd'].min()} end={bt['MonthEnd'].max()}")

    # Save monthly returns
    bt_out = bt.copy()
    bt_out["MonthEnd"] = pd.to_datetime(bt_out["MonthEnd"]).dt.date.astype(str)
    bt_out.to_csv(OUT_MONTHLY_RET, index=False, encoding="utf-8-sig")

    # Cumulative curve + drawdown
    bt2 = bt.copy()
    bt2 = bt2.dropna(subset=["portfolio_ret"]).copy()
    bt2["cum"] = (1.0 + bt2["portfolio_ret"]).cumprod()
    bt2["dd"] = compute_drawdown(bt2["cum"])

    cum_out = bt2[["MonthEnd", "portfolio_ret", "cum", "dd", "n_selected", "n_used_ret", "avg_alpha_selected"]].copy()
    cum_out["MonthEnd"] = pd.to_datetime(cum_out["MonthEnd"]).dt.date.astype(str)
    cum_out.to_csv(OUT_CUMCURVE, index=False, encoding="utf-8-sig")

    # Performance summary
    stats = perf_stats(bt2["portfolio_ret"])
    perf_df = pd.DataFrame([stats])
    perf_df.insert(0, "lookback_months_for_selection", LOOKBACK_M)
    perf_df.to_csv(OUT_PERF, index=False, encoding="utf-8-sig")

    # Portfolio regression vs factors
    reg_df = regress_portfolio_vs_factors(bt, fac)
    reg_df.to_csv(OUT_REG, index=False, encoding="utf-8-sig")

    log("-" * 110)
    log("✅ SAVED")
    log(f"  - monthly returns : {OUT_MONTHLY_RET}")
    log(f"  - cumulative curve: {OUT_CUMCURVE}")
    log(f"  - perf summary    : {OUT_PERF}")
    log(f"  - regression      : {OUT_REG}")
    log(f"  - diag selected   : {OUT_DIAG_SEL}")
    log(f"  - diag coverage   : {OUT_DIAG_COV}")
    log("-" * 110)

    log("[PERF SUMMARY]")
    for k, v in stats.items():
        log(f"  {k}: {v}")

    log("[REGRESSION SUMMARY]")
    if len(reg_df):
        row = reg_df.iloc[0].to_dict()
        keys = ["lookback_months_for_selection", "n_months", "R2", "alpha_monthly", "alpha_annualized_approx", "se_alpha"] \
               + [f"beta_{c}" for c in FACTOR_COLS]
        for k in keys:
            if k in row:
                log(f"  {k}: {row[k]}")

if __name__ == "__main__":
    main()


Backtest: Top20 by rolling alpha (FF5+MOM) -> next-month equal-weight returns
PRICE_PATH : C:\Users\yongr\Project\merged_data_all_stocks\factors\price_month_end.parquet
SNAP_PATH  : C:\Users\yongr\Project\merged_data_all_stocks\factors\month_end_snapshot.parquet
FACTOR_PATH: C:\Users\yongr\Project\merged_data_all_stocks\factors\ff5_mom_factors_monthly.parquet
PERIOD     : 2016-10-31 .. 2025-09-30
LOOKBACK   : 24 months | TOP_K=20 | MIN_OBS=18
OUT_DIR    : C:\Users\yongr\Project\merged_data_all_stocks\factors\bt_top20_rolling_alpha_ff5mom_lb24_201610_202509
[UNIVERSE] latest MonthEnd=2025-09-30 codes=3,428 (MAX_CODES=None)
[PANEL] rows=363,954 codes=3,428 months=117
[ALPHA] rows=269,234 months=87
[BT] months=87 start=2018-07-31 00:00:00 end=2025-09-30 00:00:00
--------------------------------------------------------------------------------------------------------------
✅ SAVED
  - monthly returns : C:\Users\yongr\Project\merged_data_all_stocks\factors\bt_top20_rolling_alpha_ff5mom_lb24_

In [3]:
# -*- coding: utf-8 -*-
"""
FF5+MOM だけで
「各月末に、直近 LOOKBACK_M ヶ月の銘柄別FF5+MOM回帰の t統計（alpha/se_alpha）」上位20銘柄を等金額で買い、
翌月リターンで評価する月次バックテスト（2016-10〜2025-09）

前回との差分（重要）:
- 銘柄スコアを alpha ではなく tstat = alpha / se_alpha でランキング
  -> ノイズで跳ねたalphaを抑制し、安定したalphaを優先

Outputs（前回と同等）:
- monthly returns CSV
- cumulative curve CSV
- performance summary CSV (CAGR, vol, sharpe(0), maxDD etc.)
- portfolio regression vs FF5+MOM CSV
- diag_selected_top20_by_month.csv（tstatも出す）
- diag_coverage_by_month.csv

依存:
  pip install pandas numpy pyarrow
（回帰は numpy でOLS実装 / statsmodels不要）
"""

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
from typing import Tuple, Dict, Optional, List

import numpy as np
import pandas as pd


# ============================================================
# Paths (あなたの環境)
# ============================================================
FACTORS_DIR = Path(r"C:\Users\yongr\Project\merged_data_all_stocks\factors")
PRICE_PATH = FACTORS_DIR / "price_month_end.parquet"
SNAP_PATH  = FACTORS_DIR / "month_end_snapshot.parquet"
FACTOR_PATH = FACTORS_DIR / "ff5_mom_factors_monthly.parquet"  # 既存FF5+MOM(月次因子)

# ============================================================
# Backtest params
# ============================================================
START_MONTHEND = pd.Timestamp("2016-10-31")
END_MONTHEND   = pd.Timestamp("2025-09-30")

# 24/36にしたい場合はここを変える（前回同様）
LOOKBACK_M = 24   # 12 / 24 / 36 など

TOP_K = 20

# 欠損対策（推奨値）
if LOOKBACK_M >= 36:
    MIN_OBS = 24
elif LOOKBACK_M >= 24:
    MIN_OBS = 18
else:
    MIN_OBS = 10

# 速度/負荷対策（全銘柄が重い場合ここを 1000 などに）
MAX_CODES: Optional[int] = None

# 回帰で使う因子列（MOMはWMLとして扱う）
FACTOR_COLS = ["MKT", "SMB", "HML", "RMW", "CMA", "WML"]

# ===== スコア設定（今回の変更点）=====
SCORE_MODE = "tstat"  # "tstat"  or  "alpha"


# ============================================================
# Outputs（スコアとlookbackが混ざらないように分ける）
# ============================================================
OUT_DIR = FACTORS_DIR / f"bt_top20_rolling_{SCORE_MODE}_ff5mom_lb{LOOKBACK_M}_201610_202509"
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_MONTHLY_RET = OUT_DIR / "bt_top20_monthly_returns.csv"
OUT_CUMCURVE    = OUT_DIR / "bt_top20_cumulative_curve.csv"
OUT_PERF        = OUT_DIR / "bt_top20_performance_summary.csv"
OUT_REG         = OUT_DIR / "bt_top20_portfolio_regression_ff5mom.csv"
OUT_DIAG_SEL    = OUT_DIR / "diag_selected_top20_by_month.csv"
OUT_DIAG_COV    = OUT_DIR / "diag_coverage_by_month.csv"


# ============================================================
# Utils
# ============================================================
BAD_CODE_STRINGS = {"None", "nan", "", "NaN", "NULL", "null"}

def normalize_code(code) -> str:
    if code is None:
        return ""
    s = str(code).strip()
    if s in BAD_CODE_STRINGS:
        return ""
    return s

def log(s: str):
    print(s)

def safe_to_datetime(s):
    return pd.to_datetime(s, errors="coerce")

def ols_alpha_beta(y: np.ndarray, X: np.ndarray) -> Tuple[float, np.ndarray, float, float]:
    """
    OLS with intercept.
    Returns alpha, betas, R2, stderr_alpha (簡易; homoskedastic)
    """
    n = y.shape[0]
    if n < 3:
        return np.nan, np.full(X.shape[1], np.nan), np.nan, np.nan

    X1 = np.column_stack([np.ones(n), X])  # intercept
    XtX = X1.T @ X1
    try:
        inv = np.linalg.inv(XtX)
    except np.linalg.LinAlgError:
        inv = np.linalg.pinv(XtX)

    b = inv @ (X1.T @ y)
    alpha = float(b[0])
    betas = b[1:].astype(float)

    yhat = X1 @ b
    resid = y - yhat
    sse = float(np.sum(resid**2))
    sst = float(np.sum((y - y.mean())**2))
    r2 = np.nan if sst <= 0 else (1.0 - sse / sst)

    k = X1.shape[1]
    dof = max(1, n - k)
    sigma2 = sse / dof
    var_b = sigma2 * inv
    se_alpha = float(np.sqrt(max(0.0, var_b[0, 0])))

    return alpha, betas, r2, se_alpha

def compute_drawdown(cum: pd.Series) -> pd.Series:
    peak = cum.cummax()
    return cum / peak - 1.0

def perf_stats(monthly_ret: pd.Series) -> Dict:
    r = monthly_ret.dropna().astype(float)
    if r.empty:
        return {
            "n_months": 0, "CAGR": np.nan, "ann_vol": np.nan, "ann_mean": np.nan,
            "sharpe0": np.nan, "maxDD": np.nan, "cum_end": np.nan
        }
    n = len(r)
    cum = (1.0 + r).cumprod()
    years = n / 12.0
    cagr = float(cum.iloc[-1] ** (1/years) - 1.0) if years > 0 else np.nan
    ann_mean = float(r.mean() * 12.0)
    ann_vol = float(r.std(ddof=1) * np.sqrt(12.0)) if n >= 2 else np.nan
    sharpe0 = float(ann_mean / ann_vol) if ann_vol and ann_vol > 0 else np.nan
    dd = compute_drawdown(cum)
    maxdd = float(dd.min())
    return {
        "n_months": int(n),
        "CAGR": cagr,
        "ann_mean": ann_mean,
        "ann_vol": ann_vol,
        "sharpe0": sharpe0,
        "maxDD": maxdd,
        "cum_end": float(cum.iloc[-1]),
    }


# ============================================================
# Load data
# ============================================================
def load_factors() -> pd.DataFrame:
    fac = pd.read_parquet(FACTOR_PATH).copy()
    fac["MonthEnd"] = safe_to_datetime(fac["MonthEnd"]).dt.normalize()
    fac = fac.sort_values("MonthEnd").reset_index(drop=True)

    need = ["MonthEnd"] + FACTOR_COLS
    missing = [c for c in need if c not in fac.columns]
    if missing:
        raise KeyError(f"Factor file missing columns: {missing} in {FACTOR_PATH}")
    return fac[need].copy()

def load_prices() -> pd.DataFrame:
    pr = pd.read_parquet(PRICE_PATH)[["Code", "MonthEnd", "AdjustedClose"]].copy()
    pr["Code"] = pr["Code"].astype(str).map(normalize_code)
    pr = pr[pr["Code"] != ""].copy()
    pr["MonthEnd"] = safe_to_datetime(pr["MonthEnd"]).dt.normalize()
    pr["AdjustedClose"] = pd.to_numeric(pr["AdjustedClose"], errors="coerce")
    pr = pr.dropna(subset=["MonthEnd", "AdjustedClose"]).copy()
    pr = pr.sort_values(["Code", "MonthEnd"], kind="mergesort").reset_index(drop=True)

    pr["Adj_next"] = pr.groupby("Code")["AdjustedClose"].shift(-1)
    pr["ret_fwd_1m"] = pr["Adj_next"] / pr["AdjustedClose"] - 1.0

    return pr[["Code", "MonthEnd", "ret_fwd_1m"]].copy()

def pick_universe_codes() -> List[str]:
    snap = pd.read_parquet(SNAP_PATH)[["Code", "MonthEnd", "MarketCap"]].copy()
    snap["Code"] = snap["Code"].astype(str).map(normalize_code)
    snap = snap[snap["Code"] != ""].copy()
    snap["MonthEnd"] = safe_to_datetime(snap["MonthEnd"]).dt.normalize()
    snap["MarketCap"] = pd.to_numeric(snap["MarketCap"], errors="coerce")
    latest = snap["MonthEnd"].max()

    s = snap[snap["MonthEnd"] == latest].dropna(subset=["MarketCap"]).copy()
    s = s.sort_values("MarketCap", ascending=False).drop_duplicates(["Code"], keep="first")

    codes = s["Code"].tolist()
    if MAX_CODES is not None:
        codes = codes[:MAX_CODES]
    log(f"[UNIVERSE] latest MonthEnd={latest.date()} codes={len(codes):,} (MAX_CODES={MAX_CODES})")
    return codes


# ============================================================
# Panel + rolling per-stock regression stats
# ============================================================
def build_panel(pr: pd.DataFrame, fac: pd.DataFrame, codes: List[str]) -> pd.DataFrame:
    p = pr[pr["Code"].isin(codes)].copy()
    p = p.merge(fac, on="MonthEnd", how="left", validate="many_to_one")
    p = p.sort_values(["Code", "MonthEnd"]).reset_index(drop=True)
    return p

def rolling_stats_by_month(panel: pd.DataFrame,
                           start_me: pd.Timestamp,
                           end_me: pd.Timestamp,
                           lookback_m: int,
                           min_obs: int) -> pd.DataFrame:
    """
    Selection at MonthEnd t uses realized returns in months (t-lookback_m .. t-1).
    realized month return at MonthEnd m is ret_fwd_1m at MonthEnd m (m->m+1).
    """
    months_all = sorted(panel["MonthEnd"].dropna().unique())
    months_all = [pd.Timestamp(x) for x in months_all]

    sel_months = [m for m in months_all if (m >= start_me) and (m <= end_me)]

    by_code = {c: g.reset_index(drop=True) for c, g in panel.groupby("Code", sort=False)}

    fac_month = (panel[["MonthEnd"] + FACTOR_COLS]
                 .drop_duplicates("MonthEnd")
                 .set_index("MonthEnd")
                 .sort_index())

    out_rows = []
    cov_rows = []

    for t in sel_months:
        prior = [m for m in months_all if m < t]
        if len(prior) < lookback_m:
            continue
        win_months = prior[-lookback_m:]

        fac_win = fac_month.reindex(win_months).dropna()
        if len(fac_win) < min_obs:
            continue

        months_aligned = fac_win.index.to_list()
        X_by_month = fac_win[FACTOR_COLS].to_numpy(dtype=float)

        n_with_stats = 0

        for code, g in by_code.items():
            gg = g[g["MonthEnd"].isin(months_aligned)][["MonthEnd", "ret_fwd_1m"]]\
                    .set_index("MonthEnd").sort_index()
            if gg.empty:
                continue

            y = gg.reindex(months_aligned)["ret_fwd_1m"].to_numpy(dtype=float)
            mask = np.isfinite(y)
            if mask.sum() < min_obs:
                continue

            y2 = y[mask]
            X2 = X_by_month[mask, :]

            alpha, betas, r2, se_a = ols_alpha_beta(y2, X2)
            if not np.isfinite(alpha):
                continue

            # ===== 핵심: t統計を計算（se=0/NaNは弾く）=====
            if (not np.isfinite(se_a)) or (se_a <= 0):
                tstat = np.nan
            else:
                tstat = float(alpha / se_a)

            out_rows.append({
                "MonthEnd": t,
                "Code": code,
                "alpha": alpha,
                "se_alpha": se_a,
                "tstat": tstat,
                "n_obs": int(mask.sum()),
                "r2": r2,
            })
            n_with_stats += 1

        cov_rows.append({
            "MonthEnd": t,
            "n_codes_in_universe": int(len(by_code)),
            "n_codes_with_stats": int(n_with_stats),
            "lookback_months": int(lookback_m),
            "min_obs": int(min_obs),
        })

    stats_df = pd.DataFrame(out_rows)
    cov_df = pd.DataFrame(cov_rows).sort_values("MonthEnd")
    cov_df.to_csv(OUT_DIAG_COV, index=False, encoding="utf-8-sig")

    return stats_df


# ============================================================
# Backtest (topK by score)
# ============================================================
def backtest_topk_equal_weight(panel: pd.DataFrame,
                               stats_df: pd.DataFrame,
                               top_k: int,
                               score_mode: str = "tstat") -> Tuple[pd.DataFrame, pd.DataFrame]:
    if stats_df.empty:
        raise RuntimeError("stats_df is empty; check coverage/period/inputs")

    ret_map = panel.set_index(["MonthEnd", "Code"])["ret_fwd_1m"]

    months = sorted(stats_df["MonthEnd"].unique())
    bt_rows = []
    sel_rows = []

    score_col = "tstat" if score_mode == "tstat" else "alpha"

    for t in months:
        s = stats_df[stats_df["MonthEnd"] == t].copy()
        s = s.dropna(subset=[score_col]).sort_values(score_col, ascending=False).head(top_k)
        if s.empty:
            continue

        codes = s["Code"].tolist()

        rets = []
        for c in codes:
            try:
                r = float(ret_map.loc[(t, c)])
            except KeyError:
                r = np.nan
            rets.append(r)

        rets = np.array(rets, dtype=float)
        used = np.isfinite(rets)
        port_ret = float(np.nanmean(rets)) if used.sum() > 0 else np.nan

        bt_rows.append({
            "MonthEnd": t,
            "portfolio_ret": port_ret,
            "n_selected": int(len(codes)),
            "n_used_ret": int(used.sum()),
            "avg_alpha_selected": float(s["alpha"].mean()),
            "avg_tstat_selected": float(s["tstat"].mean()) if "tstat" in s.columns else np.nan,
            "score_mode": score_mode,
        })

        for _, r in s.iterrows():
            sel_rows.append({
                "MonthEnd": t,
                "Code": r["Code"],
                "alpha": float(r["alpha"]),
                "se_alpha": float(r["se_alpha"]) if pd.notna(r["se_alpha"]) else np.nan,
                "tstat": float(r["tstat"]) if pd.notna(r["tstat"]) else np.nan,
                "score": float(r[score_col]),
                "n_obs": int(r["n_obs"]),
                "r2": float(r["r2"]) if pd.notna(r["r2"]) else np.nan,
            })

    bt = pd.DataFrame(bt_rows).sort_values("MonthEnd").reset_index(drop=True)
    sel = pd.DataFrame(sel_rows).sort_values(["MonthEnd", "score"], ascending=[True, False]).reset_index(drop=True)
    sel.to_csv(OUT_DIAG_SEL, index=False, encoding="utf-8-sig")

    return bt, sel


# ============================================================
# Portfolio regression vs factors
# ============================================================
def regress_portfolio_vs_factors(bt: pd.DataFrame, fac: pd.DataFrame) -> pd.DataFrame:
    df = bt.merge(fac, on="MonthEnd", how="left", validate="many_to_one").copy()
    df = df.dropna(subset=["portfolio_ret"] + FACTOR_COLS).copy()

    y = df["portfolio_ret"].to_numpy(dtype=float)
    X = df[FACTOR_COLS].to_numpy(dtype=float)

    alpha, betas, r2, se_a = ols_alpha_beta(y, X)

    out = {
        "score_mode": SCORE_MODE,
        "lookback_months_for_selection": int(LOOKBACK_M),
        "min_obs": int(MIN_OBS),
        "n_months": int(len(df)),
        "R2": float(r2),
        "alpha_monthly": float(alpha),
        "alpha_annualized_approx": float(alpha * 12.0),
        "se_alpha": float(se_a),
    }
    for c, b in zip(FACTOR_COLS, betas):
        out[f"beta_{c}"] = float(b)

    return pd.DataFrame([out])


# ============================================================
# Main
# ============================================================
def main():
    log("=" * 110)
    log("Backtest: Top20 by rolling tstat(alpha/se) (FF5+MOM) -> next-month equal-weight returns")
    log("=" * 110)
    log(f"PRICE_PATH : {PRICE_PATH}")
    log(f"SNAP_PATH  : {SNAP_PATH}")
    log(f"FACTOR_PATH: {FACTOR_PATH}")
    log(f"PERIOD     : {START_MONTHEND.date()} .. {END_MONTHEND.date()}")
    log(f"LOOKBACK   : {LOOKBACK_M} months | TOP_K={TOP_K} | MIN_OBS={MIN_OBS} | SCORE_MODE={SCORE_MODE}")
    log(f"OUT_DIR    : {OUT_DIR}")

    fac = load_factors()
    pr = load_prices()
    codes = pick_universe_codes()

    panel = build_panel(pr, fac, codes)

    # keep earlier months for lookback
    min_needed = START_MONTHEND - pd.offsets.MonthEnd(LOOKBACK_M + 2)
    panel = panel[(panel["MonthEnd"] >= min_needed) & (panel["MonthEnd"] <= END_MONTHEND)].copy()

    log(f"[PANEL] rows={len(panel):,} codes={panel['Code'].nunique():,} months={panel['MonthEnd'].nunique():,}")

    stats_df = rolling_stats_by_month(panel, START_MONTHEND, END_MONTHEND, LOOKBACK_M, MIN_OBS)
    log(f"[STATS] rows={len(stats_df):,} months={stats_df['MonthEnd'].nunique() if len(stats_df) else 0:,}")

    bt, sel = backtest_topk_equal_weight(panel, stats_df, TOP_K, SCORE_MODE)
    log(f"[BT] months={len(bt):,} start={bt['MonthEnd'].min()} end={bt['MonthEnd'].max()}")

    # Save monthly returns
    bt_out = bt.copy()
    bt_out["MonthEnd"] = pd.to_datetime(bt_out["MonthEnd"]).dt.date.astype(str)
    bt_out.to_csv(OUT_MONTHLY_RET, index=False, encoding="utf-8-sig")

    # Cumulative curve + DD
    bt2 = bt.dropna(subset=["portfolio_ret"]).copy()
    bt2["cum"] = (1.0 + bt2["portfolio_ret"]).cumprod()
    bt2["dd"] = compute_drawdown(bt2["cum"])

    cum_out = bt2[["MonthEnd", "portfolio_ret", "cum", "dd", "n_selected", "n_used_ret",
                   "avg_alpha_selected", "avg_tstat_selected", "score_mode"]].copy()
    cum_out["MonthEnd"] = pd.to_datetime(cum_out["MonthEnd"]).dt.date.astype(str)
    cum_out.to_csv(OUT_CUMCURVE, index=False, encoding="utf-8-sig")

    # Performance summary
    stats = perf_stats(bt2["portfolio_ret"])
    perf_df = pd.DataFrame([stats])
    perf_df.insert(0, "score_mode", SCORE_MODE)
    perf_df.insert(1, "lookback_months_for_selection", LOOKBACK_M)
    perf_df.insert(2, "min_obs", MIN_OBS)
    perf_df.to_csv(OUT_PERF, index=False, encoding="utf-8-sig")

    # Portfolio regression vs factors
    reg_df = regress_portfolio_vs_factors(bt, fac)
    reg_df.to_csv(OUT_REG, index=False, encoding="utf-8-sig")

    log("-" * 110)
    log("✅ SAVED")
    log(f"  - monthly returns : {OUT_MONTHLY_RET}")
    log(f"  - cumulative curve: {OUT_CUMCURVE}")
    log(f"  - perf summary    : {OUT_PERF}")
    log(f"  - regression      : {OUT_REG}")
    log(f"  - diag selected   : {OUT_DIAG_SEL}")
    log(f"  - diag coverage   : {OUT_DIAG_COV}")
    log("-" * 110)

    log("[PERF SUMMARY]")
    for k, v in stats.items():
        log(f"  {k}: {v}")

    log("[REGRESSION SUMMARY]")
    if len(reg_df):
        row = reg_df.iloc[0].to_dict()
        keys = ["score_mode", "lookback_months_for_selection", "min_obs", "n_months",
                "R2", "alpha_monthly", "alpha_annualized_approx", "se_alpha"] + [f"beta_{c}" for c in FACTOR_COLS]
        for k in keys:
            if k in row:
                log(f"  {k}: {row[k]}")

if __name__ == "__main__":
    main()


Backtest: Top20 by rolling tstat(alpha/se) (FF5+MOM) -> next-month equal-weight returns
PRICE_PATH : C:\Users\yongr\Project\merged_data_all_stocks\factors\price_month_end.parquet
SNAP_PATH  : C:\Users\yongr\Project\merged_data_all_stocks\factors\month_end_snapshot.parquet
FACTOR_PATH: C:\Users\yongr\Project\merged_data_all_stocks\factors\ff5_mom_factors_monthly.parquet
PERIOD     : 2016-10-31 .. 2025-09-30
LOOKBACK   : 24 months | TOP_K=20 | MIN_OBS=18 | SCORE_MODE=tstat
OUT_DIR    : C:\Users\yongr\Project\merged_data_all_stocks\factors\bt_top20_rolling_tstat_ff5mom_lb24_201610_202509
[UNIVERSE] latest MonthEnd=2025-09-30 codes=3,428 (MAX_CODES=None)
[PANEL] rows=363,954 codes=3,428 months=117
[STATS] rows=269,234 months=87
[BT] months=87 start=2018-07-31 00:00:00 end=2025-09-30 00:00:00
--------------------------------------------------------------------------------------------------------------
✅ SAVED
  - monthly returns : C:\Users\yongr\Project\merged_data_all_stocks\factors\bt_top

In [4]:
# -*- coding: utf-8 -*-
"""
FF5+MOM だけで
「各月末に、直近 LOOKBACK_M ヶ月の銘柄別FF5+MOM回帰 t統計（alpha/se）」上位20銘柄を等金額で買い、
翌月リターンで評価する月次バックテスト（2016-10〜2025-09）

+ Risk Control (C): レジーム減速 + β上限制約
- レジーム: 直近3ヶ月 sum(MKT)<0 or sum(WML)<0 -> invest_ratio=0.5
- β制約: beta_CMA < -0.8 or abs(beta_MKT) > 0.9 -> invest_ratio=0.5
- それ以外: invest_ratio=1.0

Outputs:
- riskcontrol無し/あり の月次リターンCSV
- riskcontrol無し/あり の累積曲線(累積,DD)CSV
- riskcontrol無し/あり の performance summary CSV（CAGR, maxDD, ann_mean, ann_vol, sharpe0）
- riskcontrol無し/あり の portfolio regression vs FF5+MOM CSV
- invest_ratio 診断CSV（発動理由つき）
- diag_selected_top20_by_month.csv
- diag_coverage_by_month.csv

依存:
  pip install pandas numpy pyarrow
"""

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
from typing import Tuple, Dict, Optional, List

import numpy as np
import pandas as pd


# ============================================================
# Paths (あなたの環境)
# ============================================================
FACTORS_DIR = Path(r"C:\Users\yongr\Project\merged_data_all_stocks\factors")
PRICE_PATH = FACTORS_DIR / "price_month_end.parquet"
SNAP_PATH  = FACTORS_DIR / "month_end_snapshot.parquet"
FACTOR_PATH = FACTORS_DIR / "ff5_mom_factors_monthly.parquet"  # 既存FF5+MOM(月次因子)

# ============================================================
# Backtest params
# ============================================================
START_MONTHEND = pd.Timestamp("2016-10-31")
END_MONTHEND   = pd.Timestamp("2025-09-30")

LOOKBACK_M = 24       # 銘柄別回帰・ポートβ推定に使う窓
TOP_K = 20

# 欠損対策（推奨値）
if LOOKBACK_M >= 36:
    MIN_OBS = 24
elif LOOKBACK_M >= 24:
    MIN_OBS = 18
else:
    MIN_OBS = 10

# 速度/負荷対策（全銘柄が重い場合ここを 1000 などに）
MAX_CODES: Optional[int] = None

# 回帰で使う因子列（MOMはWMLとして扱う）
FACTOR_COLS = ["MKT", "SMB", "HML", "RMW", "CMA", "WML"]

# スコア（選定）はt統計
SCORE_MODE = "tstat"

# ============================================================
# Risk Control params (C)
# ============================================================
INVEST_RATIO_RISKOFF = 0.5

REGIME_LOOKBACK_M = 3
REGIME_OFF_IF_SUM_MKT_LT = 0.0
REGIME_OFF_IF_SUM_WML_LT = 0.0

BETA_CAP_CMA_LT = -0.8
BETA_CAP_ABS_MKT_GT = 0.9


# ============================================================
# Outputs
# ============================================================
OUT_DIR = FACTORS_DIR / f"bt_top20_tstat_ff5mom_lb{LOOKBACK_M}_riskC_201610_202509"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# --- no risk control
OUT_MONTHLY_RET_RAW = OUT_DIR / "bt_monthly_returns_raw.csv"
OUT_CUMCURVE_RAW    = OUT_DIR / "bt_cumulative_curve_raw.csv"
OUT_PERF_RAW        = OUT_DIR / "bt_performance_summary_raw.csv"
OUT_REG_RAW         = OUT_DIR / "bt_portfolio_regression_raw_ff5mom.csv"

# --- with risk control
OUT_MONTHLY_RET_RC = OUT_DIR / "bt_monthly_returns_riskcontrol.csv"
OUT_CUMCURVE_RC    = OUT_DIR / "bt_cumulative_curve_riskcontrol.csv"
OUT_PERF_RC        = OUT_DIR / "bt_performance_summary_riskcontrol.csv"
OUT_REG_RC         = OUT_DIR / "bt_portfolio_regression_riskcontrol_ff5mom.csv"

# diagnostics
OUT_DIAG_SEL    = OUT_DIR / "diag_selected_top20_by_month.csv"
OUT_DIAG_COV    = OUT_DIR / "diag_coverage_by_month.csv"
OUT_DIAG_IR     = OUT_DIR / "diag_invest_ratio_by_month.csv"


# ============================================================
# Utils
# ============================================================
BAD_CODE_STRINGS = {"None", "nan", "", "NaN", "NULL", "null"}

def normalize_code(code) -> str:
    if code is None:
        return ""
    s = str(code).strip()
    if s in BAD_CODE_STRINGS:
        return ""
    return s

def log(s: str):
    print(s)

def safe_to_datetime(s):
    return pd.to_datetime(s, errors="coerce")

def ols_alpha_beta(y: np.ndarray, X: np.ndarray) -> Tuple[float, np.ndarray, float, float]:
    """
    OLS with intercept.
    Returns alpha, betas, R2, stderr_alpha (簡易; homoskedastic)
    """
    n = y.shape[0]
    if n < 3:
        return np.nan, np.full(X.shape[1], np.nan), np.nan, np.nan

    X1 = np.column_stack([np.ones(n), X])  # intercept
    XtX = X1.T @ X1
    try:
        inv = np.linalg.inv(XtX)
    except np.linalg.LinAlgError:
        inv = np.linalg.pinv(XtX)

    b = inv @ (X1.T @ y)
    alpha = float(b[0])
    betas = b[1:].astype(float)

    yhat = X1 @ b
    resid = y - yhat
    sse = float(np.sum(resid**2))
    sst = float(np.sum((y - y.mean())**2))
    r2 = np.nan if sst <= 0 else (1.0 - sse / sst)

    k = X1.shape[1]
    dof = max(1, n - k)
    sigma2 = sse / dof
    var_b = sigma2 * inv
    se_alpha = float(np.sqrt(max(0.0, var_b[0, 0])))

    return alpha, betas, r2, se_alpha

def compute_drawdown(cum: pd.Series) -> pd.Series:
    peak = cum.cummax()
    return cum / peak - 1.0

def perf_stats(monthly_ret: pd.Series) -> Dict:
    r = monthly_ret.dropna().astype(float)
    if r.empty:
        return {
            "n_months": 0, "CAGR": np.nan, "ann_vol": np.nan, "ann_mean": np.nan,
            "sharpe0": np.nan, "maxDD": np.nan, "cum_end": np.nan
        }
    n = len(r)
    cum = (1.0 + r).cumprod()
    years = n / 12.0
    cagr = float(cum.iloc[-1] ** (1/years) - 1.0) if years > 0 else np.nan
    ann_mean = float(r.mean() * 12.0)
    ann_vol = float(r.std(ddof=1) * np.sqrt(12.0)) if n >= 2 else np.nan
    sharpe0 = float(ann_mean / ann_vol) if ann_vol and ann_vol > 0 else np.nan
    dd = compute_drawdown(cum)
    maxdd = float(dd.min())
    return {
        "n_months": int(n),
        "CAGR": cagr,
        "ann_mean": ann_mean,
        "ann_vol": ann_vol,
        "sharpe0": sharpe0,
        "maxDD": maxdd,
        "cum_end": float(cum.iloc[-1]),
    }


# ============================================================
# Load data
# ============================================================
def load_factors() -> pd.DataFrame:
    fac = pd.read_parquet(FACTOR_PATH).copy()
    fac["MonthEnd"] = safe_to_datetime(fac["MonthEnd"]).dt.normalize()
    fac = fac.sort_values("MonthEnd").reset_index(drop=True)

    need = ["MonthEnd"] + FACTOR_COLS
    missing = [c for c in need if c not in fac.columns]
    if missing:
        raise KeyError(f"Factor file missing columns: {missing} in {FACTOR_PATH}")
    return fac[need].copy()

def load_prices() -> pd.DataFrame:
    pr = pd.read_parquet(PRICE_PATH)[["Code", "MonthEnd", "AdjustedClose"]].copy()
    pr["Code"] = pr["Code"].astype(str).map(normalize_code)
    pr = pr[pr["Code"] != ""].copy()
    pr["MonthEnd"] = safe_to_datetime(pr["MonthEnd"]).dt.normalize()
    pr["AdjustedClose"] = pd.to_numeric(pr["AdjustedClose"], errors="coerce")
    pr = pr.dropna(subset=["MonthEnd", "AdjustedClose"]).copy()
    pr = pr.sort_values(["Code", "MonthEnd"], kind="mergesort").reset_index(drop=True)

    pr["Adj_next"] = pr.groupby("Code")["AdjustedClose"].shift(-1)
    pr["ret_fwd_1m"] = pr["Adj_next"] / pr["AdjustedClose"] - 1.0

    return pr[["Code", "MonthEnd", "ret_fwd_1m"]].copy()

def pick_universe_codes() -> List[str]:
    snap = pd.read_parquet(SNAP_PATH)[["Code", "MonthEnd", "MarketCap"]].copy()
    snap["Code"] = snap["Code"].astype(str).map(normalize_code)
    snap = snap[snap["Code"] != ""].copy()
    snap["MonthEnd"] = safe_to_datetime(snap["MonthEnd"]).dt.normalize()
    snap["MarketCap"] = pd.to_numeric(snap["MarketCap"], errors="coerce")
    latest = snap["MonthEnd"].max()

    s = snap[snap["MonthEnd"] == latest].dropna(subset=["MarketCap"]).copy()
    s = s.sort_values("MarketCap", ascending=False).drop_duplicates(["Code"], keep="first")

    codes = s["Code"].tolist()
    if MAX_CODES is not None:
        codes = codes[:MAX_CODES]
    log(f"[UNIVERSE] latest MonthEnd={latest.date()} codes={len(codes):,} (MAX_CODES={MAX_CODES})")
    return codes


# ============================================================
# Panel + rolling per-stock regression stats
# ============================================================
def build_panel(pr: pd.DataFrame, fac: pd.DataFrame, codes: List[str]) -> pd.DataFrame:
    p = pr[pr["Code"].isin(codes)].copy()
    p = p.merge(fac, on="MonthEnd", how="left", validate="many_to_one")
    p = p.sort_values(["Code", "MonthEnd"]).reset_index(drop=True)
    return p

def rolling_stock_stats_by_month(panel: pd.DataFrame,
                                 start_me: pd.Timestamp,
                                 end_me: pd.Timestamp,
                                 lookback_m: int,
                                 min_obs: int) -> pd.DataFrame:
    """
    Selection at MonthEnd t uses realized returns in months (t-lookback_m .. t-1).
    realized month return at MonthEnd m is ret_fwd_1m at MonthEnd m (m->m+1).
    """
    months_all = sorted(panel["MonthEnd"].dropna().unique())
    months_all = [pd.Timestamp(x) for x in months_all]
    sel_months = [m for m in months_all if (m >= start_me) and (m <= end_me)]

    by_code = {c: g.reset_index(drop=True) for c, g in panel.groupby("Code", sort=False)}

    fac_month = (panel[["MonthEnd"] + FACTOR_COLS]
                 .drop_duplicates("MonthEnd")
                 .set_index("MonthEnd")
                 .sort_index())

    out_rows = []
    cov_rows = []

    for t in sel_months:
        prior = [m for m in months_all if m < t]
        if len(prior) < lookback_m:
            continue
        win_months = prior[-lookback_m:]

        fac_win = fac_month.reindex(win_months).dropna()
        if len(fac_win) < min_obs:
            continue

        months_aligned = fac_win.index.to_list()
        X_by_month = fac_win[FACTOR_COLS].to_numpy(dtype=float)

        n_with_stats = 0

        for code, g in by_code.items():
            gg = g[g["MonthEnd"].isin(months_aligned)][["MonthEnd", "ret_fwd_1m"]]\
                    .set_index("MonthEnd").sort_index()
            if gg.empty:
                continue

            y = gg.reindex(months_aligned)["ret_fwd_1m"].to_numpy(dtype=float)
            mask = np.isfinite(y)
            if mask.sum() < min_obs:
                continue

            y2 = y[mask]
            X2 = X_by_month[mask, :]

            alpha, betas, r2, se_a = ols_alpha_beta(y2, X2)
            if not np.isfinite(alpha):
                continue

            if (not np.isfinite(se_a)) or (se_a <= 0):
                tstat = np.nan
            else:
                tstat = float(alpha / se_a)

            out_rows.append({
                "MonthEnd": t,
                "Code": code,
                "alpha": alpha,
                "se_alpha": se_a,
                "tstat": tstat,
                "n_obs": int(mask.sum()),
                "r2": r2,
            })
            n_with_stats += 1

        cov_rows.append({
            "MonthEnd": t,
            "n_codes_in_universe": int(len(by_code)),
            "n_codes_with_stats": int(n_with_stats),
            "lookback_months": int(lookback_m),
            "min_obs": int(min_obs),
        })

    stats_df = pd.DataFrame(out_rows)
    cov_df = pd.DataFrame(cov_rows).sort_values("MonthEnd")
    cov_df.to_csv(OUT_DIAG_COV, index=False, encoding="utf-8-sig")

    return stats_df


# ============================================================
# Helpers: regime + beta cap -> invest_ratio
# ============================================================
def compute_regime_flag(fac: pd.DataFrame, t: pd.Timestamp, lookback_m: int = 3) -> Tuple[bool, Dict]:
    """
    Regime OFF if sum of last `lookback_m` months (ending at t-1) is negative
    for MKT or WML.
    """
    fac2 = fac.sort_values("MonthEnd").reset_index(drop=True).copy()
    fac2 = fac2.set_index("MonthEnd")

    # months strictly before t
    idx = fac2.index[fac2.index < t]
    if len(idx) < lookback_m:
        return False, {"regime_ready": False}

    win = idx[-lookback_m:]
    s_mkt = float(pd.to_numeric(fac2.loc[win, "MKT"], errors="coerce").sum())
    s_wml = float(pd.to_numeric(fac2.loc[win, "WML"], errors="coerce").sum())

    off = (s_mkt < REGIME_OFF_IF_SUM_MKT_LT) or (s_wml < REGIME_OFF_IF_SUM_WML_LT)
    return bool(off), {"regime_ready": True, "sum_MKT_3m": s_mkt, "sum_WML_3m": s_wml}

def estimate_portfolio_beta_from_selected(panel: pd.DataFrame,
                                          selected_codes: List[str],
                                          t: pd.Timestamp,
                                          lookback_m: int,
                                          min_obs: int) -> Tuple[Dict, Dict]:
    """
    Estimate portfolio beta using equal-weight portfolio realized returns over window (t-lookback_m .. t-1).
    Uses the same factor set FF5+MOM (MKT..WML).
    """
    months_all = sorted(panel["MonthEnd"].dropna().unique())
    months_all = [pd.Timestamp(x) for x in months_all]
    prior = [m for m in months_all if m < t]
    if len(prior) < lookback_m:
        return {"beta_ready": False}, {}

    win_months = prior[-lookback_m:]

    # build portfolio realized returns in those months (equal-weight across selected)
    sub = panel[(panel["MonthEnd"].isin(win_months)) & (panel["Code"].isin(selected_codes))].copy()
    if sub.empty:
        return {"beta_ready": False}, {}

    # equal-weight mean by MonthEnd
    port = sub.groupby("MonthEnd")["ret_fwd_1m"].mean().to_frame("port_ret").reset_index()
    # merge factors
    facm = (panel[["MonthEnd"] + FACTOR_COLS].drop_duplicates("MonthEnd"))
    df = port.merge(facm, on="MonthEnd", how="left", validate="one_to_one").dropna(subset=["port_ret"] + FACTOR_COLS)
    if len(df) < min_obs:
        return {"beta_ready": False, "n_obs_port": int(len(df))}, {}

    y = df["port_ret"].to_numpy(dtype=float)
    X = df[FACTOR_COLS].to_numpy(dtype=float)
    alpha, betas, r2, se_a = ols_alpha_beta(y, X)

    beta_dict = {f"beta_{c}": float(b) for c, b in zip(FACTOR_COLS, betas)}
    meta = {
        "beta_ready": True,
        "n_obs_port": int(len(df)),
        "r2_port": float(r2),
        "alpha_port": float(alpha),
    }
    meta.update(beta_dict)
    return meta, beta_dict

def decide_invest_ratio(fac: pd.DataFrame,
                        panel: pd.DataFrame,
                        t: pd.Timestamp,
                        selected_codes: List[str]) -> Tuple[float, Dict]:
    """
    Combine:
      (A) regime off  -> invest_ratio=0.5
      (B) beta cap    -> invest_ratio=0.5
    """
    invest_ratio = 1.0
    reasons = {
        "MonthEnd": t,
        "invest_ratio": invest_ratio,
        "regime_off": False,
        "beta_off": False,
        "beta_ready": False,
        "regime_ready": False,
        "sum_MKT_3m": np.nan,
        "sum_WML_3m": np.nan,
        "beta_MKT": np.nan,
        "beta_CMA": np.nan,
        "beta_r2": np.nan,
        "beta_n_obs": np.nan,
    }

    # (A) regime
    off_regime, info = compute_regime_flag(fac, t, REGIME_LOOKBACK_M)
    reasons["regime_off"] = bool(off_regime)
    reasons["regime_ready"] = bool(info.get("regime_ready", False))
    reasons["sum_MKT_3m"] = info.get("sum_MKT_3m", np.nan)
    reasons["sum_WML_3m"] = info.get("sum_WML_3m", np.nan)
    if off_regime:
        invest_ratio = min(invest_ratio, INVEST_RATIO_RISKOFF)

    # (B) beta cap (use past LOOKBACK_M months)
    meta, _ = estimate_portfolio_beta_from_selected(panel, selected_codes, t, LOOKBACK_M, MIN_OBS)
    reasons["beta_ready"] = bool(meta.get("beta_ready", False))
    if meta.get("beta_ready", False):
        b_mkt = float(meta.get("beta_MKT", np.nan))
        b_cma = float(meta.get("beta_CMA", np.nan))
        reasons["beta_MKT"] = b_mkt
        reasons["beta_CMA"] = b_cma
        reasons["beta_r2"] = float(meta.get("r2_port", np.nan))
        reasons["beta_n_obs"] = float(meta.get("n_obs_port", np.nan))

        off_beta = (b_cma < BETA_CAP_CMA_LT) or (abs(b_mkt) > BETA_CAP_ABS_MKT_GT)
        reasons["beta_off"] = bool(off_beta)
        if off_beta:
            invest_ratio = min(invest_ratio, INVEST_RATIO_RISKOFF)

    reasons["invest_ratio"] = float(invest_ratio)
    return float(invest_ratio), reasons


# ============================================================
# Backtest (raw + riskcontrol)
# ============================================================
def backtest_with_riskcontrol(panel: pd.DataFrame,
                              fac: pd.DataFrame,
                              stats_df: pd.DataFrame,
                              top_k: int) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    For each MonthEnd t:
      - select top_k by tstat
      - raw return = mean(ret_fwd_1m)
      - invest_ratio decided by (A)+(B)
      - riskcontrolled return = invest_ratio * raw_return
    """
    if stats_df.empty:
        raise RuntimeError("stats_df is empty; check coverage/period/inputs")

    ret_map = panel.set_index(["MonthEnd", "Code"])["ret_fwd_1m"]

    months = sorted(stats_df["MonthEnd"].unique())

    bt_rows = []
    sel_rows = []
    ir_rows = []

    for t in months:
        s = stats_df[stats_df["MonthEnd"] == t].copy()
        s = s.dropna(subset=["tstat"]).sort_values("tstat", ascending=False).head(top_k)
        if s.empty:
            continue

        codes = s["Code"].tolist()

        rets = []
        for c in codes:
            try:
                r = float(ret_map.loc[(t, c)])
            except KeyError:
                r = np.nan
            rets.append(r)

        rets = np.array(rets, dtype=float)
        used = np.isfinite(rets)
        raw_ret = float(np.nanmean(rets)) if used.sum() > 0 else np.nan

        invest_ratio, reason = decide_invest_ratio(fac, panel, pd.Timestamp(t), codes)
        ir_rows.append(reason)

        rc_ret = raw_ret * invest_ratio if np.isfinite(raw_ret) else np.nan

        bt_rows.append({
            "MonthEnd": t,
            "portfolio_ret_raw": raw_ret,
            "portfolio_ret_riskcontrol": rc_ret,
            "invest_ratio": float(invest_ratio),
            "n_selected": int(len(codes)),
            "n_used_ret": int(used.sum()),
            "avg_alpha_selected": float(s["alpha"].mean()),
            "avg_tstat_selected": float(s["tstat"].mean()),
        })

        for _, r in s.iterrows():
            sel_rows.append({
                "MonthEnd": t,
                "Code": r["Code"],
                "alpha": float(r["alpha"]),
                "se_alpha": float(r["se_alpha"]) if pd.notna(r["se_alpha"]) else np.nan,
                "tstat": float(r["tstat"]) if pd.notna(r["tstat"]) else np.nan,
                "n_obs": int(r["n_obs"]),
                "r2": float(r["r2"]) if pd.notna(r["r2"]) else np.nan,
            })

    bt = pd.DataFrame(bt_rows).sort_values("MonthEnd").reset_index(drop=True)
    sel = pd.DataFrame(sel_rows).sort_values(["MonthEnd", "tstat"], ascending=[True, False]).reset_index(drop=True)
    ir = pd.DataFrame(ir_rows).sort_values("MonthEnd").reset_index(drop=True)

    sel.to_csv(OUT_DIAG_SEL, index=False, encoding="utf-8-sig")
    ir.to_csv(OUT_DIAG_IR, index=False, encoding="utf-8-sig")

    return bt, sel, ir


# ============================================================
# Portfolio regression vs factors
# ============================================================
def regress_portfolio_vs_factors(bt: pd.DataFrame, fac: pd.DataFrame, ret_col: str) -> pd.DataFrame:
    df = bt.merge(fac, on="MonthEnd", how="left", validate="many_to_one").copy()
    df = df.dropna(subset=[ret_col] + FACTOR_COLS).copy()

    y = df[ret_col].to_numpy(dtype=float)
    X = df[FACTOR_COLS].to_numpy(dtype=float)

    alpha, betas, r2, se_a = ols_alpha_beta(y, X)

    out = {
        "ret_col": ret_col,
        "lookback_months_for_selection": int(LOOKBACK_M),
        "min_obs": int(MIN_OBS),
        "n_months": int(len(df)),
        "R2": float(r2),
        "alpha_monthly": float(alpha),
        "alpha_annualized_approx": float(alpha * 12.0),
        "se_alpha": float(se_a),
        "risk_regime_lookback_m": int(REGIME_LOOKBACK_M),
        "riskoff_ratio": float(INVEST_RATIO_RISKOFF),
        "beta_cap_CMA_lt": float(BETA_CAP_CMA_LT),
        "beta_cap_abs_MKT_gt": float(BETA_CAP_ABS_MKT_GT),
        "regime_off_if_sum_MKT_lt": float(REGIME_OFF_IF_SUM_MKT_LT),
        "regime_off_if_sum_WML_lt": float(REGIME_OFF_IF_SUM_WML_LT),
    }
    for c, b in zip(FACTOR_COLS, betas):
        out[f"beta_{c}"] = float(b)

    return pd.DataFrame([out])


# ============================================================
# Save helpers
# ============================================================
def save_bt_bundle(bt: pd.DataFrame, ret_col: str, out_ret_csv: Path, out_curve_csv: Path, out_perf_csv: Path):
    x = bt[["MonthEnd", ret_col, "invest_ratio", "n_selected", "n_used_ret",
            "avg_alpha_selected", "avg_tstat_selected"]].copy()
    x["MonthEnd"] = pd.to_datetime(x["MonthEnd"]).dt.date.astype(str)
    x = x.rename(columns={ret_col: "portfolio_ret"})
    x.to_csv(out_ret_csv, index=False, encoding="utf-8-sig")

    y = bt[["MonthEnd", ret_col]].dropna().copy()
    y = y.rename(columns={ret_col: "portfolio_ret"})
    y["cum"] = (1.0 + y["portfolio_ret"]).cumprod()
    y["dd"] = compute_drawdown(y["cum"])

    curve = y.copy()
    curve["MonthEnd"] = pd.to_datetime(curve["MonthEnd"]).dt.date.astype(str)
    curve.to_csv(out_curve_csv, index=False, encoding="utf-8-sig")

    stats = perf_stats(y["portfolio_ret"])
    perf_df = pd.DataFrame([stats])
    perf_df.insert(0, "ret_col", ret_col)
    perf_df.insert(1, "lookback_months_for_selection", LOOKBACK_M)
    perf_df.insert(2, "min_obs", MIN_OBS)
    perf_df.insert(3, "riskoff_ratio", INVEST_RATIO_RISKOFF)
    perf_df.to_csv(out_perf_csv, index=False, encoding="utf-8-sig")


# ============================================================
# Main
# ============================================================
def main():
    log("=" * 110)
    log("Backtest: Top20 by rolling tstat(alpha/se) (FF5+MOM) -> next-month equal-weight returns + RiskControl(C)")
    log("=" * 110)
    log(f"PRICE_PATH : {PRICE_PATH}")
    log(f"SNAP_PATH  : {SNAP_PATH}")
    log(f"FACTOR_PATH: {FACTOR_PATH}")
    log(f"PERIOD     : {START_MONTHEND.date()} .. {END_MONTHEND.date()}")
    log(f"LOOKBACK   : {LOOKBACK_M} months | TOP_K={TOP_K} | MIN_OBS={MIN_OBS} | SCORE_MODE={SCORE_MODE}")
    log(f"RISK(C)    : regime(3m MKT<0 or WML<0) + betaCap(CMA<-0.8 or |MKT|>0.9) -> invest_ratio=0.5")
    log(f"OUT_DIR    : {OUT_DIR}")

    fac = load_factors()
    pr = load_prices()
    codes = pick_universe_codes()

    panel = build_panel(pr, fac, codes)

    # keep earlier months for lookback (selection & beta estimation)
    min_needed = START_MONTHEND - pd.offsets.MonthEnd(LOOKBACK_M + REGIME_LOOKBACK_M + 3)
    panel = panel[(panel["MonthEnd"] >= min_needed) & (panel["MonthEnd"] <= END_MONTHEND)].copy()

    log(f"[PANEL] rows={len(panel):,} codes={panel['Code'].nunique():,} months={panel['MonthEnd'].nunique():,}")

    stats_df = rolling_stock_stats_by_month(panel, START_MONTHEND, END_MONTHEND, LOOKBACK_M, MIN_OBS)
    log(f"[STATS] rows={len(stats_df):,} months={stats_df['MonthEnd'].nunique() if len(stats_df) else 0:,}")

    bt, sel, ir = backtest_with_riskcontrol(panel, fac, stats_df, TOP_K)
    log(f"[BT] months={len(bt):,} start={bt['MonthEnd'].min()} end={bt['MonthEnd'].max()}")

    # Save raw bundle
    save_bt_bundle(bt, "portfolio_ret_raw", OUT_MONTHLY_RET_RAW, OUT_CUMCURVE_RAW, OUT_PERF_RAW)
    reg_raw = regress_portfolio_vs_factors(bt, fac, "portfolio_ret_raw")
    reg_raw.to_csv(OUT_REG_RAW, index=False, encoding="utf-8-sig")

    # Save riskcontrol bundle
    save_bt_bundle(bt, "portfolio_ret_riskcontrol", OUT_MONTHLY_RET_RC, OUT_CUMCURVE_RC, OUT_PERF_RC)
    reg_rc = regress_portfolio_vs_factors(bt, fac, "portfolio_ret_riskcontrol")
    reg_rc.to_csv(OUT_REG_RC, index=False, encoding="utf-8-sig")

    log("-" * 110)
    log("✅ SAVED (RAW)")
    log(f"  - {OUT_MONTHLY_RET_RAW}")
    log(f"  - {OUT_CUMCURVE_RAW}")
    log(f"  - {OUT_PERF_RAW}")
    log(f"  - {OUT_REG_RAW}")
    log("✅ SAVED (RISKCONTROL)")
    log(f"  - {OUT_MONTHLY_RET_RC}")
    log(f"  - {OUT_CUMCURVE_RC}")
    log(f"  - {OUT_PERF_RC}")
    log(f"  - {OUT_REG_RC}")
    log("✅ DIAGNOSTICS")
    log(f"  - selected: {OUT_DIAG_SEL}")
    log(f"  - coverage: {OUT_DIAG_COV}")
    log(f"  - invest_ratio: {OUT_DIAG_IR}")
    log("-" * 110)

    # quick summary print
    raw_stats = pd.read_csv(OUT_PERF_RAW).iloc[0].to_dict()
    rc_stats  = pd.read_csv(OUT_PERF_RC).iloc[0].to_dict()
    log("[PERF SUMMARY RAW]")
    for k in ["n_months","CAGR","ann_mean","ann_vol","sharpe0","maxDD","cum_end"]:
        if k in raw_stats:
            log(f"  {k}: {raw_stats[k]}")
    log("[PERF SUMMARY RISKCONTROL]")
    for k in ["n_months","CAGR","ann_mean","ann_vol","sharpe0","maxDD","cum_end"]:
        if k in rc_stats:
            log(f"  {k}: {rc_stats[k]}")

if __name__ == "__main__":
    main()


Backtest: Top20 by rolling tstat(alpha/se) (FF5+MOM) -> next-month equal-weight returns + RiskControl(C)
PRICE_PATH : C:\Users\yongr\Project\merged_data_all_stocks\factors\price_month_end.parquet
SNAP_PATH  : C:\Users\yongr\Project\merged_data_all_stocks\factors\month_end_snapshot.parquet
FACTOR_PATH: C:\Users\yongr\Project\merged_data_all_stocks\factors\ff5_mom_factors_monthly.parquet
PERIOD     : 2016-10-31 .. 2025-09-30
LOOKBACK   : 24 months | TOP_K=20 | MIN_OBS=18 | SCORE_MODE=tstat
RISK(C)    : regime(3m MKT<0 or WML<0) + betaCap(CMA<-0.8 or |MKT|>0.9) -> invest_ratio=0.5
OUT_DIR    : C:\Users\yongr\Project\merged_data_all_stocks\factors\bt_top20_tstat_ff5mom_lb24_riskC_201610_202509
[UNIVERSE] latest MonthEnd=2025-09-30 codes=3,428 (MAX_CODES=None)
[PANEL] rows=363,954 codes=3,428 months=117
[STATS] rows=269,234 months=87
[BT] months=87 start=2018-07-31 00:00:00 end=2025-09-30 00:00:00
----------------------------------------------------------------------------------------------

In [7]:
# -*- coding: utf-8 -*-
"""
年次（10/1）割安高質バックテスト（100株単位・税引後）に
FF5+MOMのリスク制御（C：レジーム減速＋β制約、riskoff=0.5）を統合。

【最小改修で高速化】
- prices_df を Code ごとの辞書 prices_by_code に前処理
- get_near_price() は巨大DFを毎回フィルタせず、prices_by_code[code]のみを探索

税金:
- 簡易：年次（10/1→翌10/1）の最終損益にのみ課税

出力（統合前後＋診断）:
- annual_returns_raw.csv / annual_returns_riskcontrol.csv
- cumulative_curve_raw.csv / cumulative_curve_riskcontrol.csv
- performance_summary_raw.csv / performance_summary_riskcontrol.csv
- regression_ff5mom_raw.csv / regression_ff5mom_riskcontrol.csv
- diag_invest_ratio_monthly.csv

依存:
  pip install pandas numpy pyarrow tqdm
"""

import warnings
warnings.filterwarnings("ignore")

import os
import logging
from datetime import datetime
from pathlib import Path
from typing import Dict, Tuple, List, Optional

import numpy as np
import pandas as pd
from tqdm import tqdm


# ===================================
# ロギング設定
# ===================================
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler("backtest_october_unit_with_ff5mom_riskcontrol.log", encoding="utf-8"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)


# ===================================
# パス（あなたの環境）
# ===================================
FACTORS_DIR = Path(r"C:\Users\yongr\Project\merged_data_all_stocks\factors")
FF5MOM_FACTOR_PATH = FACTORS_DIR / "ff5_mom_factors_monthly.parquet"

CACHE_FILE = "topix_quarterly_statements.csv"
OHLCV_DIR = "./OHLCV_Adjusted"

OUT_DIR = FACTORS_DIR / "bt_october_unit_with_ff5mom_riskcontrol"
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_ANNUAL_RAW = OUT_DIR / "annual_returns_raw.csv"
OUT_ANNUAL_RC  = OUT_DIR / "annual_returns_riskcontrol.csv"

OUT_CURVE_RAW  = OUT_DIR / "cumulative_curve_raw.csv"
OUT_CURVE_RC   = OUT_DIR / "cumulative_curve_riskcontrol.csv"

OUT_PERF_RAW   = OUT_DIR / "performance_summary_raw.csv"
OUT_PERF_RC    = OUT_DIR / "performance_summary_riskcontrol.csv"

OUT_REG_RAW    = OUT_DIR / "regression_ff5mom_raw.csv"
OUT_REG_RC     = OUT_DIR / "regression_ff5mom_riskcontrol.csv"

OUT_IR_DIAG    = OUT_DIR / "diag_invest_ratio_monthly.csv"


# ===================================
# 税・単位株
# ===================================
TAX_RATE = 0.20315
UNIT_SHARES = 100
INITIAL_CAPITAL = 10_000_000


# ===================================
# リスク制御パラメータ（指定）
# ===================================
RISKOFF_RATIO = 0.5

REGIME_LOOKBACK_M = 3
REGIME_OFF_IF_SUM_MKT_LT = 0.0
REGIME_OFF_IF_SUM_WML_LT = 0.0

BETA_CAP_CMA_LT = -0.8
BETA_CAP_ABS_MKT_GT = 0.9

BETA_EST_WINDOW_M = 12
BETA_EST_MIN_OBS = 10

FACTOR_COLS = ["MKT", "SMB", "HML", "RMW", "CMA", "WML"]

BAD_CODE_STRINGS = {"None", "nan", "", "NaN", "NULL", "null"}


# ===================================
# ユーティリティ
# ===================================
def normalize_code(code) -> str:
    if code is None:
        return ""
    s = str(code).strip()
    if s in BAD_CODE_STRINGS:
        return ""
    return s


def ols_alpha_beta(y: np.ndarray, X: np.ndarray) -> Tuple[float, np.ndarray, float]:
    n = len(y)
    if n < 3:
        return np.nan, np.full(X.shape[1], np.nan), np.nan

    X1 = np.column_stack([np.ones(n), X])
    XtX = X1.T @ X1
    try:
        inv = np.linalg.inv(XtX)
    except np.linalg.LinAlgError:
        inv = np.linalg.pinv(XtX)
    b = inv @ (X1.T @ y)

    yhat = X1 @ b
    resid = y - yhat
    sse = float(np.sum(resid**2))
    sst = float(np.sum((y - y.mean())**2))
    r2 = np.nan if sst <= 0 else (1.0 - sse / sst)

    alpha = float(b[0])
    betas = b[1:].astype(float)
    return alpha, betas, float(r2)


def compute_drawdown(cum: pd.Series) -> pd.Series:
    peak = cum.cummax()
    return cum / peak - 1.0


def perf_stats(annual_ret: pd.Series) -> Dict:
    r = annual_ret.dropna().astype(float)
    if r.empty:
        return {"n_years": 0, "CAGR": np.nan, "ann_mean": np.nan, "ann_vol": np.nan, "sharpe0": np.nan, "maxDD": np.nan, "cum_end": np.nan}

    n = len(r)
    cum = (1.0 + r).cumprod()
    years = n
    cagr = float(cum.iloc[-1] ** (1/years) - 1.0) if years > 0 else np.nan
    ann_mean = float(r.mean())
    ann_vol = float(r.std(ddof=1)) if n >= 2 else np.nan
    sharpe0 = float(ann_mean / ann_vol) if ann_vol and ann_vol > 0 else np.nan
    dd = compute_drawdown(cum)
    maxdd = float(dd.min())

    return {
        "n_years": int(n),
        "CAGR": cagr,
        "ann_mean": ann_mean,
        "ann_vol": ann_vol,
        "sharpe0": sharpe0,
        "maxDD": maxdd,
        "cum_end": float(cum.iloc[-1]),
    }


# ===================================
# A. 財務データ読み込み（列名自動判定）
# ===================================
def load_financial_data(cache_filename: str = CACHE_FILE) -> pd.DataFrame:
    if not os.path.exists(cache_filename):
        logger.error(f"財務データファイル '{cache_filename}' が見つかりません")
        return pd.DataFrame()

    try:
        df = pd.read_csv(cache_filename, encoding="utf-8-sig", parse_dates=["DisclosedDate"], low_memory=False)
        logger.info(f"財務データ読み込み成功: {len(df):,}件")

        column_mapping = {
            "IssuedShareTotal": ["IssuedShareTotal", "NumberOfIssuedAndOutstandingSharesAtTheEndOfFiscalYearIncludingTreasuryStock"],
            "Equity": ["Equity", "NetAssets", "TotalEquity"],
            "Profit": ["Profit", "NetIncome", "ProfitAttributableToOwnersOfParent"],
        }

        available_cols = df.columns.tolist()
        required_columns = {}
        for target_col, possible_names in column_mapping.items():
            found = False
            for possible_name in possible_names:
                if possible_name in available_cols:
                    required_columns[target_col] = possible_name
                    found = True
                    break
            if not found:
                required_columns[target_col] = None

        rename_dict = {v: k for k, v in required_columns.items() if v is not None}
        df = df.rename(columns=rename_dict)

        for col in ["IssuedShareTotal", "Equity", "Profit"]:
            if col not in df.columns:
                df[col] = 0

        base_cols = ["Code", "DisclosedDate"]
        if "CompanyName" in df.columns:
            base_cols.append("CompanyName")
        final_cols = base_cols + ["Profit", "Equity", "IssuedShareTotal"]
        df = df[final_cols].copy()

        logger.info(f"使用列: {df.columns.tolist()}")
        return df

    except Exception as e:
        logger.error(f"財務データ読み込みエラー: {e}", exc_info=True)
        return pd.DataFrame()


# ===================================
# B. 株価データ読み込み
# ===================================
def load_existing_price_data(ohlcv_dir: str = OHLCV_DIR) -> pd.DataFrame:
    if not os.path.exists(ohlcv_dir):
        logger.error(f"株価データディレクトリ '{ohlcv_dir}' が見つかりません。")
        return pd.DataFrame()

    csv_files = sorted([f for f in os.listdir(ohlcv_dir)
                        if f.startswith("OHLCV_Adjusted_") and f.endswith(".csv") and f != "OHLCV_Adjusted_TOPIX.csv"])

    if not csv_files:
        logger.error(f"ディレクトリ '{ohlcv_dir}' 内にCSVファイルが見つかりません。")
        return pd.DataFrame()

    logger.info(f"株価ファイル数: {len(csv_files)}個")

    all_dataframes = []
    usecols = ["Date", "Ticker", "AdjustmentClose"]

    for csv_file in tqdm(csv_files, desc="株価ファイル読み込み中"):
        file_path = os.path.join(ohlcv_dir, csv_file)
        try:
            df = pd.read_csv(
                file_path,
                usecols=usecols,
                parse_dates=["Date"],
                dtype={"Ticker": "Int64", "AdjustmentClose": "float32"},
            )
            df = df.drop_duplicates(subset=["Ticker", "Date"], keep="first")
            all_dataframes.append(df)
        except Exception as e:
            logger.warning(f"ファイル読み込みエラー ({csv_file}): {e}")
            continue

    if not all_dataframes:
        return pd.DataFrame()

    logger.info("全ファイルを結合中...")
    df_all = pd.concat(all_dataframes, ignore_index=True)
    logger.info(f"結合完了: {len(df_all):,}件")

    df_all = df_all.rename(columns={"Ticker": "Code", "AdjustmentClose": "Close"})
    df_all["Code"] = df_all["Code"].astype("str").str.replace("<NA>", "0").str.zfill(4)
    df_all = df_all.dropna(subset=["Close"])

    logger.info("重複除去 & ソート中...")
    df_all = df_all.sort_values(["Code", "Date"])
    df_all = df_all.drop_duplicates(subset=["Code", "Date"], keep="first")
    logger.info(f"重複除去後: {len(df_all):,}件")

    return df_all


# ===== PATCH START: 高速化（prices_by_code作成） =====
def build_prices_by_code(prices_df: pd.DataFrame) -> Dict[str, pd.DataFrame]:
    """
    Code -> その銘柄だけの価格DF(Date, Close)。Date昇順、indexはDateにしておく。
    これにより get_near_price が巨大DF全体フィルタをしなくて済む。
    """
    d = {}
    # 事前に必要列だけにして軽量化
    tmp = prices_df[["Code", "Date", "Close"]].copy()
    tmp["Code"] = tmp["Code"].astype(str).map(normalize_code)
    tmp = tmp[tmp["Code"] != ""].copy()
    tmp = tmp.sort_values(["Code", "Date"])

    for code, g in tmp.groupby("Code", sort=False):
        gg = g[["Date", "Close"]].drop_duplicates(subset=["Date"], keep="last").sort_values("Date").copy()
        gg = gg.set_index("Date", drop=False)  # Dateでスライスしやすくする
        d[code] = gg
    logger.info(f"prices_by_code built: {len(d):,} codes")
    return d


def get_near_price_fast(prices_by_code: Dict[str, pd.DataFrame],
                        code: str,
                        ref_date: pd.Timestamp,
                        kind: str = "last") -> Optional[float]:
    """
    ref_date前後±5日で、first/lastのCloseを取る（高速版）
    """
    code = normalize_code(code)
    if not code or code not in prices_by_code:
        return None
    g = prices_by_code[code]
    # Date indexで±5日を切る
    lo = ref_date - pd.Timedelta(days=5)
    hi = ref_date + pd.Timedelta(days=5)
    w = g.loc[(g["Date"] >= lo) & (g["Date"] <= hi)]
    if w.empty:
        return None
    return float(w.iloc[0]["Close"]) if kind == "first" else float(w.iloc[-1]["Close"])
# ===== PATCH END =====


# ===================================
# C. 財務指標計算（あなたの既存ロジック）
# ===================================
def safe_code_to_int(code_series: pd.Series) -> pd.Series:
    cleaned = code_series.astype(str).str.replace(r"\D", "", regex=True)
    cleaned = cleaned.replace("", "0")
    return pd.to_numeric(cleaned, errors="coerce").fillna(0).astype("int64")


def calculate_market_metrics_fast_chunked(statements_df: pd.DataFrame,
                                          prices_df: pd.DataFrame,
                                          chunk_size: int = 200) -> pd.DataFrame:
    logger.info("時価総額・PBR・ROE計算中（チャンク処理版）...")

    if prices_df.empty:
        logger.error("株価データが空です。")
        return pd.DataFrame()

    req_cols = {"Code", "Date", "Close"}
    if not req_cols.issubset(set(prices_df.columns)):
        logger.error(f"株価データに必要な列が存在しません。存在する列: {prices_df.columns.tolist()}")
        return pd.DataFrame()

    statements_df = statements_df.copy()
    statements_df["Profit"] = pd.to_numeric(statements_df["Profit"], errors="coerce").fillna(0)
    statements_df["Equity"] = pd.to_numeric(statements_df["Equity"], errors="coerce").fillna(0)
    statements_df["IssuedShareTotal"] = pd.to_numeric(statements_df["IssuedShareTotal"], errors="coerce").fillna(1)

    statements_df = statements_df[(statements_df["Equity"] > 0) & (statements_df["IssuedShareTotal"] > 0)]
    logger.info(f"有効な財務データ: {len(statements_df):,}件")

    statements_df["Code_int"] = safe_code_to_int(statements_df["Code"])
    prices_df = prices_df.copy()
    prices_df["Code_int"] = safe_code_to_int(prices_df["Code"])

    statements_df = statements_df[statements_df["Code_int"] > 0]
    prices_df = prices_df[prices_df["Code_int"] > 0]

    statements_df = statements_df.sort_values(["Code_int", "DisclosedDate"]).reset_index(drop=True)
    prices_df = prices_df.sort_values(["Code_int", "Date"]).reset_index(drop=True)

    statements_df = statements_df.drop_duplicates(subset=["Code_int", "DisclosedDate"], keep="first")
    prices_df = prices_df.drop_duplicates(subset=["Code_int", "Date"], keep="first")

    logger.info(f"ソート・重複除去後: 財務 {len(statements_df):,}件, 株価 {len(prices_df):,}件")

    statements_groups = list(statements_df.groupby("Code_int"))
    prices_dict = {code: group for code, group in prices_df.groupby("Code_int")}

    merged_list = []
    num_chunks = (len(statements_groups) + chunk_size - 1) // chunk_size

    for chunk_idx in tqdm(range(num_chunks), desc="マージ処理"):
        start_idx = chunk_idx * chunk_size
        end_idx = min((chunk_idx + 1) * chunk_size, len(statements_groups))
        chunk_groups = statements_groups[start_idx:end_idx]

        for code, stmt_code in chunk_groups:
            if code not in prices_dict:
                continue
            price_code = prices_dict[code]
            if len(price_code) == 0:
                continue

            stmt_code = stmt_code.sort_values("DisclosedDate").reset_index(drop=True)
            price_code = price_code.sort_values("Date").reset_index(drop=True)

            try:
                merged = pd.merge_asof(
                    stmt_code,
                    price_code[["Date", "Close"]],
                    left_on="DisclosedDate",
                    right_on="Date",
                    direction="backward",
                    tolerance=pd.Timedelta(days=10),
                )
                if not merged.empty:
                    merged_list.append(merged)
            except Exception:
                continue

    if not merged_list:
        logger.error("マージ結果が空です")
        return pd.DataFrame()

    df_merged = pd.concat(merged_list, ignore_index=True)
    df_merged = df_merged.dropna(subset=["Close"])
    logger.info(f"マージ完了: {len(df_merged):,}件")

    df_merged["MarketCap"] = df_merged["Close"] * df_merged["IssuedShareTotal"]
    df_merged["PBR"] = df_merged["MarketCap"] / df_merged["Equity"]
    df_merged["ROE"] = (df_merged["Profit"] / df_merged["Equity"]) * 100

    result_cols = ["Code", "DisclosedDate", "Close", "MarketCap", "PBR", "ROE", "Date"]
    if "CompanyName" in df_merged.columns:
        result_cols.insert(1, "CompanyName")

    result_df = df_merged[result_cols].copy()
    result_df = result_df.rename(columns={"Close": "StockPrice", "Date": "PriceDate"})

    mask = (
        (result_df["PBR"] > 0) &
        (result_df["PBR"] < 50) &
        (result_df["ROE"] > -100) &
        (result_df["ROE"] < 100) &
        (result_df["MarketCap"] > 1_000_000_000)
    )
    result_df = result_df[mask].copy()

    logger.info(f"計算完了: {len(result_df):,}件")
    return result_df


# ===================================
# D. 100株単位ポート構築
# ===================================
def build_unit_share_portfolio(stock_candidates: pd.DataFrame,
                               target_positions: int = 20,
                               initial_capital: float = 10_000_000) -> dict:
    if len(stock_candidates) == 0:
        return {"stocks": [], "shares": [], "prices": [], "amounts": []}

    selected = stock_candidates.head(target_positions).copy()
    capital_per_stock = initial_capital / len(selected)

    stocks, shares_list, prices_list, amounts_list = [], [], [], []

    for _, row in selected.iterrows():
        code = row["Code"]
        price = row["StockPrice"]
        required_amount = price * UNIT_SHARES

        if required_amount <= capital_per_stock:
            shares = int(capital_per_stock // required_amount) * UNIT_SHARES
            if shares > 0:
                stocks.append(code)
                shares_list.append(shares)
                prices_list.append(price)
                amounts_list.append(shares * price)

    return {"stocks": stocks, "shares": shares_list, "prices": prices_list, "amounts": amounts_list}


# ===================================
# 月次区切り
# ===================================
def make_month_ends(start: pd.Timestamp, end: pd.Timestamp) -> List[pd.Timestamp]:
    m0 = pd.Timestamp(start.year, start.month, 1) + pd.offsets.MonthEnd(0)
    m1 = pd.Timestamp(end.year, end.month, 1) + pd.offsets.MonthEnd(0)
    months = pd.date_range(m0, m1, freq="M")
    return [pd.Timestamp(x).normalize() for x in months]


# ===================================
# FF5+MOM 因子
# ===================================
def load_ff5mom_factors_monthly() -> pd.DataFrame:
    fac = pd.read_parquet(FF5MOM_FACTOR_PATH).copy()
    fac["MonthEnd"] = pd.to_datetime(fac["MonthEnd"], errors="coerce").dt.normalize()
    need = ["MonthEnd"] + FACTOR_COLS
    missing = [c for c in need if c not in fac.columns]
    if missing:
        raise KeyError(f"FF5MOM factors missing columns: {missing} in {FF5MOM_FACTOR_PATH}")
    fac = fac[need].sort_values("MonthEnd").reset_index(drop=True)
    return fac


def compute_regime_off(fac: pd.DataFrame, month_end: pd.Timestamp) -> Tuple[bool, Dict]:
    fac2 = fac.set_index("MonthEnd")
    idx = fac2.index[fac2.index < month_end]
    if len(idx) < REGIME_LOOKBACK_M:
        return False, {"regime_ready": False}

    win = idx[-REGIME_LOOKBACK_M:]
    s_mkt = float(pd.to_numeric(fac2.loc[win, "MKT"], errors="coerce").sum())
    s_wml = float(pd.to_numeric(fac2.loc[win, "WML"], errors="coerce").sum())
    off = (s_mkt < REGIME_OFF_IF_SUM_MKT_LT) or (s_wml < REGIME_OFF_IF_SUM_WML_LT)
    return bool(off), {"regime_ready": True, "sum_MKT_3m": s_mkt, "sum_WML_3m": s_wml}


def estimate_port_beta(monthly_port_rets: pd.DataFrame, fac: pd.DataFrame, month_end: pd.Timestamp) -> Tuple[bool, Dict]:
    fac2 = fac.set_index("MonthEnd")
    idx = fac2.index[fac2.index < month_end]
    if len(idx) < BETA_EST_WINDOW_M:
        return False, {"beta_ready": False}

    win = idx[-BETA_EST_WINDOW_M:]

    # ===== PATCH START: MonthEnd重複をgroupby集約して一意化（one_to_one merge維持） =====
    m = monthly_port_rets.copy()
    m["MonthEnd"] = pd.to_datetime(m["MonthEnd"], errors="coerce").dt.normalize()
    m = m.dropna(subset=["MonthEnd", "port_ret"]).copy()

    # 同一MonthEndが複数行ある場合、(1+r)の積-1 で「その月の合成リターン」にして 1行にする
    # -> MonthEnd を一意化し、merge(validate="one_to_one") を満たす
    m = (
        m.groupby("MonthEnd", as_index=False)["port_ret"]
         .apply(lambda s: (1.0 + s.astype(float)).prod() - 1.0)
    )
    # ===== PATCH END =====

    df = m[m["MonthEnd"].isin(win)].merge(
        fac, on="MonthEnd", how="left", validate="one_to_one"
    ).dropna(subset=["port_ret"] + FACTOR_COLS)

    if len(df) < BETA_EST_MIN_OBS:
        return False, {"beta_ready": False, "n_obs": int(len(df))}

    y = df["port_ret"].to_numpy(dtype=float)
    X = df[FACTOR_COLS].to_numpy(dtype=float)
    alpha, betas, r2 = ols_alpha_beta(y, X)

    out = {"beta_ready": True, "n_obs": int(len(df)), "r2": float(r2), "alpha": float(alpha)}
    for c, b in zip(FACTOR_COLS, betas):
        out[f"beta_{c}"] = float(b)
    return True, out


def decide_invest_ratio(month_end: pd.Timestamp,
                        fac: pd.DataFrame,
                        port_hist: pd.DataFrame) -> Tuple[float, Dict]:
    invest_ratio = 1.0
    diag = {
        "MonthEnd": month_end,
        "invest_ratio": 1.0,
        "regime_off": False,
        "beta_off": False,
        "regime_ready": False,
        "beta_ready": False,
        "sum_MKT_3m": np.nan,
        "sum_WML_3m": np.nan,
        "beta_MKT": np.nan,
        "beta_CMA": np.nan,
        "beta_n_obs": np.nan,
        "beta_r2": np.nan,
    }

    off_reg, info = compute_regime_off(fac, month_end)
    diag["regime_off"] = bool(off_reg)
    diag["regime_ready"] = bool(info.get("regime_ready", False))
    diag["sum_MKT_3m"] = info.get("sum_MKT_3m", np.nan)
    diag["sum_WML_3m"] = info.get("sum_WML_3m", np.nan)
    if off_reg:
        invest_ratio = min(invest_ratio, RISKOFF_RATIO)

    ok_beta, binfo = estimate_port_beta(port_hist, fac, month_end)
    diag["beta_ready"] = bool(binfo.get("beta_ready", False))
    if binfo.get("beta_ready", False):
        b_mkt = float(binfo.get("beta_MKT", np.nan))
        b_cma = float(binfo.get("beta_CMA", np.nan))
        diag["beta_MKT"] = b_mkt
        diag["beta_CMA"] = b_cma
        diag["beta_n_obs"] = float(binfo.get("n_obs", np.nan))
        diag["beta_r2"] = float(binfo.get("r2", np.nan))

        off_beta = (b_cma < BETA_CAP_CMA_LT) or (abs(b_mkt) > BETA_CAP_ABS_MKT_GT)
        diag["beta_off"] = bool(off_beta)
        if off_beta:
            invest_ratio = min(invest_ratio, RISKOFF_RATIO)

    diag["invest_ratio"] = float(invest_ratio)
    return float(invest_ratio), diag


# ===================================
# 月次リターン列（高速版 get_near_price_fast を利用）
# ===================================
def compute_monthly_portfolio_returns_with_riskcontrol_fast(
    portfolio: dict,
    start_date: pd.Timestamp,
    end_date: pd.Timestamp,
    prices_by_code: Dict[str, pd.DataFrame],
    fac: pd.DataFrame
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    stocks = portfolio["stocks"]
    shares = portfolio["shares"]
    if not stocks:
        return pd.DataFrame(), pd.DataFrame()

    month_ends = make_month_ends(start_date, end_date)
    boundaries = [start_date] + [me for me in month_ends if (me > start_date) and (me < end_date)] + [end_date]

    rows = []
    ir_rows = []

    port_hist = pd.DataFrame(columns=["MonthEnd", "port_ret"])

    for j in range(len(boundaries) - 1):
        d0 = boundaries[j]
        d1 = boundaries[j + 1]

        me = pd.Timestamp(d0.year, d0.month, 1) + pd.offsets.MonthEnd(0)
        me = pd.Timestamp(me).normalize()

        start_vals = []
        end_vals = []
        for i, code in enumerate(stocks):
            p0 = get_near_price_fast(prices_by_code, code, d0, kind="first")
            p1 = get_near_price_fast(prices_by_code, code, d1, kind="last")
            if p0 is None or p1 is None:
                continue
            sh = shares[i]
            start_vals.append(sh * p0)
            end_vals.append(sh * p1)

        if len(start_vals) == 0:
            continue

        start_v = float(np.sum(start_vals))
        end_v = float(np.sum(end_vals))
        stock_ret = (end_v / start_v) - 1.0

        port_hist = pd.concat([port_hist, pd.DataFrame([{"MonthEnd": me, "port_ret": stock_ret}])], ignore_index=True)

        invest_ratio, diag = decide_invest_ratio(me, fac, port_hist)
        ir_rows.append(diag)

        total_ret = invest_ratio * stock_ret

        rows.append({
            "MonthEnd": me,
            "period_start": d0,
            "period_end": d1,
            "port_ret_stock": stock_ret,
            "invest_ratio": invest_ratio,
            "port_ret_total": total_ret,
        })

    monthly_df = pd.DataFrame(rows)
    ir_diag_df = pd.DataFrame(ir_rows)
    return monthly_df, ir_diag_df


# ===================================
# 年次バックテスト（RAW / RiskControl）
# ===================================
def build_long_candidates(enhanced_financial_data: pd.DataFrame, rebalance_date: pd.Timestamp) -> pd.DataFrame:
    current_data = enhanced_financial_data[enhanced_financial_data["DisclosedDate"] <= rebalance_date].copy()
    current_data = current_data.sort_values("DisclosedDate").groupby("Code").tail(1)

    if len(current_data) < 100:
        return pd.DataFrame()

    current_data["PBR_Rank"] = current_data["PBR"].rank(method="first", ascending=True)
    current_data["ROE_Rank"] = current_data["ROE"].rank(method="first", ascending=False)

    current_data["PBR_Quartile"] = pd.qcut(current_data["PBR_Rank"], q=4, labels=[1, 2, 3, 4])
    current_data["ROE_Quartile"] = pd.qcut(current_data["ROE_Rank"], q=4, labels=[1, 2, 3, 4])

    long_candidates = current_data[
        (current_data["PBR_Quartile"] == 1) &
        (current_data["ROE_Quartile"] == 4)
    ].nsmallest(50, "PBR")

    return long_candidates


def annual_return_from_monthly(monthly_rets: pd.Series) -> float:
    if monthly_rets.empty:
        return 0.0
    return float((1.0 + monthly_rets).prod() - 1.0)


def apply_tax_annual(gross_return: float, initial_capital: float) -> Tuple[float, float]:
    profit = gross_return * initial_capital
    taxable = max(profit, 0.0)
    tax = taxable * TAX_RATE
    net_profit = profit - tax
    net_return = net_profit / initial_capital
    tax_rate_total = tax / initial_capital
    return float(net_return), float(tax_rate_total)


def run_annual_backtest_with_and_without_riskcontrol_fast(
    enhanced_financial_data: pd.DataFrame,
    prices_df: pd.DataFrame,
    prices_by_code: Dict[str, pd.DataFrame],
    fac: pd.DataFrame,
    initial_capital: float = INITIAL_CAPITAL
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    rebalance_dates = [pd.Timestamp(f"{year}-10-01") for year in range(2016, 2026)]

    results_raw = []
    results_rc = []
    diag_ir_all = []

    for i, rebalance_date in enumerate(tqdm(rebalance_dates[:-1], desc="10月1日リバランス（統合・高速）")):
        next_rebalance = rebalance_dates[i + 1]

        long_candidates = build_long_candidates(enhanced_financial_data, rebalance_date)
        if long_candidates.empty:
            logger.warning(f"{rebalance_date}: データ不足でスキップ")
            continue

        long_portfolio = build_unit_share_portfolio(
            long_candidates,
            target_positions=20,
            initial_capital=initial_capital
        )

        n_long = len(long_portfolio["stocks"])
        inv_amount = float(np.sum(long_portfolio["amounts"])) if n_long > 0 else 0.0
        inv_ratio_raw = inv_amount / initial_capital if initial_capital > 0 else 0.0

        logger.info(f"{rebalance_date.strftime('%Y-%m')}: long={n_long} invest={inv_amount:,.0f} ratio={inv_ratio_raw:.3f}")

        # --- RAW（年次 start->end の価格で計算）
        total_profit = 0.0
        total_start_value = 0.0

        for j, code in enumerate(long_portfolio["stocks"]):
            sh = long_portfolio["shares"][j]
            p0 = long_portfolio["prices"][j]
            p1 = get_near_price_fast(prices_by_code, code, next_rebalance, kind="last")
            if p1 is None:
                continue
            start_value = sh * p0
            end_value = sh * p1
            total_start_value += start_value
            total_profit += (end_value - start_value)

        gross_return_raw = (total_profit / initial_capital) if initial_capital > 0 else 0.0
        net_return_raw, tax_rate_raw = apply_tax_annual(gross_return_raw, initial_capital)

        results_raw.append({
            "date": next_rebalance,
            "strategy_return_gross": gross_return_raw,
            "strategy_return_net": net_return_raw,
            "tax": tax_rate_raw,
            "long_count": n_long,
            "investment_ratio": inv_ratio_raw,
        })

        # --- RiskControl（月次で縮尺）
        monthly_df, ir_df = compute_monthly_portfolio_returns_with_riskcontrol_fast(
            long_portfolio, rebalance_date, next_rebalance, prices_by_code, fac
        )

        if monthly_df.empty:
            gross_return_rc = 0.0
            inv_ratio_rc_avg = 0.0
        else:
            gross_return_rc = annual_return_from_monthly(monthly_df["port_ret_total"])
            inv_ratio_rc_avg = float(monthly_df["invest_ratio"].mean())

        net_return_rc, tax_rate_rc = apply_tax_annual(gross_return_rc, initial_capital)

        results_rc.append({
            "date": next_rebalance,
            "strategy_return_gross": gross_return_rc,
            "strategy_return_net": net_return_rc,
            "tax": tax_rate_rc,
            "long_count": n_long,
            "investment_ratio": inv_ratio_rc_avg,
        })

        if not ir_df.empty:
            ir_df = ir_df.copy()
            ir_df["rebalance_start"] = rebalance_date
            ir_df["rebalance_end"] = next_rebalance
            diag_ir_all.append(ir_df)

    df_raw = pd.DataFrame(results_raw)
    df_rc = pd.DataFrame(results_rc)
    diag_ir = pd.concat(diag_ir_all, ignore_index=True) if len(diag_ir_all) else pd.DataFrame()

    return df_raw, df_rc, diag_ir


# ===================================
# 年次回帰（FF5+MOM）
# ===================================
def build_annual_factor_from_monthly(fac: pd.DataFrame, start_date: pd.Timestamp, end_date: pd.Timestamp) -> Dict:
    fac2 = fac.copy()
    fac2 = fac2[(fac2["MonthEnd"] >= (start_date + pd.offsets.MonthEnd(0))) &
                (fac2["MonthEnd"] <= (end_date + pd.offsets.MonthEnd(-1)))].copy()
    out = {}
    for c in FACTOR_COLS:
        s = pd.to_numeric(fac2[c], errors="coerce").dropna()
        out[c] = float((1.0 + s).prod() - 1.0) if len(s) else np.nan
    return out


def run_factor_regression(results_df: pd.DataFrame, fac: pd.DataFrame) -> pd.DataFrame:
    if results_df.empty:
        return pd.DataFrame()

    rows = []
    for _, row in results_df.iterrows():
        end_date = pd.Timestamp(row["date"])
        start_date = end_date - pd.DateOffset(years=1)
        ann_fac = build_annual_factor_from_monthly(fac, start_date, end_date)
        rec = {"date": end_date}
        rec.update(ann_fac)
        rec["y"] = float(row["strategy_return_net"])
        rows.append(rec)

    df = pd.DataFrame(rows).dropna(subset=["y"] + FACTOR_COLS).copy()
    if len(df) < 3:
        return pd.DataFrame([{
            "n_years": int(len(df)),
            "R2": np.nan,
            "alpha": np.nan,
            **{f"beta_{c}": np.nan for c in FACTOR_COLS}
        }])

    y = df["y"].to_numpy(dtype=float)
    X = df[FACTOR_COLS].to_numpy(dtype=float)
    alpha, betas, r2 = ols_alpha_beta(y, X)

    out = {"n_years": int(len(df)), "R2": float(r2), "alpha": float(alpha)}
    for c, b in zip(FACTOR_COLS, betas):
        out[f"beta_{c}"] = float(b)
    return pd.DataFrame([out])


# ===================================
# 保存（累積曲線など）
# ===================================
def save_annual_bundle(df: pd.DataFrame, out_annual_csv: Path, out_curve_csv: Path, out_perf_csv: Path):
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date").reset_index(drop=True)

    df.to_csv(out_annual_csv, index=False, encoding="utf-8-sig")

    r = df["strategy_return_net"].astype(float)
    cum = (1.0 + r).cumprod()
    dd = compute_drawdown(cum)
    curve = pd.DataFrame({"date": df["date"], "ret": r, "cum": cum, "dd": dd})
    curve.to_csv(out_curve_csv, index=False, encoding="utf-8-sig")

    stats = perf_stats(r)
    perf = pd.DataFrame([stats])
    perf.to_csv(out_perf_csv, index=False, encoding="utf-8-sig")


# ===================================
# Main
# ===================================
def main():
    logger.info("=" * 110)
    logger.info("10/1 年次「割安高質」バックテスト + FF5+MOM RiskControl(C) [FAST get_near_price]")
    logger.info("=" * 110)
    logger.info(f"FF5MOM_FACTOR_PATH: {FF5MOM_FACTOR_PATH}")
    logger.info(f"OUT_DIR: {OUT_DIR}")
    logger.info(f"Tax mode: simple annual tax only (TAX_RATE={TAX_RATE})")
    logger.info(f"RiskControl: regime(3m MKT<0 or WML<0) + beta(CMA<-0.8 or |MKT|>0.9) => riskoff={RISKOFF_RATIO}")
    logger.info(f"Beta estimation window: {BETA_EST_WINDOW_M} months (min_obs={BETA_EST_MIN_OBS})")

    statements_df = load_financial_data()
    if statements_df.empty:
        logger.error("財務データ読み込み失敗")
        return

    prices_df = load_existing_price_data()
    if prices_df.empty:
        logger.error("株価データ読み込み失敗")
        return

    fac = load_ff5mom_factors_monthly()

    # ===== PATCH START: prices_by_code作成（ここが高速化の肝） =====
    prices_by_code = build_prices_by_code(prices_df)
    # ===== PATCH END =====

    enhanced_financial_data = calculate_market_metrics_fast_chunked(statements_df, prices_df, chunk_size=200)
    if enhanced_financial_data.empty:
        logger.error("財務指標計算失敗")
        return

    df_raw, df_rc, diag_ir = run_annual_backtest_with_and_without_riskcontrol_fast(
        enhanced_financial_data, prices_df, prices_by_code, fac, initial_capital=INITIAL_CAPITAL
    )

    if df_raw.empty or df_rc.empty:
        logger.error("年次バックテスト結果が空です")
        return

    save_annual_bundle(df_raw, OUT_ANNUAL_RAW, OUT_CURVE_RAW, OUT_PERF_RAW)
    save_annual_bundle(df_rc,  OUT_ANNUAL_RC,  OUT_CURVE_RC,  OUT_PERF_RC)

    reg_raw = run_factor_regression(df_raw, fac)
    reg_rc = run_factor_regression(df_rc, fac)
    reg_raw.to_csv(OUT_REG_RAW, index=False, encoding="utf-8-sig")
    reg_rc.to_csv(OUT_REG_RC, index=False, encoding="utf-8-sig")

    if not diag_ir.empty:
        diag_ir["MonthEnd"] = pd.to_datetime(diag_ir["MonthEnd"])
        diag_ir.sort_values(["rebalance_start", "MonthEnd"], inplace=True)
        diag_ir.to_csv(OUT_IR_DIAG, index=False, encoding="utf-8-sig")

    logger.info("-" * 110)
    logger.info("✅ SAVED (RAW)")
    logger.info(f"  - {OUT_ANNUAL_RAW}")
    logger.info(f"  - {OUT_CURVE_RAW}")
    logger.info(f"  - {OUT_PERF_RAW}")
    logger.info(f"  - {OUT_REG_RAW}")
    logger.info("✅ SAVED (RISKCONTROL)")
    logger.info(f"  - {OUT_ANNUAL_RC}")
    logger.info(f"  - {OUT_CURVE_RC}")
    logger.info(f"  - {OUT_PERF_RC}")
    logger.info(f"  - {OUT_REG_RC}")
    logger.info("✅ DIAGNOSTIC")
    logger.info(f"  - {OUT_IR_DIAG}")
    logger.info("-" * 110)

    raw_perf = pd.read_csv(OUT_PERF_RAW).iloc[0].to_dict()
    rc_perf  = pd.read_csv(OUT_PERF_RC).iloc[0].to_dict()
    logger.info("[PERF RAW] " + " | ".join([f"{k}={raw_perf.get(k)}" for k in ["n_years","CAGR","ann_mean","ann_vol","sharpe0","maxDD","cum_end"]]))
    logger.info("[PERF RC ] " + " | ".join([f"{k}={rc_perf.get(k)}"  for k in ["n_years","CAGR","ann_mean","ann_vol","sharpe0","maxDD","cum_end"]]))

if __name__ == "__main__":
    main()


2026-01-25 05:12:48,480 - INFO - ==============================================================================================================
2026-01-25 05:12:48,481 - INFO - 10/1 年次「割安高質」バックテスト + FF5+MOM RiskControl(C) [FAST get_near_price]
2026-01-25 05:12:48,482 - INFO - ==============================================================================================================
2026-01-25 05:12:48,483 - INFO - FF5MOM_FACTOR_PATH: C:\Users\yongr\Project\merged_data_all_stocks\factors\ff5_mom_factors_monthly.parquet
2026-01-25 05:12:48,484 - INFO - OUT_DIR: C:\Users\yongr\Project\merged_data_all_stocks\factors\bt_october_unit_with_ff5mom_riskcontrol
2026-01-25 05:12:48,484 - INFO - Tax mode: simple annual tax only (TAX_RATE=0.20315)
2026-01-25 05:12:48,484 - INFO - RiskControl: regime(3m MKT<0 or WML<0) + beta(CMA<-0.8 or |MKT|>0.9) => riskoff=0.5
2026-01-25 05:12:48,485 - INFO - Beta estimation window: 12 months (min_obs=10)
2026-01-25 05:12:49,098 - INFO - 財務データ読み込み成功: 64,222件
2

In [8]:
# -*- coding: utf-8 -*-
"""
年次（10/1）割安高質バックテスト（100株単位・税引後）に
FF5+MOMのリスク制御（C：レジーム減速＋β制約、riskoff=0.5）を統合。

【最小改修で高速化】
- prices_df を Code ごとの辞書 prices_by_code に前処理
- get_near_price() は巨大DFを毎回フィルタせず、prices_by_code[code]のみを探索

税金:
- 簡易：年次（10/1→翌10/1）の最終損益にのみ課税

出力（統合前後＋診断）:
- annual_returns_raw.csv / annual_returns_riskcontrol.csv
- cumulative_curve_raw.csv / cumulative_curve_riskcontrol.csv
- performance_summary_raw.csv / performance_summary_riskcontrol.csv
- regression_ff5mom_raw.csv / regression_ff5mom_riskcontrol.csv
- diag_invest_ratio_monthly.csv

【追加出力（既存は変更しない）】
- daily_equity_curve_raw.csv / daily_equity_curve_riskcontrol.csv
- daily_mdd_summary.csv
- daily_drawdown_events_raw.csv / daily_drawdown_events_riskcontrol.csv
- daily_drawdown_events_max_summary.csv

依存:
  pip install pandas numpy pyarrow tqdm
"""

import warnings
warnings.filterwarnings("ignore")

import os
import logging
from pathlib import Path
from typing import Dict, Tuple, List, Optional

import numpy as np
import pandas as pd
from tqdm import tqdm


# ===================================
# ロギング設定
# ===================================
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler("backtest_october_unit_with_ff5mom_riskcontrol.log", encoding="utf-8"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)


# ===================================
# パス（あなたの環境）
# ===================================
FACTORS_DIR = Path(r"C:\Users\yongr\Project\merged_data_all_stocks\factors")
FF5MOM_FACTOR_PATH = FACTORS_DIR / "ff5_mom_factors_monthly.parquet"

CACHE_FILE = "topix_quarterly_statements.csv"
OHLCV_DIR = "./OHLCV_Adjusted"

OUT_DIR = FACTORS_DIR / "bt_october_unit_with_ff5mom_riskcontrol"
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_ANNUAL_RAW = OUT_DIR / "annual_returns_raw.csv"
OUT_ANNUAL_RC  = OUT_DIR / "annual_returns_riskcontrol.csv"

OUT_CURVE_RAW  = OUT_DIR / "cumulative_curve_raw.csv"
OUT_CURVE_RC   = OUT_DIR / "cumulative_curve_riskcontrol.csv"

OUT_PERF_RAW   = OUT_DIR / "performance_summary_raw.csv"
OUT_PERF_RC    = OUT_DIR / "performance_summary_riskcontrol.csv"

OUT_REG_RAW    = OUT_DIR / "regression_ff5mom_raw.csv"
OUT_REG_RC     = OUT_DIR / "regression_ff5mom_riskcontrol.csv"

OUT_IR_DIAG    = OUT_DIR / "diag_invest_ratio_monthly.csv"

# --- 追加出力（日次）
OUT_DAILY_CURVE_RAW = OUT_DIR / "daily_equity_curve_raw.csv"
OUT_DAILY_CURVE_RC  = OUT_DIR / "daily_equity_curve_riskcontrol.csv"
OUT_DAILY_MDD_SUMMARY = OUT_DIR / "daily_mdd_summary.csv"

OUT_DD_EVENTS_RAW = OUT_DIR / "daily_drawdown_events_raw.csv"
OUT_DD_EVENTS_RC  = OUT_DIR / "daily_drawdown_events_riskcontrol.csv"
OUT_DD_EVENTS_MAX = OUT_DIR / "daily_drawdown_events_max_summary.csv"


# ===================================
# 税・単位株
# ===================================
TAX_RATE = 0.20315
UNIT_SHARES = 100
INITIAL_CAPITAL = 10_000_000


# ===================================
# リスク制御パラメータ（指定）
# ===================================
RISKOFF_RATIO = 0.5

REGIME_LOOKBACK_M = 3
REGIME_OFF_IF_SUM_MKT_LT = 0.0
REGIME_OFF_IF_SUM_WML_LT = 0.0

BETA_CAP_CMA_LT = -0.8
BETA_CAP_ABS_MKT_GT = 0.9

BETA_EST_WINDOW_M = 12
BETA_EST_MIN_OBS = 10

FACTOR_COLS = ["MKT", "SMB", "HML", "RMW", "CMA", "WML"]

BAD_CODE_STRINGS = {"None", "nan", "", "NaN", "NULL", "null"}


# ===================================
# ユーティリティ
# ===================================
def normalize_code(code) -> str:
    if code is None:
        return ""
    s = str(code).strip()
    if s in BAD_CODE_STRINGS:
        return ""
    return s


def ols_alpha_beta(y: np.ndarray, X: np.ndarray) -> Tuple[float, np.ndarray, float]:
    n = len(y)
    if n < 3:
        return np.nan, np.full(X.shape[1], np.nan), np.nan

    X1 = np.column_stack([np.ones(n), X])
    XtX = X1.T @ X1
    try:
        inv = np.linalg.inv(XtX)
    except np.linalg.LinAlgError:
        inv = np.linalg.pinv(XtX)
    b = inv @ (X1.T @ y)

    yhat = X1 @ b
    resid = y - yhat
    sse = float(np.sum(resid**2))
    sst = float(np.sum((y - y.mean())**2))
    r2 = np.nan if sst <= 0 else (1.0 - sse / sst)

    alpha = float(b[0])
    betas = b[1:].astype(float)
    return alpha, betas, float(r2)


def compute_drawdown(cum: pd.Series) -> pd.Series:
    peak = cum.cummax()
    return cum / peak - 1.0


def perf_stats(annual_ret: pd.Series) -> Dict:
    r = annual_ret.dropna().astype(float)
    if r.empty:
        return {"n_years": 0, "CAGR": np.nan, "ann_mean": np.nan, "ann_vol": np.nan, "sharpe0": np.nan, "maxDD": np.nan, "cum_end": np.nan}

    n = len(r)
    cum = (1.0 + r).cumprod()
    years = n
    cagr = float(cum.iloc[-1] ** (1/years) - 1.0) if years > 0 else np.nan
    ann_mean = float(r.mean())
    ann_vol = float(r.std(ddof=1)) if n >= 2 else np.nan
    sharpe0 = float(ann_mean / ann_vol) if ann_vol and ann_vol > 0 else np.nan
    dd = compute_drawdown(cum)
    maxdd = float(dd.min())

    return {
        "n_years": int(n),
        "CAGR": cagr,
        "ann_mean": ann_mean,
        "ann_vol": ann_vol,
        "sharpe0": sharpe0,
        "maxDD": maxdd,
        "cum_end": float(cum.iloc[-1]),
    }


def extract_max_drawdown_event_from_daily_curve(df_daily: pd.DataFrame, label: str = "") -> pd.DataFrame:
    """
    daily_equity_curve_*.csv 相当のDataFrame（列: Date, equity_total, dd_total）から
    「最大DDイベント」を抽出して 1行DataFrameで返す。

    peak_date     : 直前の最高値日
    trough_date   : DDが最小（最大下落）の底日
    recovery_date : peak_equityを再び上回った最初の日（無ければNaT）
    dd_min        : dd_totalの最小値（負の値）

    注意：
    - equity_totalは1から始まる累積（税は日次には反映しない運用で比較する想定）
    """
    if df_daily is None or df_daily.empty:
        return pd.DataFrame([{
            "label": label,
            "peak_date": pd.NaT,
            "trough_date": pd.NaT,
            "recovery_date": pd.NaT,
            "dd_min": np.nan,
            "peak_equity": np.nan,
            "trough_equity": np.nan,
            "recovery_equity": np.nan,
            "days_to_trough": np.nan,
            "days_to_recovery": np.nan,
            "dd_duration_days": np.nan,
        }])

    df = df_daily.copy()
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df = df.dropna(subset=["Date"]).sort_values("Date").reset_index(drop=True)

    for col in ["equity_total", "dd_total"]:
        if col not in df.columns:
            raise KeyError(f"daily curve missing required column: {col}")

    df["equity_total"] = pd.to_numeric(df["equity_total"], errors="coerce")
    df["dd_total"] = pd.to_numeric(df["dd_total"], errors="coerce")
    df = df.dropna(subset=["equity_total", "dd_total"]).copy()
    if df.empty:
        return pd.DataFrame([{
            "label": label,
            "peak_date": pd.NaT,
            "trough_date": pd.NaT,
            "recovery_date": pd.NaT,
            "dd_min": np.nan,
            "peak_equity": np.nan,
            "trough_equity": np.nan,
            "recovery_equity": np.nan,
            "days_to_trough": np.nan,
            "days_to_recovery": np.nan,
            "dd_duration_days": np.nan,
        }])

    # 最大DD（最小dd_total）
    trough_idx = int(df["dd_total"].idxmin())
    dd_min = float(df.loc[trough_idx, "dd_total"])
    trough_date = pd.Timestamp(df.loc[trough_idx, "Date"])
    trough_equity = float(df.loc[trough_idx, "equity_total"])

    # trough時点までのピーク（直前最高値）
    df_pre = df.loc[:trough_idx].copy()
    peak_equity = float(df_pre["equity_total"].max())
    peak_idx = int(df_pre["equity_total"].idxmax())
    peak_date = pd.Timestamp(df.loc[peak_idx, "Date"])

    # 回復日（peak_equity以上に戻る最初の日）
    df_post = df.loc[trough_idx:].copy()
    rec = df_post[df_post["equity_total"] >= peak_equity]
    if len(rec) == 0:
        recovery_date = pd.NaT
        recovery_equity = np.nan
        days_to_recovery = np.nan
        dd_duration_days = np.nan
    else:
        rec_idx = int(rec.index[0])
        recovery_date = pd.Timestamp(df.loc[rec_idx, "Date"])
        recovery_equity = float(df.loc[rec_idx, "equity_total"])
        days_to_recovery = int((recovery_date - peak_date).days)
        dd_duration_days = int((recovery_date - peak_date).days)

    days_to_trough = int((trough_date - peak_date).days)

    return pd.DataFrame([{
        "label": label,
        "peak_date": peak_date,
        "trough_date": trough_date,
        "recovery_date": recovery_date,
        "dd_min": dd_min,
        "peak_equity": peak_equity,
        "trough_equity": trough_equity,
        "recovery_equity": recovery_equity,
        "days_to_trough": days_to_trough,
        "days_to_recovery": days_to_recovery,
        "dd_duration_days": dd_duration_days,
    }])


def daily_mdd(df_daily: pd.DataFrame) -> float:
    if df_daily is None or df_daily.empty or "dd_total" not in df_daily.columns:
        return np.nan
    s = pd.to_numeric(df_daily["dd_total"], errors="coerce").dropna()
    return float(s.min()) if len(s) else np.nan


# ===================================
# A. 財務データ読み込み（列名自動判定）
# ===================================
def load_financial_data(cache_filename: str = CACHE_FILE) -> pd.DataFrame:
    if not os.path.exists(cache_filename):
        logger.error(f"財務データファイル '{cache_filename}' が見つかりません")
        return pd.DataFrame()

    try:
        df = pd.read_csv(cache_filename, encoding="utf-8-sig", parse_dates=["DisclosedDate"], low_memory=False)
        logger.info(f"財務データ読み込み成功: {len(df):,}件")

        column_mapping = {
            "IssuedShareTotal": ["IssuedShareTotal", "NumberOfIssuedAndOutstandingSharesAtTheEndOfFiscalYearIncludingTreasuryStock"],
            "Equity": ["Equity", "NetAssets", "TotalEquity"],
            "Profit": ["Profit", "NetIncome", "ProfitAttributableToOwnersOfParent"],
        }

        available_cols = df.columns.tolist()
        required_columns = {}
        for target_col, possible_names in column_mapping.items():
            found = False
            for possible_name in possible_names:
                if possible_name in available_cols:
                    required_columns[target_col] = possible_name
                    found = True
                    break
            if not found:
                required_columns[target_col] = None

        rename_dict = {v: k for k, v in required_columns.items() if v is not None}
        df = df.rename(columns=rename_dict)

        for col in ["IssuedShareTotal", "Equity", "Profit"]:
            if col not in df.columns:
                df[col] = 0

        base_cols = ["Code", "DisclosedDate"]
        if "CompanyName" in df.columns:
            base_cols.append("CompanyName")
        final_cols = base_cols + ["Profit", "Equity", "IssuedShareTotal"]
        df = df[final_cols].copy()

        logger.info(f"使用列: {df.columns.tolist()}")
        return df

    except Exception as e:
        logger.error(f"財務データ読み込みエラー: {e}", exc_info=True)
        return pd.DataFrame()


# ===================================
# B. 株価データ読み込み
# ===================================
def load_existing_price_data(ohlcv_dir: str = OHLCV_DIR) -> pd.DataFrame:
    if not os.path.exists(ohlcv_dir):
        logger.error(f"株価データディレクトリ '{ohlcv_dir}' が見つかりません。")
        return pd.DataFrame()

    csv_files = sorted([f for f in os.listdir(ohlcv_dir)
                        if f.startswith("OHLCV_Adjusted_") and f.endswith(".csv") and f != "OHLCV_Adjusted_TOPIX.csv"])

    if not csv_files:
        logger.error(f"ディレクトリ '{ohlcv_dir}' 内にCSVファイルが見つかりません。")
        return pd.DataFrame()

    logger.info(f"株価ファイル数: {len(csv_files)}個")

    all_dataframes = []
    usecols = ["Date", "Ticker", "AdjustmentClose"]

    for csv_file in tqdm(csv_files, desc="株価ファイル読み込み中"):
        file_path = os.path.join(ohlcv_dir, csv_file)
        try:
            df = pd.read_csv(
                file_path,
                usecols=usecols,
                parse_dates=["Date"],
                dtype={"Ticker": "Int64", "AdjustmentClose": "float32"},
            )
            df = df.drop_duplicates(subset=["Ticker", "Date"], keep="first")
            all_dataframes.append(df)
        except Exception as e:
            logger.warning(f"ファイル読み込みエラー ({csv_file}): {e}")
            continue

    if not all_dataframes:
        return pd.DataFrame()

    logger.info("全ファイルを結合中...")
    df_all = pd.concat(all_dataframes, ignore_index=True)
    logger.info(f"結合完了: {len(df_all):,}件")

    df_all = df_all.rename(columns={"Ticker": "Code", "AdjustmentClose": "Close"})
    df_all["Code"] = df_all["Code"].astype("str").str.replace("<NA>", "0").str.zfill(4)
    df_all = df_all.dropna(subset=["Close"])

    logger.info("重複除去 & ソート中...")
    df_all = df_all.sort_values(["Code", "Date"])
    df_all = df_all.drop_duplicates(subset=["Code", "Date"], keep="first")
    logger.info(f"重複除去後: {len(df_all):,}件")

    return df_all


# ===== PATCH START: 高速化（prices_by_code作成） =====
def build_prices_by_code(prices_df: pd.DataFrame) -> Dict[str, pd.DataFrame]:
    """
    Code -> その銘柄だけの価格DF(Date, Close)。Date昇順、indexはDateにしておく。
    これにより get_near_price が巨大DF全体フィルタをしなくて済む。
    """
    d = {}
    tmp = prices_df[["Code", "Date", "Close"]].copy()
    tmp["Code"] = tmp["Code"].astype(str).map(normalize_code)
    tmp = tmp[tmp["Code"] != ""].copy()
    tmp = tmp.sort_values(["Code", "Date"])

    for code, g in tmp.groupby("Code", sort=False):
        gg = g[["Date", "Close"]].drop_duplicates(subset=["Date"], keep="last").sort_values("Date").copy()
        gg = gg.set_index("Date", drop=False)
        d[code] = gg

    logger.info(f"prices_by_code built: {len(d):,} codes")
    return d


def get_near_price_fast(prices_by_code: Dict[str, pd.DataFrame],
                        code: str,
                        ref_date: pd.Timestamp,
                        kind: str = "last") -> Optional[float]:
    """
    ref_date前後±5日で、first/lastのCloseを取る（高速版）
    """
    code = normalize_code(code)
    if not code or code not in prices_by_code:
        return None
    g = prices_by_code[code]
    lo = ref_date - pd.Timedelta(days=5)
    hi = ref_date + pd.Timedelta(days=5)
    w = g.loc[(g["Date"] >= lo) & (g["Date"] <= hi)]
    if w.empty:
        return None
    return float(w.iloc[0]["Close"]) if kind == "first" else float(w.iloc[-1]["Close"])
# ===== PATCH END =====


# ===================================
# C. 財務指標計算（あなたの既存ロジック）
# ===================================
def safe_code_to_int(code_series: pd.Series) -> pd.Series:
    cleaned = code_series.astype(str).str.replace(r"\D", "", regex=True)
    cleaned = cleaned.replace("", "0")
    return pd.to_numeric(cleaned, errors="coerce").fillna(0).astype("int64")


def calculate_market_metrics_fast_chunked(statements_df: pd.DataFrame,
                                          prices_df: pd.DataFrame,
                                          chunk_size: int = 200) -> pd.DataFrame:
    logger.info("時価総額・PBR・ROE計算中（チャンク処理版）...")

    if prices_df.empty:
        logger.error("株価データが空です。")
        return pd.DataFrame()

    req_cols = {"Code", "Date", "Close"}
    if not req_cols.issubset(set(prices_df.columns)):
        logger.error(f"株価データに必要な列が存在しません。存在する列: {prices_df.columns.tolist()}")
        return pd.DataFrame()

    statements_df = statements_df.copy()
    statements_df["Profit"] = pd.to_numeric(statements_df["Profit"], errors="coerce").fillna(0)
    statements_df["Equity"] = pd.to_numeric(statements_df["Equity"], errors="coerce").fillna(0)
    statements_df["IssuedShareTotal"] = pd.to_numeric(statements_df["IssuedShareTotal"], errors="coerce").fillna(1)

    statements_df = statements_df[(statements_df["Equity"] > 0) & (statements_df["IssuedShareTotal"] > 0)]
    logger.info(f"有効な財務データ: {len(statements_df):,}件")

    statements_df["Code_int"] = safe_code_to_int(statements_df["Code"])
    prices_df = prices_df.copy()
    prices_df["Code_int"] = safe_code_to_int(prices_df["Code"])

    statements_df = statements_df[statements_df["Code_int"] > 0]
    prices_df = prices_df[prices_df["Code_int"] > 0]

    statements_df = statements_df.sort_values(["Code_int", "DisclosedDate"]).reset_index(drop=True)
    prices_df = prices_df.sort_values(["Code_int", "Date"]).reset_index(drop=True)

    statements_df = statements_df.drop_duplicates(subset=["Code_int", "DisclosedDate"], keep="first")
    prices_df = prices_df.drop_duplicates(subset=["Code_int", "Date"], keep="first")

    logger.info(f"ソート・重複除去後: 財務 {len(statements_df):,}件, 株価 {len(prices_df):,}件")

    statements_groups = list(statements_df.groupby("Code_int"))
    prices_dict = {code: group for code, group in prices_df.groupby("Code_int")}

    merged_list = []
    num_chunks = (len(statements_groups) + chunk_size - 1) // chunk_size

    for chunk_idx in tqdm(range(num_chunks), desc="マージ処理"):
        start_idx = chunk_idx * chunk_size
        end_idx = min((chunk_idx + 1) * chunk_size, len(statements_groups))
        chunk_groups = statements_groups[start_idx:end_idx]

        for code, stmt_code in chunk_groups:
            if code not in prices_dict:
                continue
            price_code = prices_dict[code]
            if len(price_code) == 0:
                continue

            stmt_code = stmt_code.sort_values("DisclosedDate").reset_index(drop=True)
            price_code = price_code.sort_values("Date").reset_index(drop=True)

            try:
                merged = pd.merge_asof(
                    stmt_code,
                    price_code[["Date", "Close"]],
                    left_on="DisclosedDate",
                    right_on="Date",
                    direction="backward",
                    tolerance=pd.Timedelta(days=10),
                )
                if not merged.empty:
                    merged_list.append(merged)
            except Exception:
                continue

    if not merged_list:
        logger.error("マージ結果が空です")
        return pd.DataFrame()

    df_merged = pd.concat(merged_list, ignore_index=True)
    df_merged = df_merged.dropna(subset=["Close"])
    logger.info(f"マージ完了: {len(df_merged):,}件")

    df_merged["MarketCap"] = df_merged["Close"] * df_merged["IssuedShareTotal"]
    df_merged["PBR"] = df_merged["MarketCap"] / df_merged["Equity"]
    df_merged["ROE"] = (df_merged["Profit"] / df_merged["Equity"]) * 100

    result_cols = ["Code", "DisclosedDate", "Close", "MarketCap", "PBR", "ROE", "Date"]
    if "CompanyName" in df_merged.columns:
        result_cols.insert(1, "CompanyName")

    result_df = df_merged[result_cols].copy()
    result_df = result_df.rename(columns={"Close": "StockPrice", "Date": "PriceDate"})

    mask = (
        (result_df["PBR"] > 0) &
        (result_df["PBR"] < 50) &
        (result_df["ROE"] > -100) &
        (result_df["ROE"] < 100) &
        (result_df["MarketCap"] > 1_000_000_000)
    )
    result_df = result_df[mask].copy()

    logger.info(f"計算完了: {len(result_df):,}件")
    return result_df


# ===================================
# D. 100株単位ポート構築
# ===================================
def build_unit_share_portfolio(stock_candidates: pd.DataFrame,
                               target_positions: int = 20,
                               initial_capital: float = 10_000_000) -> dict:
    if len(stock_candidates) == 0:
        return {"stocks": [], "shares": [], "prices": [], "amounts": []}

    selected = stock_candidates.head(target_positions).copy()
    capital_per_stock = initial_capital / len(selected)

    stocks, shares_list, prices_list, amounts_list = [], [], [], []

    for _, row in selected.iterrows():
        code = row["Code"]
        price = row["StockPrice"]
        required_amount = price * UNIT_SHARES

        if required_amount <= capital_per_stock:
            shares = int(capital_per_stock // required_amount) * UNIT_SHARES
            if shares > 0:
                stocks.append(code)
                shares_list.append(shares)
                prices_list.append(price)
                amounts_list.append(shares * price)

    return {"stocks": stocks, "shares": shares_list, "prices": prices_list, "amounts": amounts_list}


# ===================================
# 月次区切り
# ===================================
def make_month_ends(start: pd.Timestamp, end: pd.Timestamp) -> List[pd.Timestamp]:
    m0 = pd.Timestamp(start.year, start.month, 1) + pd.offsets.MonthEnd(0)
    m1 = pd.Timestamp(end.year, end.month, 1) + pd.offsets.MonthEnd(0)
    months = pd.date_range(m0, m1, freq="M")
    return [pd.Timestamp(x).normalize() for x in months]


# ===================================
# 日次エクイティ（追加）
# ===================================
def build_daily_equity_curve_for_period(
    portfolio: dict,
    start_date: pd.Timestamp,
    end_date: pd.Timestamp,
    prices_by_code: Dict[str, pd.DataFrame],
    invest_ratio_by_monthend: Optional[Dict[pd.Timestamp, float]] = None,
) -> pd.DataFrame:
    """
    指定期間の「日次」エクイティカーブを作成。
    invest_ratio_by_monthend が指定されれば、その月の営業日リターンに投資比率を掛ける（現金0%仮定）。
    """
    stocks = portfolio.get("stocks", [])
    shares = portfolio.get("shares", [])
    if not stocks:
        return pd.DataFrame()

    start_date = pd.to_datetime(start_date).normalize()
    end_date = pd.to_datetime(end_date).normalize()

    # 日付集合（各銘柄のDateの和集合）
    date_sets = []
    for code in stocks:
        code = normalize_code(code)
        if code in prices_by_code:
            g = prices_by_code[code]
            d = g[(g["Date"] >= start_date - pd.Timedelta(days=10)) & (g["Date"] <= end_date + pd.Timedelta(days=10))]["Date"]
            if len(d):
                date_sets.append(d)

    if not date_sets:
        return pd.DataFrame()

    dates = pd.Index(sorted(pd.unique(pd.concat(date_sets)))).astype("datetime64[ns]")
    dates = dates[(dates >= start_date) & (dates <= end_date)]
    if len(dates) == 0:
        return pd.DataFrame()

    values = []
    for dt in dates:
        v = 0.0
        ok = False
        for i, code in enumerate(stocks):
            code = normalize_code(code)
            if code not in prices_by_code:
                continue
            p = get_near_price_fast(prices_by_code, code, pd.Timestamp(dt), kind="last")
            if p is None:
                continue
            v += float(shares[i]) * float(p)
            ok = True
        values.append(v if ok else np.nan)

    df = pd.DataFrame({"Date": pd.to_datetime(dates), "equity_stock": values})
    df = df.dropna(subset=["equity_stock"]).copy()
    if df.empty:
        return df

    df = df.sort_values("Date").reset_index(drop=True)
    df["ret_stock"] = df["equity_stock"].pct_change().fillna(0.0)

    if invest_ratio_by_monthend is None:
        df["MonthEnd"] = (df["Date"] + pd.offsets.MonthEnd(0)).dt.normalize()
        df["invest_ratio"] = 1.0
        df["ret_total"] = df["ret_stock"]
    else:
        df["MonthEnd"] = (df["Date"] + pd.offsets.MonthEnd(0)).dt.normalize()
        df["invest_ratio"] = df["MonthEnd"].map(invest_ratio_by_monthend).fillna(1.0).astype(float)
        df["ret_total"] = df["invest_ratio"] * df["ret_stock"]

    df["equity_total"] = (1.0 + df["ret_total"]).cumprod()
    peak = df["equity_total"].cummax()
    df["dd_total"] = df["equity_total"] / peak - 1.0

    return df[["Date", "MonthEnd", "equity_stock", "ret_stock", "invest_ratio", "ret_total", "equity_total", "dd_total"]]


# ===================================
# FF5+MOM 因子
# ===================================
def load_ff5mom_factors_monthly() -> pd.DataFrame:
    fac = pd.read_parquet(FF5MOM_FACTOR_PATH).copy()
    fac["MonthEnd"] = pd.to_datetime(fac["MonthEnd"], errors="coerce").dt.normalize()
    need = ["MonthEnd"] + FACTOR_COLS
    missing = [c for c in need if c not in fac.columns]
    if missing:
        raise KeyError(f"FF5MOM factors missing columns: {missing} in {FF5MOM_FACTOR_PATH}")
    fac = fac[need].sort_values("MonthEnd").reset_index(drop=True)
    return fac


def compute_regime_off(fac: pd.DataFrame, month_end: pd.Timestamp) -> Tuple[bool, Dict]:
    fac2 = fac.set_index("MonthEnd")
    idx = fac2.index[fac2.index < month_end]
    if len(idx) < REGIME_LOOKBACK_M:
        return False, {"regime_ready": False}

    win = idx[-REGIME_LOOKBACK_M:]
    s_mkt = float(pd.to_numeric(fac2.loc[win, "MKT"], errors="coerce").sum())
    s_wml = float(pd.to_numeric(fac2.loc[win, "WML"], errors="coerce").sum())
    off = (s_mkt < REGIME_OFF_IF_SUM_MKT_LT) or (s_wml < REGIME_OFF_IF_SUM_WML_LT)
    return bool(off), {"regime_ready": True, "sum_MKT_3m": s_mkt, "sum_WML_3m": s_wml}


def estimate_port_beta(monthly_port_rets: pd.DataFrame, fac: pd.DataFrame, month_end: pd.Timestamp) -> Tuple[bool, Dict]:
    fac2 = fac.set_index("MonthEnd")
    idx = fac2.index[fac2.index < month_end]
    if len(idx) < BETA_EST_WINDOW_M:
        return False, {"beta_ready": False}

    win = idx[-BETA_EST_WINDOW_M:]

    # ===== PATCH START: MonthEnd重複をgroupby集約して一意化（one_to_one merge維持） =====
    m = monthly_port_rets.copy()
    m["MonthEnd"] = pd.to_datetime(m["MonthEnd"], errors="coerce").dt.normalize()
    m = m.dropna(subset=["MonthEnd", "port_ret"]).copy()

    m = (
        m.groupby("MonthEnd", as_index=False)["port_ret"]
         .apply(lambda s: (1.0 + s.astype(float)).prod() - 1.0)
    )
    # ===== PATCH END =====

    df = m[m["MonthEnd"].isin(win)].merge(
        fac, on="MonthEnd", how="left", validate="one_to_one"
    ).dropna(subset=["port_ret"] + FACTOR_COLS)

    if len(df) < BETA_EST_MIN_OBS:
        return False, {"beta_ready": False, "n_obs": int(len(df))}

    y = df["port_ret"].to_numpy(dtype=float)
    X = df[FACTOR_COLS].to_numpy(dtype=float)
    alpha, betas, r2 = ols_alpha_beta(y, X)

    out = {"beta_ready": True, "n_obs": int(len(df)), "r2": float(r2), "alpha": float(alpha)}
    for c, b in zip(FACTOR_COLS, betas):
        out[f"beta_{c}"] = float(b)
    return True, out


def decide_invest_ratio(month_end: pd.Timestamp,
                        fac: pd.DataFrame,
                        port_hist: pd.DataFrame) -> Tuple[float, Dict]:
    invest_ratio = 1.0
    diag = {
        "MonthEnd": month_end,
        "invest_ratio": 1.0,
        "regime_off": False,
        "beta_off": False,
        "regime_ready": False,
        "beta_ready": False,
        "sum_MKT_3m": np.nan,
        "sum_WML_3m": np.nan,
        "beta_MKT": np.nan,
        "beta_CMA": np.nan,
        "beta_n_obs": np.nan,
        "beta_r2": np.nan,
    }

    off_reg, info = compute_regime_off(fac, month_end)
    diag["regime_off"] = bool(off_reg)
    diag["regime_ready"] = bool(info.get("regime_ready", False))
    diag["sum_MKT_3m"] = info.get("sum_MKT_3m", np.nan)
    diag["sum_WML_3m"] = info.get("sum_WML_3m", np.nan)
    if off_reg:
        invest_ratio = min(invest_ratio, RISKOFF_RATIO)

    ok_beta, binfo = estimate_port_beta(port_hist, fac, month_end)
    diag["beta_ready"] = bool(binfo.get("beta_ready", False))
    if binfo.get("beta_ready", False):
        b_mkt = float(binfo.get("beta_MKT", np.nan))
        b_cma = float(binfo.get("beta_CMA", np.nan))
        diag["beta_MKT"] = b_mkt
        diag["beta_CMA"] = b_cma
        diag["beta_n_obs"] = float(binfo.get("n_obs", np.nan))
        diag["beta_r2"] = float(binfo.get("r2", np.nan))

        off_beta = (b_cma < BETA_CAP_CMA_LT) or (abs(b_mkt) > BETA_CAP_ABS_MKT_GT)
        diag["beta_off"] = bool(off_beta)
        if off_beta:
            invest_ratio = min(invest_ratio, RISKOFF_RATIO)

    diag["invest_ratio"] = float(invest_ratio)
    return float(invest_ratio), diag


# ===================================
# 月次リターン列（高速版 get_near_price_fast を利用）
# ===================================
def compute_monthly_portfolio_returns_with_riskcontrol_fast(
    portfolio: dict,
    start_date: pd.Timestamp,
    end_date: pd.Timestamp,
    prices_by_code: Dict[str, pd.DataFrame],
    fac: pd.DataFrame
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    stocks = portfolio["stocks"]
    shares = portfolio["shares"]
    if not stocks:
        return pd.DataFrame(), pd.DataFrame()

    month_ends = make_month_ends(start_date, end_date)
    boundaries = [start_date] + [me for me in month_ends if (me > start_date) and (me < end_date)] + [end_date]

    rows = []
    ir_rows = []

    port_hist = pd.DataFrame(columns=["MonthEnd", "port_ret"])

    for j in range(len(boundaries) - 1):
        d0 = boundaries[j]
        d1 = boundaries[j + 1]

        me = pd.Timestamp(d0.year, d0.month, 1) + pd.offsets.MonthEnd(0)
        me = pd.Timestamp(me).normalize()

        start_vals = []
        end_vals = []
        for i, code in enumerate(stocks):
            p0 = get_near_price_fast(prices_by_code, code, d0, kind="first")
            p1 = get_near_price_fast(prices_by_code, code, d1, kind="last")
            if p0 is None or p1 is None:
                continue
            sh = shares[i]
            start_vals.append(sh * p0)
            end_vals.append(sh * p1)

        if len(start_vals) == 0:
            continue

        start_v = float(np.sum(start_vals))
        end_v = float(np.sum(end_vals))
        stock_ret = (end_v / start_v) - 1.0

        port_hist = pd.concat([port_hist, pd.DataFrame([{"MonthEnd": me, "port_ret": stock_ret}])], ignore_index=True)

        invest_ratio, diag = decide_invest_ratio(me, fac, port_hist)
        ir_rows.append(diag)

        total_ret = invest_ratio * stock_ret

        rows.append({
            "MonthEnd": me,
            "period_start": d0,
            "period_end": d1,
            "port_ret_stock": stock_ret,
            "invest_ratio": invest_ratio,
            "port_ret_total": total_ret,
        })

    monthly_df = pd.DataFrame(rows)
    ir_diag_df = pd.DataFrame(ir_rows)
    return monthly_df, ir_diag_df


# ===================================
# 年次バックテスト（RAW / RiskControl）
# ===================================
def build_long_candidates(enhanced_financial_data: pd.DataFrame, rebalance_date: pd.Timestamp) -> pd.DataFrame:
    current_data = enhanced_financial_data[enhanced_financial_data["DisclosedDate"] <= rebalance_date].copy()
    current_data = current_data.sort_values("DisclosedDate").groupby("Code").tail(1)

    if len(current_data) < 100:
        return pd.DataFrame()

    current_data["PBR_Rank"] = current_data["PBR"].rank(method="first", ascending=True)
    current_data["ROE_Rank"] = current_data["ROE"].rank(method="first", ascending=False)

    current_data["PBR_Quartile"] = pd.qcut(current_data["PBR_Rank"], q=4, labels=[1, 2, 3, 4])
    current_data["ROE_Quartile"] = pd.qcut(current_data["ROE_Rank"], q=4, labels=[1, 2, 3, 4])

    long_candidates = current_data[
        (current_data["PBR_Quartile"] == 1) &
        (current_data["ROE_Quartile"] == 4)
    ].nsmallest(50, "PBR")

    return long_candidates


def annual_return_from_monthly(monthly_rets: pd.Series) -> float:
    if monthly_rets.empty:
        return 0.0
    return float((1.0 + monthly_rets).prod() - 1.0)


def apply_tax_annual(gross_return: float, initial_capital: float) -> Tuple[float, float]:
    profit = gross_return * initial_capital
    taxable = max(profit, 0.0)
    tax = taxable * TAX_RATE
    net_profit = profit - tax
    net_return = net_profit / initial_capital
    tax_rate_total = tax / initial_capital
    return float(net_return), float(tax_rate_total)


def run_annual_backtest_with_and_without_riskcontrol_fast(
    enhanced_financial_data: pd.DataFrame,
    prices_df: pd.DataFrame,
    prices_by_code: Dict[str, pd.DataFrame],
    fac: pd.DataFrame,
    initial_capital: float = INITIAL_CAPITAL
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    rebalance_dates = [pd.Timestamp(f"{year}-10-01") for year in range(2016, 2026)]

    results_raw = []
    results_rc = []
    diag_ir_all = []

    daily_curves_raw = []
    daily_curves_rc  = []

    for i, rebalance_date in enumerate(tqdm(rebalance_dates[:-1], desc="10月1日リバランス（統合・高速）")):
        next_rebalance = rebalance_dates[i + 1]

        long_candidates = build_long_candidates(enhanced_financial_data, rebalance_date)
        if long_candidates.empty:
            logger.warning(f"{rebalance_date}: データ不足でスキップ")
            continue

        long_portfolio = build_unit_share_portfolio(
            long_candidates,
            target_positions=20,
            initial_capital=initial_capital
        )

        n_long = len(long_portfolio["stocks"])
        inv_amount = float(np.sum(long_portfolio["amounts"])) if n_long > 0 else 0.0
        inv_ratio_raw = inv_amount / initial_capital if initial_capital > 0 else 0.0

        logger.info(f"{rebalance_date.strftime('%Y-%m')}: long={n_long} invest={inv_amount:,.0f} ratio={inv_ratio_raw:.3f}")

        # --- RAW（年次 start->end の価格で計算）
        total_profit = 0.0
        total_start_value = 0.0

        for j, code in enumerate(long_portfolio["stocks"]):
            sh = long_portfolio["shares"][j]
            p0 = long_portfolio["prices"][j]
            p1 = get_near_price_fast(prices_by_code, code, next_rebalance, kind="last")
            if p1 is None:
                continue
            start_value = sh * p0
            end_value = sh * p1
            total_start_value += start_value
            total_profit += (end_value - start_value)

        gross_return_raw = (total_profit / initial_capital) if initial_capital > 0 else 0.0
        net_return_raw, tax_rate_raw = apply_tax_annual(gross_return_raw, initial_capital)

        results_raw.append({
            "date": next_rebalance,
            "strategy_return_gross": gross_return_raw,
            "strategy_return_net": net_return_raw,
            "tax": tax_rate_raw,
            "long_count": n_long,
            "investment_ratio": inv_ratio_raw,
        })

        # --- RiskControl（月次で縮尺）
        monthly_df, ir_df = compute_monthly_portfolio_returns_with_riskcontrol_fast(
            long_portfolio, rebalance_date, next_rebalance, prices_by_code, fac
        )

        if monthly_df.empty:
            gross_return_rc = 0.0
            inv_ratio_rc_avg = 0.0
        else:
            gross_return_rc = annual_return_from_monthly(monthly_df["port_ret_total"])
            inv_ratio_rc_avg = float(monthly_df["invest_ratio"].mean())

        net_return_rc, tax_rate_rc = apply_tax_annual(gross_return_rc, initial_capital)

        results_rc.append({
            "date": next_rebalance,
            "strategy_return_gross": gross_return_rc,
            "strategy_return_net": net_return_rc,
            "tax": tax_rate_rc,
            "long_count": n_long,
            "investment_ratio": inv_ratio_rc_avg,
        })

        if not ir_df.empty:
            ir_df = ir_df.copy()
            ir_df["rebalance_start"] = rebalance_date
            ir_df["rebalance_end"] = next_rebalance
            diag_ir_all.append(ir_df)

        # --- 日次曲線（RAW / RC）を追加生成
        invest_ratio_map = None
        if not ir_df.empty:
            tmp_ir = ir_df.copy()
            tmp_ir["MonthEnd"] = pd.to_datetime(tmp_ir["MonthEnd"], errors="coerce").dt.normalize()
            tmp_ir = tmp_ir.dropna(subset=["MonthEnd"])
            tmp_ir = tmp_ir.sort_values("MonthEnd").drop_duplicates(subset=["MonthEnd"], keep="last")
            invest_ratio_map = dict(zip(tmp_ir["MonthEnd"], tmp_ir["invest_ratio"].astype(float)))

        daily_raw = build_daily_equity_curve_for_period(
            long_portfolio, rebalance_date, next_rebalance, prices_by_code, invest_ratio_by_monthend=None
        )
        if not daily_raw.empty:
            daily_raw["rebalance_start"] = rebalance_date
            daily_raw["rebalance_end"] = next_rebalance
            daily_curves_raw.append(daily_raw)

        daily_rc = build_daily_equity_curve_for_period(
            long_portfolio, rebalance_date, next_rebalance, prices_by_code, invest_ratio_by_monthend=invest_ratio_map
        )
        if not daily_rc.empty:
            daily_rc["rebalance_start"] = rebalance_date
            daily_rc["rebalance_end"] = next_rebalance
            daily_curves_rc.append(daily_rc)

    df_raw = pd.DataFrame(results_raw)
    df_rc = pd.DataFrame(results_rc)
    diag_ir = pd.concat(diag_ir_all, ignore_index=True) if len(diag_ir_all) else pd.DataFrame()

    daily_raw_all = pd.concat(daily_curves_raw, ignore_index=True) if len(daily_curves_raw) else pd.DataFrame()
    daily_rc_all  = pd.concat(daily_curves_rc,  ignore_index=True) if len(daily_curves_rc)  else pd.DataFrame()

    # 既存の戻り値I/Fを壊さないため、属性に退避（mainで保存）
    run_annual_backtest_with_and_without_riskcontrol_fast._daily_raw_all = daily_raw_all
    run_annual_backtest_with_and_without_riskcontrol_fast._daily_rc_all = daily_rc_all

    return df_raw, df_rc, diag_ir


# ===================================
# 年次回帰（FF5+MOM）
# ===================================
def build_annual_factor_from_monthly(fac: pd.DataFrame, start_date: pd.Timestamp, end_date: pd.Timestamp) -> Dict:
    fac2 = fac.copy()
    fac2 = fac2[(fac2["MonthEnd"] >= (start_date + pd.offsets.MonthEnd(0))) &
                (fac2["MonthEnd"] <= (end_date + pd.offsets.MonthEnd(-1)))].copy()
    out = {}
    for c in FACTOR_COLS:
        s = pd.to_numeric(fac2[c], errors="coerce").dropna()
        out[c] = float((1.0 + s).prod() - 1.0) if len(s) else np.nan
    return out


def run_factor_regression(results_df: pd.DataFrame, fac: pd.DataFrame) -> pd.DataFrame:
    if results_df.empty:
        return pd.DataFrame()

    rows = []
    for _, row in results_df.iterrows():
        end_date = pd.Timestamp(row["date"])
        start_date = end_date - pd.DateOffset(years=1)
        ann_fac = build_annual_factor_from_monthly(fac, start_date, end_date)
        rec = {"date": end_date}
        rec.update(ann_fac)
        rec["y"] = float(row["strategy_return_net"])
        rows.append(rec)

    df = pd.DataFrame(rows).dropna(subset=["y"] + FACTOR_COLS).copy()
    if len(df) < 3:
        return pd.DataFrame([{
            "n_years": int(len(df)),
            "R2": np.nan,
            "alpha": np.nan,
            **{f"beta_{c}": np.nan for c in FACTOR_COLS}
        }])

    y = df["y"].to_numpy(dtype=float)
    X = df[FACTOR_COLS].to_numpy(dtype=float)
    alpha, betas, r2 = ols_alpha_beta(y, X)

    out = {"n_years": int(len(df)), "R2": float(r2), "alpha": float(alpha)}
    for c, b in zip(FACTOR_COLS, betas):
        out[f"beta_{c}"] = float(b)
    return pd.DataFrame([out])


# ===================================
# 保存（累積曲線など）
# ===================================
def save_annual_bundle(df: pd.DataFrame, out_annual_csv: Path, out_curve_csv: Path, out_perf_csv: Path):
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date").reset_index(drop=True)

    df.to_csv(out_annual_csv, index=False, encoding="utf-8-sig")

    r = df["strategy_return_net"].astype(float)
    cum = (1.0 + r).cumprod()
    dd = compute_drawdown(cum)
    curve = pd.DataFrame({"date": df["date"], "ret": r, "cum": cum, "dd": dd})
    curve.to_csv(out_curve_csv, index=False, encoding="utf-8-sig")

    stats = perf_stats(r)
    perf = pd.DataFrame([stats])
    perf.to_csv(out_perf_csv, index=False, encoding="utf-8-sig")


# ===================================
# Main
# ===================================
def main():
    logger.info("=" * 110)
    logger.info("10/1 年次「割安高質」バックテスト + FF5+MOM RiskControl(C) [FAST get_near_price]")
    logger.info("=" * 110)
    logger.info(f"FF5MOM_FACTOR_PATH: {FF5MOM_FACTOR_PATH}")
    logger.info(f"OUT_DIR: {OUT_DIR}")
    logger.info(f"Tax mode: simple annual tax only (TAX_RATE={TAX_RATE})")
    logger.info(f"RiskControl: regime(3m MKT<0 or WML<0) + beta(CMA<-0.8 or |MKT|>0.9) => riskoff={RISKOFF_RATIO}")
    logger.info(f"Beta estimation window: {BETA_EST_WINDOW_M} months (min_obs={BETA_EST_MIN_OBS})")

    statements_df = load_financial_data()
    if statements_df.empty:
        logger.error("財務データ読み込み失敗")
        return

    prices_df = load_existing_price_data()
    if prices_df.empty:
        logger.error("株価データ読み込み失敗")
        return

    fac = load_ff5mom_factors_monthly()

    prices_by_code = build_prices_by_code(prices_df)

    enhanced_financial_data = calculate_market_metrics_fast_chunked(statements_df, prices_df, chunk_size=200)
    if enhanced_financial_data.empty:
        logger.error("財務指標計算失敗")
        return

    df_raw, df_rc, diag_ir = run_annual_backtest_with_and_without_riskcontrol_fast(
        enhanced_financial_data, prices_df, prices_by_code, fac, initial_capital=INITIAL_CAPITAL
    )

    if df_raw.empty or df_rc.empty:
        logger.error("年次バックテスト結果が空です")
        return

    save_annual_bundle(df_raw, OUT_ANNUAL_RAW, OUT_CURVE_RAW, OUT_PERF_RAW)
    save_annual_bundle(df_rc,  OUT_ANNUAL_RC,  OUT_CURVE_RC,  OUT_PERF_RC)

    reg_raw = run_factor_regression(df_raw, fac)
    reg_rc = run_factor_regression(df_rc, fac)
    reg_raw.to_csv(OUT_REG_RAW, index=False, encoding="utf-8-sig")
    reg_rc.to_csv(OUT_REG_RC, index=False, encoding="utf-8-sig")

    if not diag_ir.empty:
        diag_ir["MonthEnd"] = pd.to_datetime(diag_ir["MonthEnd"])
        diag_ir.sort_values(["rebalance_start", "MonthEnd"], inplace=True)
        diag_ir.to_csv(OUT_IR_DIAG, index=False, encoding="utf-8-sig")

    # ==============================
    # 追加：日次曲線保存 + 日次MDD + 最大DDイベント抽出
    # ==============================
    daily_raw_all = getattr(run_annual_backtest_with_and_without_riskcontrol_fast, "_daily_raw_all", pd.DataFrame())
    daily_rc_all  = getattr(run_annual_backtest_with_and_without_riskcontrol_fast, "_daily_rc_all", pd.DataFrame())

    if not daily_raw_all.empty:
        daily_raw_all.to_csv(OUT_DAILY_CURVE_RAW, index=False, encoding="utf-8-sig")
    if not daily_rc_all.empty:
        daily_rc_all.to_csv(OUT_DAILY_CURVE_RC, index=False, encoding="utf-8-sig")

    mdd_raw_d = daily_mdd(daily_raw_all)
    mdd_rc_d  = daily_mdd(daily_rc_all)

    pd.DataFrame([{
        "daily_maxDD_raw": mdd_raw_d,
        "daily_maxDD_riskcontrol": mdd_rc_d,
        "n_days_raw": int(len(daily_raw_all)) if not daily_raw_all.empty else 0,
        "n_days_riskcontrol": int(len(daily_rc_all)) if not daily_rc_all.empty else 0,
    }]).to_csv(OUT_DAILY_MDD_SUMMARY, index=False, encoding="utf-8-sig")

    # 最大DDイベント（開始/底/回復/幅）
    ev_raw = extract_max_drawdown_event_from_daily_curve(daily_raw_all, label="RAW")
    ev_rc  = extract_max_drawdown_event_from_daily_curve(daily_rc_all,  label="RISKCONTROL")

    ev_raw.to_csv(OUT_DD_EVENTS_RAW, index=False, encoding="utf-8-sig")
    ev_rc.to_csv(OUT_DD_EVENTS_RC, index=False, encoding="utf-8-sig")

    pd.concat([ev_raw, ev_rc], ignore_index=True).to_csv(OUT_DD_EVENTS_MAX, index=False, encoding="utf-8-sig")

    logger.info("-" * 110)
    logger.info("✅ SAVED (RAW)")
    logger.info(f"  - {OUT_ANNUAL_RAW}")
    logger.info(f"  - {OUT_CURVE_RAW}")
    logger.info(f"  - {OUT_PERF_RAW}")
    logger.info(f"  - {OUT_REG_RAW}")
    logger.info("✅ SAVED (RISKCONTROL)")
    logger.info(f"  - {OUT_ANNUAL_RC}")
    logger.info(f"  - {OUT_CURVE_RC}")
    logger.info(f"  - {OUT_PERF_RC}")
    logger.info(f"  - {OUT_REG_RC}")
    logger.info("✅ DIAGNOSTIC")
    logger.info(f"  - {OUT_IR_DIAG}")

    logger.info("✅ DAILY (ADDED)")
    logger.info(f"  - {OUT_DAILY_CURVE_RAW}")
    logger.info(f"  - {OUT_DAILY_CURVE_RC}")
    logger.info(f"  - {OUT_DAILY_MDD_SUMMARY}")
    logger.info("✅ DAILY DD EVENTS (ADDED)")
    logger.info(f"  - {OUT_DD_EVENTS_RAW}")
    logger.info(f"  - {OUT_DD_EVENTS_RC}")
    logger.info(f"  - {OUT_DD_EVENTS_MAX}")
    logger.info("-" * 110)

    raw_perf = pd.read_csv(OUT_PERF_RAW).iloc[0].to_dict()
    rc_perf  = pd.read_csv(OUT_PERF_RC).iloc[0].to_dict()
    logger.info("[PERF RAW] " + " | ".join([f"{k}={raw_perf.get(k)}" for k in ["n_years","CAGR","ann_mean","ann_vol","sharpe0","maxDD","cum_end"]]))
    logger.info("[PERF RC ] " + " | ".join([f"{k}={rc_perf.get(k)}"  for k in ["n_years","CAGR","ann_mean","ann_vol","sharpe0","maxDD","cum_end"]]))

    logger.info(f"[DAILY MDD] RAW={mdd_raw_d:.6f} | RC={mdd_rc_d:.6f}")
    logger.info("[DAILY MAX DD EVENT RAW] " + " | ".join([f"{k}={ev_raw.iloc[0][k]}" for k in ["peak_date","trough_date","recovery_date","dd_min"]]))
    logger.info("[DAILY MAX DD EVENT  RC] " + " | ".join([f"{k}={ev_rc.iloc[0][k]}"  for k in ["peak_date","trough_date","recovery_date","dd_min"]]))

if __name__ == "__main__":
    main()


2026-01-25 05:20:41,547 - INFO - ==============================================================================================================
2026-01-25 05:20:41,549 - INFO - 10/1 年次「割安高質」バックテスト + FF5+MOM RiskControl(C) [FAST get_near_price]
2026-01-25 05:20:41,549 - INFO - ==============================================================================================================
2026-01-25 05:20:41,550 - INFO - FF5MOM_FACTOR_PATH: C:\Users\yongr\Project\merged_data_all_stocks\factors\ff5_mom_factors_monthly.parquet
2026-01-25 05:20:41,551 - INFO - OUT_DIR: C:\Users\yongr\Project\merged_data_all_stocks\factors\bt_october_unit_with_ff5mom_riskcontrol
2026-01-25 05:20:41,551 - INFO - Tax mode: simple annual tax only (TAX_RATE=0.20315)
2026-01-25 05:20:41,552 - INFO - RiskControl: regime(3m MKT<0 or WML<0) + beta(CMA<-0.8 or |MKT|>0.9) => riskoff=0.5
2026-01-25 05:20:41,552 - INFO - Beta estimation window: 12 months (min_obs=10)
2026-01-25 05:20:42,327 - INFO - 財務データ読み込み成功: 64,222件
2

In [2]:
# -*- coding: utf-8 -*-
"""
年次（10/1）割安高質バックテスト（100株単位・税引後）に
FF5+MOMのリスク制御（C：レジーム減速＋β制約）を統合。

Option3:
- regime判定を OR に戻す
- riskoffの強さを弱める（RISKOFF_RATIO=0.7 もしくは 0.8）

税金:
- 簡易：年次（10/1→翌10/1）の最終損益にのみ課税

出力（統合前後＋診断）:
- annual_returns_raw.csv / annual_returns_riskcontrol.csv
- cumulative_curve_raw.csv / cumulative_curve_riskcontrol.csv
- performance_summary_raw.csv / performance_summary_riskcontrol.csv
- regression_ff5mom_raw.csv / regression_ff5mom_riskcontrol.csv
- diag_invest_ratio_monthly.csv

追加出力（日次）:
- daily_equity_curve_raw.csv / daily_equity_curve_riskcontrol.csv
- daily_mdd_summary.csv
- daily_drawdown_events_raw.csv / daily_drawdown_events_riskcontrol.csv
- daily_drawdown_events_max_summary.csv

依存:
  pip install pandas numpy pyarrow tqdm
"""

import warnings
warnings.filterwarnings("ignore")

import os
import logging
from pathlib import Path
from typing import Dict, Tuple, List, Optional

import numpy as np
import pandas as pd
from tqdm import tqdm


# ===================================
# ロギング設定
# ===================================
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler("backtest_october_unit_with_ff5mom_riskcontrol.log", encoding="utf-8"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)


# ===================================
# パス（あなたの環境）
# ===================================
FACTORS_DIR = Path(r"C:\Users\yongr\Project\merged_data_all_stocks\factors")
FF5MOM_FACTOR_PATH = FACTORS_DIR / "ff5_mom_factors_monthly.parquet"

CACHE_FILE = "topix_quarterly_statements.csv"
OHLCV_DIR = "./OHLCV_Adjusted"

OUT_DIR = FACTORS_DIR / "bt_october_unit_with_ff5mom_riskcontrol"
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_ANNUAL_RAW = OUT_DIR / "annual_returns_raw.csv"
OUT_ANNUAL_RC  = OUT_DIR / "annual_returns_riskcontrol.csv"

OUT_CURVE_RAW  = OUT_DIR / "cumulative_curve_raw.csv"
OUT_CURVE_RC   = OUT_DIR / "cumulative_curve_riskcontrol.csv"

OUT_PERF_RAW   = OUT_DIR / "performance_summary_raw.csv"
OUT_PERF_RC    = OUT_DIR / "performance_summary_riskcontrol.csv"

OUT_REG_RAW    = OUT_DIR / "regression_ff5mom_raw.csv"
OUT_REG_RC     = OUT_DIR / "regression_ff5mom_riskcontrol.csv"

OUT_IR_DIAG    = OUT_DIR / "diag_invest_ratio_monthly.csv"

# --- 追加出力（日次）
OUT_DAILY_CURVE_RAW = OUT_DIR / "daily_equity_curve_raw.csv"
OUT_DAILY_CURVE_RC  = OUT_DIR / "daily_equity_curve_riskcontrol.csv"
OUT_DAILY_MDD_SUMMARY = OUT_DIR / "daily_mdd_summary.csv"

OUT_DD_EVENTS_RAW = OUT_DIR / "daily_drawdown_events_raw.csv"
OUT_DD_EVENTS_RC  = OUT_DIR / "daily_drawdown_events_riskcontrol.csv"
OUT_DD_EVENTS_MAX = OUT_DIR / "daily_drawdown_events_max_summary.csv"


# ===================================
# 税・単位株
# ===================================
TAX_RATE = 0.20315
UNIT_SHARES = 100
INITIAL_CAPITAL = 10_000_000


# ===================================
# リスク制御パラメータ（Option3）
# ===================================
# ★ここだけ 0.8 に変えるだけで riskoff=0.8 版になります
RISKOFF_RATIO = 0.7

REGIME_LOOKBACK_M = 3
REGIME_OFF_IF_SUM_MKT_LT = 0.0
REGIME_OFF_IF_SUM_WML_LT = 0.0

BETA_CAP_CMA_LT = -0.8
BETA_CAP_ABS_MKT_GT = 0.9

BETA_EST_WINDOW_M = 12
BETA_EST_MIN_OBS = 10

FACTOR_COLS = ["MKT", "SMB", "HML", "RMW", "CMA", "WML"]

BAD_CODE_STRINGS = {"None", "nan", "", "NaN", "NULL", "null"}


# ===================================
# ユーティリティ
# ===================================
def normalize_code(code) -> str:
    if code is None:
        return ""
    s = str(code).strip()
    if s in BAD_CODE_STRINGS:
        return ""
    return s


def ols_alpha_beta(y: np.ndarray, X: np.ndarray) -> Tuple[float, np.ndarray, float]:
    n = len(y)
    if n < 3:
        return np.nan, np.full(X.shape[1], np.nan), np.nan

    X1 = np.column_stack([np.ones(n), X])
    XtX = X1.T @ X1
    try:
        inv = np.linalg.inv(XtX)
    except np.linalg.LinAlgError:
        inv = np.linalg.pinv(XtX)
    b = inv @ (X1.T @ y)

    yhat = X1 @ b
    resid = y - yhat
    sse = float(np.sum(resid**2))
    sst = float(np.sum((y - y.mean())**2))
    r2 = np.nan if sst <= 0 else (1.0 - sse / sst)

    alpha = float(b[0])
    betas = b[1:].astype(float)
    return alpha, betas, float(r2)


def compute_drawdown(cum: pd.Series) -> pd.Series:
    peak = cum.cummax()
    return cum / peak - 1.0


def perf_stats(annual_ret: pd.Series) -> Dict:
    r = annual_ret.dropna().astype(float)
    if r.empty:
        return {"n_years": 0, "CAGR": np.nan, "ann_mean": np.nan, "ann_vol": np.nan, "sharpe0": np.nan, "maxDD": np.nan, "cum_end": np.nan}

    n = len(r)
    cum = (1.0 + r).cumprod()
    years = n
    cagr = float(cum.iloc[-1] ** (1/years) - 1.0) if years > 0 else np.nan
    ann_mean = float(r.mean())
    ann_vol = float(r.std(ddof=1)) if n >= 2 else np.nan
    sharpe0 = float(ann_mean / ann_vol) if ann_vol and ann_vol > 0 else np.nan
    dd = compute_drawdown(cum)
    maxdd = float(dd.min())

    return {
        "n_years": int(n),
        "CAGR": cagr,
        "ann_mean": ann_mean,
        "ann_vol": ann_vol,
        "sharpe0": sharpe0,
        "maxDD": maxdd,
        "cum_end": float(cum.iloc[-1]),
    }


def extract_max_drawdown_event_from_daily_curve(df_daily: pd.DataFrame, label: str = "") -> pd.DataFrame:
    if df_daily is None or df_daily.empty:
        return pd.DataFrame([{
            "label": label,
            "peak_date": pd.NaT,
            "trough_date": pd.NaT,
            "recovery_date": pd.NaT,
            "dd_min": np.nan,
            "peak_equity": np.nan,
            "trough_equity": np.nan,
            "recovery_equity": np.nan,
            "days_to_trough": np.nan,
            "days_to_recovery": np.nan,
            "dd_duration_days": np.nan,
        }])

    df = df_daily.copy()
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df = df.dropna(subset=["Date"]).sort_values("Date").reset_index(drop=True)

    for col in ["equity_total", "dd_total"]:
        if col not in df.columns:
            raise KeyError(f"daily curve missing required column: {col}")

    df["equity_total"] = pd.to_numeric(df["equity_total"], errors="coerce")
    df["dd_total"] = pd.to_numeric(df["dd_total"], errors="coerce")
    df = df.dropna(subset=["equity_total", "dd_total"]).copy()
    if df.empty:
        return pd.DataFrame([{
            "label": label,
            "peak_date": pd.NaT,
            "trough_date": pd.NaT,
            "recovery_date": pd.NaT,
            "dd_min": np.nan,
            "peak_equity": np.nan,
            "trough_equity": np.nan,
            "recovery_equity": np.nan,
            "days_to_trough": np.nan,
            "days_to_recovery": np.nan,
            "dd_duration_days": np.nan,
        }])

    trough_idx = int(df["dd_total"].idxmin())
    dd_min = float(df.loc[trough_idx, "dd_total"])
    trough_date = pd.Timestamp(df.loc[trough_idx, "Date"])
    trough_equity = float(df.loc[trough_idx, "equity_total"])

    df_pre = df.loc[:trough_idx].copy()
    peak_equity = float(df_pre["equity_total"].max())
    peak_idx = int(df_pre["equity_total"].idxmax())
    peak_date = pd.Timestamp(df.loc[peak_idx, "Date"])

    df_post = df.loc[trough_idx:].copy()
    rec = df_post[df_post["equity_total"] >= peak_equity]
    if len(rec) == 0:
        recovery_date = pd.NaT
        recovery_equity = np.nan
        days_to_recovery = np.nan
        dd_duration_days = np.nan
    else:
        rec_idx = int(rec.index[0])
        recovery_date = pd.Timestamp(df.loc[rec_idx, "Date"])
        recovery_equity = float(df.loc[rec_idx, "equity_total"])
        days_to_recovery = int((recovery_date - peak_date).days)
        dd_duration_days = int((recovery_date - peak_date).days)

    days_to_trough = int((trough_date - peak_date).days)

    return pd.DataFrame([{
        "label": label,
        "peak_date": peak_date,
        "trough_date": trough_date,
        "recovery_date": recovery_date,
        "dd_min": dd_min,
        "peak_equity": peak_equity,
        "trough_equity": trough_equity,
        "recovery_equity": recovery_equity,
        "days_to_trough": days_to_trough,
        "days_to_recovery": days_to_recovery,
        "dd_duration_days": dd_duration_days,
    }])


def daily_mdd(df_daily: pd.DataFrame) -> float:
    if df_daily is None or df_daily.empty or "dd_total" not in df_daily.columns:
        return np.nan
    s = pd.to_numeric(df_daily["dd_total"], errors="coerce").dropna()
    return float(s.min()) if len(s) else np.nan


# ===================================
# A. 財務データ読み込み（列名自動判定）
# ===================================
def load_financial_data(cache_filename: str = CACHE_FILE) -> pd.DataFrame:
    if not os.path.exists(cache_filename):
        logger.error(f"財務データファイル '{cache_filename}' が見つかりません")
        return pd.DataFrame()

    try:
        df = pd.read_csv(cache_filename, encoding="utf-8-sig", parse_dates=["DisclosedDate"], low_memory=False)
        logger.info(f"財務データ読み込み成功: {len(df):,}件")

        column_mapping = {
            "IssuedShareTotal": ["IssuedShareTotal", "NumberOfIssuedAndOutstandingSharesAtTheEndOfFiscalYearIncludingTreasuryStock"],
            "Equity": ["Equity", "NetAssets", "TotalEquity"],
            "Profit": ["Profit", "NetIncome", "ProfitAttributableToOwnersOfParent"],
        }

        available_cols = df.columns.tolist()
        required_columns = {}
        for target_col, possible_names in column_mapping.items():
            found = False
            for possible_name in possible_names:
                if possible_name in available_cols:
                    required_columns[target_col] = possible_name
                    found = True
                    break
            if not found:
                required_columns[target_col] = None

        rename_dict = {v: k for k, v in required_columns.items() if v is not None}
        df = df.rename(columns=rename_dict)

        for col in ["IssuedShareTotal", "Equity", "Profit"]:
            if col not in df.columns:
                df[col] = 0

        base_cols = ["Code", "DisclosedDate"]
        if "CompanyName" in df.columns:
            base_cols.append("CompanyName")
        final_cols = base_cols + ["Profit", "Equity", "IssuedShareTotal"]
        df = df[final_cols].copy()

        logger.info(f"使用列: {df.columns.tolist()}")
        return df

    except Exception as e:
        logger.error(f"財務データ読み込みエラー: {e}", exc_info=True)
        return pd.DataFrame()


# ===================================
# B. 株価データ読み込み
# ===================================
def load_existing_price_data(ohlcv_dir: str = OHLCV_DIR) -> pd.DataFrame:
    if not os.path.exists(ohlcv_dir):
        logger.error(f"株価データディレクトリ '{ohlcv_dir}' が見つかりません。")
        return pd.DataFrame()

    csv_files = sorted([f for f in os.listdir(ohlcv_dir)
                        if f.startswith("OHLCV_Adjusted_") and f.endswith(".csv") and f != "OHLCV_Adjusted_TOPIX.csv"])

    if not csv_files:
        logger.error(f"ディレクトリ '{ohlcv_dir}' 内にCSVファイルが見つかりません。")
        return pd.DataFrame()

    logger.info(f"株価ファイル数: {len(csv_files)}個")

    all_dataframes = []
    usecols = ["Date", "Ticker", "AdjustmentClose"]

    for csv_file in tqdm(csv_files, desc="株価ファイル読み込み中"):
        file_path = os.path.join(ohlcv_dir, csv_file)
        try:
            df = pd.read_csv(
                file_path,
                usecols=usecols,
                parse_dates=["Date"],
                dtype={"Ticker": "Int64", "AdjustmentClose": "float32"},
            )
            df = df.drop_duplicates(subset=["Ticker", "Date"], keep="first")
            all_dataframes.append(df)
        except Exception as e:
            logger.warning(f"ファイル読み込みエラー ({csv_file}): {e}")
            continue

    if not all_dataframes:
        return pd.DataFrame()

    logger.info("全ファイルを結合中...")
    df_all = pd.concat(all_dataframes, ignore_index=True)
    logger.info(f"結合完了: {len(df_all):,}件")

    df_all = df_all.rename(columns={"Ticker": "Code", "AdjustmentClose": "Close"})
    df_all["Code"] = df_all["Code"].astype("str").str.replace("<NA>", "0").str.zfill(4)
    df_all = df_all.dropna(subset=["Close"])

    logger.info("重複除去 & ソート中...")
    df_all = df_all.sort_values(["Code", "Date"])
    df_all = df_all.drop_duplicates(subset=["Code", "Date"], keep="first")
    logger.info(f"重複除去後: {len(df_all):,}件")

    return df_all


def build_prices_by_code(prices_df: pd.DataFrame) -> Dict[str, pd.DataFrame]:
    d = {}
    tmp = prices_df[["Code", "Date", "Close"]].copy()
    tmp["Code"] = tmp["Code"].astype(str).map(normalize_code)
    tmp = tmp[tmp["Code"] != ""].copy()
    tmp = tmp.sort_values(["Code", "Date"])

    for code, g in tmp.groupby("Code", sort=False):
        gg = g[["Date", "Close"]].drop_duplicates(subset=["Date"], keep="last").sort_values("Date").copy()
        gg = gg.set_index("Date", drop=False)
        d[code] = gg

    logger.info(f"prices_by_code built: {len(d):,} codes")
    return d


def get_near_price_fast(prices_by_code: Dict[str, pd.DataFrame],
                        code: str,
                        ref_date: pd.Timestamp,
                        kind: str = "last") -> Optional[float]:
    code = normalize_code(code)
    if not code or code not in prices_by_code:
        return None
    g = prices_by_code[code]
    lo = ref_date - pd.Timedelta(days=5)
    hi = ref_date + pd.Timedelta(days=5)
    w = g.loc[(g["Date"] >= lo) & (g["Date"] <= hi)]
    if w.empty:
        return None
    return float(w.iloc[0]["Close"]) if kind == "first" else float(w.iloc[-1]["Close"])


def safe_code_to_int(code_series: pd.Series) -> pd.Series:
    cleaned = code_series.astype(str).str.replace(r"\D", "", regex=True)
    cleaned = cleaned.replace("", "0")
    return pd.to_numeric(cleaned, errors="coerce").fillna(0).astype("int64")


def calculate_market_metrics_fast_chunked(statements_df: pd.DataFrame,
                                          prices_df: pd.DataFrame,
                                          chunk_size: int = 200) -> pd.DataFrame:
    logger.info("時価総額・PBR・ROE計算中（チャンク処理版）...")

    if prices_df.empty:
        logger.error("株価データが空です。")
        return pd.DataFrame()

    req_cols = {"Code", "Date", "Close"}
    if not req_cols.issubset(set(prices_df.columns)):
        logger.error(f"株価データに必要な列が存在しません。存在する列: {prices_df.columns.tolist()}")
        return pd.DataFrame()

    statements_df = statements_df.copy()
    statements_df["Profit"] = pd.to_numeric(statements_df["Profit"], errors="coerce").fillna(0)
    statements_df["Equity"] = pd.to_numeric(statements_df["Equity"], errors="coerce").fillna(0)
    statements_df["IssuedShareTotal"] = pd.to_numeric(statements_df["IssuedShareTotal"], errors="coerce").fillna(1)

    statements_df = statements_df[(statements_df["Equity"] > 0) & (statements_df["IssuedShareTotal"] > 0)]
    logger.info(f"有効な財務データ: {len(statements_df):,}件")

    statements_df["Code_int"] = safe_code_to_int(statements_df["Code"])
    prices_df = prices_df.copy()
    prices_df["Code_int"] = safe_code_to_int(prices_df["Code"])

    statements_df = statements_df[statements_df["Code_int"] > 0]
    prices_df = prices_df[prices_df["Code_int"] > 0]

    statements_df = statements_df.sort_values(["Code_int", "DisclosedDate"]).reset_index(drop=True)
    prices_df = prices_df.sort_values(["Code_int", "Date"]).reset_index(drop=True)

    statements_df = statements_df.drop_duplicates(subset=["Code_int", "DisclosedDate"], keep="first")
    prices_df = prices_df.drop_duplicates(subset=["Code_int", "Date"], keep="first")

    logger.info(f"ソート・重複除去後: 財務 {len(statements_df):,}件, 株価 {len(prices_df):,}件")

    statements_groups = list(statements_df.groupby("Code_int"))
    prices_dict = {code: group for code, group in prices_df.groupby("Code_int")}

    merged_list = []
    num_chunks = (len(statements_groups) + chunk_size - 1) // chunk_size

    for chunk_idx in tqdm(range(num_chunks), desc="マージ処理"):
        start_idx = chunk_idx * chunk_size
        end_idx = min((chunk_idx + 1) * chunk_size, len(statements_groups))
        chunk_groups = statements_groups[start_idx:end_idx]

        for code, stmt_code in chunk_groups:
            if code not in prices_dict:
                continue
            price_code = prices_dict[code]
            if len(price_code) == 0:
                continue

            stmt_code = stmt_code.sort_values("DisclosedDate").reset_index(drop=True)
            price_code = price_code.sort_values("Date").reset_index(drop=True)

            try:
                merged = pd.merge_asof(
                    stmt_code,
                    price_code[["Date", "Close"]],
                    left_on="DisclosedDate",
                    right_on="Date",
                    direction="backward",
                    tolerance=pd.Timedelta(days=10),
                )
                if not merged.empty:
                    merged_list.append(merged)
            except Exception:
                continue

    if not merged_list:
        logger.error("マージ結果が空です")
        return pd.DataFrame()

    df_merged = pd.concat(merged_list, ignore_index=True)
    df_merged = df_merged.dropna(subset=["Close"])
    logger.info(f"マージ完了: {len(df_merged):,}件")

    df_merged["MarketCap"] = df_merged["Close"] * df_merged["IssuedShareTotal"]
    df_merged["PBR"] = df_merged["MarketCap"] / df_merged["Equity"]
    df_merged["ROE"] = (df_merged["Profit"] / df_merged["Equity"]) * 100

    result_cols = ["Code", "DisclosedDate", "Close", "MarketCap", "PBR", "ROE", "Date"]
    if "CompanyName" in df_merged.columns:
        result_cols.insert(1, "CompanyName")

    result_df = df_merged[result_cols].copy()
    result_df = result_df.rename(columns={"Close": "StockPrice", "Date": "PriceDate"})

    mask = (
        (result_df["PBR"] > 0) &
        (result_df["PBR"] < 50) &
        (result_df["ROE"] > -100) &
        (result_df["ROE"] < 100) &
        (result_df["MarketCap"] > 1_000_000_000)
    )
    result_df = result_df[mask].copy()

    logger.info(f"計算完了: {len(result_df):,}件")
    return result_df


def build_unit_share_portfolio(stock_candidates: pd.DataFrame,
                               target_positions: int = 20,
                               initial_capital: float = 10_000_000) -> dict:
    if len(stock_candidates) == 0:
        return {"stocks": [], "shares": [], "prices": [], "amounts": []}

    selected = stock_candidates.head(target_positions).copy()
    capital_per_stock = initial_capital / len(selected)

    stocks, shares_list, prices_list, amounts_list = [], [], [], []

    for _, row in selected.iterrows():
        code = row["Code"]
        price = row["StockPrice"]
        required_amount = price * UNIT_SHARES

        if required_amount <= capital_per_stock:
            shares = int(capital_per_stock // required_amount) * UNIT_SHARES
            if shares > 0:
                stocks.append(code)
                shares_list.append(shares)
                prices_list.append(price)
                amounts_list.append(shares * price)

    return {"stocks": stocks, "shares": shares_list, "prices": prices_list, "amounts": amounts_list}


def make_month_ends(start: pd.Timestamp, end: pd.Timestamp) -> List[pd.Timestamp]:
    m0 = pd.Timestamp(start.year, start.month, 1) + pd.offsets.MonthEnd(0)
    m1 = pd.Timestamp(end.year, end.month, 1) + pd.offsets.MonthEnd(0)
    months = pd.date_range(m0, m1, freq="M")
    return [pd.Timestamp(x).normalize() for x in months]


def build_daily_equity_curve_for_period(
    portfolio: dict,
    start_date: pd.Timestamp,
    end_date: pd.Timestamp,
    prices_by_code: Dict[str, pd.DataFrame],
    invest_ratio_by_monthend: Optional[Dict[pd.Timestamp, float]] = None,
) -> pd.DataFrame:
    stocks = portfolio.get("stocks", [])
    shares = portfolio.get("shares", [])
    if not stocks:
        return pd.DataFrame()

    start_date = pd.to_datetime(start_date).normalize()
    end_date = pd.to_datetime(end_date).normalize()

    date_sets = []
    for code in stocks:
        code = normalize_code(code)
        if code in prices_by_code:
            g = prices_by_code[code]
            d = g[(g["Date"] >= start_date - pd.Timedelta(days=10)) & (g["Date"] <= end_date + pd.Timedelta(days=10))]["Date"]
            if len(d):
                date_sets.append(d)

    if not date_sets:
        return pd.DataFrame()

    dates = pd.Index(sorted(pd.unique(pd.concat(date_sets)))).astype("datetime64[ns]")
    dates = dates[(dates >= start_date) & (dates <= end_date)]
    if len(dates) == 0:
        return pd.DataFrame()

    values = []
    for dt in dates:
        v = 0.0
        ok = False
        for i, code in enumerate(stocks):
            code = normalize_code(code)
            if code not in prices_by_code:
                continue
            p = get_near_price_fast(prices_by_code, code, pd.Timestamp(dt), kind="last")
            if p is None:
                continue
            v += float(shares[i]) * float(p)
            ok = True
        values.append(v if ok else np.nan)

    df = pd.DataFrame({"Date": pd.to_datetime(dates), "equity_stock": values})
    df = df.dropna(subset=["equity_stock"]).copy()
    if df.empty:
        return df

    df = df.sort_values("Date").reset_index(drop=True)
    df["ret_stock"] = df["equity_stock"].pct_change().fillna(0.0)
    df["MonthEnd"] = (df["Date"] + pd.offsets.MonthEnd(0)).dt.normalize()

    if invest_ratio_by_monthend is None:
        df["invest_ratio"] = 1.0
        df["ret_total"] = df["ret_stock"]
    else:
        df["invest_ratio"] = df["MonthEnd"].map(invest_ratio_by_monthend).fillna(1.0).astype(float)
        df["ret_total"] = df["invest_ratio"] * df["ret_stock"]

    df["equity_total"] = (1.0 + df["ret_total"]).cumprod()
    peak = df["equity_total"].cummax()
    df["dd_total"] = df["equity_total"] / peak - 1.0

    return df[["Date", "MonthEnd", "equity_stock", "ret_stock", "invest_ratio", "ret_total", "equity_total", "dd_total"]]


def load_ff5mom_factors_monthly() -> pd.DataFrame:
    fac = pd.read_parquet(FF5MOM_FACTOR_PATH).copy()
    fac["MonthEnd"] = pd.to_datetime(fac["MonthEnd"], errors="coerce").dt.normalize()
    need = ["MonthEnd"] + FACTOR_COLS
    missing = [c for c in need if c not in fac.columns]
    if missing:
        raise KeyError(f"FF5MOM factors missing columns: {missing} in {FF5MOM_FACTOR_PATH}")
    fac = fac[need].sort_values("MonthEnd").reset_index(drop=True)
    return fac


def compute_regime_off(fac: pd.DataFrame, month_end: pd.Timestamp) -> Tuple[bool, Dict]:
    fac2 = fac.set_index("MonthEnd")
    idx = fac2.index[fac2.index < month_end]
    if len(idx) < REGIME_LOOKBACK_M:
        return False, {"regime_ready": False}

    win = idx[-REGIME_LOOKBACK_M:]
    s_mkt = float(pd.to_numeric(fac2.loc[win, "MKT"], errors="coerce").sum())
    s_wml = float(pd.to_numeric(fac2.loc[win, "WML"], errors="coerce").sum())

    # Option3: ORに戻す
    off = (s_mkt < REGIME_OFF_IF_SUM_MKT_LT) or (s_wml < REGIME_OFF_IF_SUM_WML_LT)

    return bool(off), {"regime_ready": True, "sum_MKT_3m": s_mkt, "sum_WML_3m": s_wml}


def estimate_port_beta(monthly_port_rets: pd.DataFrame, fac: pd.DataFrame, month_end: pd.Timestamp) -> Tuple[bool, Dict]:
    fac2 = fac.set_index("MonthEnd")
    idx = fac2.index[fac2.index < month_end]
    if len(idx) < BETA_EST_WINDOW_M:
        return False, {"beta_ready": False}

    win = idx[-BETA_EST_WINDOW_M:]

    m = monthly_port_rets.copy()
    m["MonthEnd"] = pd.to_datetime(m["MonthEnd"], errors="coerce").dt.normalize()
    m = m.dropna(subset=["MonthEnd", "port_ret"]).copy()

    m = (
        m.groupby("MonthEnd", as_index=False)["port_ret"]
         .apply(lambda s: (1.0 + s.astype(float)).prod() - 1.0)
    )

    df = m[m["MonthEnd"].isin(win)].merge(
        fac, on="MonthEnd", how="left", validate="one_to_one"
    ).dropna(subset=["port_ret"] + FACTOR_COLS)

    if len(df) < BETA_EST_MIN_OBS:
        return False, {"beta_ready": False, "n_obs": int(len(df))}

    y = df["port_ret"].to_numpy(dtype=float)
    X = df[FACTOR_COLS].to_numpy(dtype=float)
    alpha, betas, r2 = ols_alpha_beta(y, X)

    out = {"beta_ready": True, "n_obs": int(len(df)), "r2": float(r2), "alpha": float(alpha)}
    for c, b in zip(FACTOR_COLS, betas):
        out[f"beta_{c}"] = float(b)
    return True, out


def decide_invest_ratio(month_end: pd.Timestamp,
                        fac: pd.DataFrame,
                        port_hist: pd.DataFrame) -> Tuple[float, Dict]:
    invest_ratio = 1.0
    diag = {
        "MonthEnd": month_end,
        "invest_ratio": 1.0,
        "regime_off": False,
        "beta_off": False,
        "regime_ready": False,
        "beta_ready": False,
        "sum_MKT_3m": np.nan,
        "sum_WML_3m": np.nan,
        "beta_MKT": np.nan,
        "beta_CMA": np.nan,
        "beta_n_obs": np.nan,
        "beta_r2": np.nan,
    }

    off_reg, info = compute_regime_off(fac, month_end)
    diag["regime_off"] = bool(off_reg)
    diag["regime_ready"] = bool(info.get("regime_ready", False))
    diag["sum_MKT_3m"] = info.get("sum_MKT_3m", np.nan)
    diag["sum_WML_3m"] = info.get("sum_WML_3m", np.nan)
    if off_reg:
        invest_ratio = min(invest_ratio, RISKOFF_RATIO)

    ok_beta, binfo = estimate_port_beta(port_hist, fac, month_end)
    diag["beta_ready"] = bool(binfo.get("beta_ready", False))
    if binfo.get("beta_ready", False):
        b_mkt = float(binfo.get("beta_MKT", np.nan))
        b_cma = float(binfo.get("beta_CMA", np.nan))
        diag["beta_MKT"] = b_mkt
        diag["beta_CMA"] = b_cma
        diag["beta_n_obs"] = float(binfo.get("n_obs", np.nan))
        diag["beta_r2"] = float(binfo.get("r2", np.nan))

        off_beta = (b_cma < BETA_CAP_CMA_LT) or (abs(b_mkt) > BETA_CAP_ABS_MKT_GT)
        diag["beta_off"] = bool(off_beta)
        if off_beta:
            invest_ratio = min(invest_ratio, RISKOFF_RATIO)

    diag["invest_ratio"] = float(invest_ratio)
    return float(invest_ratio), diag


def compute_monthly_portfolio_returns_with_riskcontrol_fast(
    portfolio: dict,
    start_date: pd.Timestamp,
    end_date: pd.Timestamp,
    prices_by_code: Dict[str, pd.DataFrame],
    fac: pd.DataFrame
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    stocks = portfolio["stocks"]
    shares = portfolio["shares"]
    if not stocks:
        return pd.DataFrame(), pd.DataFrame()

    month_ends = make_month_ends(start_date, end_date)
    boundaries = [start_date] + [me for me in month_ends if (me > start_date) and (me < end_date)] + [end_date]

    rows = []
    ir_rows = []
    port_hist = pd.DataFrame(columns=["MonthEnd", "port_ret"])

    for j in range(len(boundaries) - 1):
        d0 = boundaries[j]
        d1 = boundaries[j + 1]

        me = pd.Timestamp(d0.year, d0.month, 1) + pd.offsets.MonthEnd(0)
        me = pd.Timestamp(me).normalize()

        start_vals = []
        end_vals = []
        for i, code in enumerate(stocks):
            p0 = get_near_price_fast(prices_by_code, code, d0, kind="first")
            p1 = get_near_price_fast(prices_by_code, code, d1, kind="last")
            if p0 is None or p1 is None:
                continue
            sh = shares[i]
            start_vals.append(sh * p0)
            end_vals.append(sh * p1)

        if len(start_vals) == 0:
            continue

        start_v = float(np.sum(start_vals))
        end_v = float(np.sum(end_vals))
        stock_ret = (end_v / start_v) - 1.0

        port_hist = pd.concat([port_hist, pd.DataFrame([{"MonthEnd": me, "port_ret": stock_ret}])], ignore_index=True)

        invest_ratio, diag = decide_invest_ratio(me, fac, port_hist)
        ir_rows.append(diag)

        total_ret = invest_ratio * stock_ret

        rows.append({
            "MonthEnd": me,
            "period_start": d0,
            "period_end": d1,
            "port_ret_stock": stock_ret,
            "invest_ratio": invest_ratio,
            "port_ret_total": total_ret,
        })

    monthly_df = pd.DataFrame(rows)
    ir_diag_df = pd.DataFrame(ir_rows)
    return monthly_df, ir_diag_df


def build_long_candidates(enhanced_financial_data: pd.DataFrame, rebalance_date: pd.Timestamp) -> pd.DataFrame:
    current_data = enhanced_financial_data[enhanced_financial_data["DisclosedDate"] <= rebalance_date].copy()
    current_data = current_data.sort_values("DisclosedDate").groupby("Code").tail(1)

    if len(current_data) < 100:
        return pd.DataFrame()

    current_data["PBR_Rank"] = current_data["PBR"].rank(method="first", ascending=True)
    current_data["ROE_Rank"] = current_data["ROE"].rank(method="first", ascending=False)

    current_data["PBR_Quartile"] = pd.qcut(current_data["PBR_Rank"], q=4, labels=[1, 2, 3, 4])
    current_data["ROE_Quartile"] = pd.qcut(current_data["ROE_Rank"], q=4, labels=[1, 2, 3, 4])

    long_candidates = current_data[
        (current_data["PBR_Quartile"] == 1) &
        (current_data["ROE_Quartile"] == 4)
    ].nsmallest(50, "PBR")

    return long_candidates


def annual_return_from_monthly(monthly_rets: pd.Series) -> float:
    if monthly_rets.empty:
        return 0.0
    return float((1.0 + monthly_rets).prod() - 1.0)


def apply_tax_annual(gross_return: float, initial_capital: float) -> Tuple[float, float]:
    profit = gross_return * initial_capital
    taxable = max(profit, 0.0)
    tax = taxable * TAX_RATE
    net_profit = profit - tax
    net_return = net_profit / initial_capital
    tax_rate_total = tax / initial_capital
    return float(net_return), float(tax_rate_total)


def run_annual_backtest_with_and_without_riskcontrol_fast(
    enhanced_financial_data: pd.DataFrame,
    prices_df: pd.DataFrame,
    prices_by_code: Dict[str, pd.DataFrame],
    fac: pd.DataFrame,
    initial_capital: float = INITIAL_CAPITAL
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    rebalance_dates = [pd.Timestamp(f"{year}-10-01") for year in range(2016, 2026)]

    results_raw = []
    results_rc = []
    diag_ir_all = []

    daily_curves_raw = []
    daily_curves_rc  = []

    for i, rebalance_date in enumerate(tqdm(rebalance_dates[:-1], desc="10月1日リバランス（統合・高速）")):
        next_rebalance = rebalance_dates[i + 1]

        long_candidates = build_long_candidates(enhanced_financial_data, rebalance_date)
        if long_candidates.empty:
            logger.warning(f"{rebalance_date}: データ不足でスキップ")
            continue

        long_portfolio = build_unit_share_portfolio(
            long_candidates,
            target_positions=20,
            initial_capital=initial_capital
        )

        n_long = len(long_portfolio["stocks"])
        inv_amount = float(np.sum(long_portfolio["amounts"])) if n_long > 0 else 0.0
        inv_ratio_raw = inv_amount / initial_capital if initial_capital > 0 else 0.0

        logger.info(f"{rebalance_date.strftime('%Y-%m')}: long={n_long} invest={inv_amount:,.0f} ratio={inv_ratio_raw:.3f}")

        # --- RAW（年次 start->end の価格で計算）
        total_profit = 0.0
        total_start_value = 0.0

        for j, code in enumerate(long_portfolio["stocks"]):
            sh = long_portfolio["shares"][j]
            p0 = long_portfolio["prices"][j]
            p1 = get_near_price_fast(prices_by_code, code, next_rebalance, kind="last")
            if p1 is None:
                continue
            start_value = sh * p0
            end_value = sh * p1
            total_start_value += start_value
            total_profit += (end_value - start_value)

        gross_return_raw = (total_profit / initial_capital) if initial_capital > 0 else 0.0
        net_return_raw, tax_rate_raw = apply_tax_annual(gross_return_raw, initial_capital)

        results_raw.append({
            "date": next_rebalance,
            "strategy_return_gross": gross_return_raw,
            "strategy_return_net": net_return_raw,
            "tax": tax_rate_raw,
            "long_count": n_long,
            "investment_ratio": inv_ratio_raw,
        })

        # --- RiskControl（月次で縮尺）
        monthly_df, ir_df = compute_monthly_portfolio_returns_with_riskcontrol_fast(
            long_portfolio, rebalance_date, next_rebalance, prices_by_code, fac
        )

        if monthly_df.empty:
            gross_return_rc = 0.0
            inv_ratio_rc_avg = 0.0
        else:
            gross_return_rc = annual_return_from_monthly(monthly_df["port_ret_total"])
            inv_ratio_rc_avg = float(monthly_df["invest_ratio"].mean())

        net_return_rc, tax_rate_rc = apply_tax_annual(gross_return_rc, initial_capital)

        results_rc.append({
            "date": next_rebalance,
            "strategy_return_gross": gross_return_rc,
            "strategy_return_net": net_return_rc,
            "tax": tax_rate_rc,
            "long_count": n_long,
            "investment_ratio": inv_ratio_rc_avg,
        })

        if not ir_df.empty:
            ir_df = ir_df.copy()
            ir_df["rebalance_start"] = rebalance_date
            ir_df["rebalance_end"] = next_rebalance
            diag_ir_all.append(ir_df)

        invest_ratio_map = None
        if not ir_df.empty:
            tmp_ir = ir_df.copy()
            tmp_ir["MonthEnd"] = pd.to_datetime(tmp_ir["MonthEnd"], errors="coerce").dt.normalize()
            tmp_ir = tmp_ir.dropna(subset=["MonthEnd"])
            tmp_ir = tmp_ir.sort_values("MonthEnd").drop_duplicates(subset=["MonthEnd"], keep="last")
            invest_ratio_map = dict(zip(tmp_ir["MonthEnd"], tmp_ir["invest_ratio"].astype(float)))

        daily_raw = build_daily_equity_curve_for_period(
            long_portfolio, rebalance_date, next_rebalance, prices_by_code, invest_ratio_by_monthend=None
        )
        if not daily_raw.empty:
            daily_raw["rebalance_start"] = rebalance_date
            daily_raw["rebalance_end"] = next_rebalance
            daily_curves_raw.append(daily_raw)

        daily_rc = build_daily_equity_curve_for_period(
            long_portfolio, rebalance_date, next_rebalance, prices_by_code, invest_ratio_by_monthend=invest_ratio_map
        )
        if not daily_rc.empty:
            daily_rc["rebalance_start"] = rebalance_date
            daily_rc["rebalance_end"] = next_rebalance
            daily_curves_rc.append(daily_rc)

    df_raw = pd.DataFrame(results_raw)
    df_rc = pd.DataFrame(results_rc)
    diag_ir = pd.concat(diag_ir_all, ignore_index=True) if len(diag_ir_all) else pd.DataFrame()

    daily_raw_all = pd.concat(daily_curves_raw, ignore_index=True) if len(daily_curves_raw) else pd.DataFrame()
    daily_rc_all  = pd.concat(daily_curves_rc,  ignore_index=True) if len(daily_curves_rc)  else pd.DataFrame()

    run_annual_backtest_with_and_without_riskcontrol_fast._daily_raw_all = daily_raw_all
    run_annual_backtest_with_and_without_riskcontrol_fast._daily_rc_all = daily_rc_all

    return df_raw, df_rc, diag_ir


def build_annual_factor_from_monthly(fac: pd.DataFrame, start_date: pd.Timestamp, end_date: pd.Timestamp) -> Dict:
    fac2 = fac.copy()
    fac2 = fac2[(fac2["MonthEnd"] >= (start_date + pd.offsets.MonthEnd(0))) &
                (fac2["MonthEnd"] <= (end_date + pd.offsets.MonthEnd(-1)))].copy()
    out = {}
    for c in FACTOR_COLS:
        s = pd.to_numeric(fac2[c], errors="coerce").dropna()
        out[c] = float((1.0 + s).prod() - 1.0) if len(s) else np.nan
    return out


def run_factor_regression(results_df: pd.DataFrame, fac: pd.DataFrame) -> pd.DataFrame:
    if results_df.empty:
        return pd.DataFrame()

    rows = []
    for _, row in results_df.iterrows():
        end_date = pd.Timestamp(row["date"])
        start_date = end_date - pd.DateOffset(years=1)
        ann_fac = build_annual_factor_from_monthly(fac, start_date, end_date)
        rec = {"date": end_date}
        rec.update(ann_fac)
        rec["y"] = float(row["strategy_return_net"])
        rows.append(rec)

    df = pd.DataFrame(rows).dropna(subset=["y"] + FACTOR_COLS).copy()
    if len(df) < 3:
        return pd.DataFrame([{
            "n_years": int(len(df)),
            "R2": np.nan,
            "alpha": np.nan,
            **{f"beta_{c}": np.nan for c in FACTOR_COLS}
        }])

    y = df["y"].to_numpy(dtype=float)
    X = df[FACTOR_COLS].to_numpy(dtype=float)
    alpha, betas, r2 = ols_alpha_beta(y, X)

    out = {"n_years": int(len(df)), "R2": float(r2), "alpha": float(alpha)}
    for c, b in zip(FACTOR_COLS, betas):
        out[f"beta_{c}"] = float(b)
    return pd.DataFrame([out])


def save_annual_bundle(df: pd.DataFrame, out_annual_csv: Path, out_curve_csv: Path, out_perf_csv: Path):
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date").reset_index(drop=True)

    df.to_csv(out_annual_csv, index=False, encoding="utf-8-sig")

    r = df["strategy_return_net"].astype(float)
    cum = (1.0 + r).cumprod()
    dd = compute_drawdown(cum)
    curve = pd.DataFrame({"date": df["date"], "ret": r, "cum": cum, "dd": dd})
    curve.to_csv(out_curve_csv, index=False, encoding="utf-8-sig")

    stats = perf_stats(r)
    perf = pd.DataFrame([stats])
    perf.to_csv(out_perf_csv, index=False, encoding="utf-8-sig")


def main():
    logger.info("=" * 110)
    logger.info("10/1 年次「割安高質」バックテスト + FF5+MOM RiskControl(C) [FAST get_near_price]")
    logger.info("=" * 110)
    logger.info(f"FF5MOM_FACTOR_PATH: {FF5MOM_FACTOR_PATH}")
    logger.info(f"OUT_DIR: {OUT_DIR}")
    logger.info(f"Tax mode: simple annual tax only (TAX_RATE={TAX_RATE})")
    logger.info(f"RiskControl: regime(3m MKT<0 OR WML<0) + beta(CMA<-0.8 or |MKT|>0.9) => riskoff={RISKOFF_RATIO}  [Option3]")
    logger.info(f"Beta estimation window: {BETA_EST_WINDOW_M} months (min_obs={BETA_EST_MIN_OBS})")

    statements_df = load_financial_data()
    if statements_df.empty:
        logger.error("財務データ読み込み失敗")
        return

    prices_df = load_existing_price_data()
    if prices_df.empty:
        logger.error("株価データ読み込み失敗")
        return

    fac = load_ff5mom_factors_monthly()
    prices_by_code = build_prices_by_code(prices_df)

    enhanced_financial_data = calculate_market_metrics_fast_chunked(statements_df, prices_df, chunk_size=200)
    if enhanced_financial_data.empty:
        logger.error("財務指標計算失敗")
        return

    df_raw, df_rc, diag_ir = run_annual_backtest_with_and_without_riskcontrol_fast(
        enhanced_financial_data, prices_df, prices_by_code, fac, initial_capital=INITIAL_CAPITAL
    )

    if df_raw.empty or df_rc.empty:
        logger.error("年次バックテスト結果が空です")
        return

    save_annual_bundle(df_raw, OUT_ANNUAL_RAW, OUT_CURVE_RAW, OUT_PERF_RAW)
    save_annual_bundle(df_rc,  OUT_ANNUAL_RC,  OUT_CURVE_RC,  OUT_PERF_RC)

    reg_raw = run_factor_regression(df_raw, fac)
    reg_rc = run_factor_regression(df_rc, fac)
    reg_raw.to_csv(OUT_REG_RAW, index=False, encoding="utf-8-sig")
    reg_rc.to_csv(OUT_REG_RC, index=False, encoding="utf-8-sig")

    if not diag_ir.empty:
        diag_ir["MonthEnd"] = pd.to_datetime(diag_ir["MonthEnd"])
        diag_ir.sort_values(["rebalance_start", "MonthEnd"], inplace=True)
        diag_ir.to_csv(OUT_IR_DIAG, index=False, encoding="utf-8-sig")

    # 日次出力 + 日次MDD + 最大DDイベント（ログまで）
    daily_raw_all = getattr(run_annual_backtest_with_and_without_riskcontrol_fast, "_daily_raw_all", pd.DataFrame())
    daily_rc_all  = getattr(run_annual_backtest_with_and_without_riskcontrol_fast, "_daily_rc_all", pd.DataFrame())

    if not daily_raw_all.empty:
        daily_raw_all.to_csv(OUT_DAILY_CURVE_RAW, index=False, encoding="utf-8-sig")
    if not daily_rc_all.empty:
        daily_rc_all.to_csv(OUT_DAILY_CURVE_RC, index=False, encoding="utf-8-sig")

    mdd_raw_d = daily_mdd(daily_raw_all)
    mdd_rc_d  = daily_mdd(daily_rc_all)

    pd.DataFrame([{
        "daily_maxDD_raw": mdd_raw_d,
        "daily_maxDD_riskcontrol": mdd_rc_d,
        "n_days_raw": int(len(daily_raw_all)) if not daily_raw_all.empty else 0,
        "n_days_riskcontrol": int(len(daily_rc_all)) if not daily_rc_all.empty else 0,
    }]).to_csv(OUT_DAILY_MDD_SUMMARY, index=False, encoding="utf-8-sig")

    ev_raw = extract_max_drawdown_event_from_daily_curve(daily_raw_all, label="RAW")
    ev_rc  = extract_max_drawdown_event_from_daily_curve(daily_rc_all,  label="RISKCONTROL")

    ev_raw.to_csv(OUT_DD_EVENTS_RAW, index=False, encoding="utf-8-sig")
    ev_rc.to_csv(OUT_DD_EVENTS_RC, index=False, encoding="utf-8-sig")
    pd.concat([ev_raw, ev_rc], ignore_index=True).to_csv(OUT_DD_EVENTS_MAX, index=False, encoding="utf-8-sig")

    logger.info("-" * 110)
    logger.info("✅ SAVED (RAW)")
    logger.info(f"  - {OUT_ANNUAL_RAW}")
    logger.info(f"  - {OUT_CURVE_RAW}")
    logger.info(f"  - {OUT_PERF_RAW}")
    logger.info(f"  - {OUT_REG_RAW}")
    logger.info("✅ SAVED (RISKCONTROL)")
    logger.info(f"  - {OUT_ANNUAL_RC}")
    logger.info(f"  - {OUT_CURVE_RC}")
    logger.info(f"  - {OUT_PERF_RC}")
    logger.info(f"  - {OUT_REG_RC}")
    logger.info("✅ DIAGNOSTIC")
    logger.info(f"  - {OUT_IR_DIAG}")
    logger.info("✅ DAILY (ADDED)")
    logger.info(f"  - {OUT_DAILY_CURVE_RAW}")
    logger.info(f"  - {OUT_DAILY_CURVE_RC}")
    logger.info(f"  - {OUT_DAILY_MDD_SUMMARY}")
    logger.info("✅ DAILY DD EVENTS (ADDED)")
    logger.info(f"  - {OUT_DD_EVENTS_RAW}")
    logger.info(f"  - {OUT_DD_EVENTS_RC}")
    logger.info(f"  - {OUT_DD_EVENTS_MAX}")
    logger.info("-" * 110)

    raw_perf = pd.read_csv(OUT_PERF_RAW).iloc[0].to_dict()
    rc_perf  = pd.read_csv(OUT_PERF_RC).iloc[0].to_dict()
    logger.info("[PERF RAW] " + " | ".join([f"{k}={raw_perf.get(k)}" for k in ["n_years","CAGR","ann_mean","ann_vol","sharpe0","maxDD","cum_end"]]))
    logger.info("[PERF RC ] " + " | ".join([f"{k}={rc_perf.get(k)}"  for k in ["n_years","CAGR","ann_mean","ann_vol","sharpe0","maxDD","cum_end"]]))

    logger.info(f"[DAILY MDD] RAW={mdd_raw_d:.6f} | RC={mdd_rc_d:.6f}")
    logger.info("[DAILY MAX DD EVENT RAW] " + " | ".join([f"{k}={ev_raw.iloc[0][k]}" for k in ["peak_date","trough_date","recovery_date","dd_min"]]))
    logger.info("[DAILY MAX DD EVENT  RC] " + " | ".join([f"{k}={ev_rc.iloc[0][k]}"  for k in ["peak_date","trough_date","recovery_date","dd_min"]]))

if __name__ == "__main__":
    main()


2026-01-25 05:38:20,306 - INFO - ==============================================================================================================
2026-01-25 05:38:20,307 - INFO - 10/1 年次「割安高質」バックテスト + FF5+MOM RiskControl(C) [FAST get_near_price]
2026-01-25 05:38:20,308 - INFO - ==============================================================================================================
2026-01-25 05:38:20,309 - INFO - FF5MOM_FACTOR_PATH: C:\Users\yongr\Project\merged_data_all_stocks\factors\ff5_mom_factors_monthly.parquet
2026-01-25 05:38:20,310 - INFO - OUT_DIR: C:\Users\yongr\Project\merged_data_all_stocks\factors\bt_october_unit_with_ff5mom_riskcontrol
2026-01-25 05:38:20,311 - INFO - Tax mode: simple annual tax only (TAX_RATE=0.20315)
2026-01-25 05:38:20,312 - INFO - RiskControl: regime(3m MKT<0 OR WML<0) + beta(CMA<-0.8 or |MKT|>0.9) => riskoff=0.7  [Option3]
2026-01-25 05:38:20,312 - INFO - Beta estimation window: 12 months (min_obs=10)
2026-01-25 05:38:21,032 - INFO - 財務データ読み込み成功

In [4]:
# -*- coding: utf-8 -*-
"""
年次（10/1）割安高質バックテスト（100株単位・税引後）に
FF5+MOMのリスク制御（C：レジーム減速＋β制約）を統合。

【追加：グリッド実験】
RISKOFF_RATIOを 0.7 / 0.8 / 0.9 で切り替えて、
日次MDD・CAGR・回復日数(days_to_recovery)を横並び比較する。

税金:
- 簡易：年次（10/1→翌10/1）の最終損益にのみ課税
- 日次は税考慮しない（実験目的のため）

出力:
各ケース別フォルダに、従来のCSV一式を保存。
さらに、全ケース横並びの grid_summary_riskoff_ratio.csv をベースフォルダ直下へ保存。

依存:
  pip install pandas numpy pyarrow tqdm
"""

import warnings
warnings.filterwarnings("ignore")

import os
import logging
from pathlib import Path
from typing import Dict, Tuple, List, Optional

import numpy as np
import pandas as pd
from tqdm import tqdm


# ===================================
# ロギング設定
# ===================================
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler("backtest_october_unit_with_ff5mom_riskcontrol_grid.log", encoding="utf-8"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)


# ===================================
# パス（あなたの環境）
# ===================================
FACTORS_DIR = Path(r"C:\Users\yongr\Project\merged_data_all_stocks\factors")
FF5MOM_FACTOR_PATH = FACTORS_DIR / "ff5_mom_factors_monthly.parquet"

CACHE_FILE = "topix_quarterly_statements.csv"
OHLCV_DIR = "./OHLCV_Adjusted"


# ===================================
# 税・単位株
# ===================================
TAX_RATE = 0.20315
UNIT_SHARES = 100
INITIAL_CAPITAL = 10_000_000


# ===================================
# リスク制御パラメータ（Option3）
# ===================================
# ★グリッド実験でここを動的に差し替える
RISKOFF_RATIO = 0.7

REGIME_LOOKBACK_M = 3
REGIME_OFF_IF_SUM_MKT_LT = 0.0
REGIME_OFF_IF_SUM_WML_LT = 0.0

BETA_CAP_CMA_LT = -0.8
BETA_CAP_ABS_MKT_GT = 0.9

BETA_EST_WINDOW_M = 12
BETA_EST_MIN_OBS = 10

FACTOR_COLS = ["MKT", "SMB", "HML", "RMW", "CMA", "WML"]

BAD_CODE_STRINGS = {"None", "nan", "", "NaN", "NULL", "null"}


# ===================================
# 出力（グリッド用ベース）
# ===================================
BASE_OUT_DIR = FACTORS_DIR / "bt_october_unit_with_ff5mom_riskcontrol_grid"
BASE_OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_GRID_SUMMARY = BASE_OUT_DIR / "grid_summary_riskoff_ratio.csv"


# ===================================
# 出力パス（ケースごとに set_output_dir で差し替える）
# ===================================
OUT_DIR = BASE_OUT_DIR / "riskoff_0p7"
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_ANNUAL_RAW = OUT_DIR / "annual_returns_raw.csv"
OUT_ANNUAL_RC  = OUT_DIR / "annual_returns_riskcontrol.csv"

OUT_CURVE_RAW  = OUT_DIR / "cumulative_curve_raw.csv"
OUT_CURVE_RC   = OUT_DIR / "cumulative_curve_riskcontrol.csv"

OUT_PERF_RAW   = OUT_DIR / "performance_summary_raw.csv"
OUT_PERF_RC    = OUT_DIR / "performance_summary_riskcontrol.csv"

OUT_REG_RAW    = OUT_DIR / "regression_ff5mom_raw.csv"
OUT_REG_RC     = OUT_DIR / "regression_ff5mom_riskcontrol.csv"

OUT_IR_DIAG    = OUT_DIR / "diag_invest_ratio_monthly.csv"

OUT_DAILY_CURVE_RAW = OUT_DIR / "daily_equity_curve_raw.csv"
OUT_DAILY_CURVE_RC  = OUT_DIR / "daily_equity_curve_riskcontrol.csv"
OUT_DAILY_MDD_SUMMARY = OUT_DIR / "daily_mdd_summary.csv"

OUT_DD_EVENTS_RAW = OUT_DIR / "daily_drawdown_events_raw.csv"
OUT_DD_EVENTS_RC  = OUT_DIR / "daily_drawdown_events_riskcontrol.csv"
OUT_DD_EVENTS_MAX = OUT_DIR / "daily_drawdown_events_max_summary.csv"


def set_output_dir(base_dir: Path, suffix: str):
    """
    ループ実験で出力先を切り替えるための関数。
    suffix例: 'riskoff_0p7'
    """
    global OUT_DIR
    global OUT_ANNUAL_RAW, OUT_ANNUAL_RC
    global OUT_CURVE_RAW, OUT_CURVE_RC
    global OUT_PERF_RAW, OUT_PERF_RC
    global OUT_REG_RAW, OUT_REG_RC
    global OUT_IR_DIAG
    global OUT_DAILY_CURVE_RAW, OUT_DAILY_CURVE_RC, OUT_DAILY_MDD_SUMMARY
    global OUT_DD_EVENTS_RAW, OUT_DD_EVENTS_RC, OUT_DD_EVENTS_MAX

    OUT_DIR = base_dir / suffix
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    OUT_ANNUAL_RAW = OUT_DIR / "annual_returns_raw.csv"
    OUT_ANNUAL_RC  = OUT_DIR / "annual_returns_riskcontrol.csv"

    OUT_CURVE_RAW  = OUT_DIR / "cumulative_curve_raw.csv"
    OUT_CURVE_RC   = OUT_DIR / "cumulative_curve_riskcontrol.csv"

    OUT_PERF_RAW   = OUT_DIR / "performance_summary_raw.csv"
    OUT_PERF_RC    = OUT_DIR / "performance_summary_riskcontrol.csv"

    OUT_REG_RAW    = OUT_DIR / "regression_ff5mom_raw.csv"
    OUT_REG_RC     = OUT_DIR / "regression_ff5mom_riskcontrol.csv"

    OUT_IR_DIAG    = OUT_DIR / "diag_invest_ratio_monthly.csv"

    OUT_DAILY_CURVE_RAW = OUT_DIR / "daily_equity_curve_raw.csv"
    OUT_DAILY_CURVE_RC  = OUT_DIR / "daily_equity_curve_riskcontrol.csv"
    OUT_DAILY_MDD_SUMMARY = OUT_DIR / "daily_mdd_summary.csv"

    OUT_DD_EVENTS_RAW = OUT_DIR / "daily_drawdown_events_raw.csv"
    OUT_DD_EVENTS_RC  = OUT_DIR / "daily_drawdown_events_riskcontrol.csv"
    OUT_DD_EVENTS_MAX = OUT_DIR / "daily_drawdown_events_max_summary.csv"


# ===================================
# ユーティリティ
# ===================================
def normalize_code(code) -> str:
    if code is None:
        return ""
    s = str(code).strip()
    if s in BAD_CODE_STRINGS:
        return ""
    return s


def ols_alpha_beta(y: np.ndarray, X: np.ndarray) -> Tuple[float, np.ndarray, float]:
    n = len(y)
    if n < 3:
        return np.nan, np.full(X.shape[1], np.nan), np.nan

    X1 = np.column_stack([np.ones(n), X])
    XtX = X1.T @ X1
    try:
        inv = np.linalg.inv(XtX)
    except np.linalg.LinAlgError:
        inv = np.linalg.pinv(XtX)
    b = inv @ (X1.T @ y)

    yhat = X1 @ b
    resid = y - yhat
    sse = float(np.sum(resid**2))
    sst = float(np.sum((y - y.mean())**2))
    r2 = np.nan if sst <= 0 else (1.0 - sse / sst)

    alpha = float(b[0])
    betas = b[1:].astype(float)
    return alpha, betas, float(r2)


def compute_drawdown(cum: pd.Series) -> pd.Series:
    peak = cum.cummax()
    return cum / peak - 1.0


def perf_stats(annual_ret: pd.Series) -> Dict:
    r = annual_ret.dropna().astype(float)
    if r.empty:
        return {"n_years": 0, "CAGR": np.nan, "ann_mean": np.nan, "ann_vol": np.nan, "sharpe0": np.nan, "maxDD": np.nan, "cum_end": np.nan}

    n = len(r)
    cum = (1.0 + r).cumprod()
    years = n
    cagr = float(cum.iloc[-1] ** (1/years) - 1.0) if years > 0 else np.nan
    ann_mean = float(r.mean())
    ann_vol = float(r.std(ddof=1)) if n >= 2 else np.nan
    sharpe0 = float(ann_mean / ann_vol) if ann_vol and ann_vol > 0 else np.nan
    dd = compute_drawdown(cum)
    maxdd = float(dd.min())

    return {
        "n_years": int(n),
        "CAGR": cagr,
        "ann_mean": ann_mean,
        "ann_vol": ann_vol,
        "sharpe0": sharpe0,
        "maxDD": maxdd,
        "cum_end": float(cum.iloc[-1]),
    }


def extract_max_drawdown_event_from_daily_curve(df_daily: pd.DataFrame, label: str = "") -> pd.DataFrame:
    if df_daily is None or df_daily.empty:
        return pd.DataFrame([{
            "label": label,
            "peak_date": pd.NaT,
            "trough_date": pd.NaT,
            "recovery_date": pd.NaT,
            "dd_min": np.nan,
            "peak_equity": np.nan,
            "trough_equity": np.nan,
            "recovery_equity": np.nan,
            "days_to_trough": np.nan,
            "days_to_recovery": np.nan,
            "dd_duration_days": np.nan,
        }])

    df = df_daily.copy()
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df = df.dropna(subset=["Date"]).sort_values("Date").reset_index(drop=True)

    for col in ["equity_total", "dd_total"]:
        if col not in df.columns:
            raise KeyError(f"daily curve missing required column: {col}")

    df["equity_total"] = pd.to_numeric(df["equity_total"], errors="coerce")
    df["dd_total"] = pd.to_numeric(df["dd_total"], errors="coerce")
    df = df.dropna(subset=["equity_total", "dd_total"]).copy()
    if df.empty:
        return pd.DataFrame([{
            "label": label,
            "peak_date": pd.NaT,
            "trough_date": pd.NaT,
            "recovery_date": pd.NaT,
            "dd_min": np.nan,
            "peak_equity": np.nan,
            "trough_equity": np.nan,
            "recovery_equity": np.nan,
            "days_to_trough": np.nan,
            "days_to_recovery": np.nan,
            "dd_duration_days": np.nan,
        }])

    trough_idx = int(df["dd_total"].idxmin())
    dd_min = float(df.loc[trough_idx, "dd_total"])
    trough_date = pd.Timestamp(df.loc[trough_idx, "Date"])
    trough_equity = float(df.loc[trough_idx, "equity_total"])

    df_pre = df.loc[:trough_idx].copy()
    peak_equity = float(df_pre["equity_total"].max())
    peak_idx = int(df_pre["equity_total"].idxmax())
    peak_date = pd.Timestamp(df.loc[peak_idx, "Date"])

    df_post = df.loc[trough_idx:].copy()
    rec = df_post[df_post["equity_total"] >= peak_equity]
    if len(rec) == 0:
        recovery_date = pd.NaT
        recovery_equity = np.nan
        days_to_recovery = np.nan
        dd_duration_days = np.nan
    else:
        rec_idx = int(rec.index[0])
        recovery_date = pd.Timestamp(df.loc[rec_idx, "Date"])
        recovery_equity = float(df.loc[rec_idx, "equity_total"])
        days_to_recovery = int((recovery_date - peak_date).days)
        dd_duration_days = int((recovery_date - peak_date).days)

    days_to_trough = int((trough_date - peak_date).days)

    return pd.DataFrame([{
        "label": label,
        "peak_date": peak_date,
        "trough_date": trough_date,
        "recovery_date": recovery_date,
        "dd_min": dd_min,
        "peak_equity": peak_equity,
        "trough_equity": trough_equity,
        "recovery_equity": recovery_equity,
        "days_to_trough": days_to_trough,
        "days_to_recovery": days_to_recovery,
        "dd_duration_days": dd_duration_days,
    }])


def daily_mdd(df_daily: pd.DataFrame) -> float:
    if df_daily is None or df_daily.empty or "dd_total" not in df_daily.columns:
        return np.nan
    s = pd.to_numeric(df_daily["dd_total"], errors="coerce").dropna()
    return float(s.min()) if len(s) else np.nan


# ===================================
# A. 財務データ読み込み（列名自動判定）
# ===================================
def load_financial_data(cache_filename: str = CACHE_FILE) -> pd.DataFrame:
    if not os.path.exists(cache_filename):
        logger.error(f"財務データファイル '{cache_filename}' が見つかりません")
        return pd.DataFrame()

    try:
        df = pd.read_csv(cache_filename, encoding="utf-8-sig", parse_dates=["DisclosedDate"], low_memory=False)
        logger.info(f"財務データ読み込み成功: {len(df):,}件")

        column_mapping = {
            "IssuedShareTotal": ["IssuedShareTotal", "NumberOfIssuedAndOutstandingSharesAtTheEndOfFiscalYearIncludingTreasuryStock"],
            "Equity": ["Equity", "NetAssets", "TotalEquity"],
            "Profit": ["Profit", "NetIncome", "ProfitAttributableToOwnersOfParent"],
        }

        available_cols = df.columns.tolist()
        required_columns = {}
        for target_col, possible_names in column_mapping.items():
            found = False
            for possible_name in possible_names:
                if possible_name in available_cols:
                    required_columns[target_col] = possible_name
                    found = True
                    break
            if not found:
                required_columns[target_col] = None

        rename_dict = {v: k for k, v in required_columns.items() if v is not None}
        df = df.rename(columns=rename_dict)

        for col in ["IssuedShareTotal", "Equity", "Profit"]:
            if col not in df.columns:
                df[col] = 0

        base_cols = ["Code", "DisclosedDate"]
        if "CompanyName" in df.columns:
            base_cols.append("CompanyName")
        final_cols = base_cols + ["Profit", "Equity", "IssuedShareTotal"]
        df = df[final_cols].copy()

        logger.info(f"使用列: {df.columns.tolist()}")
        return df

    except Exception as e:
        logger.error(f"財務データ読み込みエラー: {e}", exc_info=True)
        return pd.DataFrame()


# ===================================
# B. 株価データ読み込み
# ===================================
def load_existing_price_data(ohlcv_dir: str = OHLCV_DIR) -> pd.DataFrame:
    if not os.path.exists(ohlcv_dir):
        logger.error(f"株価データディレクトリ '{ohlcv_dir}' が見つかりません。")
        return pd.DataFrame()

    csv_files = sorted([f for f in os.listdir(ohlcv_dir)
                        if f.startswith("OHLCV_Adjusted_") and f.endswith(".csv") and f != "OHLCV_Adjusted_TOPIX.csv"])

    if not csv_files:
        logger.error(f"ディレクトリ '{ohlcv_dir}' 内にCSVファイルが見つかりません。")
        return pd.DataFrame()

    logger.info(f"株価ファイル数: {len(csv_files)}個")

    all_dataframes = []
    usecols = ["Date", "Ticker", "AdjustmentClose"]

    for csv_file in tqdm(csv_files, desc="株価ファイル読み込み中"):
        file_path = os.path.join(ohlcv_dir, csv_file)
        try:
            df = pd.read_csv(
                file_path,
                usecols=usecols,
                parse_dates=["Date"],
                dtype={"Ticker": "Int64", "AdjustmentClose": "float32"},
            )
            df = df.drop_duplicates(subset=["Ticker", "Date"], keep="first")
            all_dataframes.append(df)
        except Exception as e:
            logger.warning(f"ファイル読み込みエラー ({csv_file}): {e}")
            continue

    if not all_dataframes:
        return pd.DataFrame()

    logger.info("全ファイルを結合中...")
    df_all = pd.concat(all_dataframes, ignore_index=True)
    logger.info(f"結合完了: {len(df_all):,}件")

    df_all = df_all.rename(columns={"Ticker": "Code", "AdjustmentClose": "Close"})
    df_all["Code"] = df_all["Code"].astype("str").str.replace("<NA>", "0").str.zfill(4)
    df_all = df_all.dropna(subset=["Close"])

    logger.info("重複除去 & ソート中...")
    df_all = df_all.sort_values(["Code", "Date"])
    df_all = df_all.drop_duplicates(subset=["Code", "Date"], keep="first")
    logger.info(f"重複除去後: {len(df_all):,}件")

    return df_all


def build_prices_by_code(prices_df: pd.DataFrame) -> Dict[str, pd.DataFrame]:
    d = {}
    tmp = prices_df[["Code", "Date", "Close"]].copy()
    tmp["Code"] = tmp["Code"].astype(str).map(normalize_code)
    tmp = tmp[tmp["Code"] != ""].copy()
    tmp = tmp.sort_values(["Code", "Date"])

    for code, g in tmp.groupby("Code", sort=False):
        gg = g[["Date", "Close"]].drop_duplicates(subset=["Date"], keep="last").sort_values("Date").copy()
        gg = gg.set_index("Date", drop=False)
        d[code] = gg

    logger.info(f"prices_by_code built: {len(d):,} codes")
    return d


def get_near_price_fast(prices_by_code: Dict[str, pd.DataFrame],
                        code: str,
                        ref_date: pd.Timestamp,
                        kind: str = "last") -> Optional[float]:
    code = normalize_code(code)
    if not code or code not in prices_by_code:
        return None
    g = prices_by_code[code]
    lo = ref_date - pd.Timedelta(days=5)
    hi = ref_date + pd.Timedelta(days=5)
    w = g.loc[(g["Date"] >= lo) & (g["Date"] <= hi)]
    if w.empty:
        return None
    return float(w.iloc[0]["Close"]) if kind == "first" else float(w.iloc[-1]["Close"])


def safe_code_to_int(code_series: pd.Series) -> pd.Series:
    cleaned = code_series.astype(str).str.replace(r"\D", "", regex=True)
    cleaned = cleaned.replace("", "0")
    return pd.to_numeric(cleaned, errors="coerce").fillna(0).astype("int64")


def calculate_market_metrics_fast_chunked(statements_df: pd.DataFrame,
                                          prices_df: pd.DataFrame,
                                          chunk_size: int = 200) -> pd.DataFrame:
    logger.info("時価総額・PBR・ROE計算中（チャンク処理版）...")

    if prices_df.empty:
        logger.error("株価データが空です。")
        return pd.DataFrame()

    req_cols = {"Code", "Date", "Close"}
    if not req_cols.issubset(set(prices_df.columns)):
        logger.error(f"株価データに必要な列が存在しません。存在する列: {prices_df.columns.tolist()}")
        return pd.DataFrame()

    statements_df = statements_df.copy()
    statements_df["Profit"] = pd.to_numeric(statements_df["Profit"], errors="coerce").fillna(0)
    statements_df["Equity"] = pd.to_numeric(statements_df["Equity"], errors="coerce").fillna(0)
    statements_df["IssuedShareTotal"] = pd.to_numeric(statements_df["IssuedShareTotal"], errors="coerce").fillna(1)

    statements_df = statements_df[(statements_df["Equity"] > 0) & (statements_df["IssuedShareTotal"] > 0)]
    logger.info(f"有効な財務データ: {len(statements_df):,}件")

    statements_df["Code_int"] = safe_code_to_int(statements_df["Code"])
    prices_df = prices_df.copy()
    prices_df["Code_int"] = safe_code_to_int(prices_df["Code"])

    statements_df = statements_df[statements_df["Code_int"] > 0]
    prices_df = prices_df[prices_df["Code_int"] > 0]

    statements_df = statements_df.sort_values(["Code_int", "DisclosedDate"]).reset_index(drop=True)
    prices_df = prices_df.sort_values(["Code_int", "Date"]).reset_index(drop=True)

    statements_df = statements_df.drop_duplicates(subset=["Code_int", "DisclosedDate"], keep="first")
    prices_df = prices_df.drop_duplicates(subset=["Code_int", "Date"], keep="first")

    logger.info(f"ソート・重複除去後: 財務 {len(statements_df):,}件, 株価 {len(prices_df):,}件")

    statements_groups = list(statements_df.groupby("Code_int"))
    prices_dict = {code: group for code, group in prices_df.groupby("Code_int")}

    merged_list = []
    num_chunks = (len(statements_groups) + chunk_size - 1) // chunk_size

    for chunk_idx in tqdm(range(num_chunks), desc="マージ処理"):
        start_idx = chunk_idx * chunk_size
        end_idx = min((chunk_idx + 1) * chunk_size, len(statements_groups))
        chunk_groups = statements_groups[start_idx:end_idx]

        for code, stmt_code in chunk_groups:
            if code not in prices_dict:
                continue
            price_code = prices_dict[code]
            if len(price_code) == 0:
                continue

            stmt_code = stmt_code.sort_values("DisclosedDate").reset_index(drop=True)
            price_code = price_code.sort_values("Date").reset_index(drop=True)

            try:
                merged = pd.merge_asof(
                    stmt_code,
                    price_code[["Date", "Close"]],
                    left_on="DisclosedDate",
                    right_on="Date",
                    direction="backward",
                    tolerance=pd.Timedelta(days=10),
                )
                if not merged.empty:
                    merged_list.append(merged)
            except Exception:
                continue

    if not merged_list:
        logger.error("マージ結果が空です")
        return pd.DataFrame()

    df_merged = pd.concat(merged_list, ignore_index=True)
    df_merged = df_merged.dropna(subset=["Close"])
    logger.info(f"マージ完了: {len(df_merged):,}件")

    df_merged["MarketCap"] = df_merged["Close"] * df_merged["IssuedShareTotal"]
    df_merged["PBR"] = df_merged["MarketCap"] / df_merged["Equity"]
    df_merged["ROE"] = (df_merged["Profit"] / df_merged["Equity"]) * 100

    result_cols = ["Code", "DisclosedDate", "Close", "MarketCap", "PBR", "ROE", "Date"]
    if "CompanyName" in df_merged.columns:
        result_cols.insert(1, "CompanyName")

    result_df = df_merged[result_cols].copy()
    result_df = result_df.rename(columns={"Close": "StockPrice", "Date": "PriceDate"})

    mask = (
        (result_df["PBR"] > 0) &
        (result_df["PBR"] < 50) &
        (result_df["ROE"] > -100) &
        (result_df["ROE"] < 100) &
        (result_df["MarketCap"] > 1_000_000_000)
    )
    result_df = result_df[mask].copy()

    logger.info(f"計算完了: {len(result_df):,}件")
    return result_df


def build_unit_share_portfolio(stock_candidates: pd.DataFrame,
                               target_positions: int = 20,
                               initial_capital: float = 10_000_000) -> dict:
    if len(stock_candidates) == 0:
        return {"stocks": [], "shares": [], "prices": [], "amounts": []}

    selected = stock_candidates.head(target_positions).copy()
    capital_per_stock = initial_capital / len(selected)

    stocks, shares_list, prices_list, amounts_list = [], [], [], []

    for _, row in selected.iterrows():
        code = row["Code"]
        price = row["StockPrice"]
        required_amount = price * UNIT_SHARES

        if required_amount <= capital_per_stock:
            shares = int(capital_per_stock // required_amount) * UNIT_SHARES
            if shares > 0:
                stocks.append(code)
                shares_list.append(shares)
                prices_list.append(price)
                amounts_list.append(shares * price)

    return {"stocks": stocks, "shares": shares_list, "prices": prices_list, "amounts": amounts_list}


def make_month_ends(start: pd.Timestamp, end: pd.Timestamp) -> List[pd.Timestamp]:
    m0 = pd.Timestamp(start.year, start.month, 1) + pd.offsets.MonthEnd(0)
    m1 = pd.Timestamp(end.year, end.month, 1) + pd.offsets.MonthEnd(0)
    months = pd.date_range(m0, m1, freq="M")
    return [pd.Timestamp(x).normalize() for x in months]


def build_daily_equity_curve_for_period(
    portfolio: dict,
    start_date: pd.Timestamp,
    end_date: pd.Timestamp,
    prices_by_code: Dict[str, pd.DataFrame],
    invest_ratio_by_monthend: Optional[Dict[pd.Timestamp, float]] = None,
) -> pd.DataFrame:
    stocks = portfolio.get("stocks", [])
    shares = portfolio.get("shares", [])
    if not stocks:
        return pd.DataFrame()

    start_date = pd.to_datetime(start_date).normalize()
    end_date = pd.to_datetime(end_date).normalize()

    date_sets = []
    for code in stocks:
        code = normalize_code(code)
        if code in prices_by_code:
            g = prices_by_code[code]
            d = g[(g["Date"] >= start_date - pd.Timedelta(days=10)) & (g["Date"] <= end_date + pd.Timedelta(days=10))]["Date"]
            if len(d):
                date_sets.append(d)

    if not date_sets:
        return pd.DataFrame()

    dates = pd.Index(sorted(pd.unique(pd.concat(date_sets)))).astype("datetime64[ns]")
    dates = dates[(dates >= start_date) & (dates <= end_date)]
    if len(dates) == 0:
        return pd.DataFrame()

    values = []
    for dt in dates:
        v = 0.0
        ok = False
        for i, code in enumerate(stocks):
            code = normalize_code(code)
            if code not in prices_by_code:
                continue
            p = get_near_price_fast(prices_by_code, code, pd.Timestamp(dt), kind="last")
            if p is None:
                continue
            v += float(shares[i]) * float(p)
            ok = True
        values.append(v if ok else np.nan)

    df = pd.DataFrame({"Date": pd.to_datetime(dates), "equity_stock": values})
    df = df.dropna(subset=["equity_stock"]).copy()
    if df.empty:
        return df

    df = df.sort_values("Date").reset_index(drop=True)
    df["ret_stock"] = df["equity_stock"].pct_change().fillna(0.0)
    df["MonthEnd"] = (df["Date"] + pd.offsets.MonthEnd(0)).dt.normalize()

    if invest_ratio_by_monthend is None:
        df["invest_ratio"] = 1.0
        df["ret_total"] = df["ret_stock"]
    else:
        df["invest_ratio"] = df["MonthEnd"].map(invest_ratio_by_monthend).fillna(1.0).astype(float)
        df["ret_total"] = df["invest_ratio"] * df["ret_stock"]

    df["equity_total"] = (1.0 + df["ret_total"]).cumprod()
    peak = df["equity_total"].cummax()
    df["dd_total"] = df["equity_total"] / peak - 1.0

    return df[["Date", "MonthEnd", "equity_stock", "ret_stock", "invest_ratio", "ret_total", "equity_total", "dd_total"]]


def load_ff5mom_factors_monthly() -> pd.DataFrame:
    fac = pd.read_parquet(FF5MOM_FACTOR_PATH).copy()
    fac["MonthEnd"] = pd.to_datetime(fac["MonthEnd"], errors="coerce").dt.normalize()
    need = ["MonthEnd"] + FACTOR_COLS
    missing = [c for c in need if c not in fac.columns]
    if missing:
        raise KeyError(f"FF5MOM factors missing columns: {missing} in {FF5MOM_FACTOR_PATH}")
    fac = fac[need].sort_values("MonthEnd").reset_index(drop=True)
    return fac


def compute_regime_off(fac: pd.DataFrame, month_end: pd.Timestamp) -> Tuple[bool, Dict]:
    fac2 = fac.set_index("MonthEnd")
    idx = fac2.index[fac2.index < month_end]
    if len(idx) < REGIME_LOOKBACK_M:
        return False, {"regime_ready": False}

    win = idx[-REGIME_LOOKBACK_M:]
    s_mkt = float(pd.to_numeric(fac2.loc[win, "MKT"], errors="coerce").sum())
    s_wml = float(pd.to_numeric(fac2.loc[win, "WML"], errors="coerce").sum())

    off = (s_mkt < REGIME_OFF_IF_SUM_MKT_LT) or (s_wml < REGIME_OFF_IF_SUM_WML_LT)
    return bool(off), {"regime_ready": True, "sum_MKT_3m": s_mkt, "sum_WML_3m": s_wml}


def estimate_port_beta(monthly_port_rets: pd.DataFrame, fac: pd.DataFrame, month_end: pd.Timestamp) -> Tuple[bool, Dict]:
    fac2 = fac.set_index("MonthEnd")
    idx = fac2.index[fac2.index < month_end]
    if len(idx) < BETA_EST_WINDOW_M:
        return False, {"beta_ready": False}

    win = idx[-BETA_EST_WINDOW_M:]

    m = monthly_port_rets.copy()
    m["MonthEnd"] = pd.to_datetime(m["MonthEnd"], errors="coerce").dt.normalize()
    m = m.dropna(subset=["MonthEnd", "port_ret"]).copy()

    m = (
        m.groupby("MonthEnd", as_index=False)["port_ret"]
         .apply(lambda s: (1.0 + s.astype(float)).prod() - 1.0)
    )

    df = m[m["MonthEnd"].isin(win)].merge(
        fac, on="MonthEnd", how="left", validate="one_to_one"
    ).dropna(subset=["port_ret"] + FACTOR_COLS)

    if len(df) < BETA_EST_MIN_OBS:
        return False, {"beta_ready": False, "n_obs": int(len(df))}

    y = df["port_ret"].to_numpy(dtype=float)
    X = df[FACTOR_COLS].to_numpy(dtype=float)
    alpha, betas, r2 = ols_alpha_beta(y, X)

    out = {"beta_ready": True, "n_obs": int(len(df)), "r2": float(r2), "alpha": float(alpha)}
    for c, b in zip(FACTOR_COLS, betas):
        out[f"beta_{c}"] = float(b)
    return True, out


def decide_invest_ratio(month_end: pd.Timestamp,
                        fac: pd.DataFrame,
                        port_hist: pd.DataFrame) -> Tuple[float, Dict]:
    invest_ratio = 1.0
    diag = {
        "MonthEnd": month_end,
        "invest_ratio": 1.0,
        "regime_off": False,
        "beta_off": False,
        "regime_ready": False,
        "beta_ready": False,
        "sum_MKT_3m": np.nan,
        "sum_WML_3m": np.nan,
        "beta_MKT": np.nan,
        "beta_CMA": np.nan,
        "beta_n_obs": np.nan,
        "beta_r2": np.nan,
    }

    off_reg, info = compute_regime_off(fac, month_end)
    diag["regime_off"] = bool(off_reg)
    diag["regime_ready"] = bool(info.get("regime_ready", False))
    diag["sum_MKT_3m"] = info.get("sum_MKT_3m", np.nan)
    diag["sum_WML_3m"] = info.get("sum_WML_3m", np.nan)
    if off_reg:
        invest_ratio = min(invest_ratio, RISKOFF_RATIO)

    ok_beta, binfo = estimate_port_beta(port_hist, fac, month_end)
    diag["beta_ready"] = bool(binfo.get("beta_ready", False))
    if binfo.get("beta_ready", False):
        b_mkt = float(binfo.get("beta_MKT", np.nan))
        b_cma = float(binfo.get("beta_CMA", np.nan))
        diag["beta_MKT"] = b_mkt
        diag["beta_CMA"] = b_cma
        diag["beta_n_obs"] = float(binfo.get("n_obs", np.nan))
        diag["beta_r2"] = float(binfo.get("r2", np.nan))

        off_beta = (b_cma < BETA_CAP_CMA_LT) or (abs(b_mkt) > BETA_CAP_ABS_MKT_GT)
        diag["beta_off"] = bool(off_beta)
        if off_beta:
            invest_ratio = min(invest_ratio, RISKOFF_RATIO)

    diag["invest_ratio"] = float(invest_ratio)
    return float(invest_ratio), diag


def compute_monthly_portfolio_returns_with_riskcontrol_fast(
    portfolio: dict,
    start_date: pd.Timestamp,
    end_date: pd.Timestamp,
    prices_by_code: Dict[str, pd.DataFrame],
    fac: pd.DataFrame
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    stocks = portfolio["stocks"]
    shares = portfolio["shares"]
    if not stocks:
        return pd.DataFrame(), pd.DataFrame()

    month_ends = make_month_ends(start_date, end_date)
    boundaries = [start_date] + [me for me in month_ends if (me > start_date) and (me < end_date)] + [end_date]

    rows = []
    ir_rows = []
    port_hist = pd.DataFrame(columns=["MonthEnd", "port_ret"])

    for j in range(len(boundaries) - 1):
        d0 = boundaries[j]
        d1 = boundaries[j + 1]

        me = pd.Timestamp(d0.year, d0.month, 1) + pd.offsets.MonthEnd(0)
        me = pd.Timestamp(me).normalize()

        start_vals = []
        end_vals = []
        for i, code in enumerate(stocks):
            p0 = get_near_price_fast(prices_by_code, code, d0, kind="first")
            p1 = get_near_price_fast(prices_by_code, code, d1, kind="last")
            if p0 is None or p1 is None:
                continue
            sh = shares[i]
            start_vals.append(sh * p0)
            end_vals.append(sh * p1)

        if len(start_vals) == 0:
            continue

        start_v = float(np.sum(start_vals))
        end_v = float(np.sum(end_vals))
        stock_ret = (end_v / start_v) - 1.0

        port_hist = pd.concat([port_hist, pd.DataFrame([{"MonthEnd": me, "port_ret": stock_ret}])], ignore_index=True)

        invest_ratio, diag = decide_invest_ratio(me, fac, port_hist)
        ir_rows.append(diag)

        total_ret = invest_ratio * stock_ret

        rows.append({
            "MonthEnd": me,
            "period_start": d0,
            "period_end": d1,
            "port_ret_stock": stock_ret,
            "invest_ratio": invest_ratio,
            "port_ret_total": total_ret,
        })

    monthly_df = pd.DataFrame(rows)
    ir_diag_df = pd.DataFrame(ir_rows)
    return monthly_df, ir_diag_df


def build_long_candidates(enhanced_financial_data: pd.DataFrame, rebalance_date: pd.Timestamp) -> pd.DataFrame:
    current_data = enhanced_financial_data[enhanced_financial_data["DisclosedDate"] <= rebalance_date].copy()
    current_data = current_data.sort_values("DisclosedDate").groupby("Code").tail(1)

    if len(current_data) < 100:
        return pd.DataFrame()

    current_data["PBR_Rank"] = current_data["PBR"].rank(method="first", ascending=True)
    current_data["ROE_Rank"] = current_data["ROE"].rank(method="first", ascending=False)

    current_data["PBR_Quartile"] = pd.qcut(current_data["PBR_Rank"], q=4, labels=[1, 2, 3, 4])
    current_data["ROE_Quartile"] = pd.qcut(current_data["ROE_Rank"], q=4, labels=[1, 2, 3, 4])

    long_candidates = current_data[
        (current_data["PBR_Quartile"] == 1) &
        (current_data["ROE_Quartile"] == 4)
    ].nsmallest(50, "PBR")

    return long_candidates


def annual_return_from_monthly(monthly_rets: pd.Series) -> float:
    if monthly_rets.empty:
        return 0.0
    return float((1.0 + monthly_rets).prod() - 1.0)


def apply_tax_annual(gross_return: float, initial_capital: float) -> Tuple[float, float]:
    profit = gross_return * initial_capital
    taxable = max(profit, 0.0)
    tax = taxable * TAX_RATE
    net_profit = profit - tax
    net_return = net_profit / initial_capital
    tax_rate_total = tax / initial_capital
    return float(net_return), float(tax_rate_total)


def run_annual_backtest_with_and_without_riskcontrol_fast(
    enhanced_financial_data: pd.DataFrame,
    prices_df: pd.DataFrame,
    prices_by_code: Dict[str, pd.DataFrame],
    fac: pd.DataFrame,
    initial_capital: float = INITIAL_CAPITAL
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    rebalance_dates = [pd.Timestamp(f"{year}-10-01") for year in range(2016, 2026)]

    results_raw = []
    results_rc = []
    diag_ir_all = []

    daily_curves_raw = []
    daily_curves_rc  = []

    for i, rebalance_date in enumerate(tqdm(rebalance_dates[:-1], desc="10月1日リバランス（統合・高速）")):
        next_rebalance = rebalance_dates[i + 1]

        long_candidates = build_long_candidates(enhanced_financial_data, rebalance_date)
        if long_candidates.empty:
            logger.warning(f"{rebalance_date}: データ不足でスキップ")
            continue

        long_portfolio = build_unit_share_portfolio(
            long_candidates,
            target_positions=20,
            initial_capital=initial_capital
        )

        n_long = len(long_portfolio["stocks"])
        inv_amount = float(np.sum(long_portfolio["amounts"])) if n_long > 0 else 0.0
        inv_ratio_raw = inv_amount / initial_capital if initial_capital > 0 else 0.0

        logger.info(f"{rebalance_date.strftime('%Y-%m')}: long={n_long} invest={inv_amount:,.0f} ratio={inv_ratio_raw:.3f}")

        # --- RAW（年次 start->end の価格で計算）
        total_profit = 0.0
        total_start_value = 0.0

        for j, code in enumerate(long_portfolio["stocks"]):
            sh = long_portfolio["shares"][j]
            p0 = long_portfolio["prices"][j]
            p1 = get_near_price_fast(prices_by_code, code, next_rebalance, kind="last")
            if p1 is None:
                continue
            start_value = sh * p0
            end_value = sh * p1
            total_start_value += start_value
            total_profit += (end_value - start_value)

        gross_return_raw = (total_profit / initial_capital) if initial_capital > 0 else 0.0
        net_return_raw, tax_rate_raw = apply_tax_annual(gross_return_raw, initial_capital)

        results_raw.append({
            "date": next_rebalance,
            "strategy_return_gross": gross_return_raw,
            "strategy_return_net": net_return_raw,
            "tax": tax_rate_raw,
            "long_count": n_long,
            "investment_ratio": inv_ratio_raw,
        })

        # --- RiskControl（月次で縮尺）
        monthly_df, ir_df = compute_monthly_portfolio_returns_with_riskcontrol_fast(
            long_portfolio, rebalance_date, next_rebalance, prices_by_code, fac
        )

        if monthly_df.empty:
            gross_return_rc = 0.0
            inv_ratio_rc_avg = 0.0
        else:
            gross_return_rc = annual_return_from_monthly(monthly_df["port_ret_total"])
            inv_ratio_rc_avg = float(monthly_df["invest_ratio"].mean())

        net_return_rc, tax_rate_rc = apply_tax_annual(gross_return_rc, initial_capital)

        results_rc.append({
            "date": next_rebalance,
            "strategy_return_gross": gross_return_rc,
            "strategy_return_net": net_return_rc,
            "tax": tax_rate_rc,
            "long_count": n_long,
            "investment_ratio": inv_ratio_rc_avg,
        })

        if not ir_df.empty:
            ir_df = ir_df.copy()
            ir_df["rebalance_start"] = rebalance_date
            ir_df["rebalance_end"] = next_rebalance
            diag_ir_all.append(ir_df)

        invest_ratio_map = None
        if not ir_df.empty:
            tmp_ir = ir_df.copy()
            tmp_ir["MonthEnd"] = pd.to_datetime(tmp_ir["MonthEnd"], errors="coerce").dt.normalize()
            tmp_ir = tmp_ir.dropna(subset=["MonthEnd"])
            tmp_ir = tmp_ir.sort_values("MonthEnd").drop_duplicates(subset=["MonthEnd"], keep="last")
            invest_ratio_map = dict(zip(tmp_ir["MonthEnd"], tmp_ir["invest_ratio"].astype(float)))

        daily_raw = build_daily_equity_curve_for_period(
            long_portfolio, rebalance_date, next_rebalance, prices_by_code, invest_ratio_by_monthend=None
        )
        if not daily_raw.empty:
            daily_raw["rebalance_start"] = rebalance_date
            daily_raw["rebalance_end"] = next_rebalance
            daily_curves_raw.append(daily_raw)

        daily_rc = build_daily_equity_curve_for_period(
            long_portfolio, rebalance_date, next_rebalance, prices_by_code, invest_ratio_by_monthend=invest_ratio_map
        )
        if not daily_rc.empty:
            daily_rc["rebalance_start"] = rebalance_date
            daily_rc["rebalance_end"] = next_rebalance
            daily_curves_rc.append(daily_rc)

    df_raw = pd.DataFrame(results_raw)
    df_rc = pd.DataFrame(results_rc)
    diag_ir = pd.concat(diag_ir_all, ignore_index=True) if len(diag_ir_all) else pd.DataFrame()

    daily_raw_all = pd.concat(daily_curves_raw, ignore_index=True) if len(daily_curves_raw) else pd.DataFrame()
    daily_rc_all  = pd.concat(daily_curves_rc,  ignore_index=True) if len(daily_curves_rc)  else pd.DataFrame()

    # 関数属性に退避（既存コード互換）
    run_annual_backtest_with_and_without_riskcontrol_fast._daily_raw_all = daily_raw_all
    run_annual_backtest_with_and_without_riskcontrol_fast._daily_rc_all = daily_rc_all

    return df_raw, df_rc, diag_ir


def build_annual_factor_from_monthly(fac: pd.DataFrame, start_date: pd.Timestamp, end_date: pd.Timestamp) -> Dict:
    fac2 = fac.copy()
    fac2 = fac2[(fac2["MonthEnd"] >= (start_date + pd.offsets.MonthEnd(0))) &
                (fac2["MonthEnd"] <= (end_date + pd.offsets.MonthEnd(-1)))].copy()
    out = {}
    for c in FACTOR_COLS:
        s = pd.to_numeric(fac2[c], errors="coerce").dropna()
        out[c] = float((1.0 + s).prod() - 1.0) if len(s) else np.nan
    return out


def run_factor_regression(results_df: pd.DataFrame, fac: pd.DataFrame) -> pd.DataFrame:
    if results_df.empty:
        return pd.DataFrame()

    rows = []
    for _, row in results_df.iterrows():
        end_date = pd.Timestamp(row["date"])
        start_date = end_date - pd.DateOffset(years=1)
        ann_fac = build_annual_factor_from_monthly(fac, start_date, end_date)
        rec = {"date": end_date}
        rec.update(ann_fac)
        rec["y"] = float(row["strategy_return_net"])
        rows.append(rec)

    df = pd.DataFrame(rows).dropna(subset=["y"] + FACTOR_COLS).copy()
    if len(df) < 3:
        return pd.DataFrame([{
            "n_years": int(len(df)),
            "R2": np.nan,
            "alpha": np.nan,
            **{f"beta_{c}": np.nan for c in FACTOR_COLS}
        }])

    y = df["y"].to_numpy(dtype=float)
    X = df[FACTOR_COLS].to_numpy(dtype=float)
    alpha, betas, r2 = ols_alpha_beta(y, X)

    out = {"n_years": int(len(df)), "R2": float(r2), "alpha": float(alpha)}
    for c, b in zip(FACTOR_COLS, betas):
        out[f"beta_{c}"] = float(b)
    return pd.DataFrame([out])


def save_annual_bundle(df: pd.DataFrame, out_annual_csv: Path, out_curve_csv: Path, out_perf_csv: Path):
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date").reset_index(drop=True)

    df.to_csv(out_annual_csv, index=False, encoding="utf-8-sig")

    r = df["strategy_return_net"].astype(float)
    cum = (1.0 + r).cumprod()
    dd = compute_drawdown(cum)
    curve = pd.DataFrame({"date": df["date"], "ret": r, "cum": cum, "dd": dd})
    curve.to_csv(out_curve_csv, index=False, encoding="utf-8-sig")

    stats = perf_stats(r)
    perf = pd.DataFrame([stats])
    perf.to_csv(out_perf_csv, index=False, encoding="utf-8-sig")


def run_one_case(
    riskoff_ratio: float,
    enhanced_financial_data: pd.DataFrame,
    prices_df: pd.DataFrame,
    prices_by_code: Dict[str, pd.DataFrame],
    fac: pd.DataFrame
) -> Dict:
    """
    1ケース実行して、比較指標を辞書で返す（加えてCSVも保存する）
    """
    global RISKOFF_RATIO
    RISKOFF_RATIO = float(riskoff_ratio)

    suffix = f"riskoff_{str(riskoff_ratio).replace('.', 'p')}"
    set_output_dir(BASE_OUT_DIR, suffix)

    logger.info("=" * 110)
    logger.info(f"[GRID] START case: RISKOFF_RATIO={RISKOFF_RATIO} | OUT_DIR={OUT_DIR}")
    logger.info("=" * 110)

    df_raw, df_rc, diag_ir = run_annual_backtest_with_and_without_riskcontrol_fast(
        enhanced_financial_data, prices_df, prices_by_code, fac, initial_capital=INITIAL_CAPITAL
    )

    if df_raw.empty or df_rc.empty:
        logger.error("年次バックテスト結果が空です")
        return {
            "riskoff_ratio": RISKOFF_RATIO,
            "daily_mdd": np.nan,
            "cagr": np.nan,
            "days_to_recovery": np.nan,
            "note": "empty_result"
        }

    # 年次出力
    save_annual_bundle(df_raw, OUT_ANNUAL_RAW, OUT_CURVE_RAW, OUT_PERF_RAW)
    save_annual_bundle(df_rc,  OUT_ANNUAL_RC,  OUT_CURVE_RC,  OUT_PERF_RC)

    # 回帰
    reg_raw = run_factor_regression(df_raw, fac)
    reg_rc  = run_factor_regression(df_rc, fac)
    reg_raw.to_csv(OUT_REG_RAW, index=False, encoding="utf-8-sig")
    reg_rc.to_csv(OUT_REG_RC, index=False, encoding="utf-8-sig")

    # 診断（月次）
    if not diag_ir.empty:
        diag_ir = diag_ir.copy()
        diag_ir["MonthEnd"] = pd.to_datetime(diag_ir["MonthEnd"])
        diag_ir.sort_values(["rebalance_start", "MonthEnd"], inplace=True)
        diag_ir.to_csv(OUT_IR_DIAG, index=False, encoding="utf-8-sig")

    # 日次出力
    daily_raw_all = getattr(run_annual_backtest_with_and_without_riskcontrol_fast, "_daily_raw_all", pd.DataFrame())
    daily_rc_all  = getattr(run_annual_backtest_with_and_without_riskcontrol_fast, "_daily_rc_all", pd.DataFrame())

    if not daily_raw_all.empty:
        daily_raw_all.to_csv(OUT_DAILY_CURVE_RAW, index=False, encoding="utf-8-sig")
    if not daily_rc_all.empty:
        daily_rc_all.to_csv(OUT_DAILY_CURVE_RC, index=False, encoding="utf-8-sig")

    # 日次MDD + DDイベント
    mdd_raw_d = daily_mdd(daily_raw_all)
    mdd_rc_d  = daily_mdd(daily_rc_all)

    ev_raw = extract_max_drawdown_event_from_daily_curve(daily_raw_all, label="RAW")
    ev_rc  = extract_max_drawdown_event_from_daily_curve(daily_rc_all,  label="RISKCONTROL")

    ev_raw.to_csv(OUT_DD_EVENTS_RAW, index=False, encoding="utf-8-sig")
    ev_rc.to_csv(OUT_DD_EVENTS_RC, index=False, encoding="utf-8-sig")
    pd.concat([ev_raw, ev_rc], ignore_index=True).to_csv(OUT_DD_EVENTS_MAX, index=False, encoding="utf-8-sig")

    pd.DataFrame([{
        "daily_maxDD_raw": mdd_raw_d,
        "daily_maxDD_riskcontrol": mdd_rc_d,
        "n_days_raw": int(len(daily_raw_all)) if not daily_raw_all.empty else 0,
        "n_days_riskcontrol": int(len(daily_rc_all)) if not daily_rc_all.empty else 0,
    }]).to_csv(OUT_DAILY_MDD_SUMMARY, index=False, encoding="utf-8-sig")

    # CAGR（RC側）
    rc_perf = pd.read_csv(OUT_PERF_RC).iloc[0].to_dict() if OUT_PERF_RC.exists() else {}
    cagr_rc = float(rc_perf.get("CAGR", np.nan))

    # 回復日数（RC側）
    days_to_recovery = float(ev_rc.iloc[0].get("days_to_recovery", np.nan)) if not ev_rc.empty else np.nan

    logger.info(f"[GRID] DONE case: RISKOFF_RATIO={RISKOFF_RATIO} | dailyMDD(RC)={mdd_rc_d:.6f} | CAGR(RC)={cagr_rc:.6f} | days_to_recovery={days_to_recovery}")

    return {
        "riskoff_ratio": float(RISKOFF_RATIO),
        "daily_mdd": float(mdd_rc_d) if np.isfinite(mdd_rc_d) else np.nan,
        "cagr": float(cagr_rc) if np.isfinite(cagr_rc) else np.nan,
        "days_to_recovery": float(days_to_recovery) if np.isfinite(days_to_recovery) else np.nan,
        "out_dir": str(OUT_DIR),
        "note": ""
    }


def main():
    logger.info("=" * 110)
    logger.info("10/1 年次「割安高質」バックテスト + FF5+MOM RiskControl(C) [GRID RISKOFF_RATIO 0.7/0.8/0.9]")
    logger.info("=" * 110)
    logger.info(f"FF5MOM_FACTOR_PATH: {FF5MOM_FACTOR_PATH}")
    logger.info(f"BASE_OUT_DIR: {BASE_OUT_DIR}")
    logger.info(f"Tax mode: annual simple tax only (TAX_RATE={TAX_RATE}) | daily tax ignored")
    logger.info("RiskControl: regime(3m MKT<0 OR WML<0) + beta(CMA<-0.8 or |MKT|>0.9) => riskoff=RISKOFF_RATIO")

    # ---- 入力データ読み込み（ここは1回だけ）
    statements_df = load_financial_data()
    if statements_df.empty:
        logger.error("財務データ読み込み失敗")
        return

    prices_df = load_existing_price_data()
    if prices_df.empty:
        logger.error("株価データ読み込み失敗")
        return

    fac = load_ff5mom_factors_monthly()
    prices_by_code = build_prices_by_code(prices_df)

    enhanced_financial_data = calculate_market_metrics_fast_chunked(statements_df, prices_df, chunk_size=200)
    if enhanced_financial_data.empty:
        logger.error("財務指標計算失敗")
        return

    # ---- グリッド実験
    grid = [0.7, 0.8, 0.9]
    rows = []
    for x in grid:
        try:
            rec = run_one_case(
                riskoff_ratio=x,
                enhanced_financial_data=enhanced_financial_data,
                prices_df=prices_df,
                prices_by_code=prices_by_code,
                fac=fac,
            )
            rows.append(rec)
        except Exception as e:
            logger.error(f"[GRID] case failed: RISKOFF_RATIO={x} | err={e}", exc_info=True)
            rows.append({
                "riskoff_ratio": float(x),
                "daily_mdd": np.nan,
                "cagr": np.nan,
                "days_to_recovery": np.nan,
                "out_dir": "",
                "note": f"error: {e}"
            })

    summary = pd.DataFrame(rows)
    summary = summary.sort_values("riskoff_ratio").reset_index(drop=True)
    summary.to_csv(OUT_GRID_SUMMARY, index=False, encoding="utf-8-sig")

    logger.info("-" * 110)
    logger.info("✅ GRID SUMMARY SAVED")
    logger.info(f"  - {OUT_GRID_SUMMARY}")
    logger.info("-" * 110)
    logger.info("[GRID TABLE]")
    logger.info(summary.to_string(index=False))


if __name__ == "__main__":
    main()


2026-01-27 03:59:16,931 - INFO - ==============================================================================================================
2026-01-27 03:59:16,932 - INFO - 10/1 年次「割安高質」バックテスト + FF5+MOM RiskControl(C) [GRID RISKOFF_RATIO 0.7/0.8/0.9]
2026-01-27 03:59:16,933 - INFO - ==============================================================================================================
2026-01-27 03:59:16,933 - INFO - FF5MOM_FACTOR_PATH: C:\Users\yongr\Project\merged_data_all_stocks\factors\ff5_mom_factors_monthly.parquet
2026-01-27 03:59:16,933 - INFO - BASE_OUT_DIR: C:\Users\yongr\Project\merged_data_all_stocks\factors\bt_october_unit_with_ff5mom_riskcontrol_grid
2026-01-27 03:59:16,934 - INFO - Tax mode: annual simple tax only (TAX_RATE=0.20315) | daily tax ignored
2026-01-27 03:59:16,934 - INFO - RiskControl: regime(3m MKT<0 OR WML<0) + beta(CMA<-0.8 or |MKT|>0.9) => riskoff=RISKOFF_RATIO
2026-01-27 03:59:17,537 - INFO - 財務データ読み込み成功: 64,222件
2026-01-27 03:59:17,565 - INFO

In [5]:
# -*- coding: utf-8 -*-
"""
年次（10/1）割安高質バックテスト（100株単位・税引後）に
FF5+MOMのリスク制御（C：レジーム減速＋β制約）を統合。

【追加：グリッド実験（Regime閾値）】
- RISKOFF_RATIO は 0.7 固定
- REGIME_OFF_IF_SUM_MKT_LT と REGIME_OFF_IF_SUM_WML_LT を
  (0.00, -0.02, -0.05) のグリッド比較（9ケース）
- 指標: 日次MDD・CAGR（年次税引後）・days_to_recovery（最大DDイベントの回復日数）
- 出力: ケース別サブフォルダ（従来CSV一式） + grid_summary.csv

税金:
- 簡易：年次（10/1→翌10/1）の最終損益にのみ課税
- 日次は税考慮しない（実験目的のため）

依存:
  pip install pandas numpy pyarrow tqdm
"""

import warnings
warnings.filterwarnings("ignore")

import os
import logging
from pathlib import Path
from typing import Dict, Tuple, List, Optional

import numpy as np
import pandas as pd
from tqdm import tqdm


# ===================================
# ロギング設定
# ===================================
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler("backtest_october_unit_with_ff5mom_regime_grid.log", encoding="utf-8"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)


# ===================================
# パス（あなたの環境）
# ===================================
FACTORS_DIR = Path(r"C:\Users\yongr\Project\merged_data_all_stocks\factors")
FF5MOM_FACTOR_PATH = FACTORS_DIR / "ff5_mom_factors_monthly.parquet"

CACHE_FILE = "topix_quarterly_statements.csv"
OHLCV_DIR = "./OHLCV_Adjusted"


# ===================================
# 出力（グリッド用ベース）
# ===================================
BASE_OUT_DIR = FACTORS_DIR / "bt_october_unit_with_ff5mom_regime_grid"
BASE_OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_GRID_SUMMARY = BASE_OUT_DIR / "grid_summary.csv"


# ===================================
# 税・単位株
# ===================================
TAX_RATE = 0.20315
UNIT_SHARES = 100
INITIAL_CAPITAL = 10_000_000


# ===================================
# リスク制御パラメータ
# ===================================
# ★固定
RISKOFF_RATIO = 0.7

REGIME_LOOKBACK_M = 3
# ★グリッド実験で動的に差し替える
REGIME_OFF_IF_SUM_MKT_LT = 0.0
REGIME_OFF_IF_SUM_WML_LT = 0.0

BETA_CAP_CMA_LT = -0.8
BETA_CAP_ABS_MKT_GT = 0.9

BETA_EST_WINDOW_M = 12
BETA_EST_MIN_OBS = 10

FACTOR_COLS = ["MKT", "SMB", "HML", "RMW", "CMA", "WML"]

BAD_CODE_STRINGS = {"None", "nan", "", "NaN", "NULL", "null"}


# ===================================
# 出力パス（ケースごとに set_output_dir で差し替える）
# ===================================
OUT_DIR = BASE_OUT_DIR / "case_tmp"
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_ANNUAL_RAW = OUT_DIR / "annual_returns_raw.csv"
OUT_ANNUAL_RC  = OUT_DIR / "annual_returns_riskcontrol.csv"

OUT_CURVE_RAW  = OUT_DIR / "cumulative_curve_raw.csv"
OUT_CURVE_RC   = OUT_DIR / "cumulative_curve_riskcontrol.csv"

OUT_PERF_RAW   = OUT_DIR / "performance_summary_raw.csv"
OUT_PERF_RC    = OUT_DIR / "performance_summary_riskcontrol.csv"

OUT_REG_RAW    = OUT_DIR / "regression_ff5mom_raw.csv"
OUT_REG_RC     = OUT_DIR / "regression_ff5mom_riskcontrol.csv"

OUT_IR_DIAG    = OUT_DIR / "diag_invest_ratio_monthly.csv"

OUT_DAILY_CURVE_RAW = OUT_DIR / "daily_equity_curve_raw.csv"
OUT_DAILY_CURVE_RC  = OUT_DIR / "daily_equity_curve_riskcontrol.csv"
OUT_DAILY_MDD_SUMMARY = OUT_DIR / "daily_mdd_summary.csv"

OUT_DD_EVENTS_RAW = OUT_DIR / "daily_drawdown_events_raw.csv"
OUT_DD_EVENTS_RC  = OUT_DIR / "daily_drawdown_events_riskcontrol.csv"
OUT_DD_EVENTS_MAX = OUT_DIR / "daily_drawdown_events_max_summary.csv"


def set_output_dir(base_dir: Path, suffix: str):
    """
    ループ実験で出力先を切り替えるための関数。
    suffix例: 'mkt0p00_wmlm0p02'
    """
    global OUT_DIR
    global OUT_ANNUAL_RAW, OUT_ANNUAL_RC
    global OUT_CURVE_RAW, OUT_CURVE_RC
    global OUT_PERF_RAW, OUT_PERF_RC
    global OUT_REG_RAW, OUT_REG_RC
    global OUT_IR_DIAG
    global OUT_DAILY_CURVE_RAW, OUT_DAILY_CURVE_RC, OUT_DAILY_MDD_SUMMARY
    global OUT_DD_EVENTS_RAW, OUT_DD_EVENTS_RC, OUT_DD_EVENTS_MAX

    OUT_DIR = base_dir / suffix
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    OUT_ANNUAL_RAW = OUT_DIR / "annual_returns_raw.csv"
    OUT_ANNUAL_RC  = OUT_DIR / "annual_returns_riskcontrol.csv"

    OUT_CURVE_RAW  = OUT_DIR / "cumulative_curve_raw.csv"
    OUT_CURVE_RC   = OUT_DIR / "cumulative_curve_riskcontrol.csv"

    OUT_PERF_RAW   = OUT_DIR / "performance_summary_raw.csv"
    OUT_PERF_RC    = OUT_DIR / "performance_summary_riskcontrol.csv"

    OUT_REG_RAW    = OUT_DIR / "regression_ff5mom_raw.csv"
    OUT_REG_RC     = OUT_DIR / "regression_ff5mom_riskcontrol.csv"

    OUT_IR_DIAG    = OUT_DIR / "diag_invest_ratio_monthly.csv"

    OUT_DAILY_CURVE_RAW = OUT_DIR / "daily_equity_curve_raw.csv"
    OUT_DAILY_CURVE_RC  = OUT_DIR / "daily_equity_curve_riskcontrol.csv"
    OUT_DAILY_MDD_SUMMARY = OUT_DIR / "daily_mdd_summary.csv"

    OUT_DD_EVENTS_RAW = OUT_DIR / "daily_drawdown_events_raw.csv"
    OUT_DD_EVENTS_RC  = OUT_DIR / "daily_drawdown_events_riskcontrol.csv"
    OUT_DD_EVENTS_MAX = OUT_DIR / "daily_drawdown_events_max_summary.csv"


def fmt_thr(x: float) -> str:
    """
    フォルダ名用の閾値表記
    0.00 -> 0p00, -0.02 -> m0p02
    """
    x = float(x)
    s = f"{abs(x):.2f}".replace(".", "p")
    return f"m{s}" if x < 0 else s


# ===================================
# ユーティリティ
# ===================================
def normalize_code(code) -> str:
    if code is None:
        return ""
    s = str(code).strip()
    if s in BAD_CODE_STRINGS:
        return ""
    return s


def ols_alpha_beta(y: np.ndarray, X: np.ndarray) -> Tuple[float, np.ndarray, float]:
    n = len(y)
    if n < 3:
        return np.nan, np.full(X.shape[1], np.nan), np.nan

    X1 = np.column_stack([np.ones(n), X])
    XtX = X1.T @ X1
    try:
        inv = np.linalg.inv(XtX)
    except np.linalg.LinAlgError:
        inv = np.linalg.pinv(XtX)
    b = inv @ (X1.T @ y)

    yhat = X1 @ b
    resid = y - yhat
    sse = float(np.sum(resid**2))
    sst = float(np.sum((y - y.mean())**2))
    r2 = np.nan if sst <= 0 else (1.0 - sse / sst)

    alpha = float(b[0])
    betas = b[1:].astype(float)
    return alpha, betas, float(r2)


def compute_drawdown(cum: pd.Series) -> pd.Series:
    peak = cum.cummax()
    return cum / peak - 1.0


def perf_stats(annual_ret: pd.Series) -> Dict:
    r = annual_ret.dropna().astype(float)
    if r.empty:
        return {"n_years": 0, "CAGR": np.nan, "ann_mean": np.nan, "ann_vol": np.nan, "sharpe0": np.nan, "maxDD": np.nan, "cum_end": np.nan}

    n = len(r)
    cum = (1.0 + r).cumprod()
    years = n
    cagr = float(cum.iloc[-1] ** (1/years) - 1.0) if years > 0 else np.nan
    ann_mean = float(r.mean())
    ann_vol = float(r.std(ddof=1)) if n >= 2 else np.nan
    sharpe0 = float(ann_mean / ann_vol) if ann_vol and ann_vol > 0 else np.nan
    dd = compute_drawdown(cum)
    maxdd = float(dd.min())

    return {
        "n_years": int(n),
        "CAGR": cagr,
        "ann_mean": ann_mean,
        "ann_vol": ann_vol,
        "sharpe0": sharpe0,
        "maxDD": maxdd,
        "cum_end": float(cum.iloc[-1]),
    }


def extract_max_drawdown_event_from_daily_curve(df_daily: pd.DataFrame, label: str = "") -> pd.DataFrame:
    if df_daily is None or df_daily.empty:
        return pd.DataFrame([{
            "label": label,
            "peak_date": pd.NaT,
            "trough_date": pd.NaT,
            "recovery_date": pd.NaT,
            "dd_min": np.nan,
            "peak_equity": np.nan,
            "trough_equity": np.nan,
            "recovery_equity": np.nan,
            "days_to_trough": np.nan,
            "days_to_recovery": np.nan,
            "dd_duration_days": np.nan,
        }])

    df = df_daily.copy()
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df = df.dropna(subset=["Date"]).sort_values("Date").reset_index(drop=True)

    for col in ["equity_total", "dd_total"]:
        if col not in df.columns:
            raise KeyError(f"daily curve missing required column: {col}")

    df["equity_total"] = pd.to_numeric(df["equity_total"], errors="coerce")
    df["dd_total"] = pd.to_numeric(df["dd_total"], errors="coerce")
    df = df.dropna(subset=["equity_total", "dd_total"]).copy()
    if df.empty:
        return pd.DataFrame([{
            "label": label,
            "peak_date": pd.NaT,
            "trough_date": pd.NaT,
            "recovery_date": pd.NaT,
            "dd_min": np.nan,
            "peak_equity": np.nan,
            "trough_equity": np.nan,
            "recovery_equity": np.nan,
            "days_to_trough": np.nan,
            "days_to_recovery": np.nan,
            "dd_duration_days": np.nan,
        }])

    trough_idx = int(df["dd_total"].idxmin())
    dd_min = float(df.loc[trough_idx, "dd_total"])
    trough_date = pd.Timestamp(df.loc[trough_idx, "Date"])
    trough_equity = float(df.loc[trough_idx, "equity_total"])

    df_pre = df.loc[:trough_idx].copy()
    peak_equity = float(df_pre["equity_total"].max())
    peak_idx = int(df_pre["equity_total"].idxmax())
    peak_date = pd.Timestamp(df.loc[peak_idx, "Date"])

    df_post = df.loc[trough_idx:].copy()
    rec = df_post[df_post["equity_total"] >= peak_equity]
    if len(rec) == 0:
        recovery_date = pd.NaT
        recovery_equity = np.nan
        days_to_recovery = np.nan
        dd_duration_days = np.nan
    else:
        rec_idx = int(rec.index[0])
        recovery_date = pd.Timestamp(df.loc[rec_idx, "Date"])
        recovery_equity = float(df.loc[rec_idx, "equity_total"])
        days_to_recovery = int((recovery_date - peak_date).days)
        dd_duration_days = int((recovery_date - peak_date).days)

    days_to_trough = int((trough_date - peak_date).days)

    return pd.DataFrame([{
        "label": label,
        "peak_date": peak_date,
        "trough_date": trough_date,
        "recovery_date": recovery_date,
        "dd_min": dd_min,
        "peak_equity": peak_equity,
        "trough_equity": trough_equity,
        "recovery_equity": recovery_equity,
        "days_to_trough": days_to_trough,
        "days_to_recovery": days_to_recovery,
        "dd_duration_days": dd_duration_days,
    }])


def daily_mdd(df_daily: pd.DataFrame) -> float:
    if df_daily is None or df_daily.empty or "dd_total" not in df_daily.columns:
        return np.nan
    s = pd.to_numeric(df_daily["dd_total"], errors="coerce").dropna()
    return float(s.min()) if len(s) else np.nan


# ===================================
# A. 財務データ読み込み（列名自動判定）
# ===================================
def load_financial_data(cache_filename: str = CACHE_FILE) -> pd.DataFrame:
    if not os.path.exists(cache_filename):
        logger.error(f"財務データファイル '{cache_filename}' が見つかりません")
        return pd.DataFrame()

    try:
        df = pd.read_csv(cache_filename, encoding="utf-8-sig", parse_dates=["DisclosedDate"], low_memory=False)
        logger.info(f"財務データ読み込み成功: {len(df):,}件")

        column_mapping = {
            "IssuedShareTotal": ["IssuedShareTotal", "NumberOfIssuedAndOutstandingSharesAtTheEndOfFiscalYearIncludingTreasuryStock"],
            "Equity": ["Equity", "NetAssets", "TotalEquity"],
            "Profit": ["Profit", "NetIncome", "ProfitAttributableToOwnersOfParent"],
        }

        available_cols = df.columns.tolist()
        required_columns = {}
        for target_col, possible_names in column_mapping.items():
            found = False
            for possible_name in possible_names:
                if possible_name in available_cols:
                    required_columns[target_col] = possible_name
                    found = True
                    break
            if not found:
                required_columns[target_col] = None

        rename_dict = {v: k for k, v in required_columns.items() if v is not None}
        df = df.rename(columns=rename_dict)

        for col in ["IssuedShareTotal", "Equity", "Profit"]:
            if col not in df.columns:
                df[col] = 0

        base_cols = ["Code", "DisclosedDate"]
        if "CompanyName" in df.columns:
            base_cols.append("CompanyName")
        final_cols = base_cols + ["Profit", "Equity", "IssuedShareTotal"]
        df = df[final_cols].copy()

        logger.info(f"使用列: {df.columns.tolist()}")
        return df

    except Exception as e:
        logger.error(f"財務データ読み込みエラー: {e}", exc_info=True)
        return pd.DataFrame()


# ===================================
# B. 株価データ読み込み
# ===================================
def load_existing_price_data(ohlcv_dir: str = OHLCV_DIR) -> pd.DataFrame:
    if not os.path.exists(ohlcv_dir):
        logger.error(f"株価データディレクトリ '{ohlcv_dir}' が見つかりません。")
        return pd.DataFrame()

    csv_files = sorted([f for f in os.listdir(ohlcv_dir)
                        if f.startswith("OHLCV_Adjusted_") and f.endswith(".csv") and f != "OHLCV_Adjusted_TOPIX.csv"])

    if not csv_files:
        logger.error(f"ディレクトリ '{ohlcv_dir}' 内にCSVファイルが見つかりません。")
        return pd.DataFrame()

    logger.info(f"株価ファイル数: {len(csv_files)}個")

    all_dataframes = []
    usecols = ["Date", "Ticker", "AdjustmentClose"]

    for csv_file in tqdm(csv_files, desc="株価ファイル読み込み中"):
        file_path = os.path.join(ohlcv_dir, csv_file)
        try:
            df = pd.read_csv(
                file_path,
                usecols=usecols,
                parse_dates=["Date"],
                dtype={"Ticker": "Int64", "AdjustmentClose": "float32"},
            )
            df = df.drop_duplicates(subset=["Ticker", "Date"], keep="first")
            all_dataframes.append(df)
        except Exception as e:
            logger.warning(f"ファイル読み込みエラー ({csv_file}): {e}")
            continue

    if not all_dataframes:
        return pd.DataFrame()

    logger.info("全ファイルを結合中...")
    df_all = pd.concat(all_dataframes, ignore_index=True)
    logger.info(f"結合完了: {len(df_all):,}件")

    df_all = df_all.rename(columns={"Ticker": "Code", "AdjustmentClose": "Close"})
    df_all["Code"] = df_all["Code"].astype("str").str.replace("<NA>", "0").str.zfill(4)
    df_all = df_all.dropna(subset=["Close"])

    logger.info("重複除去 & ソート中...")
    df_all = df_all.sort_values(["Code", "Date"])
    df_all = df_all.drop_duplicates(subset=["Code", "Date"], keep="first")
    logger.info(f"重複除去後: {len(df_all):,}件")

    return df_all


def build_prices_by_code(prices_df: pd.DataFrame) -> Dict[str, pd.DataFrame]:
    d = {}
    tmp = prices_df[["Code", "Date", "Close"]].copy()
    tmp["Code"] = tmp["Code"].astype(str).map(normalize_code)
    tmp = tmp[tmp["Code"] != ""].copy()
    tmp = tmp.sort_values(["Code", "Date"])

    for code, g in tmp.groupby("Code", sort=False):
        gg = g[["Date", "Close"]].drop_duplicates(subset=["Date"], keep="last").sort_values("Date").copy()
        gg = gg.set_index("Date", drop=False)
        d[code] = gg

    logger.info(f"prices_by_code built: {len(d):,} codes")
    return d


def get_near_price_fast(prices_by_code: Dict[str, pd.DataFrame],
                        code: str,
                        ref_date: pd.Timestamp,
                        kind: str = "last") -> Optional[float]:
    code = normalize_code(code)
    if not code or code not in prices_by_code:
        return None
    g = prices_by_code[code]
    lo = ref_date - pd.Timedelta(days=5)
    hi = ref_date + pd.Timedelta(days=5)
    w = g.loc[(g["Date"] >= lo) & (g["Date"] <= hi)]
    if w.empty:
        return None
    return float(w.iloc[0]["Close"]) if kind == "first" else float(w.iloc[-1]["Close"])


def safe_code_to_int(code_series: pd.Series) -> pd.Series:
    cleaned = code_series.astype(str).str.replace(r"\D", "", regex=True)
    cleaned = cleaned.replace("", "0")
    return pd.to_numeric(cleaned, errors="coerce").fillna(0).astype("int64")


def calculate_market_metrics_fast_chunked(statements_df: pd.DataFrame,
                                          prices_df: pd.DataFrame,
                                          chunk_size: int = 200) -> pd.DataFrame:
    logger.info("時価総額・PBR・ROE計算中（チャンク処理版）...")

    if prices_df.empty:
        logger.error("株価データが空です。")
        return pd.DataFrame()

    req_cols = {"Code", "Date", "Close"}
    if not req_cols.issubset(set(prices_df.columns)):
        logger.error(f"株価データに必要な列が存在しません。存在する列: {prices_df.columns.tolist()}")
        return pd.DataFrame()

    statements_df = statements_df.copy()
    statements_df["Profit"] = pd.to_numeric(statements_df["Profit"], errors="coerce").fillna(0)
    statements_df["Equity"] = pd.to_numeric(statements_df["Equity"], errors="coerce").fillna(0)
    statements_df["IssuedShareTotal"] = pd.to_numeric(statements_df["IssuedShareTotal"], errors="coerce").fillna(1)

    statements_df = statements_df[(statements_df["Equity"] > 0) & (statements_df["IssuedShareTotal"] > 0)]
    logger.info(f"有効な財務データ: {len(statements_df):,}件")

    statements_df["Code_int"] = safe_code_to_int(statements_df["Code"])
    prices_df = prices_df.copy()
    prices_df["Code_int"] = safe_code_to_int(prices_df["Code"])

    statements_df = statements_df[statements_df["Code_int"] > 0]
    prices_df = prices_df[prices_df["Code_int"] > 0]

    statements_df = statements_df.sort_values(["Code_int", "DisclosedDate"]).reset_index(drop=True)
    prices_df = prices_df.sort_values(["Code_int", "Date"]).reset_index(drop=True)

    statements_df = statements_df.drop_duplicates(subset=["Code_int", "DisclosedDate"], keep="first")
    prices_df = prices_df.drop_duplicates(subset=["Code_int", "Date"], keep="first")

    logger.info(f"ソート・重複除去後: 財務 {len(statements_df):,}件, 株価 {len(prices_df):,}件")

    statements_groups = list(statements_df.groupby("Code_int"))
    prices_dict = {code: group for code, group in prices_df.groupby("Code_int")}

    merged_list = []
    num_chunks = (len(statements_groups) + chunk_size - 1) // chunk_size

    for chunk_idx in tqdm(range(num_chunks), desc="マージ処理"):
        start_idx = chunk_idx * chunk_size
        end_idx = min((chunk_idx + 1) * chunk_size, len(statements_groups))
        chunk_groups = statements_groups[start_idx:end_idx]

        for code, stmt_code in chunk_groups:
            if code not in prices_dict:
                continue
            price_code = prices_dict[code]
            if len(price_code) == 0:
                continue

            stmt_code = stmt_code.sort_values("DisclosedDate").reset_index(drop=True)
            price_code = price_code.sort_values("Date").reset_index(drop=True)

            try:
                merged = pd.merge_asof(
                    stmt_code,
                    price_code[["Date", "Close"]],
                    left_on="DisclosedDate",
                    right_on="Date",
                    direction="backward",
                    tolerance=pd.Timedelta(days=10),
                )
                if not merged.empty:
                    merged_list.append(merged)
            except Exception:
                continue

    if not merged_list:
        logger.error("マージ結果が空です")
        return pd.DataFrame()

    df_merged = pd.concat(merged_list, ignore_index=True)
    df_merged = df_merged.dropna(subset=["Close"])
    logger.info(f"マージ完了: {len(df_merged):,}件")

    df_merged["MarketCap"] = df_merged["Close"] * df_merged["IssuedShareTotal"]
    df_merged["PBR"] = df_merged["MarketCap"] / df_merged["Equity"]
    df_merged["ROE"] = (df_merged["Profit"] / df_merged["Equity"]) * 100

    result_cols = ["Code", "DisclosedDate", "Close", "MarketCap", "PBR", "ROE", "Date"]
    if "CompanyName" in df_merged.columns:
        result_cols.insert(1, "CompanyName")

    result_df = df_merged[result_cols].copy()
    result_df = result_df.rename(columns={"Close": "StockPrice", "Date": "PriceDate"})

    mask = (
        (result_df["PBR"] > 0) &
        (result_df["PBR"] < 50) &
        (result_df["ROE"] > -100) &
        (result_df["ROE"] < 100) &
        (result_df["MarketCap"] > 1_000_000_000)
    )
    result_df = result_df[mask].copy()

    logger.info(f"計算完了: {len(result_df):,}件")
    return result_df


def build_unit_share_portfolio(stock_candidates: pd.DataFrame,
                               target_positions: int = 20,
                               initial_capital: float = 10_000_000) -> dict:
    if len(stock_candidates) == 0:
        return {"stocks": [], "shares": [], "prices": [], "amounts": []}

    selected = stock_candidates.head(target_positions).copy()
    capital_per_stock = initial_capital / len(selected)

    stocks, shares_list, prices_list, amounts_list = [], [], [], []

    for _, row in selected.iterrows():
        code = row["Code"]
        price = row["StockPrice"]
        required_amount = price * UNIT_SHARES

        if required_amount <= capital_per_stock:
            shares = int(capital_per_stock // required_amount) * UNIT_SHARES
            if shares > 0:
                stocks.append(code)
                shares_list.append(shares)
                prices_list.append(price)
                amounts_list.append(shares * price)

    return {"stocks": stocks, "shares": shares_list, "prices": prices_list, "amounts": amounts_list}


def make_month_ends(start: pd.Timestamp, end: pd.Timestamp) -> List[pd.Timestamp]:
    m0 = pd.Timestamp(start.year, start.month, 1) + pd.offsets.MonthEnd(0)
    m1 = pd.Timestamp(end.year, end.month, 1) + pd.offsets.MonthEnd(0)
    months = pd.date_range(m0, m1, freq="M")
    return [pd.Timestamp(x).normalize() for x in months]


def build_daily_equity_curve_for_period(
    portfolio: dict,
    start_date: pd.Timestamp,
    end_date: pd.Timestamp,
    prices_by_code: Dict[str, pd.DataFrame],
    invest_ratio_by_monthend: Optional[Dict[pd.Timestamp, float]] = None,
) -> pd.DataFrame:
    stocks = portfolio.get("stocks", [])
    shares = portfolio.get("shares", [])
    if not stocks:
        return pd.DataFrame()

    start_date = pd.to_datetime(start_date).normalize()
    end_date = pd.to_datetime(end_date).normalize()

    date_sets = []
    for code in stocks:
        code = normalize_code(code)
        if code in prices_by_code:
            g = prices_by_code[code]
            d = g[(g["Date"] >= start_date - pd.Timedelta(days=10)) & (g["Date"] <= end_date + pd.Timedelta(days=10))]["Date"]
            if len(d):
                date_sets.append(d)

    if not date_sets:
        return pd.DataFrame()

    dates = pd.Index(sorted(pd.unique(pd.concat(date_sets)))).astype("datetime64[ns]")
    dates = dates[(dates >= start_date) & (dates <= end_date)]
    if len(dates) == 0:
        return pd.DataFrame()

    values = []
    for dt in dates:
        v = 0.0
        ok = False
        for i, code in enumerate(stocks):
            code = normalize_code(code)
            if code not in prices_by_code:
                continue
            p = get_near_price_fast(prices_by_code, code, pd.Timestamp(dt), kind="last")
            if p is None:
                continue
            v += float(shares[i]) * float(p)
            ok = True
        values.append(v if ok else np.nan)

    df = pd.DataFrame({"Date": pd.to_datetime(dates), "equity_stock": values})
    df = df.dropna(subset=["equity_stock"]).copy()
    if df.empty:
        return df

    df = df.sort_values("Date").reset_index(drop=True)
    df["ret_stock"] = df["equity_stock"].pct_change().fillna(0.0)
    df["MonthEnd"] = (df["Date"] + pd.offsets.MonthEnd(0)).dt.normalize()

    if invest_ratio_by_monthend is None:
        df["invest_ratio"] = 1.0
        df["ret_total"] = df["ret_stock"]
    else:
        df["invest_ratio"] = df["MonthEnd"].map(invest_ratio_by_monthend).fillna(1.0).astype(float)
        df["ret_total"] = df["invest_ratio"] * df["ret_stock"]

    df["equity_total"] = (1.0 + df["ret_total"]).cumprod()
    peak = df["equity_total"].cummax()
    df["dd_total"] = df["equity_total"] / peak - 1.0

    return df[["Date", "MonthEnd", "equity_stock", "ret_stock", "invest_ratio", "ret_total", "equity_total", "dd_total"]]


def load_ff5mom_factors_monthly() -> pd.DataFrame:
    fac = pd.read_parquet(FF5MOM_FACTOR_PATH).copy()
    fac["MonthEnd"] = pd.to_datetime(fac["MonthEnd"], errors="coerce").dt.normalize()
    need = ["MonthEnd"] + FACTOR_COLS
    missing = [c for c in need if c not in fac.columns]
    if missing:
        raise KeyError(f"FF5MOM factors missing columns: {missing} in {FF5MOM_FACTOR_PATH}")
    fac = fac[need].sort_values("MonthEnd").reset_index(drop=True)
    return fac


def compute_regime_off(fac: pd.DataFrame, month_end: pd.Timestamp) -> Tuple[bool, Dict]:
    fac2 = fac.set_index("MonthEnd")
    idx = fac2.index[fac2.index < month_end]
    if len(idx) < REGIME_LOOKBACK_M:
        return False, {"regime_ready": False}

    win = idx[-REGIME_LOOKBACK_M:]
    s_mkt = float(pd.to_numeric(fac2.loc[win, "MKT"], errors="coerce").sum())
    s_wml = float(pd.to_numeric(fac2.loc[win, "WML"], errors="coerce").sum())

    # Option3: OR
    off = (s_mkt < REGIME_OFF_IF_SUM_MKT_LT) or (s_wml < REGIME_OFF_IF_SUM_WML_LT)
    return bool(off), {"regime_ready": True, "sum_MKT_3m": s_mkt, "sum_WML_3m": s_wml}


def estimate_port_beta(monthly_port_rets: pd.DataFrame, fac: pd.DataFrame, month_end: pd.Timestamp) -> Tuple[bool, Dict]:
    fac2 = fac.set_index("MonthEnd")
    idx = fac2.index[fac2.index < month_end]
    if len(idx) < BETA_EST_WINDOW_M:
        return False, {"beta_ready": False}

    win = idx[-BETA_EST_WINDOW_M:]

    m = monthly_port_rets.copy()
    m["MonthEnd"] = pd.to_datetime(m["MonthEnd"], errors="coerce").dt.normalize()
    m = m.dropna(subset=["MonthEnd", "port_ret"]).copy()

    m = (
        m.groupby("MonthEnd", as_index=False)["port_ret"]
         .apply(lambda s: (1.0 + s.astype(float)).prod() - 1.0)
    )

    df = m[m["MonthEnd"].isin(win)].merge(
        fac, on="MonthEnd", how="left", validate="one_to_one"
    ).dropna(subset=["port_ret"] + FACTOR_COLS)

    if len(df) < BETA_EST_MIN_OBS:
        return False, {"beta_ready": False, "n_obs": int(len(df))}

    y = df["port_ret"].to_numpy(dtype=float)
    X = df[FACTOR_COLS].to_numpy(dtype=float)
    alpha, betas, r2 = ols_alpha_beta(y, X)

    out = {"beta_ready": True, "n_obs": int(len(df)), "r2": float(r2), "alpha": float(alpha)}
    for c, b in zip(FACTOR_COLS, betas):
        out[f"beta_{c}"] = float(b)
    return True, out


def decide_invest_ratio(month_end: pd.Timestamp,
                        fac: pd.DataFrame,
                        port_hist: pd.DataFrame) -> Tuple[float, Dict]:
    invest_ratio = 1.0
    diag = {
        "MonthEnd": month_end,
        "invest_ratio": 1.0,
        "regime_off": False,
        "beta_off": False,
        "regime_ready": False,
        "beta_ready": False,
        "sum_MKT_3m": np.nan,
        "sum_WML_3m": np.nan,
        "beta_MKT": np.nan,
        "beta_CMA": np.nan,
        "beta_n_obs": np.nan,
        "beta_r2": np.nan,
    }

    off_reg, info = compute_regime_off(fac, month_end)
    diag["regime_off"] = bool(off_reg)
    diag["regime_ready"] = bool(info.get("regime_ready", False))
    diag["sum_MKT_3m"] = info.get("sum_MKT_3m", np.nan)
    diag["sum_WML_3m"] = info.get("sum_WML_3m", np.nan)
    if off_reg:
        invest_ratio = min(invest_ratio, RISKOFF_RATIO)

    ok_beta, binfo = estimate_port_beta(port_hist, fac, month_end)
    diag["beta_ready"] = bool(binfo.get("beta_ready", False))
    if binfo.get("beta_ready", False):
        b_mkt = float(binfo.get("beta_MKT", np.nan))
        b_cma = float(binfo.get("beta_CMA", np.nan))
        diag["beta_MKT"] = b_mkt
        diag["beta_CMA"] = b_cma
        diag["beta_n_obs"] = float(binfo.get("n_obs", np.nan))
        diag["beta_r2"] = float(binfo.get("r2", np.nan))

        off_beta = (b_cma < BETA_CAP_CMA_LT) or (abs(b_mkt) > BETA_CAP_ABS_MKT_GT)
        diag["beta_off"] = bool(off_beta)
        if off_beta:
            invest_ratio = min(invest_ratio, RISKOFF_RATIO)

    diag["invest_ratio"] = float(invest_ratio)
    return float(invest_ratio), diag


def compute_monthly_portfolio_returns_with_riskcontrol_fast(
    portfolio: dict,
    start_date: pd.Timestamp,
    end_date: pd.Timestamp,
    prices_by_code: Dict[str, pd.DataFrame],
    fac: pd.DataFrame
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    stocks = portfolio["stocks"]
    shares = portfolio["shares"]
    if not stocks:
        return pd.DataFrame(), pd.DataFrame()

    month_ends = make_month_ends(start_date, end_date)
    boundaries = [start_date] + [me for me in month_ends if (me > start_date) and (me < end_date)] + [end_date]

    rows = []
    ir_rows = []
    port_hist = pd.DataFrame(columns=["MonthEnd", "port_ret"])

    for j in range(len(boundaries) - 1):
        d0 = boundaries[j]
        d1 = boundaries[j + 1]

        me = pd.Timestamp(d0.year, d0.month, 1) + pd.offsets.MonthEnd(0)
        me = pd.Timestamp(me).normalize()

        start_vals = []
        end_vals = []
        for i, code in enumerate(stocks):
            p0 = get_near_price_fast(prices_by_code, code, d0, kind="first")
            p1 = get_near_price_fast(prices_by_code, code, d1, kind="last")
            if p0 is None or p1 is None:
                continue
            sh = shares[i]
            start_vals.append(sh * p0)
            end_vals.append(sh * p1)

        if len(start_vals) == 0:
            continue

        start_v = float(np.sum(start_vals))
        end_v = float(np.sum(end_vals))
        stock_ret = (end_v / start_v) - 1.0

        port_hist = pd.concat([port_hist, pd.DataFrame([{"MonthEnd": me, "port_ret": stock_ret}])], ignore_index=True)

        invest_ratio, diag = decide_invest_ratio(me, fac, port_hist)
        ir_rows.append(diag)

        total_ret = invest_ratio * stock_ret

        rows.append({
            "MonthEnd": me,
            "period_start": d0,
            "period_end": d1,
            "port_ret_stock": stock_ret,
            "invest_ratio": invest_ratio,
            "port_ret_total": total_ret,
        })

    monthly_df = pd.DataFrame(rows)
    ir_diag_df = pd.DataFrame(ir_rows)
    return monthly_df, ir_diag_df


def build_long_candidates(enhanced_financial_data: pd.DataFrame, rebalance_date: pd.Timestamp) -> pd.DataFrame:
    current_data = enhanced_financial_data[enhanced_financial_data["DisclosedDate"] <= rebalance_date].copy()
    current_data = current_data.sort_values("DisclosedDate").groupby("Code").tail(1)

    if len(current_data) < 100:
        return pd.DataFrame()

    current_data["PBR_Rank"] = current_data["PBR"].rank(method="first", ascending=True)
    current_data["ROE_Rank"] = current_data["ROE"].rank(method="first", ascending=False)

    current_data["PBR_Quartile"] = pd.qcut(current_data["PBR_Rank"], q=4, labels=[1, 2, 3, 4])
    current_data["ROE_Quartile"] = pd.qcut(current_data["ROE_Rank"], q=4, labels=[1, 2, 3, 4])

    long_candidates = current_data[
        (current_data["PBR_Quartile"] == 1) &
        (current_data["ROE_Quartile"] == 4)
    ].nsmallest(50, "PBR")

    return long_candidates


def annual_return_from_monthly(monthly_rets: pd.Series) -> float:
    if monthly_rets.empty:
        return 0.0
    return float((1.0 + monthly_rets).prod() - 1.0)


def apply_tax_annual(gross_return: float, initial_capital: float) -> Tuple[float, float]:
    profit = gross_return * initial_capital
    taxable = max(profit, 0.0)
    tax = taxable * TAX_RATE
    net_profit = profit - tax
    net_return = net_profit / initial_capital
    tax_rate_total = tax / initial_capital
    return float(net_return), float(tax_rate_total)


def run_annual_backtest_with_and_without_riskcontrol_fast(
    enhanced_financial_data: pd.DataFrame,
    prices_df: pd.DataFrame,
    prices_by_code: Dict[str, pd.DataFrame],
    fac: pd.DataFrame,
    initial_capital: float = INITIAL_CAPITAL
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    rebalance_dates = [pd.Timestamp(f"{year}-10-01") for year in range(2016, 2026)]

    results_raw = []
    results_rc = []
    diag_ir_all = []

    daily_curves_raw = []
    daily_curves_rc  = []

    for i, rebalance_date in enumerate(tqdm(rebalance_dates[:-1], desc="10月1日リバランス（統合・高速）")):
        next_rebalance = rebalance_dates[i + 1]

        long_candidates = build_long_candidates(enhanced_financial_data, rebalance_date)
        if long_candidates.empty:
            logger.warning(f"{rebalance_date}: データ不足でスキップ")
            continue

        long_portfolio = build_unit_share_portfolio(
            long_candidates,
            target_positions=20,
            initial_capital=initial_capital
        )

        n_long = len(long_portfolio["stocks"])
        inv_amount = float(np.sum(long_portfolio["amounts"])) if n_long > 0 else 0.0
        inv_ratio_raw = inv_amount / initial_capital if initial_capital > 0 else 0.0

        logger.info(f"{rebalance_date.strftime('%Y-%m')}: long={n_long} invest={inv_amount:,.0f} ratio={inv_ratio_raw:.3f}")

        # --- RAW（年次 start->end の価格で計算）
        total_profit = 0.0
        for j, code in enumerate(long_portfolio["stocks"]):
            sh = long_portfolio["shares"][j]
            p0 = long_portfolio["prices"][j]
            p1 = get_near_price_fast(prices_by_code, code, next_rebalance, kind="last")
            if p1 is None:
                continue
            start_value = sh * p0
            end_value = sh * p1
            total_profit += (end_value - start_value)

        gross_return_raw = (total_profit / initial_capital) if initial_capital > 0 else 0.0
        net_return_raw, tax_rate_raw = apply_tax_annual(gross_return_raw, initial_capital)

        results_raw.append({
            "date": next_rebalance,
            "strategy_return_gross": gross_return_raw,
            "strategy_return_net": net_return_raw,
            "tax": tax_rate_raw,
            "long_count": n_long,
            "investment_ratio": inv_ratio_raw,
        })

        # --- RiskControl（月次で縮尺）
        monthly_df, ir_df = compute_monthly_portfolio_returns_with_riskcontrol_fast(
            long_portfolio, rebalance_date, next_rebalance, prices_by_code, fac
        )

        if monthly_df.empty:
            gross_return_rc = 0.0
            inv_ratio_rc_avg = 0.0
        else:
            gross_return_rc = annual_return_from_monthly(monthly_df["port_ret_total"])
            inv_ratio_rc_avg = float(monthly_df["invest_ratio"].mean())

        net_return_rc, tax_rate_rc = apply_tax_annual(gross_return_rc, initial_capital)

        results_rc.append({
            "date": next_rebalance,
            "strategy_return_gross": gross_return_rc,
            "strategy_return_net": net_return_rc,
            "tax": tax_rate_rc,
            "long_count": n_long,
            "investment_ratio": inv_ratio_rc_avg,
        })

        if not ir_df.empty:
            ir_df = ir_df.copy()
            ir_df["rebalance_start"] = rebalance_date
            ir_df["rebalance_end"] = next_rebalance
            diag_ir_all.append(ir_df)

        invest_ratio_map = None
        if not ir_df.empty:
            tmp_ir = ir_df.copy()
            tmp_ir["MonthEnd"] = pd.to_datetime(tmp_ir["MonthEnd"], errors="coerce").dt.normalize()
            tmp_ir = tmp_ir.dropna(subset=["MonthEnd"])
            tmp_ir = tmp_ir.sort_values("MonthEnd").drop_duplicates(subset=["MonthEnd"], keep="last")
            invest_ratio_map = dict(zip(tmp_ir["MonthEnd"], tmp_ir["invest_ratio"].astype(float)))

        daily_raw = build_daily_equity_curve_for_period(
            long_portfolio, rebalance_date, next_rebalance, prices_by_code, invest_ratio_by_monthend=None
        )
        if not daily_raw.empty:
            daily_raw["rebalance_start"] = rebalance_date
            daily_raw["rebalance_end"] = next_rebalance
            daily_curves_raw.append(daily_raw)

        daily_rc = build_daily_equity_curve_for_period(
            long_portfolio, rebalance_date, next_rebalance, prices_by_code, invest_ratio_by_monthend=invest_ratio_map
        )
        if not daily_rc.empty:
            daily_rc["rebalance_start"] = rebalance_date
            daily_rc["rebalance_end"] = next_rebalance
            daily_curves_rc.append(daily_rc)

    df_raw = pd.DataFrame(results_raw)
    df_rc = pd.DataFrame(results_rc)
    diag_ir = pd.concat(diag_ir_all, ignore_index=True) if len(diag_ir_all) else pd.DataFrame()

    daily_raw_all = pd.concat(daily_curves_raw, ignore_index=True) if len(daily_curves_raw) else pd.DataFrame()
    daily_rc_all  = pd.concat(daily_curves_rc,  ignore_index=True) if len(daily_curves_rc)  else pd.DataFrame()

    # 既存互換
    run_annual_backtest_with_and_without_riskcontrol_fast._daily_raw_all = daily_raw_all
    run_annual_backtest_with_and_without_riskcontrol_fast._daily_rc_all = daily_rc_all

    return df_raw, df_rc, diag_ir


def build_annual_factor_from_monthly(fac: pd.DataFrame, start_date: pd.Timestamp, end_date: pd.Timestamp) -> Dict:
    fac2 = fac.copy()
    fac2 = fac2[(fac2["MonthEnd"] >= (start_date + pd.offsets.MonthEnd(0))) &
                (fac2["MonthEnd"] <= (end_date + pd.offsets.MonthEnd(-1)))].copy()
    out = {}
    for c in FACTOR_COLS:
        s = pd.to_numeric(fac2[c], errors="coerce").dropna()
        out[c] = float((1.0 + s).prod() - 1.0) if len(s) else np.nan
    return out


def run_factor_regression(results_df: pd.DataFrame, fac: pd.DataFrame) -> pd.DataFrame:
    if results_df.empty:
        return pd.DataFrame()

    rows = []
    for _, row in results_df.iterrows():
        end_date = pd.Timestamp(row["date"])
        start_date = end_date - pd.DateOffset(years=1)
        ann_fac = build_annual_factor_from_monthly(fac, start_date, end_date)
        rec = {"date": end_date}
        rec.update(ann_fac)
        rec["y"] = float(row["strategy_return_net"])
        rows.append(rec)

    df = pd.DataFrame(rows).dropna(subset=["y"] + FACTOR_COLS).copy()
    if len(df) < 3:
        return pd.DataFrame([{
            "n_years": int(len(df)),
            "R2": np.nan,
            "alpha": np.nan,
            **{f"beta_{c}": np.nan for c in FACTOR_COLS}
        }])

    y = df["y"].to_numpy(dtype=float)
    X = df[FACTOR_COLS].to_numpy(dtype=float)
    alpha, betas, r2 = ols_alpha_beta(y, X)

    out = {"n_years": int(len(df)), "R2": float(r2), "alpha": float(alpha)}
    for c, b in zip(FACTOR_COLS, betas):
        out[f"beta_{c}"] = float(b)
    return pd.DataFrame([out])


def save_annual_bundle(df: pd.DataFrame, out_annual_csv: Path, out_curve_csv: Path, out_perf_csv: Path):
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date").reset_index(drop=True)

    df.to_csv(out_annual_csv, index=False, encoding="utf-8-sig")

    r = df["strategy_return_net"].astype(float)
    cum = (1.0 + r).cumprod()
    dd = compute_drawdown(cum)
    curve = pd.DataFrame({"date": df["date"], "ret": r, "cum": cum, "dd": dd})
    curve.to_csv(out_curve_csv, index=False, encoding="utf-8-sig")

    stats = perf_stats(r)
    perf = pd.DataFrame([stats])
    perf.to_csv(out_perf_csv, index=False, encoding="utf-8-sig")


def run_one_case(
    mkt_thr: float,
    wml_thr: float,
    enhanced_financial_data: pd.DataFrame,
    prices_df: pd.DataFrame,
    prices_by_code: Dict[str, pd.DataFrame],
    fac: pd.DataFrame
) -> Dict:
    """
    1ケース実行して、比較指標を辞書で返す（加えてCSVも保存する）
    """
    global REGIME_OFF_IF_SUM_MKT_LT, REGIME_OFF_IF_SUM_WML_LT

    REGIME_OFF_IF_SUM_MKT_LT = float(mkt_thr)
    REGIME_OFF_IF_SUM_WML_LT = float(wml_thr)

    suffix = f"mkt_{fmt_thr(mkt_thr)}__wml_{fmt_thr(wml_thr)}"
    set_output_dir(BASE_OUT_DIR, suffix)

    logger.info("=" * 110)
    logger.info(f"[GRID] START case: RISKOFF_RATIO={RISKOFF_RATIO} | MKT_THR={REGIME_OFF_IF_SUM_MKT_LT} | WML_THR={REGIME_OFF_IF_SUM_WML_LT}")
    logger.info(f"[GRID] OUT_DIR={OUT_DIR}")
    logger.info("=" * 110)

    df_raw, df_rc, diag_ir = run_annual_backtest_with_and_without_riskcontrol_fast(
        enhanced_financial_data, prices_df, prices_by_code, fac, initial_capital=INITIAL_CAPITAL
    )

    if df_raw.empty or df_rc.empty:
        logger.error("年次バックテスト結果が空です")
        return {
            "mkt_thr": REGIME_OFF_IF_SUM_MKT_LT,
            "wml_thr": REGIME_OFF_IF_SUM_WML_LT,
            "daily_mdd": np.nan,
            "cagr": np.nan,
            "days_to_recovery": np.nan,
            "out_dir": str(OUT_DIR),
            "note": "empty_result"
        }

    # 年次出力
    save_annual_bundle(df_raw, OUT_ANNUAL_RAW, OUT_CURVE_RAW, OUT_PERF_RAW)
    save_annual_bundle(df_rc,  OUT_ANNUAL_RC,  OUT_CURVE_RC,  OUT_PERF_RC)

    # 回帰
    reg_raw = run_factor_regression(df_raw, fac)
    reg_rc  = run_factor_regression(df_rc, fac)
    reg_raw.to_csv(OUT_REG_RAW, index=False, encoding="utf-8-sig")
    reg_rc.to_csv(OUT_REG_RC, index=False, encoding="utf-8-sig")

    # 診断（月次）
    if not diag_ir.empty:
        diag_ir = diag_ir.copy()
        diag_ir["MonthEnd"] = pd.to_datetime(diag_ir["MonthEnd"])
        diag_ir.sort_values(["rebalance_start", "MonthEnd"], inplace=True)
        diag_ir.to_csv(OUT_IR_DIAG, index=False, encoding="utf-8-sig")

    # 日次出力
    daily_raw_all = getattr(run_annual_backtest_with_and_without_riskcontrol_fast, "_daily_raw_all", pd.DataFrame())
    daily_rc_all  = getattr(run_annual_backtest_with_and_without_riskcontrol_fast, "_daily_rc_all", pd.DataFrame())

    if not daily_raw_all.empty:
        daily_raw_all.to_csv(OUT_DAILY_CURVE_RAW, index=False, encoding="utf-8-sig")
    if not daily_rc_all.empty:
        daily_rc_all.to_csv(OUT_DAILY_CURVE_RC, index=False, encoding="utf-8-sig")

    # 日次MDD + DDイベント
    mdd_rc_d = daily_mdd(daily_rc_all)

    ev_rc = extract_max_drawdown_event_from_daily_curve(daily_rc_all, label="RISKCONTROL")
    ev_raw = extract_max_drawdown_event_from_daily_curve(daily_raw_all, label="RAW")

    ev_raw.to_csv(OUT_DD_EVENTS_RAW, index=False, encoding="utf-8-sig")
    ev_rc.to_csv(OUT_DD_EVENTS_RC, index=False, encoding="utf-8-sig")
    pd.concat([ev_raw, ev_rc], ignore_index=True).to_csv(OUT_DD_EVENTS_MAX, index=False, encoding="utf-8-sig")

    pd.DataFrame([{
        "daily_maxDD_raw": daily_mdd(daily_raw_all),
        "daily_maxDD_riskcontrol": mdd_rc_d,
        "n_days_raw": int(len(daily_raw_all)) if not daily_raw_all.empty else 0,
        "n_days_riskcontrol": int(len(daily_rc_all)) if not daily_rc_all.empty else 0,
        "RISKOFF_RATIO": RISKOFF_RATIO,
        "REGIME_OFF_IF_SUM_MKT_LT": REGIME_OFF_IF_SUM_MKT_LT,
        "REGIME_OFF_IF_SUM_WML_LT": REGIME_OFF_IF_SUM_WML_LT,
    }]).to_csv(OUT_DAILY_MDD_SUMMARY, index=False, encoding="utf-8-sig")

    # CAGR（RC側）
    rc_perf = pd.read_csv(OUT_PERF_RC).iloc[0].to_dict() if OUT_PERF_RC.exists() else {}
    cagr_rc = float(rc_perf.get("CAGR", np.nan))

    # 回復日数（RC側）
    days_to_recovery = float(ev_rc.iloc[0].get("days_to_recovery", np.nan)) if not ev_rc.empty else np.nan

    logger.info(f"[GRID] DONE case: MKT_THR={REGIME_OFF_IF_SUM_MKT_LT} | WML_THR={REGIME_OFF_IF_SUM_WML_LT} | dailyMDD={mdd_rc_d:.6f} | CAGR={cagr_rc:.6f} | days_to_recovery={days_to_recovery}")

    return {
        "riskoff_ratio": float(RISKOFF_RATIO),
        "mkt_thr": float(REGIME_OFF_IF_SUM_MKT_LT),
        "wml_thr": float(REGIME_OFF_IF_SUM_WML_LT),
        "daily_mdd": float(mdd_rc_d) if np.isfinite(mdd_rc_d) else np.nan,
        "cagr": float(cagr_rc) if np.isfinite(cagr_rc) else np.nan,
        "days_to_recovery": float(days_to_recovery) if np.isfinite(days_to_recovery) else np.nan,
        "out_dir": str(OUT_DIR),
        "note": ""
    }


def main():
    logger.info("=" * 110)
    logger.info("10/1 年次「割安高質」バックテスト + FF5+MOM RiskControl(C) [REGIME THRESHOLD GRID]")
    logger.info("=" * 110)
    logger.info(f"FF5MOM_FACTOR_PATH: {FF5MOM_FACTOR_PATH}")
    logger.info(f"BASE_OUT_DIR: {BASE_OUT_DIR}")
    logger.info(f"Tax mode: annual simple tax only (TAX_RATE={TAX_RATE}) | daily tax ignored")
    logger.info(f"Fixed: RISKOFF_RATIO={RISKOFF_RATIO}")
    logger.info("Grid: REGIME_OFF_IF_SUM_MKT_LT x REGIME_OFF_IF_SUM_WML_LT in (0.00, -0.02, -0.05)")
    logger.info("Regime: off if (sum_MKT_3m < MKT_THR) OR (sum_WML_3m < WML_THR)")
    logger.info("=" * 110)

    # ---- 入力データ読み込み（ここは1回だけ）
    statements_df = load_financial_data()
    if statements_df.empty:
        logger.error("財務データ読み込み失敗")
        return

    prices_df = load_existing_price_data()
    if prices_df.empty:
        logger.error("株価データ読み込み失敗")
        return

    fac = load_ff5mom_factors_monthly()
    prices_by_code = build_prices_by_code(prices_df)

    enhanced_financial_data = calculate_market_metrics_fast_chunked(statements_df, prices_df, chunk_size=200)
    if enhanced_financial_data.empty:
        logger.error("財務指標計算失敗")
        return

    # ---- グリッド実験（9ケース）
    grid_vals = [0.00, -0.02, -0.05]
    rows = []
    for mkt_thr in grid_vals:
        for wml_thr in grid_vals:
            try:
                rec = run_one_case(
                    mkt_thr=mkt_thr,
                    wml_thr=wml_thr,
                    enhanced_financial_data=enhanced_financial_data,
                    prices_df=prices_df,
                    prices_by_code=prices_by_code,
                    fac=fac,
                )
                rows.append(rec)
            except Exception as e:
                logger.error(f"[GRID] case failed: MKT_THR={mkt_thr} | WML_THR={wml_thr} | err={e}", exc_info=True)
                rows.append({
                    "riskoff_ratio": float(RISKOFF_RATIO),
                    "mkt_thr": float(mkt_thr),
                    "wml_thr": float(wml_thr),
                    "daily_mdd": np.nan,
                    "cagr": np.nan,
                    "days_to_recovery": np.nan,
                    "out_dir": "",
                    "note": f"error: {e}"
                })

    summary = pd.DataFrame(rows)
    summary = summary.sort_values(["mkt_thr", "wml_thr"]).reset_index(drop=True)
    summary.to_csv(OUT_GRID_SUMMARY, index=False, encoding="utf-8-sig")

    logger.info("-" * 110)
    logger.info("✅ GRID SUMMARY SAVED")
    logger.info(f"  - {OUT_GRID_SUMMARY}")
    logger.info("-" * 110)
    logger.info("[GRID TABLE]")
    logger.info(summary.to_string(index=False))


if __name__ == "__main__":
    main()


2026-01-27 04:06:07,443 - INFO - ==============================================================================================================
2026-01-27 04:06:07,444 - INFO - 10/1 年次「割安高質」バックテスト + FF5+MOM RiskControl(C) [REGIME THRESHOLD GRID]
2026-01-27 04:06:07,445 - INFO - ==============================================================================================================
2026-01-27 04:06:07,446 - INFO - FF5MOM_FACTOR_PATH: C:\Users\yongr\Project\merged_data_all_stocks\factors\ff5_mom_factors_monthly.parquet
2026-01-27 04:06:07,447 - INFO - BASE_OUT_DIR: C:\Users\yongr\Project\merged_data_all_stocks\factors\bt_october_unit_with_ff5mom_regime_grid
2026-01-27 04:06:07,448 - INFO - Tax mode: annual simple tax only (TAX_RATE=0.20315) | daily tax ignored
2026-01-27 04:06:07,449 - INFO - Fixed: RISKOFF_RATIO=0.7
2026-01-27 04:06:07,450 - INFO - Grid: REGIME_OFF_IF_SUM_MKT_LT x REGIME_OFF_IF_SUM_WML_LT in (0.00, -0.02, -0.05)
2026-01-27 04:06:07,451 - INFO - Regime: off if (sum